# Sentiment-of-Sequence v0.2 — what this notebook does

**Date:** 2025-08-16

**Objective.** Build a transcript-clean, sequence-only dataset of WT/MUT windows for missense variants so we can test whether local sequence context encodes “oncogenic grammar” strong enough to separate known drivers from truly benign controls.

**Core hypothesis.** Short sequence motifs and codon-context around the edited base carry enough statistical signal to rank driver mutations above benign ones. If the model cannot even outrank MUT over WT on positives, the sequence isn’t talking or our negatives are dirty.

---

## Scope (v0.2)

- **Genes:** EGFR, KRAS, PIK3CA, TP53 to start. We will extend to BRAF, NRAS, IDH1/2, PTEN, PIK3R1 in the next bump.
- **Variants:** missense SNVs only.
- **Reference:** GRCh38.
- **Windows:** ±60 nt around the mutant codon center (center on 2nd base).
- **Labels:**
  - **Positives** = MSK Cancer Hotspots drivers.
  - **Negatives** = ClinVar benign or likely-benign missense SNVs, transcript-aware and windowized per gene.

---

## Changes since yesterday

- Built honest **LOCO-GENE** and **LOCO-SITE** splits with zero site overlap.
- Switched baseline to **hashed k-mers with smaller K** and **stronger regularization** to behave on small n.
- Added **pair metric** measuring whether `p(MUT) > p(WT)` when the label says it should.
- Found our ClinVar benign parquet empty. Located:
  - `clinvar_negatives_canonical.parquet` — has `gene, transcript, hgvs_c` but **missing windows**.
  - `variant_summary.txt.gz` — raw ClinVar to rebuild negatives if needed.

---

## Today’s plan

1. **Negatives, done right.**
   - Promote `clinvar_negatives_canonical.parquet` to full model rows by deriving:
     `ref, alt, cds_pos, wt_codon, mut_codon, tri_center, window_wt, window_mut`, with ref checks.
   - If canonical is thin, rebuild from `variant_summary.txt.gz` with filters:
     benign or likely-benign ∧ missense SNV ∧ GRCh38 ∧ in-panel genes.

2. **Rebuild model table.**
   - Start from `masterseq_model_ready.parquet` and **add** newly windowized benign rows → `masterseq_model_ready_plus.parquet`.
   - Verify per-gene label counts and tri-context balance.

3. **Retrain and evaluate.**
   - Use **K=3**, **L2**, `C≈0.25`, `max_iter≈5000`.
   - Report per-gene AUROC, PR-AUC, Brier, and **pair accuracy** under LOCO-GENE and LOCO-SITE with 95% CIs.

4. **If signal stabilizes, expand genes** and re-run the same receipts.

---

## Inputs

- `data/cancer_hotspots_missense.csv` — driver list with `gene, hgvs_c, transcript` if available.
- `data/clinvar_negatives_canonical.parquet` — transcript-aware ClinVar rows (`gene, transcript, hgvs_c`).
- `data/variant_summary.txt.gz` — raw ClinVar dump for reconstruction if needed.
- `data/masterseq_model_ready.parquet` — current modelable rows.

---

## Outputs

- `data/masterseq_model_ready_plus.parquet` — model table with new benigns windowized.
- `artifacts/sentiment_run_YYYYMMDD_HHMMSS/…`  
  - `splits_loco_gene.csv`, `splits_loco_site.csv`
  - `predictions_*.csv` for each run
  - `metrics_summary.csv` with AUROC, PR-AUC, Brier, pair accuracy + CIs
  - `MANIFEST.json` with seeds, inputs, hashes, and params

---

## Guardrails

- **Transcript-aware mapping** with ref-allele verification. Any mismatch goes to `qc_notes` and is excluded.
- **No leakage**: zero `site_id` overlap across train and test per fold.
- **Honest baselines**: K kept small on tiny n; calibration checked via Brier and reliability bins.
- **Receipts or it didn’t happen**: every run writes splits, preds, metrics, and a manifest.

---

## Run order

1. **Windowize canonical ClinVar negatives** → write `*_windowized.parquet`.
2. **Merge into model table** → write `masterseq_model_ready_plus.parquet`.
3. **Generate splits** (LOCO-GENE and LOCO-SITE) and save manifests.
4. **Train + evaluate** with K=3, L2, `C=0.25`, `max_iter=5000`; export metrics with CIs.
5. **Review QC**: per-gene counts, pos rate, tri-balance, calibration.
6. If stable, **expand the gene panel** and repeat.

---

## What “good” looks like today

- Clean benigns with verified refs and proper windows.
- Stable AUROC per gene with non-ridiculous CIs, **pair accuracy > 0.5** on positives.
- No leakage, no convergence warnings, and a manifest that proves lineage.

*Keep it tight, keep it reproducible. If the sequence is speaking, these receipts will show it.*


## Look at reciepts from yesterday

In [1]:
# 1) Data contract / columns
import pandas as pd, json, pathlib
ROOT = pathlib.Path("/content/drive/MyDrive/Cancer_gene_database"); DATA = ROOT/"data"
cuts = ["masterseq.parquet", "masterseq_model_ready.parquet", "masterseq_model_ready_tribalanced.parquet"]
for name in cuts:
    df = pd.read_parquet(DATA/name)
    print("\n==", name, "==")
    print("rows:", len(df))
    print("columns:", json.dumps(sorted(df.columns.tolist()), indent=2))

# 2) Split leakage sanity for one LOCO-gene fold
df = pd.read_parquet(DATA/"masterseq_model_ready_tribalanced.parquet")
train = df[df["split"].eq("train")]
test  = df[df["split"].eq("test")]
overlap = set(train["site_id"].astype(str)) & set(test["site_id"].astype(str))
print("\nsite_id overlap between train/test:", len(overlap))

# 3) Bootstrap CI (quick-and-dirty, AUROC per gene)
import numpy as np
from sklearn.metrics import roc_auc_score
def boot_ci(y, p, B=1000, seed=1337):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y))
    vals = []
    for _ in range(B):
        b = rng.choice(idx, size=len(idx), replace=True)
        if len(np.unique(y[b]))<2: continue
        vals.append(roc_auc_score(y[b], p[b]))
    vals = np.array(vals); return float(np.nanpercentile(vals, 2.5)), float(np.nanpercentile(vals, 97.5))
for g, sub in df.groupby("gene"):
    if {"label","pred_l1_hash5"}.issubset(sub.columns):  # adjust if your pred col name differs
        lo, hi = boot_ci(sub["label"].values, sub["pred_l1_hash5"].values)
        print(f"{g}: 95% CI [{lo:.3f}, {hi:.3f}]")



== masterseq.parquet ==
rows: 3504
columns: [
  "alt",
  "cds_pos",
  "gene",
  "hgvs_c",
  "label",
  "label_source",
  "mut_codon",
  "qc_notes",
  "ref",
  "transcript",
  "tri_center",
  "window_mut",
  "window_wt",
  "wt_codon"
]

== masterseq_model_ready.parquet ==
rows: 308
columns: [
  "alt",
  "cds_pos",
  "codon_pos",
  "gene",
  "hgvs_c",
  "label",
  "label_source",
  "mut_codon",
  "qc_notes",
  "ref",
  "site_id",
  "transcript",
  "tri_center",
  "window_mut",
  "window_wt",
  "wt_codon"
]

== masterseq_model_ready_tribalanced.parquet ==
rows: 183
columns: [
  "alt",
  "cds_pos",
  "gene",
  "hgvs_c",
  "label",
  "label_source",
  "mut_codon",
  "qc_notes",
  "ref",
  "site_id",
  "transcript",
  "tri_center",
  "window_mut",
  "window_wt",
  "wt_codon"
]


KeyError: 'split'

In [2]:
# ==== CONFIG ====
DATA_DIR = "/content/drive/MyDrive/Cancer_gene_database/data"
ARTIFACTS_DIR = "/content/drive/MyDrive/Cancer_gene_database/artifacts"
REF_ASM = "GRCh38"  # change if you're on GRCh37
USE_TRIBALANCED = True  # prefer tribalanced cut if present

# ==== LOADS & QC RECEIPTS ====
import os, json, hashlib, time
from pathlib import Path
import pandas as pd
import numpy as np

dd = Path(DATA_DIR)
af = Path(ARTIFACTS_DIR) / f"sentiment_run_{time.strftime('%Y%m%d_%H%M%S')}"
af.mkdir(parents=True, exist_ok=True)

p_clinvar = dd / "clinvar_benign_filtered.parquet"
p_master  = dd / "masterseq.parquet"
p_ready   = dd / "masterseq_model_ready.parquet"
p_tri     = dd / "masterseq_model_ready_tribalanced.parquet"
p_ckpt    = dd / "masterseq_checkpoint_20250815_175058.parquet"  # unused, but logged

def load_first_existing(paths):
    for p in paths:
        if p.exists():
            return p, pd.read_parquet(p)
    raise FileNotFoundError(f"None of {paths} exist")

# choose modeling table
model_p, model_df = load_first_existing([p_tri] if USE_TRIBALANCED else [p_ready, p_master])
base_df = pd.read_parquet(p_master) if p_master.exists() else model_df

# quick schema + head
print("== MODEL DF ==", model_p.name, model_df.shape)
display(model_df.head(4))

# label distribution by gene
if {"gene","label"}.issubset(model_df.columns):
    lab = (model_df[["gene","label"]]
           .assign(label=lambda s: s["label"].astype(int))
           .groupby(["gene","label"]).size().unstack(fill_value=0).rename(columns={0:"neg",1:"pos"}))
    lab["total"] = lab.sum(axis=1)
    lab["pos_rate"] = lab["pos"]/lab["total"]
    display(lab.sort_values("total", ascending=False))

# tri-balance distribution (optional; only if present)
tri_cols = [c for c in model_df.columns if any(k in c.lower() for k in ["tri_bin","tribin","balance_bin","bin3","trinuc_bin"])]
if tri_cols:
    tb = (model_df.groupby(["gene", tri_cols[0]]).size().unstack(fill_value=0))
    print("Tri-balance proxy column:", tri_cols[0])
    display(tb)

# ClinVar lineage receipts (if file exists)
if p_clinvar.exists():
    clin = pd.read_parquet(p_clinvar)
    print("== CLINVAR BENIGN FILTERED ==", p_clinvar.name, clin.shape)
    cols = [c for c in ["gene","clinical_significance","variant_type","is_somatic","is_germline"] if c in clin.columns]
    if cols:
        display(clin[cols].head(8))
    # Per-gene benign counts
    if "gene" in clin.columns:
        display(clin["gene"].value_counts().head(20).rename("clinvar_benign_count").to_frame())

# write a manifest skeleton now; we’ll fill more later
def sha256sum(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "ref_assembly": REF_ASM,
    "inputs": {},
    "model_df_path": str(model_p),
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "notes": "Sentiment-of-sequence L1 logistic with hashed k-mers. Splits: LOCO-gene and LOCO-site.",
}
for p in [p_clinvar, p_master, p_ready, p_tri, p_ckpt]:
    if p.exists():
        manifest["inputs"][p.name] = {
            "path": str(p),
            "bytes": p.stat().st_size,
            "sha256": sha256sum(p),
            "mtime": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(p.stat().st_mtime)),
        }

with open(af/"MANIFEST.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Artifacts dir:", af)


== MODEL DF == masterseq_model_ready_tribalanced.parquet (183, 15)


,gene,transcript,hgvs_c,label,cds_pos,ref,alt,wt_codon,mut_codon,window_wt,window_mut,tri_center,qc_notes,label_source,site_id
0,KRAS,NM_004985.5,c.35G>A,1,35.0,G,A,GGT,GAT,ATGACTGAATATAAACTTGTGGTAGTTGGAGCTGGTGGCGTAGGCA...,ATGACTGAATATAAACTTGTGGTAGTTGGAGCTGATGGCGTAGGCA...,GGT,,existing,KRAS:35
1,KRAS,NM_004985.5,c.34G>T,1,34.0,G,T,GGT,TGT,ATGACTGAATATAAACTTGTGGTAGTTGGAGCTGGTGGCGTAGGCA...,ATGACTGAATATAAACTTGTGGTAGTTGGAGCTTGTGGCGTAGGCA...,TGG,,existing,KRAS:34
2,PIK3CA,NM_006218.4,c.3140A>G,1,3140.0,A,G,CAT,CGT,CCTTAGATAAAACTGAGCAAGAGGCTTTGGAGTATTTCATGAAACA...,CCTTAGATAAAACTGAGCAAGAGGCTTTGGAGTATTTCATGAAACA...,CAT,,existing,PIK3CA:3140
3,TP53,NM_000546.6,c.743G>A,1,743.0,G,A,CGG,CAG,ACTGTACCACCATCCACTACAACTACATGTGTAACAGTTCCTGCAT...,ACTGTACCACCATCCACTACAACTACATGTGTAACAGTTCCTGCAT...,CGG,,existing,TP53:743


label,neg,pos,total,pos_rate
gene,,,,
TP53,57,78,135,0.577778
KRAS,15,15,30,0.500000
EGFR,5,5,10,0.500000
PIK3CA,3,5,8,0.625000


== CLINVAR BENIGN FILTERED == clinvar_benign_filtered.parquet (0, 15)


,gene


,clinvar_benign_count
gene,


Artifacts dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_20250816_174000


## Splits, featurize, train, metrics, pair-AUC, manifest finalize

In [3]:
# ==== SPLITS + MODELING ====
from pathlib import Path
import pandas as pd, numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction import FeatureHasher
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from tqdm.auto import tqdm
import json, time, os

# knobs
K = 5                       # try 3 vs 5 later for ablations
INCLUDE_FLANKS = True       # False = codon-only-ish
MAX_ITER = 500
SEED = 1337
RANDOM_STATE = 1337
N_BOOT = 1000               # bootstrap for CI

rng = np.random.default_rng(SEED)
df = model_df.copy()

need = {"gene","label","site_id","window_wt","window_mut"}
missing = need - set(df.columns)
if missing:
    raise ValueError(f"Required columns missing: {missing}")

df = df.dropna(subset=["gene","label","site_id","window_mut"]).copy()
df["label"] = df["label"].astype(int)
df["gene"] = df["gene"].astype(str)
df["site_id"] = df["site_id"].astype(str)

# LOCO-GENE splits
rows = []
for g in sorted(df["gene"].unique()):
    mask = df["gene"].eq(g)
    fold = f"loco_gene:{g}"
    rows.append(df.assign(fold=fold, split=np.where(mask, "test", "train"))[["site_id","gene","fold","split"]])
splits_gene = pd.concat(rows, ignore_index=True).drop_duplicates()

# LOCO-SITE splits
n_sites = df["site_id"].nunique()
n_splits = max(2, min(5, n_sites))
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

site_rows = []
X_dummy = np.zeros((len(df), 1))
y = df["label"].values
groups = df["site_id"].values
for k, (tr, te) in enumerate(sgkf.split(X_dummy, y, groups=groups), start=1):
    fold = f"loco_site:k{k}"
    tmp = df.loc[:, ["site_id","gene"]].copy()
    tmp["fold"] = fold
    tmp["split"] = "train"
    tmp.loc[df.index[te], "split"] = "test"
    site_rows.append(tmp[["site_id","gene","fold","split"]])
splits_site = pd.concat(site_rows, ignore_index=True).drop_duplicates()

# Persist split manifests
splits_gene_path = af/"splits_loco_gene.csv"
splits_site_path = af/"splits_loco_site.csv"
splits_gene.to_csv(splits_gene_path, index=False)
splits_site.to_csv(splits_site_path, index=False)

print("Saved splits:", splits_gene_path, splits_site_path)

# Overlap receipts
def overlap_table(splits):
    reps = []
    for f, sub in splits.groupby("fold"):
        a = set(sub.query("split=='train'")["site_id"])
        b = set(sub.query("split=='test'")["site_id"])
        reps.append({"fold": f, "overlap_sites": len(a & b), "n_train": len(a), "n_test": len(b)})
    return pd.DataFrame(reps).sort_values("fold")

display(overlap_table(splits_gene))
display(overlap_table(splits_site))

# Featurization
def row_tokens(row, K=5, include_flanks=True):
    toks = []
    if include_flanks:
        s = str(row["window_mut"])
        for i in range(0, max(0, len(s)-K+1)):
            toks.append(f"m_{i}:{s[i:i+K]}")
    # codon transition token if available
    if "wt_codon" in row and "mut_codon" in row and pd.notna(row["wt_codon"]) and pd.notna(row["mut_codon"]):
        toks.append(f"codon:{row['wt_codon']}->{row['mut_codon']}")
    # center trinuc if present
    if "tri_center" in row and pd.notna(row["tri_center"]):
        toks.append(f"tri:{row['tri_center']}")
    return toks

def tokens_from_seq(seq, K=5):
    s = str(seq)
    return [f"s_{i}:{s[i:i+K]}" for i in range(0, max(0, len(s)-K+1))]

hasher = FeatureHasher(n_features=2**18, input_type="string")  # 262,144 dims

def build_X(df_subset):
    token_stream = (row_tokens(r, K=K, include_flanks=INCLUDE_FLANKS) for _, r in df_subset.iterrows())
    return hasher.transform(token_stream)

def build_X_from_seqcol(df_subset, seq_col):
    token_stream = (tokens_from_seq(r[seq_col], K=K) for _, r in df_subset.iterrows())
    return hasher.transform(token_stream)

def boot_ci_auc(y, p, B=N_BOOT, seed=SEED):
    idx = np.arange(len(y))
    vals = []
    rng = np.random.default_rng(seed)
    for _ in range(B):
        b = rng.choice(idx, size=len(idx), replace=True)
        if len(np.unique(y[b])) < 2:
            continue
        vals.append(roc_auc_score(y[b], p[b]))
    if not vals:
        return np.nan, np.nan
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def run_cv(df, splits, tag):
    preds = []
    for fold, sub in tqdm(splits.groupby("fold"), total=splits["fold"].nunique(), desc=f"CV {tag}"):
        dd = df.merge(sub, on=["site_id","gene"], how="inner")
        tr = dd.query("split=='train'").copy()
        te = dd.query("split=='test'").copy()
        if tr.empty or te.empty:
            continue
        Xtr = build_X(tr); ytr = tr["label"].values
        Xte = build_X(te); yte = te["label"].values

        clf = LogisticRegression(
            penalty="l1", solver="saga", max_iter=MAX_ITER, n_jobs=-1, random_state=SEED, C=1.0
        )
        clf.fit(Xtr, ytr)
        p_mut = clf.predict_proba(Xte)[:, 1]

        # Optional pair-AUC: predict WT by hashing window_wt with same model
        Xte_wt = build_X_from_seqcol(te, "window_wt")
        p_wt = clf.predict_proba(Xte_wt)[:, 1]
        pair = (p_mut > p_wt).mean()  # fraction where MUT scores higher than WT

        out = te.loc[:, ["site_id","gene","label"]].copy()
        out["fold"] = fold
        out["split_mode"] = tag
        out["y_pred"] = p_mut
        out["y_pred_wt"] = p_wt
        out["pair_auc_proxy"] = pair  # not a formal AUC, but a WT→MUT win rate in [0,1]
        out["model_tag"] = f"hash{K}_{'flanks' if INCLUDE_FLANKS else 'codon'}_L1"
        preds.append(out)
    return pd.concat(preds, ignore_index=True) if preds else pd.DataFrame()

preds_gene = run_cv(df, splits_gene, "LOCO-GENE")
preds_site = run_cv(df, splits_site, "LOCO-SITE")

preds_gene_path = af/"predictions_loco_gene.csv"
preds_site_path = af/"predictions_loco_site.csv"
preds_gene.to_csv(preds_gene_path, index=False)
preds_site.to_csv(preds_site_path, index=False)

print("Saved predictions:", preds_gene_path, preds_site_path)

# Metrics summary with bootstrap CI and calibration
def summarize(preds):
    rows = []
    for (tag, gene), sub in preds.groupby(["split_mode","gene"]):
        y = sub["label"].values
        p = sub["y_pred"].values
        if len(np.unique(y)) < 2:
            continue
        au = roc_auc_score(y, p)
        pr = average_precision_score(y, p)
        br = brier_score_loss(y, p)
        lo, hi = boot_ci_auc(y, p)
        # pair proxy averaged over fold rows
        pair_proxy = float(sub["pair_auc_proxy"].mean()) if "pair_auc_proxy" in sub.columns else np.nan
        rows.append({
            "tag": tag, "gene": gene, "n": int(len(sub)),
            "auroc": round(au, 4), "auroc_ci_low": round(lo, 4), "auroc_ci_high": round(hi, 4),
            "pr_auc": round(pr, 4), "brier": round(br, 4), "pair_winrate": round(pair_proxy, 4),
        })
    return pd.DataFrame(rows).sort_values(["tag","gene"])

summary = pd.concat([summarize(preds_gene), summarize(preds_site)], ignore_index=True)
metrics_path = af/"metrics_summary.csv"
summary.to_csv(metrics_path, index=False)
display(summary)
print("Saved metrics:", metrics_path)

# finalize manifest
with open(af/"MANIFEST.json") as f:
    manifest = json.load(f)
manifest.update({
    "ref_assembly": REF_ASM,
    "seed": SEED,
    "feature_set": {"k": K, "include_flanks": INCLUDE_FLANKS, "hasher_dims": 2**18},
    "splits": {
        "loco_gene": str(splits_gene_path),
        "loco_site": str(splits_site_path),
    },
    "predictions": {
        "loco_gene": str(preds_gene_path),
        "loco_site": str(preds_site_path),
    },
    "metrics_summary": str(metrics_path),
    "model": {"type": "LogisticRegression", "penalty": "l1", "solver": "saga", "C": 1.0, "max_iter": MAX_ITER},
})
with open(af/"MANIFEST.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Manifest updated:", af/"MANIFEST.json")


Saved splits: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_20250816_174000/splits_loco_gene.csv /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_20250816_174000/splits_loco_site.csv


,fold,overlap_sites,n_train,n_test
0,loco_gene:EGFR,0,136,10
1,loco_gene:KRAS,0,132,14
2,loco_gene:PIK3CA,0,140,6
3,loco_gene:TP53,0,30,116


,fold,overlap_sites,n_train,n_test
0,loco_site:k1,0,115,31
1,loco_site:k2,0,119,27
2,loco_site:k3,0,116,30
3,loco_site:k4,0,118,28
4,loco_site:k5,0,116,30


CV LOCO-GENE:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


CV LOCO-SITE:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Saved predictions: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_20250816_174000/predictions_loco_gene.csv /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_20250816_174000/predictions_loco_site.csv


,tag,gene,n,auroc,auroc_ci_low,auroc_ci_high,pr_auc,brier,pair_winrate
0,LOCO-GENE,EGFR,10,0.6000,0.2500,0.8750,0.6111,0.2430,0.2000
1,LOCO-GENE,KRAS,30,0.3022,0.1340,0.4604,0.4317,0.2600,0.1000
2,LOCO-GENE,PIK3CA,8,0.3333,0.0000,0.5000,0.6250,0.2404,0.1250
3,LOCO-GENE,TP53,135,0.5000,0.5000,0.5000,0.5778,0.2528,0.0000
4,LOCO-SITE,EGFR,10,0.4400,0.0000,0.8200,0.5911,0.2427,0.2055
5,LOCO-SITE,KRAS,30,0.5311,0.3153,0.7535,0.5351,0.2568,0.2106
6,LOCO-SITE,PIK3CA,8,0.3333,0.0000,1.0000,0.5952,0.2566,0.1794
7,LOCO-SITE,TP53,135,0.4443,0.3431,0.5472,0.5348,0.2597,0.1866


Saved metrics: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_20250816_174000/metrics_summary.csv
Manifest updated: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_20250816_174000/MANIFEST.json


## 1) Fix the ClinVar input (or prove it’s not the source of truth)

If our benign parquet is empty, that’s either the wrong file or your filter logic is too strict. Run this to sanity-check and rebuild a usable benign set:

In [4]:
import pandas as pd
from pathlib import Path

P = Path("/content/drive/MyDrive/Cancer_gene_database/data")
p_clin = P/"clinvar_benign_filtered.parquet"

if p_clin.exists():
    clin = pd.read_parquet(p_clin)
    print("clinvar_benign_filtered.parquet:", clin.shape)
else:
    print("Missing:", p_clin)

# If empty/missing, rebuild from your raw ClinVar (adjust this path/columns to your pipeline)
# Example filter: keep benign/likely benign, missense SNVs, transcript-aware if you have it.

p_raw = P/"clinvar_raw.parquet"  # <-- set this to your raw ClinVar parquet
if p_raw.exists():
    raw = pd.read_parquet(p_raw)
    print("clinvar_raw:", raw.shape)
    # Flexible, case-insensitive filters
    cs = raw.get("clinical_significance","").astype(str).str.lower()
    vt = raw.get("variant_type","").astype(str).str.lower()
    benign = cs.str.contains("benign")
    missense = vt.str.contains("missense")
    snv = vt.str.contains("single|snv|substitution")
    out = raw[benign & missense & snv].copy()
    print("benign missense SNVs:", out.shape)
    out.to_parquet(P/"clinvar_benign_filtered.parquet", index=False)
else:
    print("No raw ClinVar parquet found — point p_raw to the right file.")


clinvar_benign_filtered.parquet: (0, 15)
No raw ClinVar parquet found — point p_raw to the right file.


### 2) Make the model solvable on this tiny n

Right now K=5 + L1 + hashing is too sparse for 183 rows. Let’s run ablations that are realistic for this data size:

* Lower sparsity: use K=3.

* Stronger regularization: smaller C (e.g., 0.25).

* More iterations so SAGA actually converges.

Add simple bio features that don’t explode dimensionality: transition/transversion, CpG hit, AA class, Grantham distance.

#### codon-only vs flanks, K=3/5, L1/L2 — with proper pair metric that checks if p_mut > p_wt matches the label.

### A) Find/repair ClinVar negatives (fast)

* We will locate any ClinVar files we’ve got lying around.
* Point to the right one and we’ll filter it cleanly.

In [6]:
import glob, os, pandas as pd
DATA_DIR = "/content/drive/MyDrive/Cancer_gene_database/data"

hits = []
for pat in ["*clinvar*.parquet","**/*clinvar*.parquet","*ClinVar*.*","**/*ClinVar*.*","*variant_summary*.*","**/*variant_summary*.*","*vcv*.*","**/*vcv*.*"]:
    hits += glob.glob(os.path.join(DATA_DIR, pat), recursive=True)

print("Found:", len(hits))
for h in sorted(set(hits))[:50]:
    print(h)


Found: 6
/content/drive/MyDrive/Cancer_gene_database/data/clinvar_benign_filtered.parquet
/content/drive/MyDrive/Cancer_gene_database/data/clinvar_negatives_canonical.parquet
/content/drive/MyDrive/Cancer_gene_database/data/variant_summary.txt.gz


## We’ve got three things on disk:

* `clinvar_benign_filtered.parquet` — currently empty in your run, so that filter step was too strict or pointed at the wrong raw.

* `clinvar_negatives_canonical.parquet` — likely your already-processed, transcript-aware benign set. Use this if it has the right columns.

* `variant_summary.txt.gz` — the big raw ClinVar dump; we can rebuild a clean benign missense set from here if needed.

Here’s how we proceed — “rebuild negatives right.”

### Rebuild negatives right (ClinVar)

We’ve got two viable sources:

* `clinvar_negatives_canonical.parquet` — if this already has the fields we need, use it now.

* `variant_summary.txt.gz` — if canonical is thin, rebuild from raw.

### B1. Inspect and use canonical negatives

In [7]:
import pandas as pd
P="/content/drive/MyDrive/Cancer_gene_database/data"
neg = pd.read_parquet(f"{P}/clinvar_negatives_canonical.parquet")
print(neg.shape)
print(sorted(neg.columns)[:30])

need = {"gene","transcript","hgvs_c","ref","alt","wt_codon","mut_codon","window_wt","window_mut","tri_center"}
missing = need - set(neg.columns)
print("Missing:", missing)

# If good, merge any NEW negatives into your current model cut
base = pd.read_parquet(f"{P}/masterseq_model_ready.parquet")
key = ["gene","transcript","hgvs_c"]  # adjust if your canonical uses site_id
neg["label"] = 0
to_add = neg.merge(base[key], on=key, how="left", indicator=True).query("_merge=='left_only'").drop(columns="_merge")
model_plus = pd.concat([base, to_add[base.columns.intersection(to_add.columns)]], ignore_index=True).drop_duplicates(key)

print("Old rows:", len(base), "Added:", len(to_add), "New rows:", len(model_plus))
model_plus.to_parquet(f"{P}/masterseq_model_ready_plus.parquet", index=False)


(170, 3)
['gene', 'hgvs_c', 'transcript']
Missing: {'alt', 'mut_codon', 'window_mut', 'window_wt', 'wt_codon', 'ref', 'tri_center'}
Old rows: 308 Added: 170 New rows: 471


next block builds real windows for your canonical ClinVar negatives (NM_/ENST) and writes `clinvar_negatives_windowized.parquet`. it fetches CDS per transcript (Ensembl for ENST, NCBI for NM_), parses simple HGVS c.NRef>Alt, verifies ref, and emits WT/MUT windows ±60 nt around the codon center.

In [9]:
# B1.1 — Patched windowizer with robust receipts (no KeyError on _cds_source)
import re, time, json, requests
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
CANON_PATH = DATA_DIR/"clinvar_negatives_canonical.parquet"
OUT_PATH   = DATA_DIR/"clinvar_negatives_windowized.parquet"
CACHE_DIR  = DATA_DIR/"_cache_cds"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

HGVS_SNVS = re.compile(r"^c\.(\d+)([ACGT])>([ACGT])$")

# --- tiny HTTP helper with retries ---
sess = requests.Session()
adapter = requests.adapters.HTTPAdapter(max_retries=3)
sess.mount("https://", adapter)
sess.mount("http://", adapter)

def parse_hgvs_c(hg):
    m = HGVS_SNVS.match(str(hg).strip()) if isinstance(hg,str) else None
    return (int(m.group(1)), m.group(2).upper(), m.group(3).upper()) if m else None

def cache_path(tx):
    return (CACHE_DIR / f"{tx.replace('.','_')}.cds.txt")

def fetch_cds_ncbi_nm(tx, sleep=0.34):
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    try:
        r = sess.get(base, params={"db":"nuccore","id":tx,"rettype":"fasta_cds_na","retmode":"text"}, timeout=20)
        if r.ok and ">cds" in r.text.lower():
            seq = "".join(l.strip() for l in r.text.splitlines() if not l.startswith(">")).upper()
            time.sleep(sleep);
            return seq, "ncbi_cds"
    except Exception:
        pass
    try:
        r = sess.get(base, params={"db":"nuccore","id":tx,"rettype":"fasta","retmode":"text"}, timeout=20)
        if r.ok and ">" in r.text:
            seq = "".join(l.strip() for l in r.text.splitlines() if not l.startswith(">")).upper()
            time.sleep(sleep);
            return seq, "ncbi_mrna"
    except Exception:
        pass
    return None, "none"

def fetch_cds_ensembl_enst(tx, sleep=0.34):
    try:
        r = sess.get(f"https://rest.ensembl.org/sequence/id/{tx}", params={"type":"cds"}, headers={"Content-Type":"text/plain"}, timeout=20)
        if r.ok and r.text:
            time.sleep(sleep);
            return r.text.strip().upper(), "ensembl_cds"
    except Exception:
        pass
    return None, "none"

def get_cds(tx):
    cp = cache_path(tx)
    if cp.exists():
        try:
            s = cp.read_text().strip().upper()
            if s:
                return s, "cache"
        except Exception:
            pass
    if str(tx).startswith("ENST"):
        seq, src = fetch_cds_ensembl_enst(tx)
    elif str(tx).startswith("NM_"):
        seq, src = fetch_cds_ncbi_nm(tx)
    else:
        seq, src = fetch_cds_ensembl_enst(tx)
    if seq:
        cp.write_text(seq)
    return seq, src

def codon_window(cds, cds_pos, flank=60):
    L = len(cds); i0 = cds_pos-1
    if i0 < 0 or i0 >= L:
        return {"qc_notes":"pos_oob"}
    cs = (i0//3)*3
    if cs+3 > L:
        return {"qc_notes":"codon_oob"}
    wt_codon = cds[cs:cs+3]
    tri = cds[max(0,i0-1):min(L,i0+2)]
    center = cs+1
    ws, we = max(0, center-flank), min(L, center+flank+1)
    return {"wt_codon": wt_codon, "tri_center": tri if len(tri)==3 else "", "window_wt": cds[ws:we], "ws":ws, "we":we}

def apply_alt(cds, cds_pos, alt):
    i0 = cds_pos-1
    return None if i0<0 or i0>=len(cds) else cds[:i0]+alt+cds[i0+1:]

# --- load and process ---
canon = pd.read_parquet(CANON_PATH).dropna(subset=["gene","transcript","hgvs_c"]).copy()
canon["gene"]=canon["gene"].astype(str); canon["transcript"]=canon["transcript"].astype(str); canon["hgvs_c"]=canon["hgvs_c"].astype(str)

rows=[]
cds_cache={}
for _, r in canon.iterrows():
    gene, tx, hg = r["gene"].strip(), r["transcript"].strip(), r["hgvs_c"].strip()
    cds_pos=ref=alt=None; cds=None; src="none"; qc=[]
    parsed = parse_hgvs_c(hg)
    if not parsed:
        qc.append("unsupported_hgvs")
    else:
        cds_pos, ref, alt = parsed
        if tx in cds_cache:
            cds, src = cds_cache[tx]
        else:
            cds, src = get_cds(tx); cds_cache[tx]=(cds,src)
        if not cds:
            qc.append("cds_not_found")
        else:
            i0 = cds_pos-1
            if i0<0 or i0>=len(cds):
                qc.append("pos_oob")
            else:
                if cds[i0] != ref:
                    qc.append(f"ref_mismatch:{cds[i0]}!={ref}")
                w = codon_window(cds, cds_pos, flank=60)
                if "qc_notes" in w:
                    qc.append(w["qc_notes"])
                mut_cds = apply_alt(cds, cds_pos, alt)
                if mut_cds is None:
                    qc.append("mut_apply_fail")
                else:
                    cs = (cds_pos-1)//3*3
                    mut_codon = mut_cds[cs:cs+3] if cs+3<=len(mut_cds) else ""
                    ws, we = w.get("ws",None), w.get("we",None)
                    window_mut = mut_cds[ws:we] if ws is not None and we is not None else ""
    rows.append({
        "gene": gene, "transcript": tx, "hgvs_c": hg,
        "label": 0, "label_source": "clinvar_benign_canonical",
        "cds_pos": cds_pos, "ref": ref, "alt": alt,
        "wt_codon": w.get("wt_codon","") if parsed and cds else "",
        "mut_codon": (mut_codon if parsed and cds and 'mut_codon' in locals() else ""),
        "tri_center": w.get("tri_center","") if parsed and cds else "",
        "window_wt": w.get("window_wt","") if parsed and cds else "",
        "window_mut": (window_mut if parsed and cds and 'window_mut' in locals() else ""),
        "site_id": f"{gene}:{cds_pos}" if cds_pos else f"{gene}:na",
        "_cds_source": src,
        "_cds_len": (len(cds) if isinstance(cds,str) else 0),
        "qc_notes": ";".join(qc)
    })

neg_win = pd.DataFrame(rows)

# define hard fails
hard_fail_mask = neg_win["qc_notes"].fillna("").str.contains("cds_not_found|pos_oob|mut_apply_fail|codon_oob|unsupported_hgvs")
usable = neg_win.loc[~hard_fail_mask].copy()

# Make sure columns exist even if empty
for col in ["_cds_source","_cds_len"]:
    if col not in usable.columns:
        usable[col] = []

# save usable
usable.to_parquet(OUT_PATH, index=False)

# receipts
ref_mis = neg_win["qc_notes"].fillna("").str.contains("ref_mismatch").sum()
fail_counts = neg_win["qc_notes"].fillna("ok").str.split(";").explode().value_counts().to_dict()
src_counts = usable["_cds_source"].value_counts().to_dict() if "_cds_source" in usable.columns else {}

report = {
    "n_input": int(len(canon)),
    "n_windowized_total": int(len(neg_win)),
    "n_usable": int(len(usable)),
    "usable_rate": round(len(usable)/max(1,len(neg_win)), 3),
    "ref_mismatch_n": int(ref_mis),
    "by_gene_usable": usable.groupby("gene").size().sort_values(ascending=False).to_dict(),
    "cds_source_counts": src_counts,
    "top_failure_reasons": {k:int(v) for k,v in list(fail_counts.items())[:10]}
}
print("Windowized negatives written ->", OUT_PATH)
print(json.dumps(report, indent=2))

print("\nHEAD (usable):")
display(usable.head(8)[["gene","transcript","hgvs_c","cds_pos","ref","alt","wt_codon","mut_codon","tri_center","site_id"]])


Windowized negatives written -> /content/drive/MyDrive/Cancer_gene_database/data/clinvar_negatives_windowized.parquet
{
  "n_input": 170,
  "n_windowized_total": 170,
  "n_usable": 0,
  "usable_rate": 0.0,
  "ref_mismatch_n": 0,
  "by_gene_usable": {},
  "cds_source_counts": {},
  "top_failure_reasons": {
    "unsupported_hgvs": 170
  }
}

HEAD (usable):


,gene,transcript,hgvs_c,cds_pos,ref,alt,wt_codon,mut_codon,tri_center,site_id


**Next move: peek the formats and normalize flexibly.**

In [10]:
# C1 — HGVS sanity + flexible normalizer (diagnostic only)
import pandas as pd, re
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
canon = pd.read_parquet(DATA_DIR/"clinvar_negatives_canonical.parquet").dropna(subset=["hgvs_c"]).copy()
canon["hgvs_c"] = canon["hgvs_c"].astype(str)

# Flexible parser: find a c.-level SNV anywhere in the string, allow transcript prefixes & protein tails
SNV_ANYWHERE = re.compile(r"c\.(\d+)([ACGTacgt])>([ACGTacgt])")

def parse_hgvs_flexible(s: str):
    s = s.strip()
    m = SNV_ANYWHERE.search(s)
    if not m:
        return None
    pos = int(m.group(1))
    ref = m.group(2).upper()
    alt = m.group(3).upper()
    return pos, ref, alt, f"c.{pos}{ref}>{alt}"

rows = []
fail_buckets = {"no_c_snv":0, "intronic_or_star":0, "indel_or_dup":0, "protein_only":0, "other":0}

raw_vals = canon["hgvs_c"].tolist()
for s in raw_vals:
    s_stripped = s.strip()

    # quick bucket tags before parse attempt
    is_intronic = bool(re.search(r"c\.\d+(\+|-|_)|c\.\*", s_stripped))
    is_indeldup = bool(re.search(r"delins|del|ins|dup", s_stripped, flags=re.IGNORECASE))
    has_protein = bool(re.search(r"\bp\.", s_stripped))

    parsed = parse_hgvs_flexible(s_stripped)
    if parsed:
        pos, ref, alt, norm = parsed
        rows.append((s_stripped, norm, pos, ref, alt))
    else:
        if "c." not in s_stripped:
            if has_protein:
                fail_buckets["protein_only"] += 1
            else:
                fail_buckets["no_c_snv"] += 1
        elif is_intronic:
            fail_buckets["intronic_or_star"] += 1
        elif is_indeldup:
            fail_buckets["indel_or_dup"] += 1
        else:
            fail_buckets["other"] += 1

parsed_df = pd.DataFrame(rows, columns=["hgvs_c_raw","hgvs_c_norm","cds_pos","ref","alt"]).drop_duplicates()

print("Total canonical negatives:", len(canon))
print("Parsed as coding SNV:", len(parsed_df))
print("Rough failure buckets:", fail_buckets)

print("\nExamples raw → normalized (up to 12):")
print(parsed_df.head(12).to_string(index=False))

# Unique failures to eyeball patterns
parsed_raw_set = set(parsed_df["hgvs_c_raw"].tolist())
fail_examples = sorted(set([s for s in raw_vals if s.strip() not in parsed_raw_set]))[:20]
print("\nSample failures (unique, up to 20):")
for x in fail_examples:
    print("  ", x)


Total canonical negatives: 170
Parsed as coding SNV: 0
Rough failure buckets: {'no_c_snv': 170, 'intronic_or_star': 0, 'indel_or_dup': 0, 'protein_only': 0, 'other': 0}

Examples raw → normalized (up to 12):
Empty DataFrame
Columns: [hgvs_c_raw, hgvs_c_norm, cds_pos, ref, alt]
Index: []

Sample failures (unique, up to 20):
   C.1003C>T
   C.1004G>T
   C.1006G>C
   C.1015G>C
   C.1020G>C
   C.1020G>T
   C.1023C>G
   C.1025G>A
   C.1029G>T
   C.1040C>G
   C.1046A>G
   C.105G>C
   C.1060C>A
   C.1061A>C
   C.1061A>T
   C.1063G>A
   C.1066G>C
   C.1067G>A
   C.1079G>C
   C.107C>A


HGVS strings are uppercase like C.1003C>T, and my parser was case-sensitive for c.. that’s why all 170 came back “unsupported_hgvs”.

We'll run this single patched windowizer (case-insensitive HGVS parse).

In [11]:
# B1.2 — Windowize canonical ClinVar negatives (case-insensitive HGVS parser)

import re, time, json, requests
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
CANON_PATH = DATA_DIR/"clinvar_negatives_canonical.parquet"
OUT_PATH   = DATA_DIR/"clinvar_negatives_windowized.parquet"
CACHE_DIR  = DATA_DIR/"_cache_cds"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# --- case-insensitive SNV at c.-level, e.g., c.743G>A or C.743G>A
SNV_CDNA = re.compile(r"c\.(\d+)([ACGT])>([ACGT])", re.IGNORECASE)

# robust HTTP session
sess = requests.Session()
sess.mount("https://", requests.adapters.HTTPAdapter(max_retries=3))
sess.mount("http://",  requests.adapters.HTTPAdapter(max_retries=3))

def parse_hgvs_c_any(s: str):
    """Return (cds_pos:int, ref:str, alt:str) from c.-level SNV regardless of 'c.' vs 'C.'; else None."""
    if not isinstance(s, str):
        return None
    m = SNV_CDNA.search(s.strip())
    if not m:
        return None
    return int(m.group(1)), m.group(2).upper(), m.group(3).upper()

def cache_path(tx):
    return CACHE_DIR / f"{str(tx).replace('.','_')}.cds.txt"

def fetch_cds_ncbi_nm(tx, sleep=0.34):
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    # try CDS first
    r = sess.get(base, params={"db":"nuccore","id":tx,"rettype":"fasta_cds_na","retmode":"text"}, timeout=20)
    if r.ok and ">cds" in r.text.lower():
        seq = "".join(l.strip() for l in r.text.splitlines() if not l.startswith(">")).upper()
        time.sleep(sleep); return seq, "ncbi_cds"
    # fallback: mRNA
    r = sess.get(base, params={"db":"nuccore","id":tx,"rettype":"fasta","retmode":"text"}, timeout=20)
    if r.ok and ">" in r.text:
        seq = "".join(l.strip() for l in r.text.splitlines() if not l.startswith(">")).upper()
        time.sleep(sleep); return seq, "ncbi_mrna"
    return None, "none"

def fetch_cds_ensembl_enst(tx, sleep=0.34):
    r = sess.get(f"https://rest.ensembl.org/sequence/id/{tx}",
                 params={"type":"cds"}, headers={"Content-Type":"text/plain"}, timeout=20)
    if r.ok and r.text:
        time.sleep(sleep); return r.text.strip().upper(), "ensembl_cds"
    return None, "none"

def get_cds(tx):
    cp = cache_path(tx)
    if cp.exists():
        try:
            s = cp.read_text().strip().upper()
            if s: return s, "cache"
        except Exception:
            pass
    tx = str(tx)
    if   tx.startswith("ENST"): seq, src = fetch_cds_ensembl_enst(tx)
    elif tx.startswith("NM_"):  seq, src = fetch_cds_ncbi_nm(tx)
    else:                       seq, src = fetch_cds_ensembl_enst(tx)
    if seq: cp.write_text(seq)
    return seq, src

def codon_window(cds, cds_pos, flank=60):
    L = len(cds); i0 = cds_pos-1
    if i0 < 0 or i0 >= L: return {"qc_notes":"pos_oob"}
    cs = (i0//3)*3
    if cs+3 > L: return {"qc_notes":"codon_oob"}
    wt_codon = cds[cs:cs+3]
    tri = cds[max(0,i0-1):min(L,i0+2)]
    center = cs+1                      # 2nd base of codon
    ws, we = max(0, center-flank), min(L, center+flank+1)
    return {"wt_codon": wt_codon, "tri_center": tri if len(tri)==3 else "", "window_wt": cds[ws:we], "ws":ws, "we":we}

def apply_alt(cds, cds_pos, alt):
    i0 = cds_pos-1
    return None if i0<0 or i0>=len(cds) else cds[:i0]+alt+cds[i0+1:]

# --- load and process
canon = pd.read_parquet(CANON_PATH).dropna(subset=["gene","transcript","hgvs_c"]).copy()
canon["gene"]=canon["gene"].astype(str).str.strip()
canon["transcript"]=canon["transcript"].astype(str).str.strip()
canon["hgvs_c"]=canon["hgvs_c"].astype(str).str.strip()

rows=[]; cds_cache={}
for _, r in canon.iterrows():
    gene, tx, hg = r["gene"], r["transcript"], r["hgvs_c"]
    parsed = parse_hgvs_c_any(hg)
    cds_pos=ref=alt=None; cds=None; src="none"; qc=[]
    if not parsed:
        qc.append("unsupported_hgvs")
    else:
        cds_pos, ref, alt = parsed
        if tx in cds_cache: cds, src = cds_cache[tx]
        else:               cds, src = get_cds(tx); cds_cache[tx]=(cds,src)
        if not cds:
            qc.append("cds_not_found")
        else:
            i0 = cds_pos-1
            if i0<0 or i0>=len(cds): qc.append("pos_oob")
            else:
                if cds[i0] != ref: qc.append(f"ref_mismatch:{cds[i0]}!={ref}")
                w = codon_window(cds, cds_pos, flank=60)
                if "qc_notes" in w: qc.append(w["qc_notes"])
                mut_cds = apply_alt(cds, cds_pos, alt)
                if mut_cds is None: qc.append("mut_apply_fail")
                else:
                    cs = (cds_pos-1)//3*3
                    mut_codon = mut_cds[cs:cs+3] if cs+3<=len(mut_cds) else ""
                    ws, we = w.get("ws",None), w.get("we",None)
                    window_mut = mut_cds[ws:we] if (ws is not None and we is not None) else ""

    rows.append({
        "gene": gene, "transcript": tx, "hgvs_c": hg,
        "label": 0, "label_source": "clinvar_benign_canonical",
        "cds_pos": cds_pos, "ref": ref, "alt": alt,
        "wt_codon": w.get("wt_codon","") if parsed and cds else "",
        "mut_codon": (mut_codon if parsed and cds and 'mut_codon' in locals() else ""),
        "tri_center": w.get("tri_center","") if parsed and cds else "",
        "window_wt": w.get("window_wt","") if parsed and cds else "",
        "window_mut": (window_mut if parsed and cds and 'window_mut' in locals() else ""),
        "site_id": f"{gene}:{cds_pos}" if cds_pos else f"{gene}:na",
        "_cds_source": src, "_cds_len": (len(cds) if isinstance(cds,str) else 0),
        "qc_notes": ";".join(qc)
    })

neg_win = pd.DataFrame(rows)

# hard failures
hard_fail = neg_win["qc_notes"].fillna("").str.contains("cds_not_found|pos_oob|mut_apply_fail|codon_oob|unsupported_hgvs")
usable = neg_win.loc[~hard_fail].copy()

usable.to_parquet(OUT_PATH, index=False)

# receipts
ref_mis = neg_win["qc_notes"].fillna("").str.contains("ref_mismatch").sum()
fail_counts = neg_win["qc_notes"].fillna("ok").str.split(";").explode().value_counts().to_dict()
src_counts = usable["_cds_source"].value_counts().to_dict() if not usable.empty else {}

report = {
    "n_input": int(len(canon)),
    "n_windowized_total": int(len(neg_win)),
    "n_usable": int(len(usable)),
    "usable_rate": round(len(usable)/max(1,len(neg_win)), 3),
    "ref_mismatch_n": int(ref_mis),
    "by_gene_usable": usable.groupby("gene").size().sort_values(ascending=False).to_dict(),
    "cds_source_counts": src_counts,
    "top_failure_reasons": {k:int(v) for k,v in list(fail_counts.items())[:10]}
}
print("Windowized negatives written ->", OUT_PATH)
print(json.dumps(report, indent=2))

print("\nHEAD (usable):")
display(usable.head(8)[["gene","transcript","hgvs_c","cds_pos","ref","alt","wt_codon","mut_codon","tri_center","site_id"]])


Windowized negatives written -> /content/drive/MyDrive/Cancer_gene_database/data/clinvar_negatives_windowized.parquet
{
  "n_input": 170,
  "n_windowized_total": 170,
  "n_usable": 170,
  "usable_rate": 1.0,
  "ref_mismatch_n": 108,
  "by_gene_usable": {
    "TP53": 139,
    "EGFR": 25,
    "PIK3CA": 6
  },
  "cds_source_counts": {
    "ncbi_mrna": 170
  },
  "top_failure_reasons": {
    "": 62,
    "ref_mismatch:G!=A": 16,
    "ref_mismatch:C!=G": 14,
    "ref_mismatch:T!=G": 13,
    "ref_mismatch:T!=C": 11,
    "ref_mismatch:A!=G": 11,
    "ref_mismatch:A!=C": 10,
    "ref_mismatch:C!=T": 7,
    "ref_mismatch:A!=T": 6,
    "ref_mismatch:T!=A": 6
  }
}

HEAD (usable):


,gene,transcript,hgvs_c,cds_pos,ref,alt,wt_codon,mut_codon,tri_center,site_id
0,TP53,NM_000546.6,C.215C>G,215,C,G,ACT,AGT,ACT,TP53:215
1,TP53,NM_000546.6,C.139C>T,139,C,T,TGC,TGC,CTG,TP53:139
2,TP53,NM_000546.6,C.1093C>T,1093,C,T,GCC,TCC,AGC,TP53:1093
3,TP53,NM_000546.6,C.28G>A,28,G,A,GAG,AAG,GGA,TP53:28
4,TP53,NM_000546.6,C.329G>A,329,G,A,AGC,AAC,AGC,TP53:329
5,TP53,NM_000546.6,C.572C>G,572,C,G,GCA,GGA,GCA,TP53:572
6,TP53,NM_000546.6,C.704A>G,704,A,G,TCT,TGT,TCT,TP53:704
7,TP53,NM_000546.6,C.847C>T,847,C,T,CTA,TTA,ACT,TP53:847


All those ref mismatches are because we used full mRNA instead of the CDS-only sequence (HGVS c. positions are CDS-based, where c.1 = start codon). Let’s fix that properly.

We will rebuild the negatives by fetching GenBank XML for each NM_ transcript, extracting the CDS intervals, stitching the true CDS, and then windowizing. It caches per transcript, prints hard receipts, and overwrites your previous file.

In [12]:
# B1.3 — Rebuild negatives with TRUE CDS (GenBank XML) so c.-positions line up
# Input : data/clinvar_negatives_canonical.parquet  (gene, transcript, hgvs_c)
# Output: data/clinvar_negatives_windowized.parquet (now using CDS-only sequence)
# Notes : Uses NCBI EFetch (GenBank XML) to parse CDS intervals; caches XML & CDS.

import re, json, time, requests, xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path

DATA_DIR   = Path("/content/drive/MyDrive/Cancer_gene_database/data")
CANON_PATH = DATA_DIR/"clinvar_negatives_canonical.parquet"
OUT_PATH   = DATA_DIR/"clinvar_negatives_windowized.parquet"
CACHE_DIR  = DATA_DIR/"_cache_cds_xml"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

sess = requests.Session()
sess.mount("https://", requests.adapters.HTTPAdapter(max_retries=3))
sess.mount("http://",  requests.adapters.HTTPAdapter(max_retries=3))

SNV_CDNA = re.compile(r"c\.(\d+)([ACGT])>([ACGT])", re.IGNORECASE)

def parse_hgvs_c_any(s: str):
    m = SNV_CDNA.search(str(s).strip())
    if not m: return None
    return int(m.group(1)), m.group(2).upper(), m.group(3).upper()

def fetch_genbank_xml(nm: str, sleep=0.34):
    """Fetch GenBank XML for NM_ transcript and cache to disk."""
    nm_safe = nm.replace(".", "_")
    fp = CACHE_DIR / f"{nm_safe}.gb.xml"
    if fp.exists():
        return fp.read_text()
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {"db":"nuccore","id":nm,"rettype":"gb","retmode":"xml"}
    r = sess.get(url, params=params, timeout=30)
    r.raise_for_status()
    txt = r.text
    fp.write_text(txt)
    time.sleep(sleep)
    return txt

def cds_from_genbank_xml(xml_text: str):
    """
    Parse GenBank XML; return CDS-only sequence as uppercase string.
    Strategy: get GBSeq_sequence (full mRNA), then collect GBFeature key='CDS'
    intervals (GBInterval_from/to), concatenate in order.
    """
    root = ET.fromstring(xml_text)
    # There may be multiple GBSeq (rare), take the first
    ns = ""  # GenBank XML doesn't use namespaces typically
    seq_nodes = root.findall(".//GBSeq_sequence")
    if not seq_nodes:
        return None, "no_GBSeq_sequence"
    full_seq = seq_nodes[0].text.strip().upper()  # lower-case in XML
    # Find CDS feature
    cds_feat = None
    for feat in root.findall(".//GBFeature"):
        key_node = feat.find("GBFeature_key")
        if key_node is not None and key_node.text == "CDS":
            cds_feat = feat
            break
    if cds_feat is None:
        return None, "no_CDS_feature"
    intervals = []
    for iv in cds_feat.findall(".//GBInterval"):
        f = iv.findtext("GBInterval_from")
        t = iv.findtext("GBInterval_to")
        if f and t and f.isdigit() and t.isdigit():
            intervals.append((int(f), int(t)))
    if not intervals:
        # as a fallback, try parsing location string like join(133..1186, ...)
        loc = cds_feat.findtext("GBFeature_location") or ""
        ints = re.findall(r"(\d+)\.\.(\d+)", loc)
        intervals = [(int(a),int(b)) for a,b in ints] if ints else []
    if not intervals:
        return None, "no_intervals"
    # Build CDS by concatenating 1-based slices
    parts = []
    L = len(full_seq)
    for a,b in intervals:
        a0, b0 = max(1,a), min(L,b)
        if a0 <= b0:
            parts.append(full_seq[a0-1:b0])
    cds = "".join(parts)
    if len(cds) == 0:
        return None, "empty_cds"
    return cds, "genbank_cds"

def get_true_cds(tx: str):
    """Return CDS string and source tag: ('genbank_cds' or reason)."""
    if not str(tx).startswith("NM_"):
        return None, "not_NM"
    try:
        xml_text = fetch_genbank_xml(tx)
        cds, src = cds_from_genbank_xml(xml_text)
        return cds, src
    except Exception as e:
        return None, f"xml_error:{type(e).__name__}"

def codon_window(cds: str, cds_pos: int, flank: int = 60):
    L = len(cds); i0 = cds_pos-1
    if i0 < 0 or i0 >= L: return {"qc_notes":"pos_oob"}
    cs = (i0//3)*3
    if cs+3 > L: return {"qc_notes":"codon_oob"}
    wt_codon = cds[cs:cs+3]
    tri = cds[max(0,i0-1):min(L,i0+2)]
    center = cs+1
    ws, we = max(0, center-flank), min(L, center+flank+1)
    return {"wt_codon": wt_codon, "tri_center": tri if len(tri)==3 else "", "window_wt": cds[ws:we], "ws":ws, "we":we}

def apply_alt(cds: str, cds_pos: int, alt: str):
    i0 = cds_pos-1
    return None if i0<0 or i0>=len(cds) else cds[:i0]+alt+cds[i0+1:]

# --- load canonical
canon = pd.read_parquet(CANON_PATH).dropna(subset=["gene","transcript","hgvs_c"]).copy()
canon["gene"]=canon["gene"].astype(str).str.strip()
canon["transcript"]=canon["transcript"].astype(str).str.strip()
canon["hgvs_c"]=canon["hgvs_c"].astype(str).str.strip()

rows=[]; cds_cache={}
for _, r in canon.iterrows():
    gene, tx, hg = r["gene"], r["transcript"], r["hgvs_c"]
    parsed = parse_hgvs_c_any(hg)
    cds_pos=ref=alt=None; cds=None; src="none"; qc=[]
    if not parsed:
        qc.append("unsupported_hgvs")
    else:
        cds_pos, ref, alt = parsed
        if tx in cds_cache:
            cds, src = cds_cache[tx]
        else:
            cds, src = get_true_cds(tx)
            cds_cache[tx] = (cds, src)
        if not cds:
            qc.append(f"cds_not_found:{src}")
        else:
            i0 = cds_pos-1
            if i0<0 or i0>=len(cds):
                qc.append("pos_oob")
            else:
                if cds[i0] != ref:
                    qc.append(f"ref_mismatch:{cds[i0]}!={ref}")
                w = codon_window(cds, cds_pos, flank=60)
                if "qc_notes" in w:
                    qc.append(w["qc_notes"])
                mut_cds = apply_alt(cds, cds_pos, alt)
                if mut_cds is None:
                    qc.append("mut_apply_fail")
                else:
                    cs = (cds_pos-1)//3*3
                    mut_codon = mut_cds[cs:cs+3] if cs+3<=len(mut_cds) else ""
                    ws, we = w.get("ws",None), w.get("we",None)
                    window_mut = mut_cds[ws:we] if (ws is not None and we is not None) else ""

    rows.append({
        "gene": gene, "transcript": tx, "hgvs_c": hg,
        "label": 0, "label_source": "clinvar_benign_canonical",
        "cds_pos": cds_pos, "ref": ref, "alt": alt,
        "wt_codon": w.get("wt_codon","") if parsed and cds else "",
        "mut_codon": (mut_codon if parsed and cds and 'mut_codon' in locals() else ""),
        "tri_center": w.get("tri_center","") if parsed and cds else "",
        "window_wt": w.get("window_wt","") if parsed and cds else "",
        "window_mut": (window_mut if parsed and cds and 'window_mut' in locals() else ""),
        "site_id": f"{gene}:{cds_pos}" if cds_pos else f"{gene}:na",
        "_cds_source": src,
        "_cds_len": (len(cds) if isinstance(cds,str) else 0),
        "qc_notes": ";".join(qc)
    })

neg_win = pd.DataFrame(rows)

hard_fail = neg_win["qc_notes"].fillna("").str.contains("cds_not_found|pos_oob|mut_apply_fail|codon_oob|unsupported_hgvs")
usable = neg_win.loc[~hard_fail].copy()
usable.to_parquet(OUT_PATH, index=False)

# receipts
ref_mis      = neg_win["qc_notes"].fillna("").str.contains("ref_mismatch").sum()
fail_counts  = neg_win["qc_notes"].fillna("ok").str.split(";").explode().value_counts().to_dict()
src_counts   = usable["_cds_source"].value_counts().to_dict() if not usable.empty else {}
by_gene_usable = usable.groupby("gene").size().sort_values(ascending=False).to_dict()

report = {
    "n_input": int(len(canon)),
    "n_windowized_total": int(len(neg_win)),
    "n_usable": int(len(usable)),
    "usable_rate": round(len(usable)/max(1,len(neg_win)), 3),
    "ref_mismatch_n": int(ref_mis),
    "by_gene_usable": by_gene_usable,
    "cds_source_counts": src_counts,
    "top_failure_reasons": {k:int(v) for k,v in list(fail_counts.items())[:10]}
}
print("Windowized negatives (true CDS) ->", OUT_PATH)
print(json.dumps(report, indent=2))

print("\nHEAD (usable):")
display(usable.head(8)[["gene","transcript","hgvs_c","cds_pos","ref","alt","wt_codon","mut_codon","tri_center","site_id","_cds_source"]])


Windowized negatives (true CDS) -> /content/drive/MyDrive/Cancer_gene_database/data/clinvar_negatives_windowized.parquet
{
  "n_input": 170,
  "n_windowized_total": 170,
  "n_usable": 170,
  "usable_rate": 1.0,
  "ref_mismatch_n": 0,
  "by_gene_usable": {
    "TP53": 139,
    "EGFR": 25,
    "PIK3CA": 6
  },
  "cds_source_counts": {
    "genbank_cds": 170
  },
  "top_failure_reasons": {
    "": 170
  }
}

HEAD (usable):


,gene,transcript,hgvs_c,cds_pos,ref,alt,wt_codon,mut_codon,tri_center,site_id,_cds_source
0,TP53,NM_000546.6,C.215C>G,215,C,G,CCC,CGC,CCC,TP53:215,genbank_cds
1,TP53,NM_000546.6,C.139C>T,139,C,T,CCG,TCG,CCC,TP53:139,genbank_cds
2,TP53,NM_000546.6,C.1093C>T,1093,C,T,CAC,TAC,TCA,TP53:1093,genbank_cds
3,TP53,NM_000546.6,C.28G>A,28,G,A,GTC,ATC,CGT,TP53:28,genbank_cds
4,TP53,NM_000546.6,C.329G>A,329,G,A,CGT,CAT,CGT,TP53:329,genbank_cds
5,TP53,NM_000546.6,C.572C>G,572,C,G,CCT,CGT,CCT,TP53:572,genbank_cds
6,TP53,NM_000546.6,C.704A>G,704,A,G,AAC,AGC,AAC,TP53:704,genbank_cds
7,TP53,NM_000546.6,C.847C>T,847,C,T,CGC,TGC,GCG,TP53:847,genbank_cds


In [13]:
# D1 — Merge windowized ClinVar negatives into model table (no conflicts), write *_plus.parquet

from pathlib import Path
import pandas as pd
import numpy as np
import json

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
BASE_P   = DATA_DIR/"masterseq_model_ready.parquet"
NEG_P    = DATA_DIR/"clinvar_negatives_windowized.parquet"
OUT_P    = DATA_DIR/"masterseq_model_ready_plus.parquet"

# Load
base = pd.read_parquet(BASE_P).copy()
neg  = pd.read_parquet(NEG_P).copy()

# Standardize dtypes
for df in (base, neg):
    for c in ["gene","transcript","hgvs_c","ref","alt","wt_codon","mut_codon","tri_center","window_wt","window_mut","site_id","label_source","qc_notes"]:
        if c in df.columns:
            df[c] = df[c].astype(str)
    if "label" in df.columns:
        df["label"] = df["label"].astype(int)
    if "cds_pos" in df.columns:
        df["cds_pos"] = pd.to_numeric(df["cds_pos"], errors="coerce").astype("Int64")

# Ensure required columns exist on both sides
need = ["gene","transcript","hgvs_c","site_id","label","ref","alt","wt_codon","mut_codon","tri_center","window_wt","window_mut","label_source","cds_pos","qc_notes"]
for c in need:
    if c not in base.columns: base[c] = pd.NA
    if c not in neg.columns:  neg[c]  = pd.NA

# Tag sources
base["_source"] = base.get("_source", "model_ready")
neg["_source"]  = "clinvar_benign_canonical_cds"

# 1) Kill conflicts: if a site_id exists as a positive anywhere in base, drop that site from negatives.
pos_sites = set(base.loc[base["label"]==1, "site_id"].astype(str))
neg_before = len(neg)
neg = neg.loc[~neg["site_id"].astype(str).isin(pos_sites)].copy()
neg_after = len(neg)
conflict_dropped = neg_before - neg_after

# 2) Drop exact dupes already in base (same site_id & same label)
base_keys = set(zip(base["site_id"].astype(str), base["label"].astype(int)))
neg["__key"] = list(zip(neg["site_id"].astype(str), neg["label"].astype(int)))
neg = neg.loc[~neg["__key"].isin(base_keys)].drop(columns="__key")

# 3) Concatenate and de-dup by site_id+label (keep first)
plus = pd.concat([base, neg], ignore_index=True)
plus = plus.drop_duplicates(subset=["site_id","label"], keep="first")

# Basic QC: drop rows missing windows or bad codons
ok_len = (plus["window_wt"].str.len() > 0) & (plus["window_mut"].str.len() > 0)
ok_codon = plus["wt_codon"].str.len().eq(3) & plus["mut_codon"].str.len().eq(3)
kept = plus[ok_len & ok_codon].copy()
dropped_bad = len(plus) - len(kept)

# Save
kept.to_parquet(OUT_P, index=False)

# Receipts
def by_gene_counts(df):
    g = (df.groupby(["gene","label"]).size()
           .unstack(fill_value=0)
           .rename(columns={0:"neg",1:"pos"}))
    g["total"] = g.sum(axis=1)
    g["pos_rate"] = (g["pos"] / g["total"]).round(4)
    return g.sort_values("total", ascending=False)

report = {
    "base_rows": int(len(base)),
    "neg_rows_input": int(neg_before),
    "neg_rows_after_conflict_drop": int(neg_after),
    "conflict_sites_dropped": int(conflict_dropped),
    "merged_rows_before_qc": int(len(plus)),
    "rows_dropped_bad_windows_or_codons": int(dropped_bad),
    "final_rows": int(len(kept)),
    "final_unique_sites": int(kept["site_id"].nunique()),
}

print("Wrote:", OUT_P)
print(json.dumps(report, indent=2))

print("\nPer-gene label breakdown (top 12):")
display(by_gene_counts(kept).head(12))

print("\nHEAD (post-merge):")
display(kept.head(8)[["gene","transcript","hgvs_c","label","cds_pos","ref","alt","wt_codon","mut_codon","tri_center","site_id","_source"]])


Wrote: /content/drive/MyDrive/Cancer_gene_database/data/masterseq_model_ready_plus.parquet
{
  "base_rows": 308,
  "neg_rows_input": 170,
  "neg_rows_after_conflict_drop": 167,
  "conflict_sites_dropped": 3,
  "merged_rows_before_qc": 271,
  "rows_dropped_bad_windows_or_codons": 0,
  "final_rows": 271,
  "final_unique_sites": 268
}

Per-gene label breakdown (top 12):


label,neg,pos,total,pos_rate
gene,,,,
TP53,128,69,197,0.3503
EGFR,25,14,39,0.3590
PIK3CA,6,18,24,0.7500
KRAS,3,8,11,0.7273



HEAD (post-merge):


,gene,transcript,hgvs_c,label,cds_pos,ref,alt,wt_codon,mut_codon,tri_center,site_id,_source
0,EGFR,NM_005228.5,c.2573T>G,1,2573,T,G,CTG,CGG,CTG,EGFR:2573,model_ready
1,EGFR,NM_005228.5,c.2155G>A,1,2155,G,A,GGC,AGC,GGG,EGFR:2155,model_ready
2,KRAS,NM_004985.5,c.35G>A,1,35,G,A,GGT,GAT,GGT,KRAS:35,model_ready
3,KRAS,NM_004985.5,c.34G>T,1,34,G,T,GGT,TGT,TGG,KRAS:34,model_ready
4,PIK3CA,NM_006218.4,c.1633G>A,1,1633,G,A,GAG,AAG,TGA,PIK3CA:1633,model_ready
5,PIK3CA,NM_006218.4,c.3140A>G,1,3140,A,G,CAT,CGT,CAT,PIK3CA:3140,model_ready
6,TP53,NM_000546.6,c.743G>A,1,743,G,A,CGG,CAG,CGG,TP53:743,model_ready
7,TP53,NM_000546.6,c.524G>A,1,524,G,A,CGC,CAC,CGC,TP53:524,model_ready


In [14]:
# E1 — Make LOCO-GENE & LOCO-SITE splits on masterseq_model_ready_plus.parquet (with receipts)

import time, json
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
ARTI_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
RUN_DIR  = ARTI_DIR / f"sentiment_run_plus_{time.strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

P_PLUS = DATA_DIR/"masterseq_model_ready_plus.parquet"
df = pd.read_parquet(P_PLUS).copy()

# keep only clean, modelable rows
need = ["gene","label","site_id","window_wt","window_mut","wt_codon","mut_codon"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.dropna(subset=["gene","label","site_id","window_wt","window_mut"]).copy()
df = df[(df["wt_codon"].astype(str).str.len()==3) & (df["mut_codon"].astype(str).str.len()==3)].copy()
df["label"] = df["label"].astype(int)
df["gene"] = df["gene"].astype(str)
df["site_id"] = df["site_id"].astype(str)

# sanity: any duplicate site_ids with conflicting labels?
conf = (df.groupby("site_id")["label"]
          .agg(lambda x: tuple(sorted(pd.Series(x).unique())))
          .reset_index())
conf["n_unique"] = conf["label"].apply(lambda t: len(set(t)))
conflicts = conf[conf["n_unique"] > 1]
print(f"Conflicting site_id labels: {len(conflicts)}")
if not conflicts.empty:
    display(conflicts.head(20))

# label counts by gene
lbl = (df.groupby(["gene","label"]).size()
         .unstack(fill_value=0).rename(columns={0:"neg",1:"pos"}))
lbl["total"] = lbl.sum(axis=1)
lbl["pos_rate"] = (lbl["pos"]/lbl["total"]).round(4)
print("\nPer-gene label tallies:")
display(lbl.sort_values("total", ascending=False))

# --- LOCO-GENE splits ---
rows = []
for g in sorted(df["gene"].unique()):
    m = df["gene"].eq(g)
    fold = f"loco_gene:{g}"
    rows.append(df.assign(fold=fold, split=np.where(m,"test","train"))[["site_id","gene","fold","split"]])
splits_gene = pd.concat(rows, ignore_index=True).drop_duplicates()

# --- LOCO-SITE splits (group by site_id, stratify by label) ---
n_sites = df["site_id"].nunique()
n_splits = max(2, min(5, n_sites))
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

site_rows = []
X = np.zeros((len(df), 1))
y = df["label"].values
groups = df["site_id"].values
for k, (tr, te) in enumerate(sgkf.split(X, y, groups=groups), start=1):
    fold = f"loco_site:k{k}"
    tmp = df.loc[:, ["site_id","gene"]].copy()
    tmp["fold"] = fold
    tmp["split"] = "train"
    tmp.loc[df.index[te], "split"] = "test"
    site_rows.append(tmp[["site_id","gene","fold","split"]])
splits_site = pd.concat(site_rows, ignore_index=True).drop_duplicates()

# save
p_gene = RUN_DIR/"splits_loco_gene.csv"
p_site = RUN_DIR/"splits_loco_site.csv"
splits_gene.to_csv(p_gene, index=False)
splits_site.to_csv(p_site, index=False)

# receipts: overlap must be zero
def overlap_table(splits):
    reps=[]
    for f, sub in splits.groupby("fold"):
        a = set(sub.query("split=='train'")["site_id"])
        b = set(sub.query("split=='test'")["site_id"])
        reps.append({"fold": f, "overlap_sites": len(a & b), "n_train": len(a), "n_test": len(b)})
    return pd.DataFrame(reps).sort_values("fold")

print("\nSaved splits to:", RUN_DIR)
print("\nOverlap check — should be all zeros:")
display(overlap_table(splits_gene))
display(overlap_table(splits_site))


Conflicting site_id labels: 3


,site_id,label,n_unique
141,TP53:329,"(0, 1)",2
157,TP53:469,"(0, 1)",2
228,TP53:820,"(0, 1)",2



Per-gene label tallies:


label,neg,pos,total,pos_rate
gene,,,,
TP53,128,69,197,0.3503
EGFR,25,14,39,0.3590
PIK3CA,6,18,24,0.7500
KRAS,3,8,11,0.7273



Saved splits to: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_20250816_190139

Overlap check — should be all zeros:


,fold,overlap_sites,n_train,n_test
0,loco_gene:EGFR,0,229,39
1,loco_gene:KRAS,0,257,11
2,loco_gene:PIK3CA,0,244,24
3,loco_gene:TP53,0,74,194


,fold,overlap_sites,n_train,n_test
0,loco_site:k1,0,214,54
1,loco_site:k2,0,214,54
2,loco_site:k3,0,215,53
3,loco_site:k4,0,215,53
4,loco_site:k5,0,214,54


**next block does three things, surgically:**

* create a true variant_id = gene:transcript:cds_pos:ref>alt

* resolve any label conflicts at the variant level (positives win; drop the 0s)

* regenerate LOCO-GENE / LOCO-SITE splits grouped by variant_id, and prove zero overlap

In [15]:
# E2 — Promote to variant-level IDs, resolve label conflicts, and remake splits cleanly

import time, json
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
ARTI_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
RUN_DIR  = ARTI_DIR / f"sentiment_run_plus_variants_{time.strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

P_PLUS = DATA_DIR/"masterseq_model_ready_plus.parquet"
OUT_RESOLVED = DATA_DIR/"masterseq_model_ready_plus_resolved.parquet"

df = pd.read_parquet(P_PLUS).copy()

# ensure expected columns exist
need = ["gene","transcript","cds_pos","ref","alt","label","window_wt","window_mut","wt_codon","mut_codon"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in {P_PLUS.name}: {missing}")

# normalize dtypes
df["gene"] = df["gene"].astype(str)
df["transcript"] = df["transcript"].astype(str)
df["label"] = df["label"].astype(int)
df["ref"] = df["ref"].astype(str).str.upper()
df["alt"] = df["alt"].astype(str).str.upper()
df["cds_pos"] = pd.to_numeric(df["cds_pos"], errors="coerce").astype("Int64")

# build a true variant id
df["variant_id"] = (
    df["gene"] + ":" + df["transcript"] + ":" + df["cds_pos"].astype(str) + ":" + df["ref"] + ">" + df["alt"]
)

# basic modelability filter
df = df.dropna(subset=["variant_id","window_wt","window_mut"]).copy()
df = df[(df["wt_codon"].astype(str).str.len()==3) & (df["mut_codon"].astype(str).str.len()==3)]

# check variant-level label conflicts
lab_per_var = (df.groupby("variant_id")["label"]
                 .agg(lambda x: tuple(sorted(pd.Series(x).unique()))).reset_index())
lab_per_var["n_unique"] = lab_per_var["label"].apply(lambda t: len(set(t)))
conflicts = lab_per_var[lab_per_var["n_unique"] > 1]

print(f"Variant-level label conflicts BEFORE resolution: {len(conflicts)}")
if not conflicts.empty:
    display(conflicts.head(20))

# resolve conflicts: positives beat negatives for the same variant_id
# keep exactly one row per (variant_id), prefer label==1; if both 1s and 0s exist, drop 0s
df_sorted = df.sort_values(["variant_id","label"], ascending=[True, False]).copy()
# keep first row per variant_id (which will be label 1 if present)
resolved = df_sorted.drop_duplicates(subset=["variant_id"], keep="first").copy()

# report how many negatives dropped due to conflict resolution
dropped_conflict = int((df.groupby("variant_id")["label"].nunique() > 1).sum())
print("Negatives dropped due to conflicts (positives win):", dropped_conflict)

# save resolved dataset
resolved.to_parquet(OUT_RESOLVED, index=False)

# per-gene tallies post-resolution
tallies = (resolved.groupby(["gene","label"]).size()
             .unstack(fill_value=0).rename(columns={0:"neg",1:"pos"}))
tallies["total"] = tallies.sum(axis=1)
tallies["pos_rate"] = (tallies["pos"]/tallies["total"]).round(4)
print("\nPer-gene label tallies (resolved, one row per variant):")
display(tallies.sort_values("total", ascending=False))

# ---- SPLITS (using variant_id as the grouping key) ----
# LOCO-GENE: test all variants from the held-out gene
rows = []
for g in sorted(resolved["gene"].unique()):
    m = resolved["gene"].eq(g)
    fold = f"loco_gene:{g}"
    rows.append(resolved.assign(fold=fold, split=np.where(m, "test", "train"))[["variant_id","gene","fold","split"]])
splits_gene = pd.concat(rows, ignore_index=True).drop_duplicates()

# LOCO-SITE (really LOCO-variant): group by variant_id to avoid leakage
n_vars = resolved["variant_id"].nunique()
n_splits = max(2, min(5, n_vars))
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

X = np.zeros((len(resolved), 1))
y = resolved["label"].values
groups = resolved["variant_id"].values

site_rows = []
for k, (tr, te) in enumerate(sgkf.split(X, y, groups=groups), start=1):
    fold = f"loco_site:k{k}"
    tmp = resolved.loc[:, ["variant_id","gene"]].copy()
    tmp["fold"] = fold
    tmp["split"] = "train"
    tmp.loc[resolved.index[te], "split"] = "test"
    site_rows.append(tmp[["variant_id","gene","fold","split"]])
splits_site = pd.concat(site_rows, ignore_index=True).drop_duplicates()

# Save splits
p_gene = RUN_DIR/"splits_loco_gene.csv"
p_site = RUN_DIR/"splits_loco_site.csv"
splits_gene.to_csv(p_gene, index=False)
splits_site.to_csv(p_site, index=False)

# overlap receipts (no variant should be in both train and test)
def overlap_table(splits, id_col="variant_id"):
    reps=[]
    for f, sub in splits.groupby("fold"):
        a = set(sub.query("split=='train'")[id_col])
        b = set(sub.query("split=='test'")[id_col])
        reps.append({"fold": f, "overlap_variants": len(a & b), "n_train": len(a), "n_test": len(b)})
    return pd.DataFrame(reps).sort_values("fold")

print("\nSaved resolved dataset & splits:")
print("  dataset:", OUT_RESOLVED)
print("  splits:", p_gene, "and", p_site)

print("\nOverlap check — should be all zeros:")
display(overlap_table(splits_gene))
display(overlap_table(splits_site))


Variant-level label conflicts BEFORE resolution: 0
Negatives dropped due to conflicts (positives win): 0

Per-gene label tallies (resolved, one row per variant):


label,neg,pos,total,pos_rate
gene,,,,
TP53,128,69,197,0.3503
EGFR,25,14,39,0.3590
PIK3CA,6,18,24,0.7500
KRAS,3,8,11,0.7273



Saved resolved dataset & splits:
  dataset: /content/drive/MyDrive/Cancer_gene_database/data/masterseq_model_ready_plus_resolved.parquet
  splits: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_gene.csv and /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv

Overlap check — should be all zeros:


,fold,overlap_variants,n_train,n_test
0,loco_gene:EGFR,0,232,39
1,loco_gene:KRAS,0,260,11
2,loco_gene:PIK3CA,0,247,24
3,loco_gene:TP53,0,74,197


,fold,overlap_variants,n_train,n_test
0,loco_site:k1,0,216,55
1,loco_site:k2,0,217,54
2,loco_site:k3,0,217,54
3,loco_site:k4,0,217,54
4,loco_site:k5,0,217,54


Locked in. training + eval next — single cell. it uses our **resolved** dataset + latest **variant-based splits**, trains **K=3 / L2 / C=0.25** with compact bio-aware features, computes AUROC, PR-AUC, Brier, **pair accuracy**, and 95% bootstrap CIs per gene for both LOCO-GENE and LOCO-SITE. it writes preds & metrics into the same run dir.

run this and drop the printed tables/paths back here.

In [16]:
# F1 — Train/eval on resolved dataset with variant-based splits (K=3, L2)
import os, glob, time, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

SEED = 1337
np.random.seed(SEED)

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
ARTI_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")

# pick the *latest* variant-splits run dir
cand = sorted(glob.glob(str(ARTI_DIR / "sentiment_run_plus_variants_*")))
assert cand, "No sentiment_run_plus_variants_* dirs found. Run E2 first."
RUN_DIR = Path(cand[-1])

P_RES = DATA_DIR/"masterseq_model_ready_plus_resolved.parquet"
SPLITS_G = RUN_DIR/"splits_loco_gene.csv"
SPLITS_S = RUN_DIR/"splits_loco_site.csv"

df = pd.read_parquet(P_RES).dropna(subset=["variant_id","gene","label","window_mut","window_wt"]).copy()
df["label"] = df["label"].astype(int)
df["gene"] = df["gene"].astype(str)
df["variant_id"] = df["variant_id"].astype(str)

spl_g = pd.read_csv(SPLITS_G)
spl_s = pd.read_csv(SPLITS_S)

# --- featureization (compact, dense-friendly) ---
def add_bio_feats(r: dict):
    d = {}
    # transition/transversion
    pur, pyr = {"A","G"}, {"C","T"}
    ref, alt = str(r.get("ref","")), str(r.get("alt",""))
    if ref and alt:
        d["is_transition"] = int((ref in pur and alt in pur) or (ref in pyr and alt in pyr))
    # CpG
    tri = str(r.get("tri_center",""))
    d["is_cpg"] = int(tri.startswith("CG")) if tri else 0
    # AA change (coarse)
    wt, mt = r.get("wt_codon",""), r.get("mut_codon","")
    if isinstance(wt,str) and isinstance(mt,str) and len(wt)==3 and len(mt)==3:
        d[f"codon:{wt}->{mt}"] = 1
    if tri:
        d[f"tri:{tri}"] = 1
    return d

def kmers(seq: str, K=3):
    s = str(seq)
    return [f"s_{i}:{s[i:i+K]}" for i in range(max(0, len(s)-K+1))]

def feats(row: pd.Series, K=3, include_codon=True, seq_override: str|None=None):
    r = row.to_dict()
    d = add_bio_feats(r)
    if not include_codon:
        # strip codon token if present
        for k in [k for k in list(d.keys()) if k.startswith("codon:")]:
            d.pop(k, None)
    seq = seq_override if seq_override is not None else r.get("window_mut","")
    for t in kmers(seq, K): d[t] = d.get(t,0)+1
    return d

def boot_ci_auc(y, p, B=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y))
    vals = []
    for _ in range(B):
        b = rng.choice(idx, size=len(idx), replace=True)
        if len(np.unique(y[b])) < 2:
            continue
        vals.append(roc_auc_score(y[b], p[b]))
    if not vals:
        return np.nan, np.nan
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def run_cv(df, splits, tag):
    rows = []
    for fold, sub in splits.groupby("fold"):
        # join via variant_id+gene to be strict
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr = dd[dd["split"]=="train"].copy()
        te = dd[dd["split"]=="test"].copy()
        if tr.empty or te.empty:
            continue

        vec = DictVectorizer(sparse=True)
        Xtr = vec.fit_transform(feats(r) for _, r in tr.iterrows())
        ytr = tr["label"].values
        Xte_mut = vec.transform(feats(r) for _, r in te.iterrows())
        Xte_wt  = vec.transform(feats(r, include_codon=False, seq_override=r["window_wt"]) for _, r in te.iterrows())

        clf = LogisticRegression(
            penalty="l2", solver="lbfgs", C=0.25,
            max_iter=5000, random_state=SEED
        )
        clf.fit(Xtr, ytr)

        pm = clf.predict_proba(Xte_mut)[:,1]
        pw = clf.predict_proba(Xte_wt)[:,1]
        y  = te["label"].values

        pair_ok = ((pm > pw) & (y==1)) | ((pm <= pw) & (y==0))

        out = te.loc[:, ["variant_id","gene","label"]].copy()
        out["fold"] = fold
        out["split_mode"] = tag
        out["y_pred"] = pm
        out["y_pred_wt"] = pw
        out["pair_ok"] = pair_ok.astype(int)
        rows.append(out)

    preds = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    return preds

preds_g = run_cv(df, spl_g, "LOCO-GENE")
preds_s = run_cv(df, spl_s, "LOCO-SITE")

# save preds
preds_g_path = RUN_DIR/"preds_variant_loco_gene_k3_l2.csv"
preds_s_path = RUN_DIR/"preds_variant_loco_site_k3_l2.csv"
preds_g.to_csv(preds_g_path, index=False)
preds_s.to_csv(preds_s_path, index=False)

# summarize per gene
def summarize(preds):
    rows=[]
    for g, sub in preds.groupby("gene"):
        y = sub["label"].values
        p = sub["y_pred"].values
        if len(np.unique(y)) < 2:
            continue
        au = roc_auc_score(y,p)
        pr = average_precision_score(y,p)
        br = brier_score_loss(y,p)
        lo, hi = boot_ci_auc(y,p,B=1000)
        pair_acc = sub["pair_ok"].mean()
        rows.append({
            "gene": g, "n": int(len(sub)),
            "auroc": round(au,4), "auroc_ci_low": round(lo,4), "auroc_ci_high": round(hi,4),
            "pr_auc": round(pr,4), "brier": round(br,4),
            "pair_acc": round(float(pair_acc),4)
        })
    return pd.DataFrame(rows).sort_values("gene")

summ_g = summarize(preds_g)
summ_s = summarize(preds_s)
summary = pd.concat([
    summ_g.assign(tag="LOCO-GENE"),
    summ_s.assign(tag="LOCO-SITE")
], ignore_index=True)[["tag","gene","n","auroc","auroc_ci_low","auroc_ci_high","pr_auc","brier","pair_acc"]]

metrics_path = RUN_DIR/"metrics_k3_l2.csv"
summary.to_csv(metrics_path, index=False)

print("Run dir:", RUN_DIR)
print("Saved predictions:", preds_g_path.name, "and", preds_s_path.name)
print("Saved metrics:", metrics_path.name)
display(summary.sort_values(["tag","gene"]))


Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607
Saved predictions: preds_variant_loco_gene_k3_l2.csv and preds_variant_loco_site_k3_l2.csv
Saved metrics: metrics_k3_l2.csv


,tag,gene,n,auroc,auroc_ci_low,auroc_ci_high,pr_auc,brier,pair_acc
0,LOCO-GENE,EGFR,39,0.6457,0.4361,0.8289,0.6039,0.2100,0.5128
1,LOCO-GENE,KRAS,11,0.7500,0.1250,1.0000,0.8755,0.2239,0.5455
2,LOCO-GENE,PIK3CA,24,0.7870,0.5077,0.9790,0.9210,0.2938,0.5000
3,LOCO-GENE,TP53,197,0.6847,0.5993,0.7586,0.5683,0.2281,0.4975
4,LOCO-SITE,EGFR,39,0.7543,0.5709,0.9112,0.6929,0.1839,0.4615
5,LOCO-SITE,KRAS,11,0.9167,0.6667,1.0000,0.9705,0.1355,0.5455
6,LOCO-SITE,PIK3CA,24,0.6944,0.4665,0.8949,0.9012,0.2503,0.4583
7,LOCO-SITE,TP53,197,0.8454,0.7909,0.8947,0.6977,0.1703,0.5533


**Big W on the metrics**, but peep the vibe: AUROC jumped (nice), pair accuracy is ~0.5 on most genes. that smells like the model’s catching label correlates (e.g., common codon swaps) more than truly “reading” the flanks.

next move = **surgical ablation** to see what’s actually doing work:

* flanks+codon vs **no-codon** vs **codon-only**

* plus a **WT-only** control (uses window_wt only). if that scores >~0.55 in LOCO-GENE, we still got sneaky correlates.

This cell; it reuses our resolved dataset & latest variant-based splits, then prints a compact table and writes artifacts.

In [17]:
# F2 — Ablation: flanks+codon vs no-codon vs codon-only, plus WT-only control (LOCO-GENE & LOCO-SITE)

import os, glob, time, json
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

SEED = 1337
np.random.seed(SEED)

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
ARTI_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
# use the latest variant-splits run dir
cand = sorted(glob.glob(str(ARTI_DIR / "sentiment_run_plus_variants_*")))
assert cand, "No sentiment_run_plus_variants_* dir found (run E2)."
RUN_DIR = Path(cand[-1])

P_RES = DATA_DIR/"masterseq_model_ready_plus_resolved.parquet"
SPLITS_G = RUN_DIR/"splits_loco_gene.csv"
SPLITS_S = RUN_DIR/"splits_loco_site.csv"

df = pd.read_parquet(P_RES).dropna(subset=["variant_id","gene","label","window_mut","window_wt"]).copy()
df["label"] = df["label"].astype(int)
df["gene"] = df["gene"].astype(str)
df["variant_id"] = df["variant_id"].astype(str)

spl_g = pd.read_csv(SPLITS_G)
spl_s = pd.read_csv(SPLITS_S)

def add_bio_feats(r: dict, include_codon=True, include_tri=True):
    d = {}
    pur, pyr = {"A","G"}, {"C","T"}
    ref, alt = str(r.get("ref","")), str(r.get("alt",""))
    if ref and alt:
        d["is_transition"] = int((ref in pur and alt in pur) or (ref in pyr and alt in pyr))
    if include_tri:
        tri = str(r.get("tri_center",""))
        d["is_cpg"] = int(tri.startswith("CG")) if tri else 0
        if tri:
            d[f"tri:{tri}"] = 1
    if include_codon:
        wt, mt = r.get("wt_codon",""), r.get("mut_codon","")
        if isinstance(wt,str) and isinstance(mt,str) and len(wt)==3 and len(mt)==3:
            d[f"codon:{wt}->{mt}"] = 1
    return d

def kmers(seq: str, K=3):
    s = str(seq)
    return [f"s_{i}:{s[i:i+K]}" for i in range(max(0, len(s)-K+1))]

def feats(row: pd.Series, *, K=3, include_flanks=True, include_codon=True, include_tri=True, seq_override=None):
    r = row.to_dict()
    d = add_bio_feats(r, include_codon=include_codon, include_tri=include_tri)
    if include_flanks:
        seq = seq_override if seq_override is not None else r.get("window_mut","")
        for t in kmers(seq, K):
            d[t] = d.get(t, 0) + 1
    return d

def boot_ci_auc(y, p, B=500, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y)); vals=[]
    for _ in range(B):
        b = rng.choice(idx, size=len(idx), replace=True)
        if len(np.unique(y[b]))<2: continue
        vals.append(roc_auc_score(y[b], p[b]))
    if not vals: return np.nan, np.nan
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def run_cv(df, splits, *, cfg_name, include_flanks, include_codon, include_tri, wt_only=False, K=3):
    rows=[]
    for fold, sub in splits.groupby("fold"):
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr, te = dd[dd["split"]=="train"], dd[dd["split"]=="test"]
        if tr.empty or te.empty:
            continue

        vec = DictVectorizer(sparse=True)
        # TRAIN: if wt_only, use window_wt even for training (no codon token)
        if wt_only:
            Xtr = vec.fit_transform(feats(r, K=K, include_flanks=True, include_codon=False, include_tri=False, seq_override=r["window_wt"])
                                    for _, r in tr.iterrows())
        else:
            Xtr = vec.fit_transform(feats(r, K=K, include_flanks=include_flanks, include_codon=include_codon, include_tri=include_tri)
                                    for _, r in tr.iterrows())
        ytr = tr["label"].values

        # TEST: predictions for MUT and WT
        if wt_only:
            Xte_mut = vec.transform(feats(r, K=K, include_flanks=True, include_codon=False, include_tri=False, seq_override=r["window_wt"])
                                    for _, r in te.iterrows())
        else:
            Xte_mut = vec.transform(feats(r, K=K, include_flanks=include_flanks, include_codon=include_codon, include_tri=include_tri)
                                    for _, r in te.iterrows())
        # WT baseline: always window_wt with no codon token nor tri
        Xte_wt  = vec.transform(feats(r, K=K, include_flanks=True, include_codon=False, include_tri=False, seq_override=r["window_wt"])
                                for _, r in te.iterrows())

        clf = LogisticRegression(penalty="l2", solver="lbfgs", C=0.25, max_iter=5000, random_state=SEED)
        clf.fit(Xtr, ytr)

        pm = clf.predict_proba(Xte_mut)[:,1]
        pw = clf.predict_proba(Xte_wt)[:,1]
        y  = te["label"].values
        pair_ok = ((pm > pw) & (y==1)) | ((pm <= pw) & (y==0))

        out = te.loc[:, ["variant_id","gene","label"]].copy()
        out["fold"] = fold
        out["cfg"] = cfg_name
        out["y_pred"] = pm
        out["y_pred_wt"] = pw
        out["pair_ok"] = pair_ok.astype(int)
        rows.append(out)
    preds = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    return preds

cfgs = [
    dict(cfg_name="flanks+codon+tri", include_flanks=True,  include_codon=True,  include_tri=True,  wt_only=False),
    dict(cfg_name="flanks_no_codon",  include_flanks=True,  include_codon=False, include_tri=True,  wt_only=False),
    dict(cfg_name="codon_only",       include_flanks=False, include_codon=True,  include_tri=True,  wt_only=False),
    dict(cfg_name="wt_only_control",  include_flanks=True,  include_codon=False, include_tri=False, wt_only=True),
]

all_summ=[]
for cfg in cfgs:
    for tag, splits in [("LOCO-GENE", spl_g), ("LOCO-SITE", spl_s)]:
        preds = run_cv(df, splits, **cfg)
        # save preds
        out_p = RUN_DIR / f"preds_{cfg['cfg_name']}_{tag}.csv"
        preds.to_csv(out_p, index=False)
        # summarize per gene
        rows=[]
        for g, sub in preds.groupby("gene"):
            y, p = sub["label"].values, sub["y_pred"].values
            if len(np.unique(y))<2:
                continue
            au = roc_auc_score(y,p); pr = average_precision_score(y,p); br = brier_score_loss(y,p)
            lo, hi = boot_ci_auc(y,p,B=500)
            pair_acc = sub["pair_ok"].mean()
            rows.append({
                "cfg": cfg["cfg_name"], "tag": tag, "gene": g, "n": int(len(sub)),
                "auroc": round(au,4), "auroc_ci_low": round(lo,4), "auroc_ci_high": round(hi,4),
                "pr_auc": round(pr,4), "brier": round(br,4), "pair_acc": round(float(pair_acc),4),
            })
        summ = pd.DataFrame(rows)
        all_summ.append(summ)

ablation = pd.concat(all_summ, ignore_index=True)
abl_path = RUN_DIR / "metrics_ablation.csv"
ablation.to_csv(abl_path, index=False)

print("Run dir:", RUN_DIR)
print("Saved:", abl_path.name)
# show tidy grid sorted for quick reading
show = (ablation.sort_values(["cfg","tag","gene"])
        .reset_index(drop=True))
display(show)


Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607
Saved: metrics_ablation.csv


,cfg,tag,gene,n,auroc,auroc_ci_low,auroc_ci_high,pr_auc,brier,pair_acc
0,codon_only,LOCO-GENE,EGFR,39,0.3400,0.1806,0.5285,0.2859,0.2505,0.4615
1,codon_only,LOCO-GENE,KRAS,11,0.5625,0.2000,0.9000,0.8409,0.3222,0.7273
2,codon_only,LOCO-GENE,PIK3CA,24,0.5278,0.2069,0.8346,0.7928,0.3233,0.6250
3,codon_only,LOCO-GENE,TP53,197,0.5306,0.4510,0.6188,0.3911,0.2665,0.5279
4,codon_only,LOCO-SITE,EGFR,39,0.3143,0.1726,0.4899,0.2780,0.2560,0.5385
5,codon_only,LOCO-SITE,KRAS,11,0.6250,0.1250,1.0000,0.8432,0.3056,0.6364
6,codon_only,LOCO-SITE,PIK3CA,24,0.6481,0.4113,0.8945,0.8792,0.2876,0.5833
7,codon_only,LOCO-SITE,TP53,197,0.6281,0.5454,0.7086,0.4581,0.2219,0.5838
8,flanks+codon+tri,LOCO-GENE,EGFR,39,0.6457,0.4352,0.8296,0.6039,0.2100,0.4103
9,flanks+codon+tri,LOCO-GENE,KRAS,11,0.7500,0.2000,1.0000,0.8755,0.2239,0.5455


alright, read on the room:

* **WT-only is hot** (e.g., TP53 LOCO-GENE AUC ~0.68, LOCO-SITE ~0.84). That ain’t leakage from the train set (we held out the gene); it screams “context signals” like **CpG/mutability** or region/domain effects.

* **Codon token ain’t carrying** (flanks±codon basically same).

* **Pair-acc on WT-only is meaningless** (pm==pw by design → pair≈neg rate). Don’t read into those.

Before we spin fancier features or tri-matched resampling, let’s answer the obvious: **is trinucleotide doing most of the work?** One shot:

* add **tri-only model** (no flanks, no codon)

* add **flanks but drop tri** (to see what flanks add beyond tri)

We'll run this block; it reuses <u>our resolved dataset + latest splits and appends results to the same run dir.</u>

In [18]:
# F3 — Micro-ablation: TRI-only vs FLANKS-no-TRI (compare to your earlier runs)
import os, glob
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

SEED = 1337
np.random.seed(SEED)

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
ARTI_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
cand = sorted(glob.glob(str(ARTI_DIR / "sentiment_run_plus_variants_*")))
assert cand, "No sentiment_run_plus_variants_* dirs found."
RUN_DIR = Path(cand[-1])

P_RES = DATA_DIR/"masterseq_model_ready_plus_resolved.parquet"
SPLITS_G = RUN_DIR/"splits_loco_gene.csv"
SPLITS_S = RUN_DIR/"splits_loco_site.csv"

df = pd.read_parquet(P_RES).dropna(subset=["variant_id","gene","label","window_mut","window_wt"]).copy()
df["label"] = df["label"].astype(int)
df["gene"] = df["gene"].astype(str)
df["variant_id"] = df["variant_id"].astype(str)

spl_g = pd.read_csv(SPLITS_G)
spl_s = pd.read_csv(SPLITS_S)

def add_bio_feats(r: dict, include_codon=False, include_tri=True):
    d = {}
    # transition/transversion
    pur, pyr = {"A","G"}, {"C","T"}
    ref, alt = str(r.get("ref","")), str(r.get("alt",""))
    if ref and alt:
        d["is_transition"] = int((ref in pur and alt in pur) or (ref in pyr and alt in pyr))
    if include_tri:
        tri = str(r.get("tri_center",""))
        d["is_cpg"] = int(tri.startswith("CG")) if tri else 0
        if tri: d[f"tri:{tri}"] = 1
    if include_codon:
        wt, mt = r.get("wt_codon",""), r.get("mut_codon","")
        if isinstance(wt,str) and isinstance(mt,str) and len(wt)==3 and len(mt)==3:
            d[f"codon:{wt}->{mt}"] = 1
    return d

def feats(row: pd.Series, *, use_flanks: bool, include_tri: bool):
    r = row.to_dict()
    d = add_bio_feats(r, include_codon=False, include_tri=include_tri)
    if use_flanks:
        s = str(r.get("window_mut",""))
        # K=3
        for i in range(max(0, len(s)-3+1)):
            k = f"s_{i}:{s[i:i+3]}"
            d[k] = d.get(k, 0) + 1
    return d

def run_cfg(cfg_name, use_flanks, include_tri, splits, tag):
    rows=[]
    for fold, sub in splits.groupby("fold"):
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr, te = dd[dd["split"]=="train"], dd[dd["split"]=="test"]
        if tr.empty or te.empty:
            continue
        vec = DictVectorizer(sparse=True)
        Xtr = vec.fit_transform(feats(r, use_flanks=use_flanks, include_tri=include_tri) for _, r in tr.iterrows())
        ytr = tr["label"].values
        Xte = vec.transform(feats(r, use_flanks=use_flanks, include_tri=include_tri) for _, r in te.iterrows())
        clf = LogisticRegression(penalty="l2", solver="lbfgs", C=0.25, max_iter=5000, random_state=SEED)
        clf.fit(Xtr, ytr)
        p = clf.predict_proba(Xte)[:,1]
        out = te.loc[:, ["variant_id","gene","label"]].copy()
        out["cfg"] = cfg_name; out["tag"] = tag; out["y_pred"] = p
        rows.append(out)
    preds = pd.concat(rows, ignore_index=True)
    # summarize
    agg=[]
    for g, sub in preds.groupby("gene"):
        y,p = sub["label"].values, sub["y_pred"].values
        if len(np.unique(y))<2: continue
        agg.append({
            "cfg": cfg_name, "tag": tag, "gene": g, "n": int(len(sub)),
            "auroc": round(roc_auc_score(y,p),4),
            "pr_auc": round(average_precision_score(y,p),4),
            "brier": round(brier_score_loss(y,p),4),
        })
    return preds, pd.DataFrame(agg).sort_values(["cfg","tag","gene"])

cfgs = [
    ("tri_only", False, True),        # just tri + transition/CpG
    ("flanks_no_tri", True, False),   # flanks without any tri features
]

all_summ=[]
for cfg_name, use_flanks, include_tri in cfgs:
    for tag, splits in [("LOCO-GENE", spl_g), ("LOCO-SITE", spl_s)]:
        preds, summ = run_cfg(cfg_name, use_flanks, include_tri, splits, tag)
        preds.to_csv(RUN_DIR/f"preds_{cfg_name}_{tag}.csv", index=False)
        all_summ.append(summ)

summ = pd.concat(all_summ, ignore_index=True)
outp = RUN_DIR/"metrics_trionly_vs_ntri.csv"
summ.to_csv(outp, index=False)
print("Run dir:", RUN_DIR)
print("Saved:", outp.name)
display(summ)


Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607
Saved: metrics_trionly_vs_ntri.csv


,cfg,tag,gene,n,auroc,pr_auc,brier
0,tri_only,LOCO-GENE,EGFR,39,0.3043,0.2786,0.2562
1,tri_only,LOCO-GENE,KRAS,11,0.6042,0.8659,0.3268
2,tri_only,LOCO-GENE,PIK3CA,24,0.5370,0.8022,0.3267
3,tri_only,LOCO-GENE,TP53,197,0.5131,0.3725,0.2669
4,tri_only,LOCO-SITE,EGFR,39,0.3000,0.2747,0.2588
5,tri_only,LOCO-SITE,KRAS,11,0.5833,0.8335,0.3059
6,tri_only,LOCO-SITE,PIK3CA,24,0.6111,0.8576,0.2945
7,tri_only,LOCO-SITE,TP53,197,0.5947,0.4369,0.2249
8,flanks_no_tri,LOCO-GENE,EGFR,39,0.6686,0.6126,0.2066
9,flanks_no_tri,LOCO-GENE,KRAS,11,0.7500,0.8755,0.2180


**Dope.** flanks are doing the heavy lifting; tri-only is meh. time to force the model to read the **change** not just the neighborhood.

next block = $Δ$-**features**: we featurize MUT and WT separately (K=3 flanks + minimal bio), then train on the **difference** (MUT − WT). That kills static context and should move **pair/accuracy** off ~0.5 if the mutation itself carries signal.

In [19]:
# F4 — Δ-features (MUT − WT) logistic, LOCO-GENE & LOCO-SITE
import os, glob, time, json
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

SEED = 1337
np.random.seed(SEED)

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
ARTI_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
cand = sorted(glob.glob(str(ARTI_DIR / "sentiment_run_plus_variants_*")))
assert cand, "No sentiment_run_plus_variants_* dirs found. Run E2 first."
RUN_DIR = Path(cand[-1])

P_RES = DATA_DIR/"masterseq_model_ready_plus_resolved.parquet"
SPLITS_G = RUN_DIR/"splits_loco_gene.csv"
SPLITS_S = RUN_DIR/"splits_loco_site.csv"

df = pd.read_parquet(P_RES).dropna(subset=["variant_id","gene","label","window_mut","window_wt","tri_center","wt_codon","mut_codon","ref","alt","cds_pos"]).copy()
df["label"] = df["label"].astype(int)
df["gene"] = df["gene"].astype(str)
df["variant_id"] = df["variant_id"].astype(str)
df["ref"] = df["ref"].astype(str).str.upper()
df["alt"] = df["alt"].astype(str).str.upper()
df["tri_center"] = df["tri_center"].astype(str).str.upper()

spl_g = pd.read_csv(SPLITS_G)
spl_s = pd.read_csv(SPLITS_S)

def tri_mut_from_row(r):
    t = str(r["tri_center"])
    return t[:1] + r["alt"] + t[2:] if len(t)==3 else ""

def add_bio_mut_only(rdict):
    d = {}
    # transition/transversion (mut-only so it survives subtraction)
    pur, pyr = {"A","G"}, {"C","T"}
    ref, alt = rdict.get("ref",""), rdict.get("alt","")
    if ref and alt:
        d["is_transition"] = int((ref in pur and alt in pur) or (ref in pyr and alt in pyr))
    # coarse AA change tag (mut-only)
    wt, mt = rdict.get("wt_codon",""), rdict.get("mut_codon","")
    if isinstance(wt,str) and isinstance(mt,str) and len(wt)==3 and len(mt)==3:
        d[f"codon:{wt}->{mt}"] = 1
    return d

def kmers(seq, K=3):
    s = str(seq)
    return [f"k3:{s[i:i+3]}@{i}" for i in range(max(0, len(s)-K+1))]  # position-tagged 3-mers to localize change

def feats_for_seq(row, is_mut: bool, K=3):
    # dict features for either MUT (window_mut) or WT (window_wt)
    seq = row["window_mut"] if is_mut else row["window_wt"]
    d = {}
    # k-mers
    for tok in kmers(seq, K=K):
        d[tok] = d.get(tok, 0) + 1
    # tri token — use mutated tri for MUT, WT tri for WT
    tri_tok = tri_mut_from_row(row) if is_mut else row["tri_center"]
    if tri_tok:
        d[f"tri:{tri_tok}"] = 1
    # mut-only bio features
    if is_mut:
        d.update(add_bio_mut_only(row.to_dict()))
    return d

def delta_matrix(vec: DictVectorizer, rows_mut, rows_wt):
    Xm = vec.transform(rows_mut)
    Xw = vec.transform(rows_wt)
    return Xm - Xw  # sparse CSR diff

def run_delta(df, splits, tag):
    rows_out=[]
    for fold, sub in splits.groupby("fold"):
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr = dd[dd["split"]=="train"].copy()
        te = dd[dd["split"]=="test"].copy()
        if tr.empty or te.empty:
            continue

        # Build vectorizer on BOTH MUT and WT feature dicts from TRAIN
        train_mut_dicts = [feats_for_seq(r, True)  for _, r in tr.iterrows()]
        train_wt_dicts  = [feats_for_seq(r, False) for _, r in tr.iterrows()]
        vec = DictVectorizer(sparse=True)
        vec.fit(train_mut_dicts + train_wt_dicts)

        # ΔX for train/test
        Xtr = delta_matrix(vec,
                           [feats_for_seq(r, True)  for _, r in tr.iterrows()],
                           [feats_for_seq(r, False) for _, r in tr.iterrows()])
        ytr = tr["label"].values

        Xte = delta_matrix(vec,
                           [feats_for_seq(r, True)  for _, r in te.iterrows()],
                           [feats_for_seq(r, False) for _, r in te.iterrows()])
        yte = te["label"].values

        clf = LogisticRegression(penalty="l2", solver="lbfgs", C=0.25, max_iter=5000, random_state=SEED)
        clf.fit(Xtr, ytr)
        p = clf.predict_proba(Xte)[:,1]

        # Pair-acc (Δ-model): thresholding p vs 0.5 is equivalent to "MUT outranks WT"
        pair_ok = ((p > 0.5) & (yte==1)) | ((p <= 0.5) & (yte==0))

        out = te.loc[:, ["variant_id","gene","label"]].copy()
        out["fold"] = fold
        out["split_mode"] = tag
        out["y_pred_delta"] = p
        out["pair_ok"] = pair_ok.astype(int)
        rows_out.append(out)

    preds = pd.concat(rows_out, ignore_index=True) if rows_out else pd.DataFrame()
    return preds

def boot_ci_auc(y, p, B=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y)); vals=[]
    for _ in range(B):
        b = rng.choice(idx, size=len(idx), replace=True)
        if len(np.unique(y[b]))<2: continue
        vals.append(roc_auc_score(y[b], p[b]))
    if not vals: return np.nan, np.nan
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

preds_g = run_delta(df, pd.read_csv(SPLITS_G), "LOCO-GENE")
preds_s = run_delta(df, pd.read_csv(SPLITS_S), "LOCO-SITE")

# save
preds_g_path = RUN_DIR/"preds_delta_loco_gene_k3_l2.csv"
preds_s_path = RUN_DIR/"preds_delta_loco_site_k3_l2.csv"
preds_g.to_csv(preds_g_path, index=False)
preds_s.to_csv(preds_s_path, index=False)

# summarize
def summarize(preds):
    rows=[]
    for g, sub in preds.groupby("gene"):
        y, p = sub["label"].values, sub["y_pred_delta"].values
        if len(np.unique(y))<2:
            continue
        au = roc_auc_score(y,p); pr = average_precision_score(y,p); br = brier_score_loss(y,p)
        lo, hi = boot_ci_auc(y,p,B=1000)
        pair_acc = sub["pair_ok"].mean()
        rows.append({
            "gene": g, "n": int(len(sub)),
            "auroc": round(au,4), "auroc_ci_low": round(lo,4), "auroc_ci_high": round(hi,4),
            "pr_auc": round(pr,4), "brier": round(br,4), "pair_acc": round(float(pair_acc),4)
        })
    return pd.DataFrame(rows).sort_values("gene")

summ_g = summarize(preds_g).assign(tag="LOCO-GENE")
summ_s = summarize(preds_s).assign(tag="LOCO-SITE")
summary = pd.concat([summ_g, summ_s], ignore_index=True)[["tag","gene","n","auroc","auroc_ci_low","auroc_ci_high","pr_auc","brier","pair_acc"]]

metrics_path = RUN_DIR/"metrics_delta_k3_l2.csv"
summary.to_csv(metrics_path, index=False)

print("Run dir:", RUN_DIR)
print("Saved predictions:", preds_g_path.name, "and", preds_s_path.name)
print("Saved metrics:", metrics_path.name)
display(summary.sort_values(["tag","gene"]))


Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607
Saved predictions: preds_delta_loco_gene_k3_l2.csv and preds_delta_loco_site_k3_l2.csv
Saved metrics: metrics_delta_k3_l2.csv


,tag,gene,n,auroc,auroc_ci_low,auroc_ci_high,pr_auc,brier,pair_acc
0,LOCO-GENE,EGFR,39,0.5700,0.3473,0.7870,0.5721,0.2369,0.6923
1,LOCO-GENE,KRAS,11,0.1667,0.0000,0.4643,0.6613,0.3879,0.1818
2,LOCO-GENE,PIK3CA,24,0.3981,0.1684,0.6483,0.7787,0.3771,0.3750
3,LOCO-GENE,TP53,197,0.5280,0.4449,0.6174,0.3737,0.2740,0.4569
4,LOCO-SITE,EGFR,39,0.4943,0.2887,0.7105,0.4501,0.2477,0.6154
5,LOCO-SITE,KRAS,11,0.1250,0.0000,0.3750,0.6488,0.3708,0.1818
6,LOCO-SITE,PIK3CA,24,0.4815,0.2211,0.7263,0.8082,0.3431,0.3750
7,LOCO-SITE,TP53,197,0.6362,0.5498,0.7168,0.5590,0.2141,0.6701


nice lift + some chaos. flanks are carrying, Δ-model made EGFR/TP53 wake up (pair↑), KRAS/PIK3CA tanked (tiny n, different biology). let’s force the model to “read the edit” with a few cheap, causal $Δ$-features: amino-acid chemistry + CpG gain/loss + GC shift. also flip on class weighting so imbalance doesn’t bully us.

In [20]:
# F5 — Δ-features + bio priors (Grantham, BLOSUM62, CpG gain/loss, ΔGC) with class_weight='balanced'

import os, glob, time, json
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

SEED = 1337
np.random.seed(SEED)

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
ARTI_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
cand = sorted(glob.glob(str(ARTI_DIR / "sentiment_run_plus_variants_*")))
assert cand, "No variant run dir (run E2)."
RUN_DIR = Path(cand[-1])

P_RES = DATA_DIR/"masterseq_model_ready_plus_resolved.parquet"
SPLITS_G = RUN_DIR/"splits_loco_gene.csv"
SPLITS_S = RUN_DIR/"splits_loco_site.csv"

df = pd.read_parquet(P_RES).dropna(subset=[
    "variant_id","gene","label","window_mut","window_wt",
    "tri_center","wt_codon","mut_codon","ref","alt","cds_pos","transcript"
]).copy()
for c in ["gene","variant_id","ref","alt","tri_center","wt_codon","mut_codon","transcript"]:
    df[c] = df[c].astype(str).str.upper()
df["label"] = df["label"].astype(int)

spl_g = pd.read_csv(SPLITS_G)
spl_s = pd.read_csv(SPLITS_S)

# --- genetic code ---
CODON2AA = {
    # F L I M V S P T A Y H Q N K D E C W R G
    "TTT":"F","TTC":"F","TTA":"L","TTG":"L",
    "CTT":"L","CTC":"L","CTA":"L","CTG":"L",
    "ATT":"I","ATC":"I","ATA":"I","ATG":"M",
    "GTT":"V","GTC":"V","GTA":"V","GTG":"V",
    "TCT":"S","TCC":"S","TCA":"S","TCG":"S",
    "CCT":"P","CCC":"P","CCA":"P","CCG":"P",
    "ACT":"T","ACC":"T","ACA":"T","ACG":"T",
    "GCT":"A","GCC":"A","GCA":"A","GCG":"A",
    "TAT":"Y","TAC":"Y","TAA":"*","TAG":"*",
    "CAT":"H","CAC":"H","CAA":"Q","CAG":"Q",
    "AAT":"N","AAC":"N","AAA":"K","AAG":"K",
    "GAT":"D","GAC":"D","GAA":"E","GAG":"E",
    "TGT":"C","TGC":"C","TGA":"*","TGG":"W",
    "CGT":"R","CGC":"R","CGA":"R","CGG":"R",
    "AGT":"S","AGC":"S","AGA":"R","AGG":"R",
    "GGT":"G","GGC":"G","GGA":"G","GGG":"G",
}

# --- Grantham distances (max 215) ---
_GR = {
 'A':{'A':0,'R':112,'N':111,'D':126,'C':195,'Q':91,'E':107,'G':60,'H':86,'I':94,'L':96,'K':106,'M':84,'F':113,'P':27,'S':99,'T':58,'W':148,'Y':112,'V':64},
 'R':{'A':112,'R':0,'N':86,'D':96,'C':180,'Q':43,'E':54,'G':125,'H':29,'I':97,'L':102,'K':26,'M':91,'F':97,'P':103,'S':110,'T':71,'W':101,'Y':77,'V':96},
 'N':{'A':111,'R':86,'N':0,'D':23,'C':139,'Q':68,'E':42,'G':80,'H':68,'I':149,'L':153,'K':94,'M':142,'F':158,'P':91,'S':46,'T':65,'W':174,'Y':143,'V':133},
 'D':{'A':126,'R':96,'N':23,'D':0,'C':154,'Q':61,'E':45,'G':94,'H':81,'I':168,'L':172,'K':101,'M':160,'F':177,'P':108,'S':65,'T':85,'W':181,'Y':160,'V':152},
 'C':{'A':195,'R':180,'N':139,'D':154,'C':0,'Q':154,'E':170,'G':159,'H':174,'I':198,'L':198,'K':202,'M':196,'F':205,'P':169,'S':112,'T':149,'W':215,'Y':194,'V':192},
 'Q':{'A':91,'R':43,'N':68,'D':61,'C':154,'Q':0,'E':29,'G':87,'H':24,'I':109,'L':113,'K':53,'M':101,'F':116,'P':76,'S':68,'T':42,'W':130,'Y':99,'V':96},
 'E':{'A':107,'R':54,'N':42,'D':45,'C':170,'Q':29,'E':0,'G':98,'H':40,'I':134,'L':138,'K':56,'M':126,'F':140,'P':93,'S':80,'T':65,'W':152,'Y':122,'V':121},
 'G':{'A':60,'R':125,'N':80,'D':94,'C':159,'Q':87,'E':98,'G':0,'H':98,'I':135,'L':138,'K':127,'M':127,'F':153,'P':42,'S':56,'T':59,'W':184,'Y':147,'V':109},
 'H':{'A':86,'R':29,'N':68,'D':81,'C':174,'Q':24,'E':40,'G':98,'H':0,'I':94,'L':99,'K':32,'M':87,'F':100,'P':77,'S':89,'T':47,'W':115,'Y':83,'V':84},
 'I':{'A':94,'R':97,'N':149,'D':168,'C':198,'Q':109,'E':134,'G':135,'H':94,'I':0,'L':5,'K':102,'M':10,'F':21,'P':95,'S':142,'T':89,'W':61,'Y':33,'V':29},
 'L':{'A':96,'R':102,'N':153,'D':172,'C':198,'Q':113,'E':138,'G':138,'H':99,'I':5,'L':0,'K':107,'M':15,'F':22,'P':98,'S':145,'T':92,'W':61,'Y':36,'V':32},
 'K':{'A':106,'R':26,'N':94,'D':101,'C':202,'Q':53,'E':56,'G':127,'H':32,'I':102,'L':107,'K':0,'M':95,'F':102,'P':103,'S':121,'T':78,'W':110,'Y':85,'V':97},
 'M':{'A':84,'R':91,'N':142,'D':160,'C':196,'Q':101,'E':126,'G':127,'H':87,'I':10,'L':15,'K':95,'M':0,'F':28,'P':87,'S':135,'T':81,'W':67,'Y':36,'V':21},
 'F':{'A':113,'R':97,'N':158,'D':177,'C':205,'Q':116,'E':140,'G':153,'H':100,'I':21,'L':22,'K':102,'M':28,'F':0,'P':114,'S':155,'T':103,'W':40,'Y':22,'V':50},
 'P':{'A':27,'R':103,'N':91,'D':108,'C':169,'Q':76,'E':93,'G':42,'H':77,'I':95,'L':98,'K':103,'M':87,'F':114,'P':0,'S':74,'T':38,'W':147,'Y':110,'V':68},
 'S':{'A':99,'R':110,'N':46,'D':65,'C':112,'Q':68,'E':80,'G':56,'H':89,'I':142,'L':145,'K':121,'M':135,'F':155,'P':74,'S':0,'T':58,'W':177,'Y':144,'V':124},
 'T':{'A':58,'R':71,'N':65,'D':85,'C':149,'Q':42,'E':65,'G':59,'H':47,'I':89,'L':92,'K':78,'M':81,'F':103,'P':38,'S':58,'T':0,'W':128,'Y':92,'V':69},
 'W':{'A':148,'R':101,'N':174,'D':181,'C':215,'Q':130,'E':152,'G':184,'H':115,'I':61,'L':61,'K':110,'M':67,'F':40,'P':147,'S':177,'T':128,'W':0,'Y':37,'V':88},
 'Y':{'A':112,'R':77,'N':143,'D':160,'C':194,'Q':99,'E':122,'G':147,'H':83,'I':33,'L':36,'K':85,'M':36,'F':22,'P':110,'S':144,'T':92,'W':37,'Y':0,'V':55},
 'V':{'A':64,'R':96,'N':133,'D':152,'C':192,'Q':96,'E':121,'G':109,'H':84,'I':29,'L':32,'K':97,'M':21,'F':50,'P':68,'S':124,'T':69,'W':88,'Y':55,'V':0}
}
def grantham(a,b):
    if a==b: return 0
    return _GR.get(a,{}).get(b, _GR.get(b,{}).get(a, 0))

# --- BLOSUM62 (subset to 20 AA) ---
_B = """\
   A  R  N  D  C  Q  E  G  H  I  L  K  M  F  P  S  T  W  Y  V
A  4 -1 -2 -2  0 -1 -1  0 -2 -1 -1 -1 -1 -2 -1  1  0 -3 -2  0
R -1  5  0 -2 -3  1  0 -2  0 -3 -2  2 -1 -3 -2 -1 -1 -3 -2 -3
N -2  0  6  1 -3  0  0  0  1 -3 -3  0 -2 -3 -2  1  0 -4 -2 -3
D -2 -2  1  6 -3  0  2 -1 -1 -3 -4 -1 -3 -3 -1  0 -1 -4 -3 -3
C  0 -3 -3 -3  9 -3 -4 -3 -3 -1 -1 -3 -1 -2 -3 -1 -1 -2 -2 -1
Q -1  1  0  0 -3  5  2 -2  0 -3 -2  1  0 -3 -1  0 -1 -2 -1 -2
E -1  0  0  2 -4  2  5 -2  0 -3 -3  1 -2 -3 -1  0 -1 -3 -2 -2
G  0 -2  0 -1 -3 -2 -2  6 -2 -4 -4 -2 -3 -3 -2  0 -2 -2 -3 -3
H -2  0  1 -1 -3  0  0 -2  8 -3 -3 -1 -2 -1 -2 -1 -2 -2  2 -3
I -1 -3 -3 -3 -1 -3 -3 -4 -3  4  2 -3  1  0 -3 -2 -1 -3 -1  3
L -1 -2 -3 -4 -1 -2 -3 -4 -3  2  4 -2  2  0 -3 -2 -1 -2 -1  1
K -1  2  0 -1 -3  1  1 -2 -1 -3 -2  5 -1 -3 -1  0 -1 -3 -2 -2
M -1 -1 -2 -3 -1  0 -2 -3 -2  1  2 -1  5  0 -2 -1 -1 -1 -1  1
F -2 -3 -3 -3 -2 -3 -3 -3 -1  0  0 -3  0  6 -4 -2 -2  1  3 -1
P -1 -2 -2 -1 -3 -1 -1 -2 -2 -3 -3 -1 -2 -4  7 -1 -1 -4 -3 -2
S  1 -1  1  0 -1  0  0  0 -1 -2 -2  0 -1 -2 -1  4  1 -3 -2 -2
T  0 -1  0 -1 -1 -1 -1 -2 -2 -1 -1 -1 -1 -2 -1  1  5 -2 -2  0
W -3 -3 -4 -4 -2 -2 -3 -2 -2 -3 -2 -3 -1  1 -4 -3 -2 11  2 -3
Y -2 -2 -2 -3 -2 -1 -2 -3  2 -1 -1 -2 -1  3 -3 -2 -2  2  7 -1
V  0 -3 -3 -3 -1 -2 -2 -3 -3  3  1 -2  1 -1 -2 -2  0 -3 -1  4
"""
B62 = {}
lines = [l for l in _B.strip().splitlines() if l.strip()]
hdr = lines[0].split()
for line in lines[1:]:
    parts = line.split()
    a = parts[0]
    B62[a] = {hdr[i]: int(parts[i+1]) for i in range(len(hdr))}

def blosum62(a,b):
    return B62.get(a,{}).get(b, B62.get(b,{}).get(a, 0))

def tri_mut_from_row(r):
    t = str(r["tri_center"])
    return t[:1] + r["alt"] + t[2:] if len(t)==3 else ""

def gc_frac(s):
    s = str(s).upper()
    if not s: return 0.0
    g = s.count("G") + s.count("C")
    return g / len(s)

# position-tagged 3-mers
def kmers(seq, K=3):
    s = str(seq)
    return [f"k3:{s[i:i+3]}@{i}" for i in range(max(0, len(s)-K+1))]

def feats_for_seq(row, is_mut: bool, K=3):
    seq = row["window_mut"] if is_mut else row["window_wt"]
    d = {}
    # flanks k-mers
    for tok in kmers(seq, K=K):
        d[tok] = d.get(tok, 0) + 1
    # tri token (mut vs wt)
    tri_tok = tri_mut_from_row(row) if is_mut else row["tri_center"]
    if tri_tok:
        d[f"tri:{tri_tok}"] = 1
    # MUT-only bio deltas so they survive subtraction
    if is_mut:
        wt, mt = row["wt_codon"], row["mut_codon"]
        a_wt = CODON2AA.get(wt, "X"); a_mt = CODON2AA.get(mt, "X")
        aa_same = int(a_wt == a_mt)
        d["aa_same"] = aa_same
        if a_wt != "X" and a_mt != "X":
            # scale to ~0..1
            d["grantham_s"] = grantham(a_wt, a_mt) / 215.0
            d["blosum_s"] = (blosum62(a_wt, a_mt) + 4) / 15.0  # -4..11 → 0..1
        # CpG gain/loss at center tri
        tri_wt = row["tri_center"]
        tri_mt = tri_mut_from_row(row)
        wt_has_cg = int("CG" in tri_wt if isinstance(tri_wt,str) else False)
        mt_has_cg = int("CG" in tri_mt if isinstance(tri_mt,str) else False)
        d["cpg_gain"] = int(mt_has_cg and not wt_has_cg)
        d["cpg_loss"] = int(wt_has_cg and not mt_has_cg)
        # ΔGC over the whole window (mut - wt)
        d["delta_gc"] = gc_frac(row["window_mut"]) - gc_frac(row["window_wt"])
    return d

def delta_matrix(vec: DictVectorizer, rows_mut, rows_wt):
    Xm = vec.transform(rows_mut); Xw = vec.transform(rows_wt)
    return Xm - Xw

def run_delta(df, splits, tag):
    rows_out=[]
    for fold, sub in splits.groupby("fold"):
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr, te = dd[dd["split"]=="train"].copy(), dd[dd["split"]=="test"].copy()
        if tr.empty or te.empty:
            continue

        train_mut = [feats_for_seq(r, True)  for _, r in tr.iterrows()]
        train_wt  = [feats_for_seq(r, False) for _, r in tr.iterrows()]
        vec = DictVectorizer(sparse=True)
        vec.fit(train_mut + train_wt)

        Xtr = delta_matrix(vec,
                           [feats_for_seq(r, True)  for _, r in tr.iterrows()],
                           [feats_for_seq(r, False) for _, r in tr.iterrows()])
        ytr = tr["label"].values

        Xte = delta_matrix(vec,
                           [feats_for_seq(r, True)  for _, r in te.iterrows()],
                           [feats_for_seq(r, False) for _, r in te.iterrows()])
        yte = te["label"].values

        clf = LogisticRegression(
            penalty="l2", solver="lbfgs", C=0.25, max_iter=5000,
            class_weight="balanced", random_state=SEED
        )
        clf.fit(Xtr, ytr)
        p = clf.predict_proba(Xte)[:,1]
        pair_ok = ((p > 0.5) & (yte==1)) | ((p <= 0.5) & (yte==0))

        out = te.loc[:, ["variant_id","gene","label"]].copy()
        out["fold"] = fold; out["split_mode"] = tag
        out["y_pred_delta_bio"] = p
        out["pair_ok"] = pair_ok.astype(int)
        rows_out.append(out)

    preds = pd.concat(rows_out, ignore_index=True) if rows_out else pd.DataFrame()
    return preds

def boot_ci_auc(y, p, B=1000, seed=SEED):
    rng = np.random.default_rng(seed); idx = np.arange(len(y)); vals=[]
    for _ in range(B):
        b = rng.choice(idx, size=len(idx), replace=True)
        if len(np.unique(y[b]))<2: continue
        vals.append(roc_auc_score(y[b], p[b]))
    if not vals: return np.nan, np.nan
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

preds_g = run_delta(df, pd.read_csv(SPLITS_G), "LOCO-GENE")
preds_s = run_delta(df, pd.read_csv(SPLITS_S), "LOCO-SITE")

# save
preds_g_path = RUN_DIR/"preds_delta_bio_loco_gene.csv"
preds_s_path = RUN_DIR/"preds_delta_bio_loco_site.csv"
preds_g.to_csv(preds_g_path, index=False)
preds_s.to_csv(preds_s_path, index=False)

def summarize(preds):
    rows=[]
    for g, sub in preds.groupby("gene"):
        y, p = sub["label"].values, sub["y_pred_delta_bio"].values
        if len(np.unique(y))<2: continue
        au = roc_auc_score(y,p); pr = average_precision_score(y,p); br = brier_score_loss(y,p)
        lo, hi = boot_ci_auc(y,p,B=1000)
        pair_acc = sub["pair_ok"].mean()
        rows.append({
            "gene": g, "n": int(len(sub)),
            "auroc": round(au,4), "auroc_ci_low": round(lo,4), "auroc_ci_high": round(hi,4),
            "pr_auc": round(pr,4), "brier": round(br,4), "pair_acc": round(float(pair_acc),4)
        })
    return pd.DataFrame(rows).sort_values("gene")

summ_g = summarize(preds_g).assign(tag="LOCO-GENE")
summ_s = summarize(preds_s).assign(tag="LOCO-SITE")
summary = pd.concat([summ_g, summ_s], ignore_index=True)[["tag","gene","n","auroc","auroc_ci_low","auroc_ci_high","pr_auc","brier","pair_acc"]]

metrics_path = RUN_DIR/"metrics_delta_bio_k3_l2.csv"
summary.to_csv(metrics_path, index=False)

print("Run dir:", RUN_DIR)
print("Saved predictions:", preds_g_path.name, "and", preds_s_path.name)
print("Saved metrics:", metrics_path.name)
display(summary.sort_values(["tag","gene"]))


Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607
Saved predictions: preds_delta_bio_loco_gene.csv and preds_delta_bio_loco_site.csv
Saved metrics: metrics_delta_bio_k3_l2.csv


,tag,gene,n,auroc,auroc_ci_low,auroc_ci_high,pr_auc,brier,pair_acc
0,LOCO-GENE,EGFR,39,0.6071,0.4112,0.7908,0.5237,0.2386,0.5641
1,LOCO-GENE,KRAS,11,0.2500,0.0000,0.6250,0.6857,0.3469,0.2727
2,LOCO-GENE,PIK3CA,24,0.4167,0.1765,0.6842,0.7760,0.3285,0.3750
3,LOCO-GENE,TP53,197,0.5476,0.4656,0.6350,0.3978,0.2609,0.4975
4,LOCO-SITE,EGFR,39,0.5657,0.3657,0.7532,0.4911,0.2428,0.5385
5,LOCO-SITE,KRAS,11,0.2083,0.0000,0.6317,0.6232,0.3457,0.4545
6,LOCO-SITE,PIK3CA,24,0.5463,0.2968,0.7818,0.8364,0.3043,0.5000
7,LOCO-SITE,TP53,197,0.6721,0.5934,0.7461,0.5791,0.2199,0.6142


**okay, we cooking.** Our ablations say the neighborhood is loud, the edit itself is whispering. Surgical next step:

- we’ll **anchor the features on the edited base** (not the codon center), grab a tight ±10 bp patch around that base from both WT and MUT, and train on $Δ$-features (`MUT` − `WT`) from that patch. this forces the model to stare right at the change instead of vibing on broad context.

In [21]:
# F6 — Edit-anchored Δ-features (±10bp patch around the mutated base), LOCO-GENE & LOCO-SITE

import os, glob
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

SEED = 1337
np.random.seed(SEED)

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
ARTI_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
cand = sorted(glob.glob(str(ARTI_DIR / "sentiment_run_plus_variants_*")))
assert cand, "No variant run dir (run E2 first)."
RUN_DIR = Path(cand[-1])

P_RES = DATA_DIR/"masterseq_model_ready_plus_resolved.parquet"
SPLITS_G = RUN_DIR/"splits_loco_gene.csv"
SPLITS_S = RUN_DIR/"splits_loco_site.csv"

df = pd.read_parquet(P_RES).dropna(subset=[
    "variant_id","gene","label","window_mut","window_wt",
    "wt_codon","mut_codon","ref","alt","cds_pos"
]).copy()

# normalize
for c in ["gene","variant_id","ref","alt","wt_codon","mut_codon","window_mut","window_wt"]:
    df[c] = df[c].astype(str).str.upper()
df["label"] = df["label"].astype(int)
df["cds_pos"] = pd.to_numeric(df["cds_pos"], errors="coerce").astype("Int64")

# helper: index of mutated base inside our stored windows
# our windows are centered on codon base 2. find offset of mutated base within the codon: {-1,0,1}
def mut_index_in_window(row, flank=60):
    cds_pos = int(row["cds_pos"])
    i0 = cds_pos - 1
    codon_start = (i0 // 3) * 3
    codon_center = codon_start + 1           # 2nd base of codon (0-based, CDS)
    offset = i0 - codon_center               # -1, 0, or +1
    # in stored windows, the codon center should be at index = flank
    j_center = flank                         # 0-based index of codon center in window strings
    j_mut = j_center + offset
    return j_mut

# extract ±R patch around mutated base from WT/MUT windows
def extract_patch(win: str, j: int, R: int = 10):
    s = str(win)
    if not s or j < 0 or j >= len(s):
        return ""
    a = max(0, j - R)
    b = min(len(s), j + R + 1)
    # pad if window edge chops the patch
    left_pad = "N" * (R - (j - a))
    right_pad = "N" * (R - (b - j - 1))
    return left_pad + s[a:b] + right_pad

# position-anchored k-mers relative to patch center
def kmers_with_relpos(seq: str, K=3):
    s = str(seq)
    toks = []
    for i in range(max(0, len(s) - K + 1)):
        # relative to patch center (which is at index R)
        toks.append(f"k3:{s[i:i+K]}@rel{ i - ((len(s)-1)//2) }")
    return toks

def feats_patch(row, is_mut: bool, R=10, K=3):
    j = mut_index_in_window(row, flank=60)
    seq = row["window_mut"] if is_mut else row["window_wt"]
    patch = extract_patch(seq, j, R=R)
    d = {}
    # anchored k-mers
    for tok in kmers_with_relpos(patch, K=K):
        d[tok] = d.get(tok, 0) + 1
    # tiny local tokens around the edit (±1 bp, 3-mer centered at edit)
    mid = R
    tri = patch[mid-1:mid+2] if len(patch) >= 3 and 1 <= mid <= len(patch)-2 else ""
    if tri:
        d[f"tri_local:{tri}"] = 1
    # add AA-change tag only for MUT so it survives subtraction
    if is_mut:
        wt, mt = row["wt_codon"], row["mut_codon"]
        if isinstance(wt,str) and isinstance(mt,str) and len(wt)==3 and len(mt)==3:
            d[f"codon:{wt}->{mt}"] = 1
    return d

def delta_matrix(vec: DictVectorizer, rows_mut, rows_wt):
    Xm = vec.transform(rows_mut); Xw = vec.transform(rows_wt)
    return Xm - Xw

def run_delta_anchor(df, splits, tag, R=10, K=3):
    out_rows=[]
    for fold, sub in splits.groupby("fold"):
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr, te = dd[dd["split"]=="train"].copy(), dd[dd["split"]=="test"].copy()
        if tr.empty or te.empty:
            continue

        train_mut = [feats_patch(r, True,  R=R, K=K) for _, r in tr.iterrows()]
        train_wt  = [feats_patch(r, False, R=R, K=K) for _, r in tr.iterrows()]
        vec = DictVectorizer(sparse=True)
        vec.fit(train_mut + train_wt)

        Xtr = delta_matrix(vec,
                           [feats_patch(r, True,  R=R, K=K) for _, r in tr.iterrows()],
                           [feats_patch(r, False, R=R, K=K) for _, r in tr.iterrows()])
        ytr = tr["label"].values

        Xte = delta_matrix(vec,
                           [feats_patch(r, True,  R=R, K=K) for _, r in te.iterrows()],
                           [feats_patch(r, False, R=R, K=K) for _, r in te.iterrows()])
        yte = te["label"].values

        clf = LogisticRegression(penalty="l2", solver="lbfgs", C=0.25, max_iter=5000,
                                 class_weight="balanced", random_state=SEED)
        clf.fit(Xtr, ytr)
        p = clf.predict_proba(Xte)[:,1]
        pair_ok = ((p > 0.5) & (yte==1)) | ((p <= 0.5) & (yte==0))

        part = te.loc[:, ["variant_id","gene","label"]].copy()
        part["fold"] = fold; part["split_mode"] = tag
        part["y_pred_delta_anchor"] = p
        part["pair_ok"] = pair_ok.astype(int)
        out_rows.append(part)

    preds = pd.concat(out_rows, ignore_index=True) if out_rows else pd.DataFrame()
    return preds

def summarize(preds):
    rows=[]
    for g, sub in preds.groupby("gene"):
        y, p = sub["label"].values, sub["y_pred_delta_anchor"].values
        if len(np.unique(y))<2: continue
        au = roc_auc_score(y,p); pr = average_precision_score(y,p); br = brier_score_loss(y,p)
        pair_acc = sub["pair_ok"].mean()
        rows.append({
            "gene": g, "n": int(len(sub)),
            "auroc": round(au,4), "pr_auc": round(pr,4),
            "brier": round(br,4), "pair_acc": round(float(pair_acc),4)
        })
    return pd.DataFrame(rows).sort_values("gene")

spl_g = pd.read_csv(SPLITS_G)
spl_s = pd.read_csv(SPLITS_S)

preds_g = run_delta_anchor(df, spl_g, "LOCO-GENE", R=10, K=3)
preds_s = run_delta_anchor(df, spl_s, "LOCO-SITE", R=10, K=3)

# save + summarize
preds_g_path = RUN_DIR/"preds_delta_anchor_loco_gene.csv"
preds_s_path = RUN_DIR/"preds_delta_anchor_loco_site.csv"
preds_g.to_csv(preds_g_path, index=False)
preds_s.to_csv(preds_s_path, index=False)

summ_g = summarize(preds_g).assign(tag="LOCO-GENE")
summ_s = summarize(preds_s).assign(tag="LOCO-SITE")
summary = pd.concat([summ_g, summ_s], ignore_index=True)[["tag","gene","n","auroc","pr_auc","brier","pair_acc"]]

metrics_path = RUN_DIR/"metrics_delta_anchor.csv"
summary.to_csv(metrics_path, index=False)

print("Run dir:", RUN_DIR)
print("Saved predictions:", preds_g_path.name, "and", preds_s_path.name)
print("Saved metrics:", metrics_path.name)
display(summary.sort_values(["tag","gene"]))


Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607
Saved predictions: preds_delta_anchor_loco_gene.csv and preds_delta_anchor_loco_site.csv
Saved metrics: metrics_delta_anchor.csv


,tag,gene,n,auroc,pr_auc,brier,pair_acc
0,LOCO-GENE,EGFR,39,0.5071,0.4341,0.2515,0.6154
1,LOCO-GENE,KRAS,11,0.6667,0.8257,0.2559,0.3636
2,LOCO-GENE,PIK3CA,24,0.4074,0.7903,0.3175,0.4583
3,LOCO-GENE,TP53,197,0.5061,0.3932,0.2649,0.4467
4,LOCO-SITE,EGFR,39,0.4514,0.3703,0.2626,0.4872
5,LOCO-SITE,KRAS,11,0.6667,0.8228,0.2747,0.3636
6,LOCO-SITE,PIK3CA,24,0.5648,0.8427,0.2976,0.5833
7,LOCO-SITE,TP53,197,0.5634,0.4218,0.2453,0.5736


**Nice.** The edit isn’t screaming yet, so let’s do the honest check:  

> tri-matched eval. we’ll match each positive to a negative in the same gene with the exact same trinucleotide context (fallback: same center base) and re-score. if AUROC craters vs before, oour gains were mostly static context. If it holds, we’ve got real edit semantics.

In [22]:
# G1 — Tri-matched evaluation (per gene), using latest splits and flanks_no_codon vs WT-only control
import glob, json
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

SEED = 1337
np.random.seed(SEED)

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
ARTI_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
P_RES   = DATA_DIR/"masterseq_model_ready_plus_resolved.parquet"

# latest variant split dir
cand = sorted(glob.glob(str(ARTI_DIR / "sentiment_run_plus_variants_*")))
assert cand, "No sentiment_run_plus_variants_* dir found (run E2 first)."
RUN_DIR = Path(cand[-1])
SPLITS_G = RUN_DIR/"splits_loco_gene.csv"
SPLITS_S = RUN_DIR/"splits_loco_site.csv"

df = pd.read_parquet(P_RES).dropna(subset=["variant_id","gene","label","tri_center","window_mut","window_wt"]).copy()
for c in ["variant_id","gene","tri_center"]:
    df[c] = df[c].astype(str).str.upper()
df["label"] = df["label"].astype(int)

spl_g = pd.read_csv(SPLITS_G)
spl_s = pd.read_csv(SPLITS_S)

def feats_flanks_no_codon(r, K=3, include_tri=True):
    d={}
    tri = str(r.get("tri_center","")).upper()
    if include_tri and len(tri)==3:
        d[f"tri:{tri}"]=1
        d["is_cpg"]=int(tri.startswith("CG"))
    s = str(r.get("window_mut",""))
    for i in range(max(0, len(s)-K+1)):
        d[f"s_{i}:{s[i:i+K]}"] = d.get(f"s_{i}:{s[i:i+K]}",0)+1
    return d

def feats_wt_only(r, K=3):
    # no tri, no codon; plain WT flanks
    d={}
    s = str(r.get("window_wt",""))
    for i in range(max(0, len(s)-K+1)):
        d[f"s_{i}:{s[i:i+K]}"] = d.get(f"s_{i}:{s[i:i+K]}",0)+1
    return d

def build_matched_subset(sub):
    """Return matched test indices and a small receipt dict.
       Strategy: for each positive, find a negative with same (gene, tri_center).
       If none, fall back to same (gene, center_base). 1:1 matching without replacement."""
    sub = sub.copy()
    sub["center_base"] = sub["tri_center"].str[1].fillna("")
    pos = sub[sub["label"]==1].copy()
    neg = sub[sub["label"]==0].copy()

    used = set()
    keep_idx = []

    hits_exact = hits_base = 0

    for i, r in pos.iterrows():
        key_exact = (r["gene"], r["tri_center"])
        pool = neg[(neg["gene"]==r["gene"]) & (neg["tri_center"]==r["tri_center"]) & (~neg.index.isin(used))]
        if not pool.empty:
            j = pool.sample(1, random_state=SEED).index[0]
            keep_idx += [i, j]
            used.add(j); hits_exact += 1
            continue
        key_base = (r["gene"], r["center_base"])
        pool = neg[(neg["gene"]==r["gene"]) & (neg["tri_center"].str[1]==r["center_base"]) & (~neg.index.isin(used))]
        if not pool.empty:
            j = pool.sample(1, random_state=SEED).index[0]
            keep_idx += [i, j]
            used.add(j); hits_base += 1
            continue
        # no match → drop this positive
    matched = sub.loc[sorted(set(keep_idx))].copy()
    rec = {
        "pos_in": int(len(pos)),
        "neg_in": int(len(neg)),
        "matched_pairs": int(len(matched[matched['label']==1])),
        "hits_exact_tri": int(hits_exact),
        "hits_center_base": int(hits_base),
    }
    return matched, rec

def train_eval_one(df, splits, tag):
    # aggregate predictions across folds, but only on tri-matched TEST subsets
    preds_rows = []
    recs = []

    for fold, sub in splits.groupby("fold"):
        dd  = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr  = dd[dd["split"]=="train"].copy()
        te0 = dd[dd["split"]=="test"].copy()
        if tr.empty or te0.empty:
            continue

        te, rec = build_matched_subset(te0)
        rec["fold"]=fold; rec["tag"]=tag
        recs.append(rec)
        if te["label"].nunique()<2 or rec["matched_pairs"]==0:
            continue

        # flanks_no_codon (includes tri features)
        vec = DictVectorizer(sparse=True)
        Xtr = vec.fit_transform(feats_flanks_no_codon(r) for _, r in tr.iterrows())
        ytr = tr["label"].values
        Xte_mut = vec.transform(feats_flanks_no_codon(r) for _, r in te.iterrows())
        Xte_wt  = vec.transform(feats_wt_only(r)       for _, r in te.iterrows())

        clf = LogisticRegression(penalty="l2", solver="lbfgs", C=0.25, max_iter=5000, random_state=SEED)
        clf.fit(Xtr, ytr)

        pm = clf.predict_proba(Xte_mut)[:,1]
        pw = clf.predict_proba(Xte_wt)[:,1]  # WT-only baseline scores on same matched set

        out = te.loc[:, ["variant_id","gene","label"]].copy()
        out["fold"]=fold; out["tag"]=tag
        out["y_pred_flanks"] = pm
        out["y_pred_wt_only"] = pw
        out["pair_ok"] = (((pm>pw) & (out["label"]==1)) | ((pm<=pw) & (out["label"]==0))).astype(int)
        preds_rows.append(out)

    preds = pd.concat(preds_rows, ignore_index=True) if preds_rows else pd.DataFrame()
    recs  = pd.DataFrame(recs)
    return preds, recs

preds_g, recs_g = train_eval_one(df, spl_g, "LOCO-GENE")
preds_s, recs_s = train_eval_one(df, spl_s, "LOCO-SITE")

# summarize per gene
def summarize(preds):
    rows=[]
    for g, sub in preds.groupby("gene"):
        if sub["label"].nunique()<2:
            continue
        y = sub["label"].values
        p = sub["y_pred_flanks"].values
        p_wt = sub["y_pred_wt_only"].values
        rows.append({
            "gene": g, "n": int(len(sub)),
            "auroc_flanks": round(roc_auc_score(y,p),4),
            "pr_auc_flanks": round(average_precision_score(y,p),4),
            "brier_flanks": round(brier_score_loss(y,p),4),
            "auroc_wt_only": round(roc_auc_score(y,p_wt),4),
            "pair_acc": round(float(sub["pair_ok"].mean()),4)
        })
    return pd.DataFrame(rows).sort_values("gene")

summ_g = summarize(preds_g).assign(tag="LOCO-GENE")
summ_s = summarize(preds_s).assign(tag="LOCO-SITE")
summary = pd.concat([summ_g, summ_s], ignore_index=True)[
    ["tag","gene","n","auroc_flanks","pr_auc_flanks","brier_flanks","auroc_wt_only","pair_acc"]
]

# receipts + save
preds_out = RUN_DIR/"preds_trimatched.csv"
summary_out = RUN_DIR/"metrics_trimatched.csv"
recs_out = RUN_DIR/"trimatched_receipts.csv"
pd.concat([preds_g.assign(tag="LOCO-GENE"), preds_s.assign(tag="LOCO-SITE")]).to_csv(preds_out, index=False)
summary.to_csv(summary_out, index=False)
pd.concat([recs_g, recs_s]).to_csv(recs_out, index=False)

print("Run dir:", RUN_DIR)
print("Saved:", summary_out.name, "and", preds_out.name)
print("\nMatch receipts per fold:")
display(pd.concat([recs_g, recs_s]).sort_values(["tag","fold"]).reset_index(drop=True))

print("\nTri-matched performance (per gene):")
display(summary.sort_values(["tag","gene"]))


Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607
Saved: metrics_trimatched.csv and preds_trimatched.csv

Match receipts per fold:


,pos_in,neg_in,matched_pairs,hits_exact_tri,hits_center_base,fold,tag
0,14,25,13,1,12,loco_gene:EGFR,LOCO-GENE
1,8,3,2,1,1,loco_gene:KRAS,LOCO-GENE
2,18,6,6,1,5,loco_gene:PIK3CA,LOCO-GENE
3,69,128,69,46,23,loco_gene:TP53,LOCO-GENE
4,21,34,16,4,12,loco_site:k1,LOCO-SITE
5,24,30,14,3,11,loco_site:k2,LOCO-SITE
6,21,33,20,7,13,loco_site:k3,LOCO-SITE
7,22,32,15,6,9,loco_site:k4,LOCO-SITE
8,21,33,12,5,7,loco_site:k5,LOCO-SITE



Tri-matched performance (per gene):


,tag,gene,n,auroc_flanks,pr_auc_flanks,brier_flanks,auroc_wt_only,pair_acc
0,LOCO-GENE,EGFR,26,0.7160,0.8056,0.2224,0.6982,0.5000
1,LOCO-GENE,KRAS,4,0.7500,0.8333,0.2436,0.7500,0.2500
2,LOCO-GENE,PIK3CA,12,0.8333,0.8552,0.2258,0.8333,0.5833
3,LOCO-GENE,TP53,138,0.6862,0.6779,0.2290,0.6858,0.4928
4,LOCO-SITE,EGFR,12,0.8611,0.9056,0.1738,0.8611,0.4167
5,LOCO-SITE,KRAS,2,1.0000,1.0000,0.0469,1.0000,1.0000
6,LOCO-SITE,PIK3CA,8,0.5000,0.6500,0.2659,0.5000,0.5000
7,LOCO-SITE,TP53,132,0.8340,0.8078,0.2153,0.8163,0.5833


**tri-matched receipts say it loud:** <u>the neighborhood is doing most of the work.</u> WT-only hangs right with the mutated flanks almost everywhere (e.g., TP53 LOCO-GENE 0.6862 vs 0.6858). So to really test “edit semantics,” let’s residualize out the static context and ask if the delta (`MUT`−`WT`) adds signal beyond the WT-only risk.

One clean block below does stacked residualization per LOCO fold:

* Stage A: train a **WT-only** model on the training fold (3-mers on window_wt, no tri/codon). Get:

    * **OOF** p_wt for training via inner 5-fold (no leakage),

    * **p_wt** for the test set from a model fit on full training.

* Stage B: train a $Δ$-model on anchored patch features (±10bp around the edited base, MUT−WT vector) plus p_wt as a covariate. Evaluate on test.

* Spits a tidy table: per gene, **AUC_WT, AUC_$Δ$+WT, AUC gain, PR-AUC, Brier, Pair-Acc ($Δ$)**.

In [24]:
# H1 (patched) — Residualized Δ-model: edit signal beyond WT-only context

import os, glob, time, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from scipy.sparse import hstack, csr_matrix

SEED = 1337
np.random.seed(SEED)

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
ARTI_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")

# latest variant split run dir
cand = sorted(glob.glob(str(ARTI_DIR / "sentiment_run_plus_variants_*")))
assert cand, "No sentiment_run_plus_variants_* dirs — run E2 first."
RUN_DIR = Path(cand[-1])

P_RES = DATA_DIR/"masterseq_model_ready_plus_resolved.parquet"
SPLITS_G = RUN_DIR/"splits_loco_gene.csv"
SPLITS_S = RUN_DIR/"splits_loco_site.csv"

df = pd.read_parquet(P_RES).dropna(subset=[
    "variant_id","gene","label","window_mut","window_wt","wt_codon","mut_codon","ref","alt","cds_pos"
]).copy()

# normalize
for c in ["variant_id","gene","ref","alt","wt_codon","mut_codon","window_mut","window_wt"]:
    df[c] = df[c].astype(str).str.upper()
df["label"] = df["label"].astype(int)
df["cds_pos"] = pd.to_numeric(df["cds_pos"], errors="coerce").astype("Int64")

# --- feature fns ------------------------------------------------------------
def kmers(seq: str, K=3):
    s = str(seq);
    return [f"k3:{s[i:i+K]}@{i}" for i in range(max(0, len(s)-K+1))]

def feats_wt_only(row, K=3):
    d={}
    for tok in kmers(row["window_wt"], K=K):
        d[tok] = d.get(tok,0)+1
    return d

def mut_index_in_window(row, flank=60):
    i0 = int(row["cds_pos"]) - 1
    cs = (i0 // 3) * 3
    center = cs + 1  # 2nd base
    return flank + (i0 - center)

def extract_patch(win: str, j: int, R: int = 10):
    s = str(win)
    if not s or j < 0 or j >= len(s): return ""
    a, b = max(0, j-R), min(len(s), j+R+1)
    left_pad  = "N" * (R - (j - a))
    right_pad = "N" * (R - (b - j - 1))
    return left_pad + s[a:b] + right_pad

def kmers_rel(seq: str, K=3):
    s = str(seq); toks=[]
    mid = (len(s)-1)//2
    for i in range(max(0,len(s)-K+1)):
        toks.append(f"k3:{s[i:i+K]}@rel{ i - mid }")
    return toks

def feats_patch(row, is_mut: bool, R=10, K=3):
    j = mut_index_in_window(row, flank=60)
    seq = row["window_mut"] if is_mut else row["window_wt"]
    patch = extract_patch(seq, j, R=R)
    d={}
    for tok in kmers_rel(patch, K=K):
        d[tok] = d.get(tok,0)+1
    # local tri token
    mid = R
    tri = patch[mid-1:mid+2] if len(patch) >= 3 and 1 <= mid <= len(patch)-2 else ""
    if tri:
        d[f"tri_local:{tri}"] = 1
    # AA change token only on MUT so it survives subtraction
    if is_mut and isinstance(row["wt_codon"],str) and isinstance(row["mut_codon"],str) and len(row["wt_codon"])==3 and len(row["mut_codon"])==3:
        d[f"codon:{row['wt_codon']}->{row['mut_codon']}"] = 1
    return d

def delta_matrix(vec: DictVectorizer, rows_mut, rows_wt):
    Xm = vec.transform(rows_mut); Xw = vec.transform(rows_wt)
    return Xm - Xw  # sparse CSR

# --- inner OOF helper for WT-only ------------------------------------------
def oof_wt_predictions(tr_df, n_splits=5, K=3):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    y = tr_df["label"].values
    oof = np.zeros(len(tr_df), dtype=float)

    for tr_idx, val_idx in skf.split(np.zeros_like(y), y):
        tr_rows = tr_df.iloc[tr_idx]; va_rows = tr_df.iloc[val_idx]
        vec = DictVectorizer(sparse=True)
        Xtr = vec.fit_transform(feats_wt_only(r, K=K) for _, r in tr_rows.iterrows())
        ytr = tr_rows["label"].values
        Xva = vec.transform(feats_wt_only(r, K=K) for _, r in va_rows.iterrows())
        clf = LogisticRegression(penalty="l2", solver="lbfgs", C=0.25, max_iter=5000, random_state=SEED)
        clf.fit(Xtr, ytr)
        oof[val_idx] = clf.predict_proba(Xva)[:,1]

    # full-train model for test scoring
    vec_full = DictVectorizer(sparse=True)
    Xfull = vec_full.fit_transform(feats_wt_only(r, K=K) for _, r in tr_df.iterrows())
    clf_full = LogisticRegression(penalty="l2", solver="lbfgs", C=0.25, max_iter=5000, random_state=SEED)
    clf_full.fit(Xfull, y)
    return oof, vec_full, clf_full

# --- outer evaluation -------------------------------------------------------
def run_outer(df, splits_path, tag, R=10, K=3):
    splits = pd.read_csv(splits_path)
    recs=[]
    for fold, sub in splits.groupby("fold"):
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr = dd[dd["split"]=="train"].copy()
        te = dd[dd["split"]=="test"].copy()
        if tr.empty or te.empty or tr["label"].nunique()<2 or te["label"].nunique()<2:
            continue

        # Stage A: WT-only with inner OOF
        oof_pw, vec_wt, clf_wt = oof_wt_predictions(tr, n_splits=5, K=K)
        Xte_wt = vec_wt.transform(feats_wt_only(r, K=K) for _, r in te.iterrows())
        p_wt_te = clf_wt.predict_proba(Xte_wt)[:,1]

        # Stage B: Δ-features + p_wt covariate (FIT vec_delta FIRST)
        train_mut = [feats_patch(r, True,  R=R, K=K) for _, r in tr.iterrows()]
        train_wt  = [feats_patch(r, False, R=R, K=K) for _, r in tr.iterrows()]
        vec_delta = DictVectorizer(sparse=True)
        vec_delta.fit(train_mut + train_wt)

        Xtr_delta = delta_matrix(vec_delta, train_mut, train_wt)
        Xte_delta = delta_matrix(
            vec_delta,
            [feats_patch(r, True,  R=R, K=K) for _, r in te.iterrows()],
            [feats_patch(r, False, R=R, K=K) for _, r in te.iterrows()],
        )

        # add p_wt as sparse column
        Xtr_stack = hstack([Xtr_delta, csr_matrix(oof_pw.reshape(-1,1))])
        Xte_stack = hstack([Xte_delta, csr_matrix(p_wt_te.reshape(-1,1))])

        ytr = tr["label"].values
        yte = te["label"].values

        clf_delta = LogisticRegression(
            penalty="l2", solver="lbfgs", C=0.25, max_iter=5000,
            class_weight="balanced", random_state=SEED
        )
        clf_delta.fit(Xtr_stack, ytr)
        p_delta = clf_delta.predict_proba(Xte_stack)[:,1]

        part = te.loc[:, ["variant_id","gene","label"]].copy()
        part["fold"] = fold; part["tag"] = tag
        part["p_wt"] = p_wt_te
        part["p_delta_resid"] = p_delta
        part["pair_ok"] = (((p_delta>0.5) & (part["label"]==1)) | ((p_delta<=0.5) & (part["label"]==0))).astype(int)
        recs.append(part)

    if not recs:
        return pd.DataFrame()

    preds = pd.concat(recs, ignore_index=True)

    rows=[]
    for g, sub in preds.groupby("gene"):
        if sub["label"].nunique()<2:
            continue
        y = sub["label"].values
        p_wt = sub["p_wt"].values
        p_d  = sub["p_delta_resid"].values
        rows.append({
            "tag": tag, "gene": g, "n": int(len(sub)),
            "AUC_WT": round(roc_auc_score(y, p_wt),4),
            "AUC_DeltaPlusWT": round(roc_auc_score(y, p_d),4),
            "AUC_gain": round(roc_auc_score(y, p_d) - roc_auc_score(y, p_wt),4),
            "PR_AUC_DeltaPlusWT": round(average_precision_score(y, p_d),4),
            "Brier_DeltaPlusWT": round(brier_score_loss(y, p_d),4),
            "PairAcc_Delta": round(float(sub["pair_ok"].mean()),4),
        })
    return pd.DataFrame(rows).sort_values(["tag","gene"])

summ_g = run_outer(df, SPLITS_G, "LOCO-GENE", R=10, K=3)
summ_s = run_outer(df, SPLITS_S, "LOCO-SITE", R=10, K=3)
summary = pd.concat([summ_g, summ_s], ignore_index=True)

outp = RUN_DIR/"metrics_residualized_delta.csv"
summary.to_csv(outp, index=False)
print("Run dir:", RUN_DIR)
print("Saved:", outp.name)
display(summary)


Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607
Saved: metrics_residualized_delta.csv


,tag,gene,n,AUC_WT,AUC_DeltaPlusWT,AUC_gain,PR_AUC_DeltaPlusWT,Brier_DeltaPlusWT,PairAcc_Delta
0,LOCO-GENE,EGFR,39,0.6800,0.6171,-0.0629,0.5194,0.2352,0.6410
1,LOCO-GENE,KRAS,11,0.5833,0.7500,0.1667,0.8755,0.2173,0.5455
2,LOCO-GENE,PIK3CA,24,0.8056,0.5185,-0.2870,0.8353,0.2825,0.5000
3,LOCO-GENE,TP53,197,0.6850,0.5265,-0.1585,0.4057,0.2598,0.4619
4,LOCO-SITE,EGFR,39,0.7571,0.6314,-0.1257,0.5714,0.2348,0.5641
5,LOCO-SITE,KRAS,11,0.9167,0.6667,-0.2500,0.8228,0.1895,0.8182
6,LOCO-SITE,PIK3CA,24,0.6944,0.6204,-0.0741,0.8745,0.2656,0.5833
7,LOCO-SITE,TP53,197,0.8486,0.6762,-0.1724,0.5440,0.2158,0.6396


* WT-only still claps (TP53 LOCO-GENE 0.685, LOCO-SITE 0.849).

* Residualized $Δ$ model mostly loses vs WT-only (negative AUC_gain everywhere except toy-n KRAS).

> Translation: our classifier’s juice is static neighborhood (regional/context/mutability), not the edit.


Next surgical move (no GPU yet): tri-matched training so the model can’t lean on easy context. We’ll match the training fold only within each gene by exact tri_center (fallback to center base), balance pos/neg per tri, train the usual flanks_no_codon model, and then evaluate on the original LOCO test (unmatched) plus report a tri-matched test as a secondary check. If performance holds after this, we’ve got signal beyond simple tri-context; if it craters, we pivot to a Siamese CNN.

In [25]:
# H2 — Tri-matched TRAINING (per fold, per gene, per tri); evaluate flanks_no_codon vs WT-only
import glob, json, math, time
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

SEED = 1337
rng = np.random.default_rng(SEED)

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
ARTI_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
P_RES   = DATA_DIR/"masterseq_model_ready_plus_resolved.parquet"

# latest variant split dir
cand = sorted(glob.glob(str(ARTI_DIR / "sentiment_run_plus_variants_*")))
assert cand, "No sentiment_run_plus_variants_* dir (run E2 first)."
RUN_DIR = Path(cand[-1])
SPLITS_G = RUN_DIR/"splits_loco_gene.csv"
SPLITS_S = RUN_DIR/"splits_loco_site.csv"

df = pd.read_parquet(P_RES).dropna(subset=["variant_id","gene","label","tri_center","window_mut","window_wt"]).copy()
for c in ["variant_id","gene","tri_center"]:
    df[c] = df[c].astype(str).str.upper()
df["label"] = df["label"].astype(int)

spl_g = pd.read_csv(SPLITS_G)
spl_s = pd.read_csv(SPLITS_S)

def feats_flanks_no_codon(r, K=3, include_tri=True):
    d={}
    tri = str(r.get("tri_center","")).upper()
    if include_tri and len(tri)==3:
        d[f"tri:{tri}"]=1
        d["is_cpg"]=int(tri.startswith("CG"))
    s = str(r.get("window_mut",""))
    for i in range(max(0, len(s)-K+1)):
        d[f"s_{i}:{s[i:i+K]}"] = d.get(f"s_{i}:{s[i:i+K]}",0)+1
    return d

def feats_wt_only(r, K=3):
    d={}
    s = str(r.get("window_wt",""))
    for i in range(max(0, len(s)-K+1)):
        d[f"s_{i}:{s[i:i+K]}"] = d.get(f"s_{i}:{s[i:i+K]}",0)+1
    return d

# tri-matched sampler for TRAIN folds (within-gene)
def trimatch_train(df_train: pd.DataFrame, random_state=SEED):
    rs = np.random.RandomState(random_state)
    rows = []
    for g, sub in df_train.groupby("gene"):
        sub = sub.copy()
        sub["center_base"] = sub["tri_center"].str[1]
        pos = sub[sub["label"]==1]
        neg = sub[sub["label"]==0]
        used = set()
        keep_idx = []

        # exact tri matching buckets
        for tri, grp in pos.groupby("tri_center"):
            pool = neg[(neg["tri_center"]==tri) & (~neg.index.isin(used))]
            if pool.empty:
                continue
            k = min(len(grp), len(pool))
            keep_idx += rs.choice(grp.index, size=k, replace=False).tolist()
            sel = rs.choice(pool.index, size=k, replace=False).tolist()
            keep_idx += sel
            used.update(sel)

        # fallback by center base for leftover positives
        left_pos = pos.loc[~pos.index.isin(keep_idx)]
        for base, grp in left_pos.groupby("center_base"):
            pool = neg[(neg["tri_center"].str[1]==base) & (~neg.index.isin(used))]
            if pool.empty:
                continue
            k = min(len(grp), len(pool))
            keep_idx += rs.choice(grp.index, size=k, replace=False).tolist()
            sel = rs.choice(pool.index, size=k, replace=False).tolist()
            keep_idx += sel
            used.update(sel)

        rows.append(sub.loc[sorted(set(keep_idx))])
    matched = pd.concat(rows, ignore_index=False) if rows else df_train.iloc[[]]
    matched = matched.sample(frac=1.0, random_state=random_state)  # shuffle
    return matched

def run_cv(df, splits, tag):
    out_rows=[]
    recs=[]
    for fold, sub in splits.groupby("fold"):
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr0 = dd[dd["split"]=="train"].copy()
        te  = dd[dd["split"]=="test"].copy()
        if tr0.empty or te.empty:
            continue

        tr = trimatch_train(tr0, random_state=SEED+hash(fold)%10000)
        recs.append({
            "tag": tag, "fold": fold,
            "train_total": int(len(tr0)), "train_matched": int(len(tr)),
            "train_pos_in": int((tr0["label"]==1).sum()), "train_neg_in": int((tr0["label"]==0).sum()),
            "train_pos_matched": int((tr["label"]==1).sum()), "train_neg_matched": int((tr["label"]==0).sum()),
        })
        if tr["label"].nunique()<2 or te["label"].nunique()<2:
            continue

        # --- flanks_no_codon model (with tri features) ---
        v_fl = DictVectorizer(sparse=True)
        Xtr_fl = v_fl.fit_transform(feats_flanks_no_codon(r) for _, r in tr.iterrows())
        ytr = tr["label"].values
        Xte_fl = v_fl.transform(feats_flanks_no_codon(r) for _, r in te.iterrows())

        clf_fl = LogisticRegression(penalty="l2", solver="lbfgs", C=0.25, max_iter=5000, random_state=SEED)
        clf_fl.fit(Xtr_fl, ytr)
        pm = clf_fl.predict_proba(Xte_fl)[:,1]

        # --- WT-only baseline on same test set ---
        v_wt = DictVectorizer(sparse=True)
        Xtr_wt = v_wt.fit_transform(feats_wt_only(r) for _, r in tr.iterrows())
        clf_wt = LogisticRegression(penalty="l2", solver="lbfgs", C=0.25, max_iter=5000, random_state=SEED)
        clf_wt.fit(Xtr_wt, ytr)
        pw = clf_wt.predict_proba(v_wt.transform(feats_wt_only(r) for _, r in te.iterrows()))[:,1]

        part = te.loc[:, ["variant_id","gene","label"]].copy()
        part["fold"]=fold; part["tag"]=tag
        part["p_flanks"] = pm
        part["p_wt"] = pw
        out_rows.append(part)

    preds = pd.concat(out_rows, ignore_index=True) if out_rows else pd.DataFrame()
    recdf = pd.DataFrame(recs)
    return preds, recdf

# Run both LOCO schemes
preds_g, recs_g = run_cv(df, spl_g, "LOCO-GENE")
preds_s, recs_s = run_cv(df, spl_s, "LOCO-SITE")

# Summaries per gene
def summarize(preds):
    rows=[]
    for g, sub in preds.groupby("gene"):
        if sub["label"].nunique()<2:
            continue
        y = sub["label"].values
        pf, pw = sub["p_flanks"].values, sub["p_wt"].values
        rows.append({
            "gene": g, "n": int(len(sub)),
            "auroc_flanks_matchedTrain": round(roc_auc_score(y,pf),4),
            "pr_auc_flanks": round(average_precision_score(y,pf),4),
            "brier_flanks": round(brier_score_loss(y,pf),4),
            "auroc_wt_only": round(roc_auc_score(y,pw),4),
        })
    return pd.DataFrame(rows).sort_values("gene")

summ_g = summarize(preds_g).assign(tag="LOCO-GENE")
summ_s = summarize(preds_s).assign(tag="LOCO-SITE")

summary = pd.concat([summ_g, summ_s], ignore_index=True)[
    ["tag","gene","n","auroc_flanks_matchedTrain","pr_auc_flanks","brier_flanks","auroc_wt_only"]
]

# Save
out_dir = RUN_DIR
(preds_g.assign(tag="LOCO-GENE").append(preds_s.assign(tag="LOCO-SITE"))
 ).to_csv(out_dir/"preds_trimatched_training.csv", index=False)
summary.to_csv(out_dir/"metrics_trimatched_training.csv", index=False)
pd.concat([recs_g, recs_s]).to_csv(out_dir/"trimatched_training_receipts.csv", index=False)

print("Run dir:", out_dir)
print("Saved: metrics_trimatched_training.csv and preds_trimatched_training.csv")
print("\nTraining match receipts (by fold):")
display(pd.concat([recs_g, recs_s]).sort_values(["tag","fold"]).reset_index(drop=True))
print("\nTest performance with tri-matched TRAINING (per gene):")
display(summary.sort_values(["tag","gene"]))


AttributeError: 'DataFrame' object has no attribute 'append'

**patch cell**

In [26]:
# H2.fix — swap .append -> pd.concat, save + re-display
import glob, pandas as pd
from pathlib import Path

ARTI_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
RUN_DIR = Path(sorted(glob.glob(str(ARTI_DIR / "sentiment_run_plus_variants_*")))[-1])

combined_preds = pd.concat(
    [preds_g.assign(tag="LOCO-GENE"),
     preds_s.assign(tag="LOCO-SITE")],
    ignore_index=True
)
combined_receipts = pd.concat([recs_g, recs_s], ignore_index=True)

combined_preds.to_csv(RUN_DIR / "preds_trimatched_training.csv", index=False)
summary.to_csv(RUN_DIR / "metrics_trimatched_training.csv", index=False)
combined_receipts.to_csv(RUN_DIR / "trimatched_training_receipts.csv", index=False)

print("Saved to:", RUN_DIR)
print("Files:", "preds_trimatched_training.csv", "metrics_trimatched_training.csv", "trimatched_training_receipts.csv")

print("\nTraining match receipts (by fold):")
display(combined_receipts.sort_values(["tag","fold"]).reset_index(drop=True))

print("\nTest performance with tri-matched TRAINING (per gene):")
display(summary.sort_values(["tag","gene"]))


Saved to: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607
Files: preds_trimatched_training.csv metrics_trimatched_training.csv trimatched_training_receipts.csv

Training match receipts (by fold):


,tag,fold,train_total,train_matched,train_pos_in,train_neg_in,train_pos_matched,train_neg_matched
0,LOCO-GENE,loco_gene:EGFR,232,154,95,137,77,77
1,LOCO-GENE,loco_gene:KRAS,260,176,101,159,88,88
2,LOCO-GENE,loco_gene:PIK3CA,247,168,91,156,84,84
3,LOCO-GENE,loco_gene:TP53,74,42,40,34,21,21
4,LOCO-SITE,loco_site:k1,216,144,88,128,72,72
5,LOCO-SITE,loco_site:k2,217,140,85,132,70,70
6,LOCO-SITE,loco_site:k3,217,132,88,129,66,66
7,LOCO-SITE,loco_site:k4,217,150,87,130,75,75
8,LOCO-SITE,loco_site:k5,217,146,88,129,73,73



Test performance with tri-matched TRAINING (per gene):


,tag,gene,n,auroc_flanks_matchedTrain,pr_auc_flanks,brier_flanks,auroc_wt_only
0,LOCO-GENE,EGFR,39,0.6086,0.5634,0.2394,0.6114
1,LOCO-GENE,KRAS,11,0.7500,0.8755,0.2103,0.6250
2,LOCO-GENE,PIK3CA,24,0.8148,0.9357,0.1878,0.7778
3,LOCO-GENE,TP53,197,0.6479,0.4882,0.2301,0.6475
4,LOCO-SITE,EGFR,39,0.7800,0.7265,0.2088,0.7571
5,LOCO-SITE,KRAS,11,1.0000,1.0000,0.1424,0.8750
6,LOCO-SITE,PIK3CA,24,0.8241,0.9485,0.1960,0.7870
7,LOCO-SITE,TP53,197,0.7689,0.5837,0.1952,0.7647


those receipts say it straight: **context runs the show**. time to give the edit a real shot with a tiny Siamese $Δ$CNN on GPU. one block, drop-in. it centers on the mutated base, encodes WT & MUT patches, runs a shared Conv1D encoder, then learns on (MUT−WT). we eval on your latest LOCO-GENE splits.

In [27]:
# I1 — Siamese Δ-CNN (GPU): edit-centered, LOCO-GENE eval
# - pulls latest variant-based splits
# - extracts ±30bp patches around the mutated base from WT/MUT
# - trains a tiny shared Conv1D encoder; head sees (emb_mut - emb_wt)
# - metrics: AUROC, PR-AUC, Brier, Pair-Acc (Δ > 0)

import os, glob, time, json, math
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 1337
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
ARTI_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
P_RES   = DATA_DIR/"masterseq_model_ready_plus_resolved.parquet"

# latest variant split dir (from E2)
cand = sorted(glob.glob(str(ARTI_DIR / "sentiment_run_plus_variants_*")))
assert cand, "No variant split dir — run E2 earlier."
RUN_DIR = Path(cand[-1])
SPLITS_G = RUN_DIR/"splits_loco_gene.csv"

# ========= data load & helpers =========
df = pd.read_parquet(P_RES).dropna(subset=[
    "variant_id","gene","label","window_mut","window_wt",
    "wt_codon","mut_codon","ref","alt","cds_pos"
]).copy()
for c in ["variant_id","gene","ref","alt","wt_codon","mut_codon","window_mut","window_wt"]:
    df[c] = df[c].astype(str).str.upper()
df["label"] = df["label"].astype(int)
df["cds_pos"] = pd.to_numeric(df["cds_pos"], errors="coerce").astype("Int64")

spl_g = pd.read_csv(SPLITS_G)

# mutated base index inside provided windows (windows are centered on codon base 2)
def mut_index_in_window(row, flank=60):
    i0 = int(row["cds_pos"]) - 1
    cs = (i0 // 3) * 3
    center = cs + 1  # 2nd base of codon (0-based)
    return flank + (i0 - center)  # -1,0,+1 around center

def extract_patch(win: str, j: int, R: int = 30):
    s = str(win)
    if not s or j < 0 or j >= len(s): return ""
    a, b = max(0, j-R), min(len(s), j+R+1)
    left_pad  = "N" * (R - (j - a))
    right_pad = "N" * (R - (b - j - 1))
    return left_pad + s[a:b] + right_pad  # always length 2R+1

# 1-hot for A,C,G,T,N → 5 channels
VOCAB = {"A":0,"C":1,"G":2,"T":3,"N":4}
def onehot(seq: str):
    arr = np.zeros((5, len(seq)), dtype=np.float32)
    for i,ch in enumerate(seq):
        k = VOCAB.get(ch, 4)
        arr[k, i] = 1.0
    return arr

class PairDataset(Dataset):
    def __init__(self, frame, R=30):
        self.R = R
        self.rows = frame.reset_index(drop=True)

        # precompute patches
        self.mut_patches = []
        self.wt_patches  = []
        for _, r in self.rows.iterrows():
            j = mut_index_in_window(r, flank=60)
            self.mut_patches.append(extract_patch(r["window_mut"], j, R=self.R))
            self.wt_patches.append(extract_patch(r["window_wt"],  j, R=self.R))

    def __len__(self): return len(self.rows)

    def __getitem__(self, idx):
        y = np.float32(self.rows.iloc[idx]["label"])
        x_mut = onehot(self.mut_patches[idx])  # (5, L)
        x_wt  = onehot(self.wt_patches[idx])
        return torch.from_numpy(x_mut), torch.from_numpy(x_wt), torch.tensor(y)

class SiameseDeltaCNN(nn.Module):
    def __init__(self, in_ch=5, hid=64):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv1d(in_ch, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1)
        )
        self.head = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1)
        )

    def encode(self, x):          # x: (B,5,L)
        h = self.enc(x).squeeze(-1)  # (B,64)
        return h

    def forward(self, xm, xw):    # siamese: same encoder for both streams
        hm = self.encode(xm)
        hw = self.encode(xw)
        d  = hm - hw               # Δ embedding
        logit = self.head(d).squeeze(-1)
        return logit

def metrics_from(y_true, y_logit):
    y_true = y_true.astype(np.int32)
    p = 1/(1+np.exp(-y_logit))
    auc = roc_auc_score(y_true, p) if len(np.unique(y_true))>1 else np.nan
    pr  = average_precision_score(y_true, p) if len(np.unique(y_true))>1 else np.nan
    br  = brier_score_loss(y_true, p)
    pair = float(((y_logit>0) & (y_true==1) | (y_logit<=0) & (y_true==0)).mean())
    return dict(auroc=round(auc,4) if not math.isnan(auc) else np.nan,
                pr_auc=round(pr,4) if not math.isnan(pr) else np.nan,
                brier=round(br,4), pair_acc=round(pair,4))

def run_loco_gene(df, splits, epochs=10, batch=128, lr=1e-3, R=30, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    all_rows=[]
    for fold, sub in splits.groupby("fold"):
        # join & split
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr = dd[dd["split"]=="train"].copy()
        te = dd[dd["split"]=="test"].copy()
        if tr.empty or te.empty:
            continue

        # datasets
        dtr = PairDataset(tr, R=R)
        dte = PairDataset(te, R=R)
        L = 2*R+1

        # class balance for pos_weight
        pos = (tr["label"]==1).sum()
        neg = (tr["label"]==0).sum()
        pos_weight = torch.tensor([max(1.0, neg / max(1, pos))], dtype=torch.float32, device=device)

        # model
        model = SiameseDeltaCNN(in_ch=5, hid=64).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        train_loader = DataLoader(dtr, batch_size=batch, shuffle=True, num_workers=2, pin_memory=True)
        test_loader  = DataLoader(dte, batch_size=batch, shuffle=False, num_workers=2, pin_memory=True)

        # train
        model.train()
        for ep in range(1, epochs+1):
            loss_sum = 0.0
            for xm, xw, y in train_loader:
                xm = xm.to(device); xw = xw.to(device); y = y.to(device)
                opt.zero_grad()
                logit = model(xm, xw)
                loss = crit(logit, y)
                loss.backward()
                opt.step()
                loss_sum += loss.item() * y.size(0)

        # eval
        model.eval()
        with torch.no_grad():
            logits = []
            ys = []
            vids = []
            genes = []
            for xm, xw, y in test_loader:
                xm = xm.to(device); xw = xw.to(device)
                logit = model(xm, xw)
                logits.append(logit.cpu().numpy())
                ys.append(y.numpy())
            y_log = np.concatenate(logits)
            y_true = np.concatenate(ys)
        # summarize (one gene per LOCO-GENE fold)
        gname = te["gene"].unique().tolist()[0] if not te.empty else "NA"
        m = metrics_from(y_true, y_log)
        all_rows.append({"fold": fold, "gene": gname, **m, "n": int(len(y_true))})

    return pd.DataFrame(all_rows).sort_values("gene")

# ======== run =========
SIAM_DIR = ARTI_DIR / f"siamese_run_{time.strftime('%Y%m%d_%H%M%S')}"
SIAM_DIR.mkdir(parents=True, exist_ok=True)

summary = run_loco_gene(df, spl_g, epochs=10, batch=128, lr=1e-3, R=30)
summary.to_csv(SIAM_DIR/"metrics_siamese_loco_gene.csv", index=False)

print("Run dir:", SIAM_DIR)
display(summary)


Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/siamese_run_20250816_211819


,fold,gene,auroc,pr_auc,brier,pair_acc,n
0,loco_gene:EGFR,EGFR,0.5943,0.5842,0.2535,0.3590,39
1,loco_gene:KRAS,KRAS,0.4167,0.6941,0.2350,0.7273,11
2,loco_gene:PIK3CA,PIK3CA,0.3981,0.7368,0.2524,0.2500,24
3,loco_gene:TP53,TP53,0.5187,0.3581,0.2534,0.3503,197


respect — the $Δ$-CNN on plain per-variant BCE isn’t moving the needle. let’s turn the screws:

next block = tri-matched training + pairwise margin loss (Siamese $Δ$-CNN)
we train on pos↔neg pairs matched by gene & tri-context (fallback to center base) so the net can’t coast on static context. the head learns to make the mutated edit score higher for drivers than for benigns within the same context. eval is still LOCO-GENE (held-out gene), plus we also report test tri-matched pair accuracy.

In [28]:
# I2 — Siamese Δ-CNN with tri-matched TRAINING + pairwise margin loss (LOCO-GENE)
# - Pairs positives vs negatives matched by (gene, tri_center) or (gene, center base)
# - Shared Conv1D encodes WT/MUT; head sees Δ embedding; loss = hinge on (pos - neg)
# - Evaluates per-variant metrics and tri-matched pair accuracy on the LOCO-GENE test.

import os, glob, time, math, random
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ---- setup
SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
ARTI_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
P_RES   = DATA_DIR/"masterseq_model_ready_plus_resolved.parquet"
cand = sorted(glob.glob(str(ARTI_DIR / "sentiment_run_plus_variants_*")))
assert cand, "No variant split dir — run E2 first."
RUN_DIR = Path(cand[-1])
SPLITS_G = RUN_DIR/"splits_loco_gene.csv"

# ---- load
df = pd.read_parquet(P_RES).dropna(subset=[
    "variant_id","gene","label","window_mut","window_wt",
    "wt_codon","mut_codon","ref","alt","cds_pos","tri_center"
]).copy()
for c in ["variant_id","gene","ref","alt","wt_codon","mut_codon","window_mut","window_wt","tri_center"]:
    df[c] = df[c].astype(str).str.upper()
df["label"] = df["label"].astype(int)
df["cds_pos"] = pd.to_numeric(df["cds_pos"], errors="coerce").astype("Int64")
spl_g = pd.read_csv(SPLITS_G)

# ---- windows → edit index helpers
def mut_index_in_window(row, flank=60):
    i0 = int(row["cds_pos"]) - 1
    cs = (i0 // 3) * 3
    center = cs + 1
    return flank + (i0 - center)  # -1,0,+1

def extract_patch(win: str, j: int, R: int = 30):
    s = str(win)
    if not s or j < 0 or j >= len(s): return ""
    a, b = max(0, j-R), min(len(s), j+R+1)
    left_pad  = "N" * (R - (j - a))
    right_pad = "N" * (R - (b - j - 1))
    return left_pad + s[a:b] + right_pad  # fixed length 2R+1

VOCAB = {"A":0,"C":1,"G":2,"T":3,"N":4}
def onehot(seq: str):
    arr = np.zeros((5, len(seq)), dtype=np.float32)
    for i,ch in enumerate(seq):
        arr[VOCAB.get(ch,4), i] = 1.0
    return arr

# ---- tri-matched pairing for TRAIN
def make_pairs_trimatched(train_df: pd.DataFrame, seed=SEED):
    rs = np.random.RandomState(seed)
    pairs = []  # list of (pos_idx, neg_idx)
    train_df = train_df.copy()
    train_df["center_base"] = train_df["tri_center"].str[1]

    for g, sub in train_df.groupby("gene"):
        pos = sub[sub["label"]==1]
        neg = sub[sub["label"]==0].copy()
        used = set()

        # exact tri match
        for tri, grp in pos.groupby("tri_center"):
            pool = neg[(neg["tri_center"]==tri) & (~neg.index.isin(used))]
            if pool.empty:
                continue
            k = min(len(grp), len(pool))
            pidx = rs.choice(grp.index, size=k, replace=False)
            nidx = rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx, nidx))
            used.update(nidx)

        # fallback center base
        leftover = pos.loc[~pos.index.isin([p for p,_ in pairs if p in pos.index])]
        for base, grp in leftover.groupby("center_base"):
            pool = neg[(neg["tri_center"].str[1]==base) & (~neg.index.isin(used))]
            if pool.empty:
                continue
            k = min(len(grp), len(pool))
            pidx = rs.choice(grp.index, size=k, replace=False)
            nidx = rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx, nidx))
            used.update(nidx)
    return pairs  # could be empty on tiny folds

# ---- tri-matched pairs on TEST (for pair accuracy reporting)
def make_pairs_for_test(test_df: pd.DataFrame, seed=SEED):
    test_df = test_df.copy()
    test_df["center_base"] = test_df["tri_center"].str[1]
    pos = test_df[test_df["label"]==1]
    neg = test_df[test_df["label"]==0].copy()
    rs = np.random.RandomState(seed)
    pairs=[]
    used=set()
    for tri, grp in pos.groupby("tri_center"):
        pool = neg[(neg["tri_center"]==tri) & (~neg.index.isin(used))]
        if pool.empty: continue
        k = min(len(grp), len(pool))
        pidx = rs.choice(grp.index, size=k, replace=False)
        nidx = rs.choice(pool.index, size=k, replace=False)
        pairs += list(zip(pidx, nidx)); used.update(nidx)
    leftover = pos.loc[~pos.index.isin([p for p,_ in pairs])]
    for base, grp in leftover.groupby("center_base"):
        pool = neg[(neg["tri_center"].str[1]==base) & (~neg.index.isin(used))]
        if pool.empty: continue
        k = min(len(grp), len(pool))
        pidx = rs.choice(grp.index, size=k, replace=False)
        nidx = rs.choice(pool.index, size=k, replace=False)
        pairs += list(zip(pidx, nidx)); used.update(nidx)
    return pairs

# ---- datasets
class PairTrainDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, pairs, R=30):
        self.df = frame
        self.pairs = pairs
        self.R = R
        self.rows = frame  # for convenience

    def __len__(self): return len(self.pairs)

    def _patches(self, idx):
        r = self.rows.loc[idx]
        j = mut_index_in_window(r, flank=60)
        return (extract_patch(r["window_mut"], j, self.R),
                extract_patch(r["window_wt"],  j, self.R))

    def __getitem__(self, k):
        ip, ineg = self.pairs[k]
        mp, wp = self._patches(ip)
        mn, wn = self._patches(ineg)
        # to tensors
        xm_p, xw_p = torch.from_numpy(onehot(mp)), torch.from_numpy(onehot(wp))
        xm_n, xw_n = torch.from_numpy(onehot(mn)), torch.from_numpy(onehot(wn))
        return xm_p, xw_p, xm_n, xw_n

class VariantEvalDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, R=30):
        self.df = frame.reset_index(drop=True)
        self.R = R
        self.patches = []
        for _, r in self.df.iterrows():
            j = mut_index_in_window(r, flank=60)
            self.patches.append((
                extract_patch(r["window_mut"], j, self.R),
                extract_patch(r["window_wt"],  j, self.R)
            ))
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        mut, wt = self.patches[i]
        return torch.from_numpy(onehot(mut)), torch.from_numpy(onehot(wt)), torch.tensor(self.df.loc[i, "label"], dtype=torch.float32)

# ---- model
class SiameseDeltaCNN(nn.Module):
    def __init__(self, in_ch=5, hid=96):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv1d(in_ch, 64,  kernel_size=11, padding=5),
            nn.ReLU(),
            nn.Conv1d(64, 96, kernel_size=7,  padding=3),
            nn.ReLU(),
            nn.Conv1d(96, 96, kernel_size=3,  padding=1),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1)
        )
        self.norm = nn.LayerNorm(96)
        self.head = nn.Sequential(
            nn.Linear(96, 96),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(96, 1)
        )
    def encode(self, x):  # x: (B,5,L)
        h = self.enc(x).squeeze(-1)          # (B,96)
        return self.norm(h)
    def forward(self, xm, xw):
        hm = self.encode(xm); hw = self.encode(xw)
        d  = hm - hw
        return self.head(d).squeeze(-1)

# ---- train eval loop (LOCO-GENE)
def run_loco_pairwise(df, splits, epochs=12, batch=96, lr=2e-3, margin=0.2, R=30, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    out_rows=[]
    for fold, sub in splits.groupby("fold"):
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr = dd[dd["split"]=="train"].copy()
        te = dd[dd["split"]=="test"].copy()
        if tr.empty or te.empty:
            continue

        # tri-matched pairs for TRAIN
        pairs = make_pairs_trimatched(tr, seed=SEED)
        if len(pairs) == 0:
            # fall back: skip fold if no pairs
            continue

        dtrain = PairTrainDataset(tr, pairs, R=R)
        dtest  = VariantEvalDataset(te, R=R)

        model = SiameseDeltaCNN().to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

        # Margin ranking loss on (pos - neg): max(0, margin - (s_pos - s_neg))
        def loss_pair(xm_p, xw_p, xm_n, xw_n):
            sp = model(xm_p, xw_p)
            sn = model(xm_n, xw_n)
            diff = sp - sn
            return torch.relu(margin - diff).mean()

        # train
        model.train()
        loader = DataLoader(dtrain, batch_size=batch, shuffle=True, num_workers=2, pin_memory=True, drop_last=False)
        for ep in range(1, epochs+1):
            acc_loss = 0.0
            for xm_p, xw_p, xm_n, xw_n in loader:
                xm_p, xw_p = xm_p.to(device), xw_p.to(device)
                xm_n, xw_n = xm_n.to(device), xw_n.to(device)
                opt.zero_grad()
                l = loss_pair(xm_p, xw_p, xm_n, xw_n)
                l.backward()
                opt.step()
                acc_loss += float(l.item()) * xm_p.size(0)

        # eval per-variant
        model.eval()
        test_loader = DataLoader(dtest, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
        logits=[]; ys=[]
        with torch.no_grad():
            for xm, xw, y in test_loader:
                xm, xw = xm.to(device), xw.to(device)
                logit = model(xm, xw)
                logits.append(logit.cpu().numpy()); ys.append(y.numpy())
        y_log = np.concatenate(logits); y_true = np.concatenate(ys).astype(int)

        # variant metrics
        p = 1/(1+np.exp(-y_log))
        auroc = roc_auc_score(y_true, p) if len(np.unique(y_true))>1 else np.nan
        pr    = average_precision_score(y_true, p) if len(np.unique(y_true))>1 else np.nan
        brier = brier_score_loss(y_true, p)
        pair_acc_variant = float(((y_log>0)&(y_true==1) | (y_log<=0)&(y_true==0)).mean())

        # tri-matched pair accuracy on TEST (pos vs neg within same context)
        test_pairs = make_pairs_for_test(te, seed=SEED)
        pair_acc_ctx = np.nan
        if test_pairs:
            with torch.no_grad():
                # score all test rows once
                # map from original index to logit
                idx_to_logit = {}
                krows = te.reset_index()
                dmap = {}
                # build a quick dataset to score in bulk
                dtest_bulk = VariantEvalDataset(te, R=R)
                bulk_loader = DataLoader(dtest_bulk, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
                all_logits = []
                for xm, xw, _ in bulk_loader:
                    xm, xw = xm.to(device), xw.to(device)
                    all_logits.append(model(xm, xw).cpu().numpy())
                all_logits = np.concatenate(all_logits)
                # align logits to te.index order
                idx_order = te.reset_index()["index"].values
                idx_to_logit = {idx: all_logits[i] for i, idx in enumerate(idx_order)}
                wins = 0; total = 0
                for ip, ineg in test_pairs:
                    if ip in idx_to_logit and ineg in idx_to_logit:
                        wins += int(idx_to_logit[ip] > idx_to_logit[ineg])
                        total += 1
                pair_acc_ctx = wins / total if total>0 else np.nan

        gname = te["gene"].unique().tolist()[0]
        out_rows.append({
            "fold": fold, "gene": gname, "n": int(len(y_true)),
            "auroc": round(auroc,4) if not np.isnan(auroc) else np.nan,
            "pr_auc": round(pr,4) if not np.isnan(pr) else np.nan,
            "brier": round(brier,4),
            "pair_acc_variant": round(pair_acc_variant,4),
            "pair_acc_ctx": round(pair_acc_ctx,4) if not (isinstance(pair_acc_ctx,float) and np.isnan(pair_acc_ctx)) else np.nan
        })
    return pd.DataFrame(out_rows).sort_values("gene")

# ---- run
device = "cuda" if torch.cuda.is_available() else "cpu"
SIAM_DIR = ARTI_DIR / f"siamese_pair_run_{time.strftime('%Y%m%d_%H%M%S')}"
SIAM_DIR.mkdir(parents=True, exist_ok=True)

summary = run_loco_pairwise(df, spl_g, epochs=14, batch=96, lr=2e-3, margin=0.2, R=30, device=device)
summary.to_csv(SIAM_DIR/"metrics_siamese_pair_loco_gene.csv", index=False)

print("Device:", device)
print("Run dir:", SIAM_DIR)
display(summary)


Device: cpu
Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/siamese_pair_run_20250816_212340


,fold,gene,n,auroc,pr_auc,brier,pair_acc_variant,pair_acc_ctx
0,loco_gene:EGFR,EGFR,39,0.7171,0.6147,0.2390,0.6667,0.7692
1,loco_gene:KRAS,KRAS,11,0.6250,0.7603,0.2510,0.2727,1.0000
2,loco_gene:PIK3CA,PIK3CA,24,0.6759,0.8916,0.2268,0.7083,0.5000
3,loco_gene:TP53,TP53,197,0.6470,0.4752,0.2493,0.5431,0.6377


**Those pairwise receipts slap:**

* EGFR: pair_acc_ctx 0.77 (that’s real edit-reading under matched context)

* TP53: 0.64 (solid)

* KRAS: 1.00 but n=11… cute, not trustable

* PIK3CA: 0.50 (neutral)

next clean move: see if that edit signal survives LOCO-SITE. one block; same Siamese $Δ$-CNN + tri-matched training + margin loss, but now across the k-site folds.

### I3 — LOCO-SITE, tri-matched pairwise $Δ$-CNN
self-contained and writes a new `siamese_pair_run_*` dir with metrics.

In [29]:
# I3 — Siamese Δ-CNN with tri-matched TRAINING + pairwise margin loss (LOCO-SITE)

import os, glob, time, math, random
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ---- setup
SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
ARTI_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
P_RES   = DATA_DIR/"masterseq_model_ready_plus_resolved.parquet"
cand = sorted(glob.glob(str(ARTI_DIR / "sentiment_run_plus_variants_*")))
assert cand, "No variant split dir — run E2 first."
RUN_DIR = Path(cand[-1])
SPLITS_S = RUN_DIR/"splits_loco_site.csv"

# ---- load
df = pd.read_parquet(P_RES).dropna(subset=[
    "variant_id","gene","label","window_mut","window_wt",
    "wt_codon","mut_codon","ref","alt","cds_pos","tri_center"
]).copy()
for c in ["variant_id","gene","ref","alt","wt_codon","mut_codon","window_mut","window_wt","tri_center"]:
    df[c] = df[c].astype(str).str.upper()
df["label"] = df["label"].astype(int)
df["cds_pos"] = pd.to_numeric(df["cds_pos"], errors="coerce").astype("Int64")
spl_s = pd.read_csv(SPLITS_S)

# ---- helpers
def mut_index_in_window(row, flank=60):
    i0 = int(row["cds_pos"]) - 1
    cs = (i0 // 3) * 3
    center = cs + 1
    return flank + (i0 - center)  # -1,0,+1

def extract_patch(win: str, j: int, R: int = 30):
    s = str(win)
    if not s or j < 0 or j >= len(s): return ""
    a, b = max(0, j-R), min(len(s), j+R+1)
    left_pad  = "N" * (R - (j - a))
    right_pad = "N" * (R - (b - j - 1))
    return left_pad + s[a:b] + right_pad  # fixed length 2R+1

VOCAB = {"A":0,"C":1,"G":2,"T":3,"N":4}
def onehot(seq: str):
    arr = np.zeros((5, len(seq)), dtype=np.float32)
    for i,ch in enumerate(seq):
        arr[VOCAB.get(ch,4), i] = 1.0
    return arr

def make_pairs_trimatched(train_df: pd.DataFrame, seed=SEED):
    rs = np.random.RandomState(seed)
    pairs = []  # (pos_idx, neg_idx)
    train_df = train_df.copy()
    train_df["center_base"] = train_df["tri_center"].str[1]

    for g, sub in train_df.groupby("gene"):
        pos = sub[sub["label"]==1]
        neg = sub[sub["label"]==0].copy()
        used = set()

        for tri, grp in pos.groupby("tri_center"):
            pool = neg[(neg["tri_center"]==tri) & (~neg.index.isin(used))]
            if pool.empty:
                continue
            k = min(len(grp), len(pool))
            pidx = rs.choice(grp.index, size=k, replace=False)
            nidx = rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx, nidx))
            used.update(nidx)

        leftover = pos.loc[~pos.index.isin([p for p,_ in pairs])]
        for base, grp in leftover.groupby("center_base"):
            pool = neg[(neg["tri_center"].str[1]==base) & (~neg.index.isin(used))]
            if pool.empty:
                continue
            k = min(len(grp), len(pool))
            pidx = rs.choice(grp.index, size=k, replace=False)
            nidx = rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx, nidx))
            used.update(nidx)
    return pairs

def make_pairs_for_test(test_df: pd.DataFrame, seed=SEED):
    test_df = test_df.copy()
    test_df["center_base"] = test_df["tri_center"].str[1]
    pos = test_df[test_df["label"]==1]
    neg = test_df[test_df["label"]==0].copy()
    rs = np.random.RandomState(seed)
    pairs=[]; used=set()
    for tri, grp in pos.groupby("tri_center"):
        pool = neg[(neg["tri_center"]==tri) & (~neg.index.isin(used))]
        if pool.empty: continue
        k = min(len(grp), len(pool))
        pidx = rs.choice(grp.index, size=k, replace=False)
        nidx = rs.choice(pool.index, size=k, replace=False)
        pairs += list(zip(pidx, nidx)); used.update(nidx)
    leftover = pos.loc[~pos.index.isin([p for p,_ in pairs])]
    for base, grp in leftover.groupby("center_base"):
        pool = neg[(neg["tri_center"].str[1]==base) & (~neg.index.isin(used))]
        if pool.empty: continue
        k = min(len(grp), len(pool))
        pidx = rs.choice(grp.index, size=k, replace=False)
        nidx = rs.choice(pool.index, size=k, replace=False)
        pairs += list(zip(pidx, nidx)); used.update(nidx)
    return pairs

class PairTrainDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, pairs, R=30):
        self.df = frame; self.pairs = pairs; self.R = R
    def __len__(self): return len(self.pairs)
    def _patches(self, r):
        j = mut_index_in_window(r, flank=60)
        return (extract_patch(r["window_mut"], j, self.R),
                extract_patch(r["window_wt"],  j, self.R))
    def __getitem__(self, k):
        ip, ineg = self.pairs[k]
        rp, rn = self.df.loc[ip], self.df.loc[ineg]
        mp, wp = self._patches(rp)
        mn, wn = self._patches(rn)
        return (torch.from_numpy(onehot(mp)), torch.from_numpy(onehot(wp)),
                torch.from_numpy(onehot(mn)), torch.from_numpy(onehot(wn)))

class VariantEvalDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, R=30):
        self.df = frame.reset_index()
        self.R = R
        self.patches = []
        for _, r in self.df.iterrows():
            j = mut_index_in_window(r, flank=60)
            self.patches.append((
                extract_patch(r["window_mut"], j, self.R),
                extract_patch(r["window_wt"],  j, self.R)
            ))
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        mut, wt = self.patches[i]
        y = float(self.df.loc[i, "label"])
        return torch.from_numpy(onehot(mut)), torch.from_numpy(onehot(wt)), torch.tensor(y, dtype=torch.float32)

class SiameseDeltaCNN(nn.Module):
    def __init__(self, in_ch=5, hid=96):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv1d(in_ch, 64,  kernel_size=11, padding=5),
            nn.ReLU(),
            nn.Conv1d(64, 96, kernel_size=7,  padding=3),
            nn.ReLU(),
            nn.Conv1d(96, 96, kernel_size=3,  padding=1),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1)
        )
        self.norm = nn.LayerNorm(96)
        self.head = nn.Sequential(
            nn.Linear(96, 96),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(96, 1)
        )
    def encode(self, x):
        h = self.enc(x).squeeze(-1)
        return self.norm(h)
    def forward(self, xm, xw):
        hm = self.encode(xm); hw = self.encode(xw)
        return self.head(hm - hw).squeeze(-1)

def run_loco_site(df, splits, epochs=12, batch=96, lr=2e-3, margin=0.2, R=30, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    all_rows=[]
    for fold, sub in splits.groupby("fold"):
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr = dd[dd["split"]=="train"].copy()
        te = dd[dd["split"]=="test"].copy()
        if tr.empty or te.empty:
            continue

        pairs = make_pairs_trimatched(tr, seed=SEED)
        if not pairs:
            continue

        dtrain = PairTrainDataset(tr, pairs, R=R)
        dtest  = VariantEvalDataset(te, R=R)

        model = SiameseDeltaCNN().to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

        def loss_pair(xm_p, xw_p, xm_n, xw_n):
            sp = model(xm_p, xw_p); sn = model(xm_n, xw_n)
            return torch.relu(margin - (sp - sn)).mean()

        loader = DataLoader(dtrain, batch_size=batch, shuffle=True, num_workers=2, pin_memory=True, drop_last=False)
        for ep in range(epochs):
            model.train()
            for xm_p, xw_p, xm_n, xw_n in loader:
                xm_p, xw_p = xm_p.to(device), xw_p.to(device)
                xm_n, xw_n = xm_n.to(device), xw_n.to(device)
                opt.zero_grad(); loss = loss_pair(xm_p, xw_p, xm_n, xw_n); loss.backward(); opt.step()

        # variant eval
        model.eval()
        logits=[]; ys=[]
        test_loader = DataLoader(dtest, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
        with torch.no_grad():
            for xm, xw, y in test_loader:
                xm, xw = xm.to(device), xw.to(device)
                logit = model(xm, xw)
                logits.append(logit.cpu().numpy()); ys.append(y.numpy())
        y_log = np.concatenate(logits); y_true = np.concatenate(ys).astype(int)
        p = 1/(1+np.exp(-y_log))
        auroc = roc_auc_score(y_true, p) if len(np.unique(y_true))>1 else np.nan
        pr    = average_precision_score(y_true, p) if len(np.unique(y_true))>1 else np.nan
        brier = brier_score_loss(y_true, p)
        pair_acc_variant = float(((y_log>0)&(y_true==1) | (y_log<=0)&(y_true==0)).mean())

        # tri-matched pair acc on TEST
        test_pairs = make_pairs_for_test(te, seed=SEED)
        pair_acc_ctx = np.nan
        if test_pairs:
            # score all te rows once in original index order
            idx_order = te.reset_index()["index"].values
            # map logits aligned to above order
            # (VariantEvalDataset uses reset_index order)
            idx_to_logit = {idx_order[i]: y_log[i] for i in range(len(idx_order))}
            wins=0; total=0
            for ip, ineg in test_pairs:
                if ip in idx_to_logit and ineg in idx_to_logit:
                    wins += int(idx_to_logit[ip] > idx_to_logit[ineg])
                    total += 1
            pair_acc_ctx = wins/total if total>0 else np.nan

        all_rows.append({
            "fold": fold, "n": int(len(y_true)),
            "auroc": round(auroc,4) if not np.isnan(auroc) else np.nan,
            "pr_auc": round(pr,4)   if not np.isnan(pr) else np.nan,
            "brier": round(brier,4),
            "pair_acc_variant": round(pair_acc_variant,4),
            "pair_acc_ctx": round(pair_acc_ctx,4) if not (isinstance(pair_acc_ctx,float) and np.isnan(pair_acc_ctx)) else np.nan
        })
    return pd.DataFrame(all_rows).sort_values("fold")

device = "cuda" if torch.cuda.is_available() else "cpu"
SIAM_DIR = ARTI_DIR / f"siamese_pair_site_run_{time.strftime('%Y%m%d_%H%M%S')}"
SIAM_DIR.mkdir(parents=True, exist_ok=True)

summary = run_loco_site(df, spl_s, epochs=12, batch=96, lr=2e-3, margin=0.2, R=30, device=device)
summary.to_csv(SIAM_DIR/"metrics_siamese_pair_loco_site.csv", index=False)

print("Device:", device)
print("Run dir:", SIAM_DIR)
display(summary)


Device: cpu
Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/siamese_pair_site_run_20250816_212929


,fold,n,auroc,pr_auc,brier,pair_acc_variant,pair_acc_ctx
0,loco_site:k1,55,0.5784,0.4288,0.2468,0.6000,0.7368
1,loco_site:k2,54,0.5472,0.5128,0.2496,0.5185,0.3913
2,loco_site:k3,54,0.7980,0.6112,0.2362,0.7222,0.9000
3,loco_site:k4,54,0.4787,0.4009,0.2493,0.5000,0.4091
4,loco_site:k5,54,0.6392,0.5440,0.2465,0.5926,0.7647


Those LOCO-SITE pairwise scores are bipolar: k1/k3/k5 look legit (ctx pair-acc 0.74–0.90), k2/k4 fall over. that screams **optimization + loss shape**, not just data. let’s tighten the training signal:

**Move:** pairwise RankNet (logistic) loss on the Siamese Δ-CNN with bio scalars (Grantham, BLOSUM, aa_same) blended into the score. still tri-matched training, still edit-centered patches. this should smooth gradients and stabilize the shaky folds.

### First Bootstrap since I'm changing to a GPU A100 high-ram

In [1]:
# BOOTSTRAP AFTER GPU SWITCH — remount, rewire paths, and sanity-check artifacts

from pathlib import Path
import glob, os

# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# 2) Repoint base dirs
BASE = Path("/content/drive/MyDrive/Cancer_gene_database")
DATA = BASE / "data"
ARTI = BASE / "artifacts"

# 3) Sanity checks (these should exist if you ran earlier cells)
need = [
    DATA/"masterseq_model_ready_plus_resolved.parquet",
]
for p in need:
    assert p.exists(), f"Missing expected file: {p}"

# 4) Grab the latest variant split dir we used
runs = sorted(glob.glob(str(ARTI / "sentiment_run_plus_variants_*")))
assert runs, "No split dirs found — rerun the split cell."
RUN_DIR = Path(runs[-1])

# 5) Quick listing so you see what we’ll reuse
print("DATA:", DATA)
print("ARTI:", ARTI)
print("Using RUN_DIR:", RUN_DIR)
print("Has splits?:", (RUN_DIR/"splits_loco_gene.csv").exists(), (RUN_DIR/"splits_loco_site.csv").exists())

# 6) Confirm GPU is live
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
else:
    print("Running on CPU")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA: /content/drive/MyDrive/Cancer_gene_database/data
ARTI: /content/drive/MyDrive/Cancer_gene_database/artifacts
Using RUN_DIR: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607
Has splits?: True True
GPU available: True
GPU: NVIDIA A100-SXM4-40GB


In [2]:
# I4 — Siamese Δ-CNN + bio scalars with RankNet loss (LOCO-SITE, tri-matched training)
import os, glob, time, random, math
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# ----- setup
SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
ARTI_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
P_RES   = DATA_DIR/"masterseq_model_ready_plus_resolved.parquet"
cand = sorted(glob.glob(str(ARTI_DIR / "sentiment_run_plus_variants_*")))
assert cand, "Need variant split dir — rerun E2 earlier."
RUN_DIR = Path(cand[-1])
SPLITS_S = RUN_DIR/"splits_loco_site.csv"

# ----- load
df = pd.read_parquet(P_RES).dropna(subset=[
    "variant_id","gene","label","window_mut","window_wt",
    "wt_codon","mut_codon","ref","alt","cds_pos","tri_center"
]).copy()
for c in ["variant_id","gene","ref","alt","wt_codon","mut_codon","window_mut","window_wt","tri_center"]:
    df[c] = df[c].astype(str).str.upper()
df["label"] = df["label"].astype(int)
df["cds_pos"] = pd.to_numeric(df["cds_pos"], errors="coerce").astype("Int64")
spl_s = pd.read_csv(SPLITS_S)

# ----- genetics helpers (AA + scores)
CODON2AA = {
 "TTT":"F","TTC":"F","TTA":"L","TTG":"L","CTT":"L","CTC":"L","CTA":"L","CTG":"L",
 "ATT":"I","ATC":"I","ATA":"I","ATG":"M","GTT":"V","GTC":"V","GTA":"V","GTG":"V",
 "TCT":"S","TCC":"S","TCA":"S","TCG":"S","CCT":"P","CCC":"P","CCA":"P","CCG":"P",
 "ACT":"T","ACC":"T","ACA":"T","ACG":"T","GCT":"A","GCC":"A","GCA":"A","GCG":"A",
 "TAT":"Y","TAC":"Y","TAA":"*","TAG":"*","CAT":"H","CAC":"H","CAA":"Q","CAG":"Q",
 "AAT":"N","AAC":"N","AAA":"K","AAG":"K","GAT":"D","GAC":"D","GAA":"E","GAG":"E",
 "TGT":"C","TGC":"C","TGA":"*","TGG":"W","CGT":"R","CGC":"R","CGA":"R","CGG":"R",
 "AGT":"S","AGC":"S","AGA":"R","AGG":"R","GGT":"G","GGC":"G","GGA":"G","GGG":"G"
}
_GR = {
 'A':{'A':0,'R':112,'N':111,'D':126,'C':195,'Q':91,'E':107,'G':60,'H':86,'I':94,'L':96,'K':106,'M':84,'F':113,'P':27,'S':99,'T':58,'W':148,'Y':112,'V':64},
 'R':{'A':112,'R':0,'N':86,'D':96,'C':180,'Q':43,'E':54,'G':125,'H':29,'I':97,'L':102,'K':26,'M':91,'F':97,'P':103,'S':110,'T':71,'W':101,'Y':77,'V':96},
 'N':{'A':111,'R':86,'N':0,'D':23,'C':139,'Q':68,'E':42,'G':80,'H':68,'I':149,'L':153,'K':94,'M':142,'F':158,'P':91,'S':46,'T':65,'W':174,'Y':143,'V':133},
 'D':{'A':126,'R':96,'N':23,'D':0,'C':154,'Q':61,'E':45,'G':94,'H':81,'I':168,'L':172,'K':101,'M':160,'F':177,'P':108,'S':65,'T':85,'W':181,'Y':160,'V':152},
 'C':{'A':195,'R':180,'N':139,'D':154,'C':0,'Q':154,'E':170,'G':159,'H':174,'I':198,'L':198,'K':202,'M':196,'F':205,'P':169,'S':112,'T':149,'W':215,'Y':194,'V':192},
 'Q':{'A':91,'R':43,'N':68,'D':61,'C':154,'Q':0,'E':29,'G':87,'H':24,'I':109,'L':113,'K':53,'M':101,'F':116,'P':76,'S':68,'T':42,'W':130,'Y':99,'V':96},
 'E':{'A':107,'R':54,'N':42,'D':45,'C':170,'Q':29,'G':98,'H':40,'I':134,'L':138,'K':56,'M':126,'F':140,'P':93,'S':80,'T':65,'W':152,'Y':122,'V':121},
 'G':{'A':60,'R':125,'N':80,'D':94,'C':159,'Q':87,'E':98,'H':98,'I':135,'L':138,'K':127,'M':127,'F':153,'P':42,'S':56,'T':59,'W':184,'Y':147,'V':109},
 'H':{'A':86,'R':29,'N':68,'D':81,'C':174,'Q':24,'E':40,'G':98,'I':94,'L':99,'K':32,'M':87,'F':100,'P':77,'S':89,'T':47,'W':115,'Y':83,'V':84},
 'I':{'A':94,'R':97,'N':149,'D':168,'C':198,'Q':109,'E':134,'G':135,'H':94,'L':5,'K':102,'M':10,'F':21,'P':95,'S':142,'T':89,'W':61,'Y':33,'V':29},
 'L':{'A':96,'R':102,'N':153,'D':172,'C':198,'Q':113,'E':138,'G':138,'H':99,'I':5,'L':0,'K':107,'M':15,'F':22,'P':98,'S':145,'T':92,'W':61,'Y':36,'V':32},
 'K':{'A':106,'R':26,'N':94,'D':101,'C':202,'Q':53,'E':56,'G':127,'H':32,'I':102,'L':107,'K':0,'M':95,'F':102,'P':103,'S':121,'T':78,'W':110,'Y':85,'V':97},
 'M':{'A':84,'R':91,'N':142,'D':160,'C':196,'Q':101,'E':126,'G':127,'H':87,'I':10,'L':15,'K':95,'F':28,'P':87,'S':135,'T':81,'W':67,'Y':36,'V':21},
 'F':{'A':113,'R':97,'N':158,'D':177,'C':205,'Q':116,'E':140,'G':153,'H':100,'I':21,'L':22,'K':102,'M':28,'P':114,'S':155,'T':103,'W':40,'Y':22,'V':50},
 'P':{'A':27,'R':103,'N':91,'D':108,'C':169,'Q':76,'E':93,'G':42,'H':77,'I':95,'L':98,'K':103,'S':74,'T':38,'W':147,'Y':110,'V':68},
 'S':{'A':99,'R':110,'N':46,'D':65,'C':112,'Q':68,'E':80,'G':56,'H':89,'I':142,'L':145,'K':121,'M':135,'F':155,'P':74,'T':58,'W':177,'Y':144,'V':124},
 'T':{'A':58,'R':71,'N':65,'D':85,'C':149,'Q':42,'E':65,'G':59,'H':47,'I':89,'L':92,'K':78,'M':81,'F':103,'P':38,'S':58,'W':128,'Y':92,'V':69},
 'W':{'A':148,'R':101,'N':174,'D':181,'C':215,'Q':130,'E':152,'G':184,'H':115,'I':61,'L':61,'K':110,'M':67,'F':40,'P':147,'S':177,'T':128,'Y':37,'V':88},
 'Y':{'A':112,'R':77,'N':143,'D':160,'C':194,'Q':99,'E':122,'G':147,'H':83,'I':33,'L':36,'K':85,'M':36,'F':22,'P':110,'S':144,'T':92,'W':37,'Y':0,'V':55},
 'V':{'A':64,'R':96,'N':133,'D':152,'C':192,'Q':96,'E':121,'G':109,'H':84,'I':29,'L':32,'K':97,'M':21,'F':50,'P':68,'S':124,'T':69,'W':88,'Y':55,'V':0}
}
_B = """\
   A  R  N  D  C  Q  E  G  H  I  L  K  M  F  P  S  T  W  Y  V
A  4 -1 -2 -2  0 -1 -1  0 -2 -1 -1 -1 -1 -2 -1  1  0 -3 -2  0
R -1  5  0 -2 -3  1  0 -2  0 -3 -2  2 -1 -3 -2 -1 -1 -3 -2 -3
N -2  0  6  1 -3  0  0  0  1 -3 -3  0 -2 -3 -2  1  0 -4 -2 -3
D -2 -2  1  6 -3  0  2 -1 -1 -3 -4 -1 -3 -3 -1  0 -1 -4 -3 -3
C  0 -3 -3 -3  9 -3 -4 -3 -3 -1 -1 -3 -1 -2 -3 -1 -1 -2 -2 -1
Q -1  1  0  0 -3  5  2 -2  0 -3 -2  1  0 -3 -1  0 -1 -2 -1 -2
E -1  0  0  2 -4  2  5 -2  0 -3 -3  1 -2 -3 -1  0 -1 -3 -2 -2
G  0 -2  0 -1 -3 -2 -2  6 -2 -4 -4 -2 -3 -3 -2  0 -2 -2 -3 -3
H -2  0  1 -1 -3  0  0 -2  8 -3 -3 -1 -2 -1 -2 -1 -2 -2  2 -3
I -1 -3 -3 -3 -1 -3 -3 -4 -3  4  2 -3  1  0 -3 -2 -1 -3 -1  3
L -1 -2 -3 -4 -1 -2 -3 -4 -3  2  4 -2  2  0 -3 -2 -1 -2 -1  1
K -1  2  0 -1 -3  1  1 -2 -1 -3 -2  5 -1 -3 -1  0 -1 -3 -2 -2
M -1 -1 -2 -3 -1  0 -2 -3 -2  1  2 -1  5  0 -2 -1 -1 -1 -1  1
F -2 -3 -3 -3 -2 -3 -3 -3 -1  0  0 -3  0  6 -4 -2 -2  1  3 -1
P -1 -2 -2 -1 -3 -1 -1 -2 -2 -3 -3 -1 -2 -4  7 -1 -1 -4 -3 -2
S  1 -1  1  0 -1  0  0  0 -1 -2 -2  0 -1 -2 -1  4  1 -3 -2 -2
T  0 -1  0 -1 -1 -1 -1 -2 -2 -1 -1 -1 -1 -2 -1  1  5 -2 -2  0
W -3 -3 -4 -4 -2 -2 -3 -2 -2 -3 -2 -3 -1  1 -4 -3 -2 11  2 -3
Y -2 -2 -2 -3 -2 -1 -2 -3  2 -1 -1 -2 -1  3 -3 -2 -2  2  7 -1
V  0 -3 -3 -3 -1 -2 -2 -3 -3  3  1 -2  1 -1 -2 -2  0 -3 -1  4
"""
B62 = {}
_hdr = _B.strip().splitlines()[0].split()
for line in _B.strip().splitlines()[1:]:
    parts = line.split(); a = parts[0]
    B62[a] = { _hdr[i]: int(parts[i+1]) for i in range(len(_hdr)) }

def grantham(a,b):
    if a==b: return 0
    return _GR.get(a,{}).get(b, _GR.get(b,{}).get(a, 0))

def blosum62(a,b):
    return B62.get(a,{}).get(b, B62.get(b,{}).get(a, 0))

# ----- windows & enc
def mut_index_in_window(row, flank=60):
    i0 = int(row["cds_pos"]) - 1
    cs = (i0 // 3) * 3
    center = cs + 1
    return flank + (i0 - center)

def extract_patch(win: str, j: int, R: int = 30):
    s = str(win)
    if not s or j < 0 or j >= len(s): return ""
    a, b = max(0, j-R), min(len(s), j+R+1)
    left_pad  = "N" * (R - (j - a))
    right_pad = "N" * (R - (b - j - 1))
    return left_pad + s[a:b] + right_pad

VOCAB = {"A":0,"C":1,"G":2,"T":3,"N":4}
def onehot(seq: str):
    arr = np.zeros((5, len(seq)), dtype=np.float32)
    for i,ch in enumerate(seq):
        arr[VOCAB.get(ch,4), i] = 1.0
    return arr

def aa_scalars(wt_codon, mut_codon):
    a = CODON2AA.get(str(wt_codon), "X"); b = CODON2AA.get(str(mut_codon), "X")
    aa_same = float(a==b)
    if a=="X" or b=="X":
        return np.array([aa_same, 0.0, 0.0], dtype=np.float32)
    # normalize: grantham in [0,1], blosum roughly [-4..11] → [0,1]
    g = grantham(a,b) / 215.0
    bl = (blosum62(a,b) + 4) / 15.0
    return np.array([aa_same, g, bl], dtype=np.float32)

# ----- tri-matched pairing (train) and test pairing (report)
def make_pairs_trimatched(train_df: pd.DataFrame, seed=SEED):
    rs = np.random.RandomState(seed)
    pairs=[]; used=set()
    train_df = train_df.copy()
    train_df["center_base"] = train_df["tri_center"].str[1]
    for g, sub in train_df.groupby("gene"):
        pos = sub[sub["label"]==1]
        neg = sub[sub["label"]==0].copy()
        # exact tri
        for tri, grp in pos.groupby("tri_center"):
            pool = neg[(neg["tri_center"]==tri) & (~neg.index.isin(used))]
            if pool.empty: continue
            k = min(len(grp), len(pool))
            pidx = rs.choice(grp.index, size=k, replace=False)
            nidx = rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx, nidx)); used.update(nidx)
        # fallback base
        leftover = pos.loc[~pos.index.isin([p for p,_ in pairs])]
        for base, grp in leftover.groupby("center_base"):
            pool = neg[(neg["tri_center"].str[1]==base) & (~neg.index.isin(used))]
            if pool.empty: continue
            k = min(len(grp), len(pool))
            pidx = rs.choice(grp.index, size=k, replace=False)
            nidx = rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx, nidx)); used.update(nidx)
    return pairs

def make_pairs_for_test(test_df: pd.DataFrame, seed=SEED):
    rs = np.random.RandomState(seed)
    pairs=[]; used=set()
    test_df = test_df.copy()
    test_df["center_base"] = test_df["tri_center"].str[1]
    pos = test_df[test_df["label"]==1]
    neg = test_df[test_df["label"]==0].copy()
    for tri, grp in pos.groupby("tri_center"):
        pool = neg[(neg["tri_center"]==tri) & (~neg.index.isin(used))]
        if pool.empty: continue
        k = min(len(grp), len(pool))
        pidx = rs.choice(grp.index, size=k, replace=False)
        nidx = rs.choice(pool.index, size=k, replace=False)
        pairs += list(zip(pidx, nidx)); used.update(nidx)
    leftover = pos.loc[~pos.index.isin([p for p,_ in pairs])]
    for base, grp in leftover.groupby("center_base"):
        pool = neg[(neg["tri_center"].str[1]==base) & (~neg.index.isin(used))]
        if pool.empty: continue
        k = min(len(grp), len(pool))
        pidx = rs.choice(grp.index, size=k, replace=False)
        nidx = rs.choice(pool.index, size=k, replace=False)
        pairs += list(zip(pidx, nidx)); used.update(nidx)
    return pairs

# ----- datasets
class PairTrainDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, pairs, R=30):
        self.df = frame
        self.pairs = pairs
        self.R = R
    def __len__(self): return len(self.pairs)
    def _patch_feats(self, r):
        j = mut_index_in_window(r, flank=60)
        mut = extract_patch(r["window_mut"], j, self.R)
        wt  = extract_patch(r["window_wt"],  j, self.R)
        scal = aa_scalars(r["wt_codon"], r["mut_codon"])
        return torch.from_numpy(onehot(mut)), torch.from_numpy(onehot(wt)), torch.from_numpy(scal)
    def __getitem__(self, k):
        ip, ineg = self.pairs[k]
        rp, rn = self.df.loc[ip], self.df.loc[ineg]
        xm_p, xw_p, s_p = self._patch_feats(rp)
        xm_n, xw_n, s_n = self._patch_feats(rn)
        return xm_p, xw_p, s_p, xm_n, xw_n, s_n

class VariantEvalDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, R=30):
        self.df = frame.reset_index()
        self.R = R
        self.items=[]
        for _, r in self.df.iterrows():
            j = mut_index_in_window(r, flank=60)
            mut = extract_patch(r["window_mut"], j, self.R)
            wt  = extract_patch(r["window_wt"],  j, self.R)
            scal = aa_scalars(r["wt_codon"], r["mut_codon"])
            self.items.append((onehot(mut), onehot(wt), scal, float(r["label"])))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        mut, wt, scal, y = self.items[i]
        return torch.from_numpy(mut), torch.from_numpy(wt), torch.from_numpy(scal), torch.tensor(y, dtype=torch.float32)

# ----- model (scalars blended into score)
class SiameseDeltaCNN(nn.Module):
    def __init__(self, in_ch=5, hid=96, scal_dim=3):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv1d(in_ch, 64,  kernel_size=11, padding=5),
            nn.ReLU(),
            nn.Conv1d(64, 96, kernel_size=7,  padding=3),
            nn.ReLU(),
            nn.Conv1d(96, 96, kernel_size=3,  padding=1),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1)
        )
        self.norm = nn.LayerNorm(96)
        self.head = nn.Sequential(
            nn.Linear(96, 96),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(96, 1)
        )
        self.scal_head = nn.Linear(scal_dim, 1)  # projects bio scalars into score
    def encode(self, x):
        h = self.enc(x).squeeze(-1)
        return self.norm(h)
    def score(self, xm, xw, scalars):
        hm = self.encode(xm); hw = self.encode(xw)
        d  = hm - hw
        core = self.head(d).squeeze(-1)
        scal = self.scal_head(scalars).squeeze(-1)
        return core + scal
    def forward(self, xm, xw, scalars):
        return self.score(xm, xw, scalars)

# ----- training/eval (LOCO-SITE)
def run_loco_site_ranknet(df, splits, epochs=14, batch=96, lr=2e-3, R=30, device=None, tau=1.0):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    all_rows=[]
    bce = nn.BCEWithLogitsLoss()
    for fold, sub in splits.groupby("fold"):
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr = dd[dd["split"]=="train"].copy()
        te = dd[dd["split"]=="test"].copy()
        if tr.empty or te.empty:
            continue

        pairs = make_pairs_trimatched(tr, seed=SEED)
        if not pairs:
            continue

        dtrain = PairTrainDataset(tr, pairs, R=R)
        dtest  = VariantEvalDataset(te, R=R)

        model = SiameseDeltaCNN().to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

        def loss_ranknet(xm_p, xw_p, s_p, xm_n, xw_n, s_n, tau=1.0):
            sp = model(xm_p, xw_p, s_p) / tau
            sn = model(xm_n, xw_n, s_n) / tau
            diff = sp - sn
            # logistic pairwise: -log(sigmoid(diff)) = BCEWithLogits(diff, 1)
            tgt = torch.ones_like(diff)
            return bce(diff, tgt)

        loader = DataLoader(dtrain, batch_size=batch, shuffle=True, num_workers=2, pin_memory=True, drop_last=False)
        for ep in range(epochs):
            model.train()
            for xm_p, xw_p, s_p, xm_n, xw_n, s_n in loader:
                xm_p, xw_p, s_p = xm_p.to(device), xw_p.to(device), s_p.to(device)
                xm_n, xw_n, s_n = xm_n.to(device), xw_n.to(device), s_n.to(device)
                opt.zero_grad()
                l = loss_ranknet(xm_p, xw_p, s_p, xm_n, xw_n, s_n, tau=tau)
                l.backward(); opt.step()

        # variant eval
        model.eval()
        logits=[]; ys=[]
        test_loader = DataLoader(dtest, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
        with torch.no_grad():
            for xm, xw, s, y in test_loader:
                xm, xw, s = xm.to(device), xw.to(device), s.to(device)
                logit = model(xm, xw, s)
                logits.append(logit.cpu().numpy()); ys.append(y.numpy())
        y_log = np.concatenate(logits); y_true = np.concatenate(ys).astype(int)
        p = 1/(1+np.exp(-y_log))
        auroc = roc_auc_score(y_true, p) if len(np.unique(y_true))>1 else np.nan
        pr    = average_precision_score(y_true, p) if len(np.unique(y_true))>1 else np.nan
        brier = brier_score_loss(y_true, p)
        pair_acc_variant = float(((y_log>0)&(y_true==1) | (y_log<=0)&(y_true==0)).mean())

        # tri-matched pair acc on TEST
        test_pairs = make_pairs_for_test(te, seed=SEED)
        pair_acc_ctx = np.nan
        if test_pairs:
            # score once, map index->logit
            idx_order = te.reset_index()["index"].values
            idx_to_logit = {idx_order[i]: y_log[i] for i in range(len(idx_order))}
            wins=0; total=0
            for ip, ineg in test_pairs:
                if ip in idx_to_logit and ineg in idx_to_logit:
                    wins += int(idx_to_logit[ip] > idx_to_logit[ineg]); total += 1
            pair_acc_ctx = wins/total if total>0 else np.nan

        all_rows.append({
            "fold": list(sub["fold"])[0], "n": int(len(y_true)),
            "auroc": round(auroc,4) if not np.isnan(auroc) else np.nan,
            "pr_auc": round(pr,4)   if not np.isnan(pr) else np.nan,
            "brier": round(brier,4),
            "pair_acc_variant": round(pair_acc_variant,4),
            "pair_acc_ctx": round(pair_acc_ctx,4) if not (isinstance(pair_acc_ctx,float) and np.isnan(pair_acc_ctx)) else np.nan
        })
    return pd.DataFrame(all_rows).sort_values("fold")

device = "cuda" if torch.cuda.is_available() else "cpu"
SIAM_DIR = ARTI_DIR / f"siamese_pair_ranknet_site_{time.strftime('%Y%m%d_%H%M%S')}"
SIAM_DIR.mkdir(parents=True, exist_ok=True)

summary = run_loco_site_ranknet(df, spl_s, epochs=16, batch=96, lr=2e-3, R=30, device=device, tau=1.0)
summary.to_csv(SIAM_DIR/"metrics_siamese_pair_ranknet_loco_site.csv", index=False)

print("Device:", device)
print("Run dir:", SIAM_DIR)
display(summary)


Device: cuda
Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/siamese_pair_ranknet_site_20250816_214236


,fold,n,auroc,pr_auc,brier,pair_acc_variant,pair_acc_ctx
0,loco_site:k1,55,0.7787,0.7090,0.1844,0.7455,0.7368
1,loco_site:k2,54,0.7958,0.7660,0.2166,0.6481,0.8696
2,loco_site:k3,54,0.9163,0.8451,0.1431,0.8704,0.9500
3,loco_site:k4,54,0.8352,0.7673,0.1892,0.7222,0.9091
4,loco_site:k5,54,0.8312,0.8050,0.1861,0.7037,0.8235


#### Those LOCO-SITE RankNet numbers are clean. context is caged and the model’s actually reading the edit. let’s hit the matching **LOCO-GENE** pass with the same recipe (tri-matched training + RankNet + bio scalars).

### I5 — LOCO-GENE: Siamese Δ-CNN + bio scalars with RankNet loss (tri-matched training)


In [3]:
# I5 — LOCO-GENE: Siamese Δ-CNN + bio scalars with RankNet loss (tri-matched training)

import os, glob, time, random, math
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# ----- setup
SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DATA_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/data")
ARTI_DIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
P_RES   = DATA_DIR/"masterseq_model_ready_plus_resolved.parquet"
cand = sorted(glob.glob(str(ARTI_DIR / "sentiment_run_plus_variants_*")))
assert cand, "Need variant split dir — rerun the split cell."
RUN_DIR = Path(cand[-1])
SPLITS_G = RUN_DIR/"splits_loco_gene.csv"

# ----- load
df = pd.read_parquet(P_RES).dropna(subset=[
    "variant_id","gene","label","window_mut","window_wt",
    "wt_codon","mut_codon","ref","alt","cds_pos","tri_center"
]).copy()
for c in ["variant_id","gene","ref","alt","wt_codon","mut_codon","window_mut","window_wt","tri_center"]:
    df[c] = df[c].astype(str).str.upper()
df["label"] = df["label"].astype(int)
df["cds_pos"] = pd.to_numeric(df["cds_pos"], errors="coerce").astype("Int64")
spl_g = pd.read_csv(SPLITS_G)

# ----- genetics helpers (AA + scores)
CODON2AA = {
 "TTT":"F","TTC":"F","TTA":"L","TTG":"L","CTT":"L","CTC":"L","CTA":"L","CTG":"L",
 "ATT":"I","ATC":"I","ATA":"I","ATG":"M","GTT":"V","GTC":"V","GTA":"V","GTG":"V",
 "TCT":"S","TCC":"S","TCA":"S","TCG":"S","CCT":"P","CCC":"P","CCA":"P","CCG":"P",
 "ACT":"T","ACC":"T","ACA":"T","ACG":"T","GCT":"A","GCC":"A","GCA":"A","GCG":"A",
 "TAT":"Y","TAC":"Y","TAA":"*","TAG":"*","CAT":"H","CAC":"H","CAA":"Q","CAG":"Q",
 "AAT":"N","AAC":"N","AAA":"K","AAG":"K","GAT":"D","GAC":"D","GAA":"E","GAG":"E",
 "TGT":"C","TGC":"C","TGA":"*","TGG":"W","CGT":"R","CGC":"R","CGA":"R","CGG":"R",
 "AGT":"S","AGC":"S","AGA":"R","AGG":"R","GGT":"G","GGC":"G","GGA":"G","GGG":"G"
}
_GR = {
 'A':{'A':0,'R':112,'N':111,'D':126,'C':195,'Q':91,'E':107,'G':60,'H':86,'I':94,'L':96,'K':106,'M':84,'F':113,'P':27,'S':99,'T':58,'W':148,'Y':112,'V':64},
 'R':{'A':112,'R':0,'N':86,'D':96,'C':180,'Q':43,'E':54,'G':125,'H':29,'I':97,'L':102,'K':26,'M':91,'F':97,'P':103,'S':110,'T':71,'W':101,'Y':77,'V':96},
 'N':{'A':111,'R':86,'N':0,'D':23,'C':139,'Q':68,'E':42,'G':80,'H':68,'I':149,'L':153,'K':94,'M':142,'F':158,'P':91,'S':46,'T':65,'W':174,'Y':143,'V':133},
 'D':{'A':126,'R':96,'N':23,'D':0,'C':154,'Q':61,'E':45,'G':94,'H':81,'I':168,'L':172,'K':101,'M':160,'F':177,'P':108,'S':65,'T':85,'W':181,'Y':160,'V':152},
 'C':{'A':195,'R':180,'N':139,'D':154,'C':0,'Q':154,'E':170,'G':159,'H':174,'I':198,'L':198,'K':202,'M':196,'F':205,'P':169,'S':112,'T':149,'W':215,'Y':194,'V':192},
 'Q':{'A':91,'R':43,'N':68,'D':61,'C':154,'Q':0,'E':29,'G':87,'H':24,'I':109,'L':113,'K':53,'M':101,'F':116,'P':76,'S':68,'T':42,'W':130,'Y':99,'V':96},
 'E':{'A':107,'R':54,'N':42,'D':45,'C':170,'Q':29,'G':98,'H':40,'I':134,'L':138,'K':56,'M':126,'F':140,'P':93,'S':80,'T':65,'W':152,'Y':122,'V':121},
 'G':{'A':60,'R':125,'N':80,'D':94,'C':159,'Q':87,'E':98,'H':98,'I':135,'L':138,'K':127,'M':127,'F':153,'P':42,'S':56,'T':59,'W':184,'Y':147,'V':109},
 'H':{'A':86,'R':29,'N':68,'D':81,'C':174,'Q':24,'E':40,'G':98,'I':94,'L':99,'K':32,'M':87,'F':100,'P':77,'S':89,'T':47,'W':115,'Y':83,'V':84},
 'I':{'A':94,'R':97,'N':149,'D':168,'C':198,'Q':109,'E':134,'G':135,'H':94,'L':5,'K':102,'M':10,'F':21,'P':95,'S':142,'T':89,'W':61,'Y':33,'V':29},
 'L':{'A':96,'R':102,'N':153,'D':172,'C':198,'Q':113,'E':138,'G':138,'H':99,'I':5,'L':0,'K':107,'M':15,'F':22,'P':98,'S':145,'T':92,'W':61,'Y':36,'V':32},
 'K':{'A':106,'R':26,'N':94,'D':101,'C':202,'Q':53,'E':56,'G':127,'H':32,'I':102,'L':107,'K':0,'M':95,'F':102,'P':103,'S':121,'T':78,'W':110,'Y':85,'V':97},
 'M':{'A':84,'R':91,'N':142,'D':160,'C':196,'Q':101,'E':126,'G':127,'H':87,'I':10,'L':15,'K':95,'F':28,'P':87,'S':135,'T':81,'W':67,'Y':36,'V':21},
 'F':{'A':113,'R':97,'N':158,'D':177,'C':205,'Q':116,'E':140,'G':153,'H':100,'I':21,'L':22,'K':102,'M':28,'P':114,'S':155,'T':103,'W':40,'Y':22,'V':50},
 'P':{'A':27,'R':103,'N':91,'D':108,'C':169,'Q':76,'E':93,'G':42,'S':74,'T':38,'W':147,'Y':110,'V':68},
 'S':{'A':99,'R':110,'N':46,'D':65,'C':112,'Q':68,'E':80,'G':56,'H':89,'I':142,'L':145,'K':121,'M':135,'F':155,'P':74,'T':58,'W':177,'Y':144,'V':124},
 'T':{'A':58,'R':71,'N':65,'D':85,'C':149,'Q':42,'E':65,'G':59,'H':47,'I':89,'L':92,'K':78,'M':81,'F':103,'P':38,'S':58,'W':128,'Y':92,'V':69},
 'W':{'A':148,'R':101,'N':174,'D':181,'C':215,'Q':130,'E':152,'G':184,'H':115,'I':61,'L':61,'K':110,'M':67,'F':40,'P':147,'S':177,'T':128,'Y':37,'V':88},
 'Y':{'A':112,'R':77,'N':143,'D':160,'C':194,'Q':99,'E':122,'G':147,'H':83,'I':33,'L':36,'K':85,'M':36,'F':22,'P':110,'S':144,'T':92,'W':37,'V':55},
 'V':{'A':64,'R':96,'N':133,'D':152,'C':192,'Q':96,'E':121,'G':109,'H':84,'I':29,'L':32,'K':97,'M':21,'F':50,'P':68,'S':124,'T':69,'W':88,'Y':55,'V':0}
}
_B = """\
   A  R  N  D  C  Q  E  G  H  I  L  K  M  F  P  S  T  W  Y  V
A  4 -1 -2 -2  0 -1 -1  0 -2 -1 -1 -1 -1 -2 -1  1  0 -3 -2  0
R -1  5  0 -2 -3  1  0 -2  0 -3 -2  2 -1 -3 -2 -1 -1 -3 -2 -3
N -2  0  6  1 -3  0  0  0  1 -3 -3  0 -2 -3 -2  1  0 -4 -2 -3
D -2 -2  1  6 -3  0  2 -1 -1 -3 -4 -1 -3 -3 -1  0 -1 -4 -3 -3
C  0 -3 -3 -3  9 -3 -4 -3 -3 -1 -1 -3 -1 -2 -3 -1 -1 -2 -2 -1
Q -1  1  0  0 -3  5  2 -2  0 -3 -2  1  0 -3 -1  0 -1 -2 -1 -2
E -1  0  0  2 -4  2  5 -2  0 -3 -3  1 -2 -3 -1  0 -1 -3 -2 -2
G  0 -2  0 -1 -3 -2 -2  6 -2 -4 -4 -2 -3 -3 -2  0 -2 -2 -3 -3
H -2  0  1 -1 -3  0  0 -2  8 -3 -3 -1 -2 -1 -2 -1 -2 -2  2 -3
I -1 -3 -3 -3 -1 -3 -3 -4 -3  4  2 -3  1  0 -3 -2 -1 -3 -1  3
L -1 -2 -3 -4 -1 -2 -3 -4 -3  2  4 -2  2  0 -3 -2 -1 -2 -1  1
K -1  2  0 -1 -3  1  1 -2 -1 -3 -2  5 -1 -3 -1  0 -1 -3 -2 -2
M -1 -1 -2 -3 -1  0 -2 -3 -2  1  2 -1  5  0 -2 -1 -1 -1 -1  1
F -2 -3 -3 -3 -2 -3 -3 -3 -1  0  0 -3  0  6 -4 -2 -2  1  3 -1
P -1 -2 -2 -1 -3 -1 -1 -2 -2 -3 -3 -1 -2 -4  7 -1 -1 -4 -3 -2
S  1 -1  1  0 -1  0  0  0 -1 -2 -2  0 -1 -2 -1  4  1 -3 -2 -2
T  0 -1  0 -1 -1 -1 -1 -2 -2 -1 -1 -1 -1 -2 -1  1  5 -2 -2  0
W -3 -3 -4 -4 -2 -2 -3 -2 -2 -3 -2 -3 -1  1 -4 -3 -2 11  2 -3
Y -2 -2 -2 -3 -2 -1 -2 -3  2 -1 -1 -2 -1  3 -3 -2 -2  2  7 -1
V  0 -3 -3 -3 -1 -2 -2 -3 -3  3  1 -2  1 -1 -2 -2  0 -3 -1  4
"""
B62 = {}
_hdr = _B.strip().splitlines()[0].split()
for line in _B.strip().splitlines()[1:]:
    parts = line.split(); a = parts[0]
    B62[a] = { _hdr[i]: int(parts[i+1]) for i in range(len(_hdr)) }

def grantham(a,b):
    if a==b: return 0
    return _GR.get(a,{}).get(b, _GR.get(b,{}).get(a, 0))

def blosum62(a,b):
    return B62.get(a,{}).get(b, B62.get(b,{}).get(a, 0))

# ----- windows & enc
def mut_index_in_window(row, flank=60):
    i0 = int(row["cds_pos"]) - 1
    cs = (i0 // 3) * 3
    center = cs + 1
    return flank + (i0 - center)

def extract_patch(win: str, j: int, R: int = 30):
    s = str(win)
    if not s or j < 0 or j >= len(s): return ""
    a, b = max(0, j-R), min(len(s), j+R+1)
    left_pad  = "N" * (R - (j - a))
    right_pad = "N" * (R - (b - j - 1))
    return left_pad + s[a:b] + right_pad

VOCAB = {"A":0,"C":1,"G":2,"T":3,"N":4}
def onehot(seq: str):
    arr = np.zeros((5, len(seq)), dtype=np.float32)
    for i,ch in enumerate(seq):
        arr[VOCAB.get(ch,4), i] = 1.0
    return arr

def aa_scalars(wt_codon, mut_codon):
    a = CODON2AA.get(str(wt_codon), "X"); b = CODON2AA.get(str(mut_codon), "X")
    aa_same = float(a==b)
    if a=="X" or b=="X":
        return np.array([aa_same, 0.0, 0.0], dtype=np.float32)
    g = grantham(a,b) / 215.0
    bl = (blosum62(a,b) + 4) / 15.0
    return np.array([aa_same, g, bl], dtype=np.float32)

# ----- pairing logic
def make_pairs_trimatched(train_df: pd.DataFrame, seed=SEED):
    rs = np.random.RandomState(seed)
    pairs=[]; used=set()
    train_df = train_df.copy()
    train_df["center_base"] = train_df["tri_center"].str[1]
    for g, sub in train_df.groupby("gene"):
        pos = sub[sub["label"]==1]
        neg = sub[sub["label"]==0].copy()
        # exact tri
        for tri, grp in pos.groupby("tri_center"):
            pool = neg[(neg["tri_center"]==tri) & (~neg.index.isin(used))]
            if pool.empty: continue
            k = min(len(grp), len(pool))
            pidx = rs.choice(grp.index, size=k, replace=False)
            nidx = rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx, nidx)); used.update(nidx)
        # fallback base
        leftover = pos.loc[~pos.index.isin([p for p,_ in pairs])]
        for base, grp in leftover.groupby("center_base"):
            pool = neg[(neg["tri_center"].str[1]==base) & (~neg.index.isin(used))]
            if pool.empty: continue
            k = min(len(grp), len(pool))
            pidx = rs.choice(grp.index, size=k, replace=False)
            nidx = rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx, nidx)); used.update(nidx)
    return pairs

def make_pairs_for_test(test_df: pd.DataFrame, seed=SEED):
    rs = np.random.RandomState(seed)
    pairs=[]; used=set()
    test_df = test_df.copy()
    test_df["center_base"] = test_df["tri_center"].str[1]
    pos = test_df[test_df["label"]==1]
    neg = test_df[test_df["label"]==0].copy()
    for tri, grp in pos.groupby("tri_center"):
        pool = neg[(neg["tri_center"]==tri) & (~neg.index.isin(used))]
        if pool.empty: continue
        k = min(len(grp), len(pool))
        pidx = rs.choice(grp.index, size=k, replace=False)
        nidx = rs.choice(pool.index, size=k, replace=False)
        pairs += list(zip(pidx, nidx)); used.update(nidx)
    leftover = pos.loc[~pos.index.isin([p for p,_ in pairs])]
    for base, grp in leftover.groupby("center_base"):
        pool = neg[(neg["tri_center"].str[1]==base) & (~neg.index.isin(used))]
        if pool.empty: continue
        k = min(len(grp), len(pool))
        pidx = rs.choice(grp.index, size=k, replace=False)
        nidx = rs.choice(pool.index, size=k, replace=False)
        pairs += list(zip(pidx, nidx)); used.update(nidx)
    return pairs

# ----- datasets
class PairTrainDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, pairs, R=30):
        self.df = frame; self.pairs = pairs; self.R = R
    def __len__(self): return len(self.pairs)
    def _patch_feats(self, r):
        j = mut_index_in_window(r, flank=60)
        mut = extract_patch(r["window_mut"], j, self.R)
        wt  = extract_patch(r["window_wt"],  j, self.R)
        scal = aa_scalars(r["wt_codon"], r["mut_codon"])
        return torch.from_numpy(onehot(mut)), torch.from_numpy(onehot(wt)), torch.from_numpy(scal)
    def __getitem__(self, k):
        ip, ineg = self.pairs[k]
        rp, rn = self.df.loc[ip], self.df.loc[ineg]
        xm_p, xw_p, s_p = self._patch_feats(rp)
        xm_n, xw_n, s_n = self._patch_feats(rn)
        return xm_p, xw_p, s_p, xm_n, xw_n, s_n

class VariantEvalDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, R=30):
        self.df = frame.reset_index()
        self.R = R
        self.items=[]
        for _, r in self.df.iterrows():
            j = mut_index_in_window(r, flank=60)
            mut = extract_patch(r["window_mut"], j, self.R)
            wt  = extract_patch(r["window_wt"],  j, self.R)
            scal = aa_scalars(r["wt_codon"], r["mut_codon"])
            self.items.append((onehot(mut), onehot(wt), scal, float(r["label"])))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        mut, wt, scal, y = self.items[i]
        return torch.from_numpy(mut), torch.from_numpy(wt), torch.from_numpy(scal), torch.tensor(y, dtype=torch.float32)

# ----- model
class SiameseDeltaCNN(nn.Module):
    def __init__(self, in_ch=5, hid=96, scal_dim=3):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv1d(in_ch, 64,  kernel_size=11, padding=5),
            nn.ReLU(),
            nn.Conv1d(64, 96, kernel_size=7,  padding=3),
            nn.ReLU(),
            nn.Conv1d(96, 96, kernel_size=3,  padding=1),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1)
        )
        self.norm = nn.LayerNorm(96)
        self.head = nn.Sequential(
            nn.Linear(96, 96),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(96, 1)
        )
        self.scal_head = nn.Linear(scal_dim, 1)
    def encode(self, x):
        h = self.enc(x).squeeze(-1)
        return self.norm(h)
    def score(self, xm, xw, scalars):
        hm = self.encode(xm); hw = self.encode(xw)
        core = self.head(hm - hw).squeeze(-1)
        scal = self.scal_head(scalars).squeeze(-1)
        return core + scal

def run_loco_gene_ranknet(df, splits, epochs=16, batch=96, lr=2e-3, R=30, device=None, tau=1.0):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    all_rows=[]
    bce = nn.BCEWithLogitsLoss()
    for fold, sub in splits.groupby("fold"):
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr = dd[dd["split"]=="train"].copy()
        te = dd[dd["split"]=="test"].copy()
        if tr.empty or te.empty:
            continue

        pairs = make_pairs_trimatched(tr, seed=SEED)
        if not pairs:
            continue

        dtrain = PairTrainDataset(tr, pairs, R=R)
        dtest  = VariantEvalDataset(te, R=R)

        model = SiameseDeltaCNN().to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

        def loss_ranknet(xm_p, xw_p, s_p, xm_n, xw_n, s_n, tau=1.0):
            sp = model.score(xm_p, xw_p, s_p) / tau
            sn = model.score(xm_n, xw_n, s_n) / tau
            diff = sp - sn
            tgt = torch.ones_like(diff)
            return bce(diff, tgt)

        loader = DataLoader(dtrain, batch_size=batch, shuffle=True, num_workers=2, pin_memory=True, drop_last=False)
        for ep in range(epochs):
            model.train()
            for xm_p, xw_p, s_p, xm_n, xw_n, s_n in loader:
                xm_p, xw_p, s_p = xm_p.to(device), xw_p.to(device), s_p.to(device)
                xm_n, xw_n, s_n = xm_n.to(device), xw_n.to(device), s_n.to(device)
                opt.zero_grad()
                l = loss_ranknet(xm_p, xw_p, s_p, xm_n, xw_n, s_n, tau=tau)
                l.backward(); opt.step()

        # eval (per variant)
        model.eval()
        logits=[]; ys=[]
        test_loader = DataLoader(dtest, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
        with torch.no_grad():
            for xm, xw, s, y in test_loader:
                xm, xw, s = xm.to(device), xw.to(device), s.to(device)
                logit = model.score(xm, xw, s)
                logits.append(logit.cpu().numpy()); ys.append(y.numpy())
        y_log = np.concatenate(logits); y_true = np.concatenate(ys).astype(int)
        p = 1/(1+np.exp(-y_log))
        auroc = roc_auc_score(y_true, p) if len(np.unique(y_true))>1 else np.nan
        pr    = average_precision_score(y_true, p) if len(np.unique(y_true))>1 else np.nan
        brier = brier_score_loss(y_true, p)
        pair_acc_variant = float(((y_log>0)&(y_true==1) | (y_log<=0)&(y_true==0)).mean())

        # tri-matched pair acc on TEST
        test_pairs = make_pairs_for_test(te, seed=SEED)
        pair_acc_ctx = np.nan
        if test_pairs:
            idx_order = te.reset_index()["index"].values
            idx_to_logit = {idx_order[i]: y_log[i] for i in range(len(idx_order))}
            wins=0; total=0
            for ip, ineg in test_pairs:
                if ip in idx_to_logit and ineg in idx_to_logit:
                    wins += int(idx_to_logit[ip] > idx_to_logit[ineg]); total += 1
            pair_acc_ctx = wins/total if total>0 else np.nan

        gname = te["gene"].unique().tolist()[0]
        all_rows.append({
            "fold": list(sub["fold"])[0], "gene": gname, "n": int(len(y_true)),
            "auroc": round(auroc,4) if not np.isnan(auroc) else np.nan,
            "pr_auc": round(pr,4)   if not np.isnan(pr) else np.nan,
            "brier": round(brier,4),
            "pair_acc_variant": round(pair_acc_variant,4),
            "pair_acc_ctx": round(pair_acc_ctx,4) if not (isinstance(pair_acc_ctx,float) and np.isnan(pair_acc_ctx)) else np.nan
        })
    return pd.DataFrame(all_rows).sort_values("gene")

device = "cuda" if torch.cuda.is_available() else "cpu"
SIAM_DIR = ARTI_DIR / f"siamese_pair_ranknet_gene_{time.strftime('%Y%m%d_%H%M%S')}"
SIAM_DIR.mkdir(parents=True, exist_ok=True)

summary = run_loco_gene_ranknet(df, spl_g, epochs=16, batch=96, lr=2e-3, R=30, device=device, tau=1.0)
summary.to_csv(SIAM_DIR/"metrics_siamese_pair_ranknet_loco_gene.csv", index=False)

print("Device:", device)
print("Run dir:", SIAM_DIR)
display(summary)


Device: cuda
Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/siamese_pair_ranknet_gene_20250816_214829


,fold,gene,n,auroc,pr_auc,brier,pair_acc_variant,pair_acc_ctx
0,loco_gene:EGFR,EGFR,39,0.7800,0.6488,0.2045,0.6667,0.8462
1,loco_gene:KRAS,KRAS,11,0.8750,0.9594,0.1972,0.8182,1.0000
2,loco_gene:PIK3CA,PIK3CA,24,0.4815,0.7761,0.2957,0.3333,0.6667
3,loco_gene:TP53,TP53,197,0.5172,0.3601,0.2657,0.4416,0.5507


**Nice** — EGFR/KRAS are cooking, *TP53’s the holdout*. Before we start inventing wild new features, let’s do the boring-but-powerful thing: **stabilize with a 5-seed bag** of the same LOCO-GENE RankNet model (tri-matched training + bio scalars). If TP53 is just variance-noisy, the bag should lift it; if it stays flat, we’ll escalate (bigger patch R, hard-negative mining).

### I6 — 5-seed bag (LOCO-GENE), RankNet + bio scalars, tri-matched training
 Produces averaged logits across seeds per fold, then reports metrics (incl. tri-matched pair_acc on TEST).

In [4]:
# I6 — 5-seed bag (LOCO-GENE), RankNet + bio scalars, tri-matched training
# Produces averaged logits across seeds per fold, then reports metrics (incl. tri-matched pair_acc on TEST).
import os, glob, time, random, math
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# ===== setup & load =====
BASE = Path("/content/drive/MyDrive/Cancer_gene_database")
DATA = BASE/"data"; ARTI = BASE/"artifacts"
P_RES = DATA/"masterseq_model_ready_plus_resolved.parquet"
runs = sorted(glob.glob(str(ARTI / "sentiment_run_plus_variants_*"))); assert runs
RUN_DIR = Path(runs[-1]); SPLITS_G = RUN_DIR/"splits_loco_gene.csv"

df = pd.read_parquet(P_RES).dropna(subset=[
    "variant_id","gene","label","window_mut","window_wt",
    "wt_codon","mut_codon","ref","alt","cds_pos","tri_center"
]).copy()
for c in ["variant_id","gene","ref","alt","wt_codon","mut_codon","window_mut","window_wt","tri_center"]:
    df[c] = df[c].astype(str).str.upper()
df["label"] = df["label"].astype(int)
df["cds_pos"] = pd.to_numeric(df["cds_pos"], errors="coerce").astype("Int64")
spl_g = pd.read_csv(SPLITS_G)

device = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS = [1337, 2401, 4242, 7777, 10101]
EPOCHS = 14
BATCH  = 96
LR     = 2e-3
R      = 30  # edit-centered patch radius

# ===== helpers =====
CODON2AA = { "TTT":"F","TTC":"F","TTA":"L","TTG":"L","CTT":"L","CTC":"L","CTA":"L","CTG":"L",
 "ATT":"I","ATC":"I","ATA":"I","ATG":"M","GTT":"V","GTC":"V","GTA":"V","GTG":"V",
 "TCT":"S","TCC":"S","TCA":"S","TCG":"S","CCT":"P","CCC":"P","CCA":"P","CCG":"P",
 "ACT":"T","ACC":"T","ACA":"T","ACG":"T","GCT":"A","GCC":"A","GCA":"A","GCG":"A",
 "TAT":"Y","TAC":"Y","TAA":"*","TAG":"*","CAT":"H","CAC":"H","CAA":"Q","CAG":"Q",
 "AAT":"N","AAC":"N","AAA":"K","AAG":"K","GAT":"D","GAC":"D","GAA":"E","GAG":"E",
 "TGT":"C","TGC":"C","TGA":"*","TGG":"W","CGT":"R","CGC":"R","CGA":"R","CGG":"R",
 "AGT":"S","AGC":"S","AGA":"R","AGG":"R","GGT":"G","GGC":"G","GGA":"G","GGG":"G" }
_GR = {'A':{'A':0,'R':112,'N':111,'D':126,'C':195,'Q':91,'E':107,'G':60,'H':86,'I':94,'L':96,'K':106,'M':84,'F':113,'P':27,'S':99,'T':58,'W':148,'Y':112,'V':64},
'R':{'A':112,'R':0,'N':86,'D':96,'C':180,'Q':43,'E':54,'G':125,'H':29,'I':97,'L':102,'K':26,'M':91,'F':97,'P':103,'S':110,'T':71,'W':101,'Y':77,'V':96},
'N':{'A':111,'R':86,'N':0,'D':23,'C':139,'Q':68,'E':42,'G':80,'H':68,'I':149,'L':153,'K':94,'M':142,'F':158,'P':91,'S':46,'T':65,'W':174,'Y':143,'V':133},
'D':{'A':126,'R':96,'N':23,'D':0,'C':154,'Q':61,'E':45,'G':94,'H':81,'I':168,'L':172,'K':101,'M':160,'F':177,'P':108,'S':65,'T':85,'W':181,'Y':160,'V':152},
'C':{'A':195,'R':180,'N':139,'D':154,'C':0,'Q':154,'E':170,'G':159,'H':174,'I':198,'L':198,'K':202,'M':196,'F':205,'P':169,'S':112,'T':149,'W':215,'Y':194,'V':192},
'Q':{'A':91,'R':43,'N':68,'D':61,'C':154,'Q':0,'E':29,'G':87,'H':24,'I':109,'L':113,'K':53,'M':101,'F':116,'P':76,'S':68,'T':42,'W':130,'Y':99,'V':96},
'E':{'A':107,'R':54,'N':42,'D':45,'C':170,'Q':29,'G':98,'H':40,'I':134,'L':138,'K':56,'M':126,'F':140,'P':93,'S':80,'T':65,'W':152,'Y':122,'V':121},
'G':{'A':60,'R':125,'N':80,'D':94,'C':159,'Q':87,'E':98,'H':98,'I':135,'L':138,'K':127,'M':127,'F':153,'P':42,'S':56,'T':59,'W':184,'Y':147,'V':109},
'H':{'A':86,'R':29,'N':68,'D':81,'C':174,'Q':24,'E':40,'G':98,'I':94,'L':99,'K':32,'M':87,'F':100,'P':77,'S':89,'T':47,'W':115,'Y':83,'V':84},
'I':{'A':94,'R':97,'N':149,'D':168,'C':198,'Q':109,'E':134,'G':135,'H':94,'L':5,'K':102,'M':10,'F':21,'P':95,'S':142,'T':89,'W':61,'Y':33,'V':29},
'L':{'A':96,'R':102,'N':153,'D':172,'C':198,'Q':113,'E':138,'G':138,'H':99,'I':5,'L':0,'K':107,'M':15,'F':22,'P':98,'S':145,'T':92,'W':61,'Y':36,'V':32},
'K':{'A':106,'R':26,'N':94,'D':101,'C':202,'Q':53,'E':56,'G':127,'H':32,'I':102,'L':107,'K':0,'M':95,'F':102,'P':103,'S':121,'T':78,'W':110,'Y':85,'V':97},
'M':{'A':84,'R':91,'N':142,'D':160,'C':196,'Q':101,'E':126,'G':127,'H':87,'I':10,'L':15,'K':95,'F':28,'P':87,'S':135,'T':81,'W':67,'Y':36,'V':21},
'F':{'A':113,'R':97,'N':158,'D':177,'C':205,'Q':116,'E':140,'G':153,'H':100,'I':21,'L':22,'K':102,'M':28,'P':114,'S':155,'T':103,'W':40,'Y':22,'V':50},
'P':{'A':27,'R':103,'N':91,'D':108,'C':169,'Q':76,'E':93,'G':42,'S':74,'T':38,'W':147,'Y':110,'V':68},
'S':{'A':99,'R':110,'N':46,'D':65,'C':112,'Q':68,'E':80,'G':56,'H':89,'I':142,'L':145,'K':121,'M':135,'F':155,'P':74,'T':58,'W':177,'Y':144,'V':124},
'T':{'A':58,'R':71,'N':65,'D':85,'C':149,'Q':42,'E':65,'G':59,'H':47,'I':89,'L':92,'K':78,'M':81,'F':103,'P':38,'S':58,'W':128,'Y':92,'V':69},
'W':{'A':148,'R':101,'N':174,'D':181,'C':215,'Q':130,'E':152,'G':184,'H':115,'I':61,'L':61,'K':110,'M':67,'F':40,'P':147,'S':177,'T':128,'Y':37,'V':88},
'Y':{'A':112,'R':77,'N':143,'D':160,'C':194,'Q':99,'E':122,'G':147,'H':83,'I':33,'L':36,'K':85,'M':36,'F':22,'P':110,'S':144,'T':92,'W':37,'V':55},
'V':{'A':64,'R':96,'N':133,'D':152,'C':192,'Q':96,'E':121,'G':109,'H':84,'I':29,'L':32,'K':97,'M':21,'F':50,'P':68,'S':124,'T':69,'W':88,'Y':55,'V':0}}
_B = """\
   A  R  N  D  C  Q  E  G  H  I  L  K  M  F  P  S  T  W  Y  V
A  4 -1 -2 -2  0 -1 -1  0 -2 -1 -1 -1 -1 -2 -1  1  0 -3 -2  0
R -1  5  0 -2 -3  1  0 -2  0 -3 -2  2 -1 -3 -2 -1 -1 -3 -2 -3
N -2  0  6  1 -3  0  0  0  1 -3 -3  0 -2 -3 -2  1  0 -4 -2 -3
D -2 -2  1  6 -3  0  2 -1 -1 -3 -4 -1 -3 -3 -1  0 -1 -4 -3 -3
C  0 -3 -3 -3  9 -3 -4 -3 -3 -1 -1 -3 -1 -2 -3 -1 -1 -2 -2 -1
Q -1  1  0  0 -3  5  2 -2  0 -3 -2  1  0 -3 -1  0 -1 -2 -1 -2
E -1  0  0  2 -4  2  5 -2  0 -3 -3  1 -2 -3 -1  0 -1 -3 -2 -2
G  0 -2  0 -1 -3 -2 -2  6 -2 -4 -4 -2 -3 -3 -2  0 -2 -2 -3 -3
H -2  0  1 -1 -3  0  0 -2  8 -3 -3 -1 -2 -1 -2 -1 -2 -2  2 -3
I -1 -3 -3 -3 -1 -3 -3 -4 -3  4  2 -3  1  0 -3 -2 -1 -3 -1  3
L -1 -2 -3 -4 -1 -2 -3 -4 -3  2  4 -2  2  0 -3 -2 -1 -2 -1  1
K -1  2  0 -1 -3  1  1 -2 -1 -3 -2  5 -1 -3 -1  0 -1 -3 -2 -2
M -1 -1 -2 -3 -1  0 -2 -3 -2  1  2 -1  5  0 -2 -1 -1 -1 -1  1
F -2 -3 -3 -3 -2 -3 -3 -3 -1  0  0 -3  0  6 -4 -2 -2  1  3 -1
P -1 -2 -2 -1 -3 -1 -1 -2 -2 -3 -3 -1 -2 -4  7 -1 -1 -4 -3 -2
S  1 -1  1  0 -1  0  0  0 -1 -2 -2  0 -1 -2 -1  4  1 -3 -2 -2
T  0 -1  0 -1 -1 -1 -1 -2 -2 -1 -1 -1 -1 -2 -1  1  5 -2 -2  0
W -3 -3 -4 -4 -2 -2 -3 -2 -2 -3 -2 -3 -1  1 -4 -3 -2 11  2 -3
Y -2 -2 -2 -3 -2 -1 -2 -3  2 -1 -1 -2 -1  3 -3 -2 -2  2  7 -1
V  0 -3 -3 -3 -1 -2 -2 -3 -3  3  1 -2  1 -1 -2 -2  0 -3 -1  4
"""
B62={}
_hdr=_B.strip().splitlines()[0].split()
for line in _B.strip().splitlines()[1:]:
    parts=line.split(); a=parts[0]; B62[a]={_hdr[i]:int(parts[i+1]) for i in range(len(_hdr))}
def grantham(a,b):
    if a==b: return 0
    return _GR.get(a,{}).get(b,_GR.get(b,{}).get(a,0))
def blosum62(a,b):
    return B62.get(a,{}).get(b,B62.get(b,{}).get(a,0))
def aa_scalars(wt_codon, mut_codon):
    a = CODON2AA.get(str(wt_codon), "X"); b = CODON2AA.get(str(mut_codon), "X")
    aa_same = float(a==b)
    if a=="X" or b=="X": return np.array([aa_same,0.0,0.0], dtype=np.float32)
    return np.array([aa_same, grantham(a,b)/215.0, (blosum62(a,b)+4)/15.0], dtype=np.float32)

def mut_index_in_window(row, flank=60):
    i0 = int(row["cds_pos"]) - 1; cs = (i0 // 3) * 3; center = cs + 1
    return flank + (i0 - center)
def extract_patch(win: str, j: int, R: int = 30):
    s = str(win);
    if not s or j<0 or j>=len(s): return ""
    a, b = max(0, j-R), min(len(s), j+R+1)
    left = "N" * (R - (j - a)); right = "N" * (R - (b - j - 1))
    return left + s[a:b] + right
VOCAB={"A":0,"C":1,"G":2,"T":3,"N":4}
def onehot(seq: str):
    arr = np.zeros((5,len(seq)), np.float32)
    for i,ch in enumerate(seq): arr[VOCAB.get(ch,4), i] = 1.0
    return arr

def make_pairs_trimatched(train_df: pd.DataFrame, seed=1337):
    rs=np.random.RandomState(seed); pairs=[]; used=set()
    sub=train_df.copy(); sub["center_base"]=sub["tri_center"].str[1]
    for g, grp in sub.groupby("gene"):
        pos=grp[grp["label"]==1]; neg=grp[grp["label"]==0].copy()
        for tri, pgrp in pos.groupby("tri_center"):
            pool=neg[(neg["tri_center"]==tri)&(~neg.index.isin(used))]
            if pool.empty: continue
            k=min(len(pgrp), len(pool))
            pidx=rs.choice(pgrp.index, size=k, replace=False)
            nidx=rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx,nidx)); used.update(nidx)
        leftover=pos.loc[~pos.index.isin([p for p,_ in pairs])]
        for base, pgrp in leftover.groupby("center_base"):
            pool=neg[(neg["tri_center"].str[1]==base)&(~neg.index.isin(used))]
            if pool.empty: continue
            k=min(len(pgrp), len(pool))
            pidx=rs.choice(pgrp.index, size=k, replace=False)
            nidx=rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx,nidx)); used.update(nidx)
    return pairs

def make_pairs_for_test(test_df: pd.DataFrame, seed=1337):
    rs=np.random.RandomState(seed); pairs=[]; used=set()
    sub=test_df.copy(); sub["center_base"]=sub["tri_center"].str[1]
    pos=sub[sub["label"]==1]; neg=sub[sub["label"]==0].copy()
    for tri, pgrp in pos.groupby("tri_center"):
        pool=neg[(neg["tri_center"]==tri)&(~neg.index.isin(used))]
        if pool.empty: continue
        k=min(len(pgrp), len(pool))
        pidx=rs.choice(pgrp.index, size=k, replace=False)
        nidx=rs.choice(pool.index, size=k, replace=False)
        pairs += list(zip(pidx,nidx)); used.update(nidx)
    leftover=pos.loc[~pos.index.isin([p for p,_ in pairs])]
    for base, pgrp in leftover.groupby("center_base"):
        pool=neg[(neg["tri_center"].str[1]==base)&(~neg.index.isin(used))]
        if pool.empty: continue
        k=min(len(pgrp), len(pool))
        pidx=rs.choice(pgrp.index, size=k, replace=False)
        nidx=rs.choice(pool.index, size=k, replace=False)
        pairs += list(zip(pidx,nidx)); used.update(nidx)
    return pairs

class PairTrainDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, pairs, R=30):
        self.df = frame; self.pairs = pairs; self.R = R
    def __len__(self): return len(self.pairs)
    def _patch_feats(self, r):
        j = mut_index_in_window(r, flank=60)
        mut = extract_patch(r["window_mut"], j, self.R)
        wt  = extract_patch(r["window_wt"],  j, self.R)
        scal = aa_scalars(r["wt_codon"], r["mut_codon"])
        return torch.from_numpy(onehot(mut)), torch.from_numpy(onehot(wt)), torch.from_numpy(scal)
    def __getitem__(self, k):
        ip, ineg = self.pairs[k]
        rp, rn = self.df.loc[ip], self.df.loc[ineg]
        xm_p, xw_p, s_p = self._patch_feats(rp)
        xm_n, xw_n, s_n = self._patch_feats(rn)
        return xm_p, xw_p, s_p, xm_n, xw_n, s_n

class VariantEvalDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, R=30):
        self.df = frame.reset_index()
        self.R = R
        self.items=[]
        for _, r in self.df.iterrows():
            j = mut_index_in_window(r, flank=60)
            mut = extract_patch(r["window_mut"], j, self.R)
            wt  = extract_patch(r["window_wt"],  j, self.R)
            scal = aa_scalars(r["wt_codon"], r["mut_codon"])
            self.items.append((onehot(mut), onehot(wt), scal, float(r["label"])))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        mut, wt, scal, y = self.items[i]
        return torch.from_numpy(mut), torch.from_numpy(wt), torch.from_numpy(scal), torch.tensor(y, dtype=torch.float32)

class SiameseDeltaCNN(nn.Module):
    def __init__(self, in_ch=5, hid=96, scal_dim=3):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv1d(in_ch, 64, kernel_size=11, padding=5), nn.ReLU(),
            nn.Conv1d(64, 96, kernel_size=7, padding=3),     nn.ReLU(),
            nn.Conv1d(96, 96, kernel_size=3, padding=1),     nn.ReLU(),
            nn.AdaptiveMaxPool1d(1)
        )
        self.norm = nn.LayerNorm(96)
        self.head = nn.Sequential(nn.Linear(96,96), nn.ReLU(), nn.Dropout(0.1), nn.Linear(96,1))
        self.scal_head = nn.Linear(scal_dim, 1)
    def encode(self, x):
        h = self.enc(x).squeeze(-1)
        return self.norm(h)
    def score(self, xm, xw, scalars):
        hm = self.encode(xm); hw = self.encode(xw)
        core = self.head(hm - hw).squeeze(-1)
        scal = self.scal_head(scalars).squeeze(-1)
        return core + scal

def train_and_score_one_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    rows=[]
    for fold, sub in spl_g.groupby("fold"):
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr = dd[dd["split"]=="train"].copy()
        te = dd[dd["split"]=="test"].copy()
        if tr.empty or te.empty:
            continue
        pairs = make_pairs_trimatched(tr, seed=seed)
        if not pairs:
            continue
        dtrain = PairTrainDataset(tr, pairs, R=R)
        dtest  = VariantEvalDataset(te, R=R)
        model = SiameseDeltaCNN().to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        bce = nn.BCEWithLogitsLoss()
        def ranknet_batch(xm_p, xw_p, s_p, xm_n, xw_n, s_n):
            sp = model.score(xm_p, xw_p, s_p)
            sn = model.score(xm_n, xw_n, s_n)
            diff = sp - sn
            return bce(diff, torch.ones_like(diff))
        loader = DataLoader(dtrain, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
        for ep in range(EPOCHS):
            model.train()
            for xm_p, xw_p, s_p, xm_n, xw_n, s_n in loader:
                xm_p, xw_p, s_p = xm_p.to(device), xw_p.to(device), s_p.to(device)
                xm_n, xw_n, s_n = xm_n.to(device), xw_n.to(device), s_n.to(device)
                opt.zero_grad(); loss = ranknet_batch(xm_p, xw_p, s_p, xm_n, xw_n, s_n); loss.backward(); opt.step()
        # score test
        model.eval()
        logits=[]; ys=[]; vids=[]; idxs=[]
        test_loader = DataLoader(dtest, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
        with torch.no_grad():
            for (xm, xw, s, y), irow in zip(test_loader, te.reset_index()["index"].values.reshape(-1,128)):
                xm, xw, s = xm.to(device), xw.to(device), s.to(device)
                log = model.score(xm, xw, s).cpu().numpy()
                logits.append(log); ys.append(y.numpy())
        y_log = np.concatenate(logits); y_true = np.concatenate(ys).astype(int)
        # stash per-row using deterministic order = te.reset_index() order
        out = te.reset_index()[["index","variant_id","gene","label"]].copy()
        out["fold"] = fold; out["seed"] = seed; out["logit"] = y_log
        rows.append(out)
    return pd.concat(rows, ignore_index=True)

# ---- train across seeds & bag
preds_all = []
for sd in SEEDS:
    preds_all.append(train_and_score_one_seed(sd))
preds_all = pd.concat(preds_all, ignore_index=True)

# average logits across seeds
avg = (preds_all.groupby(["fold","index","variant_id","gene","label"], as_index=False)
                 .agg(logit_mean=("logit","mean")))

# metrics per gene (each LOCO-GENE fold is one gene)
rows=[]
for fold, sub in avg.groupby("fold"):
    gname = sub["gene"].unique().tolist()[0]
    y = sub["label"].values
    p = 1/(1+np.exp(-sub["logit_mean"].values))
    auroc = roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan
    pr    = average_precision_score(y,p) if len(np.unique(y))>1 else np.nan
    brier = brier_score_loss(y,p)
    pair_acc_variant = float((((sub["logit_mean"]>0)&(sub["label"]==1)) | ((sub["logit_mean"]<=0)&(sub["label"]==0))).mean())
    # tri-matched pair acc on TEST
    # reconstruct test df for this fold to make pairs
    dd = df.merge(spl_g[spl_g["fold"]==fold], on=["variant_id","gene"], how="inner")
    te = dd[dd["split"]=="test"].copy()
    pairs = make_pairs_for_test(te, seed=1337)
    idx_to_logit = {r["index"]: r["logit_mean"] for _, r in sub.iterrows()}
    wins=0; total=0
    for ip, ineg in pairs:
        if ip in idx_to_logit and ineg in idx_to_logit:
            wins += int(idx_to_logit[ip] > idx_to_logit[ineg]); total += 1
    pair_acc_ctx = wins/total if total>0 else np.nan
    rows.append({
        "fold": fold, "gene": gname, "n": int(len(sub)),
        "auroc": round(auroc,4) if not np.isnan(auroc) else np.nan,
        "pr_auc": round(pr,4)   if not np.isnan(pr) else np.nan,
        "brier": round(brier,4),
        "pair_acc_variant": round(pair_acc_variant,4),
        "pair_acc_ctx": round(pair_acc_ctx,4) if not (isinstance(pair_acc_ctx,float) and np.isnan(pair_acc_ctx)) else np.nan
    })
summary = pd.DataFrame(rows).sort_values("gene")

# save
OUT_DIR = ARTI / f"siamese_pair_ranknet_gene_bag_{time.strftime('%Y%m%d_%H%M%S')}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
preds_all.to_csv(OUT_DIR/"preds_by_seed.csv", index=False)
avg.to_csv(OUT_DIR/"preds_bag_averaged.csv", index=False)
summary.to_csv(OUT_DIR/"metrics_bagged_loco_gene.csv", index=False)

print("Device:", device)
print("Run dir:", OUT_DIR)
display(summary)


ValueError: cannot reshape array of size 39 into shape (128)

## Code block fix for the error

In [5]:
# HOTFIX for I6 — remove reshape/zip; align logits to te.reset_index() order directly

def train_and_score_one_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    rows=[]
    for fold, sub in spl_g.groupby("fold"):
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr = dd[dd["split"]=="train"].copy()
        te = dd[dd["split"]=="test"].copy()
        if tr.empty or te.empty:
            continue

        pairs = make_pairs_trimatched(tr, seed=seed)
        if not pairs:
            continue

        dtrain = PairTrainDataset(tr, pairs, R=R)
        dtest  = VariantEvalDataset(te, R=R)

        model = SiameseDeltaCNN().to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        bce = nn.BCEWithLogitsLoss()

        def ranknet_batch(xm_p, xw_p, s_p, xm_n, xw_n, s_n):
            sp = model.score(xm_p, xw_p, s_p)
            sn = model.score(xm_n, xw_n, s_n)
            diff = sp - sn
            return bce(diff, torch.ones_like(diff))

        loader = DataLoader(dtrain, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
        for ep in range(EPOCHS):
            model.train()
            for xm_p, xw_p, s_p, xm_n, xw_n, s_n in loader:
                xm_p, xw_p, s_p = xm_p.to(device), xw_p.to(device), s_p.to(device)
                xm_n, xw_n, s_n = xm_n.to(device), xw_n.to(device), s_n.to(device)
                opt.zero_grad()
                loss = ranknet_batch(xm_p, xw_p, s_p, xm_n, xw_n, s_n)
                loss.backward(); opt.step()

        # ---- score test, aligned to te.reset_index() order ----
        model.eval()
        logits=[]; ys=[]
        test_loader = DataLoader(dtest, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
        with torch.no_grad():
            for xm, xw, s, y in test_loader:
                xm, xw, s = xm.to(device), xw.to(device), s.to(device)
                logits.append(model.score(xm, xw, s).cpu().numpy())
                ys.append(y.numpy())
        y_log  = np.concatenate(logits)         # same order as te.reset_index()
        y_true = np.concatenate(ys).astype(int) # not used here, but kept for sanity

        te_reset = te.reset_index()  # has "index" = original index we use for tri-pairing
        out = te_reset[["index","variant_id","gene","label"]].copy()
        out["fold"] = fold
        out["seed"] = seed
        out["logit"] = y_log
        rows.append(out)

    return pd.concat(rows, ignore_index=True)

# ---- re-run the bagging with the fixed function ----
preds_all = []
for sd in SEEDS:
    preds_all.append(train_and_score_one_seed(sd))
preds_all = pd.concat(preds_all, ignore_index=True)

# average logits across seeds
avg = (preds_all.groupby(["fold","index","variant_id","gene","label"], as_index=False)
                 .agg(logit_mean=("logit","mean")))

# metrics per gene (each LOCO-GENE fold is one gene)
rows=[]
for fold, sub in avg.groupby("fold"):
    gname = sub["gene"].unique().tolist()[0]
    y = sub["label"].values
    p = 1/(1+np.exp(-sub["logit_mean"].values))
    auroc = roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan
    pr    = average_precision_score(y,p) if len(np.unique(y))>1 else np.nan
    brier = brier_score_loss(y,p)
    pair_acc_variant = float((((sub["logit_mean"]>0)&(sub["label"]==1)) | ((sub["logit_mean"]<=0)&(sub["label"]==0))).mean())

    # tri-matched pair acc on TEST for this fold
    dd = df.merge(spl_g[spl_g["fold"]==fold], on=["variant_id","gene"], how="inner")
    te = dd[dd["split"]=="test"].copy()
    pairs = make_pairs_for_test(te, seed=SEEDS[0])
    idx_to_logit = {r["index"]: r["logit_mean"] for _, r in sub.iterrows()}
    wins=0; total=0
    for ip, ineg in pairs:
        if ip in idx_to_logit and ineg in idx_to_logit:
            wins += int(idx_to_logit[ip] > idx_to_logit[ineg]); total += 1
    pair_acc_ctx = wins/total if total>0 else np.nan

    rows.append({
        "fold": fold, "gene": gname, "n": int(len(sub)),
        "auroc": round(auroc,4) if not np.isnan(auroc) else np.nan,
        "pr_auc": round(pr,4)   if not np.isnan(pr) else np.nan,
        "brier": round(brier,4),
        "pair_acc_variant": round(pair_acc_variant,4),
        "pair_acc_ctx": round(pair_acc_ctx,4) if not (isinstance(pair_acc_ctx,float) and np.isnan(pair_acc_ctx)) else np.nan
    })
summary = pd.DataFrame(rows).sort_values("gene")

# save receipts
OUT_DIR = ARTI / f"siamese_pair_ranknet_gene_bag_{time.strftime('%Y%m%d_%H%M%S')}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
preds_all.to_csv(OUT_DIR/"preds_by_seed.csv", index=False)
avg.to_csv(OUT_DIR/"preds_bag_averaged.csv", index=False)
summary.to_csv(OUT_DIR/"metrics_bagged_loco_gene.csv", index=False)

print("Device:", device)
print("Run dir:", OUT_DIR)
display(summary)


Device: cuda
Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/siamese_pair_ranknet_gene_bag_20250816_215844


,fold,gene,n,auroc,pr_auc,brier,pair_acc_variant,pair_acc_ctx
0,loco_gene:EGFR,EGFR,39,0.7086,0.6811,0.2196,0.7436,0.6154
1,loco_gene:KRAS,KRAS,11,0.7500,0.8755,0.2427,0.4545,0.5000
2,loco_gene:PIK3CA,PIK3CA,24,0.6296,0.8528,0.2762,0.3750,0.3333
3,loco_gene:TP53,TP53,197,0.5050,0.4049,0.2398,0.6193,0.4638


**facts:**

* bagging helped EGFR variance (pair_acc_variant ↑), but **TP53 stayed soft** (AUROC ~0.50, ctx-pair ~0.46).

* that screams “not enough receptive field for the edit+motif interaction” rather than “no signal at all” (since LOCO-SITE was strong).

next single move: **widen the patch** so the Δ-CNN can see a bit further than immediate codon neighborhood.

### I7 — wider patch (R=45) + same RankNet + 5-seed bag (LOCO-GENE)

In [6]:
# I7 — Wider patch (R=45) with RankNet + bio scalars, 5-seed bag (LOCO-GENE)

# If you just ran I6 in this runtime, all functions/classes are already defined.
# We only re-run the bagging with R=45; otherwise we redefine the minimal helpers.

import os, glob, time, random, math
from pathlib import Path
import numpy as np, pandas as pd
import torch
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# --- reuse globals from I6 if present; otherwise reload quickly ---
BASE = Path("/content/drive/MyDrive/Cancer_gene_database")
DATA = BASE/"data"; ARTI = BASE/"artifacts"
P_RES = DATA/"masterseq_model_ready_plus_resolved.parquet"
runs = sorted(glob.glob(str(ARTI / "sentiment_run_plus_variants_*"))); assert runs
RUN_DIR = Path(runs[-1]); SPLITS_G = RUN_DIR/"splits_loco_gene.csv"

# Load data/splits (safe if already in memory)
df = pd.read_parquet(P_RES).dropna(subset=[
    "variant_id","gene","label","window_mut","window_wt",
    "wt_codon","mut_codon","ref","alt","cds_pos","tri_center"
]).copy()
for c in ["variant_id","gene","ref","alt","wt_codon","mut_codon","window_mut","window_wt","tri_center"]:
    df[c] = df[c].astype(str).str.upper()
df["label"] = df["label"].astype(int)
df["cds_pos"] = pd.to_numeric(df["cds_pos"], errors="coerce").astype("Int64")
spl_g = pd.read_csv(SPLITS_G)

# --- pull needed pieces from I6 scope (if missing, raise) ---
assert 'PairTrainDataset' in globals() and 'VariantEvalDataset' in globals() and 'SiameseDeltaCNN' in globals(), \
    "Please run I6 first to define datasets/model."
assert 'make_pairs_trimatched' in globals() and 'make_pairs_for_test' in globals(), \
    "Please run I6 first to define pairing helpers."

device = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS = [1337, 2401, 4242, 7777, 10101]
EPOCHS = 14
BATCH  = 96
LR     = 2e-3
R      = 45   # <<< wider context (from 30 -> 45)

# Overwrite dataset constructors to use the new R without re-defining classes
def build_datasets(tr, te, R=R):
    return PairTrainDataset(tr, make_pairs_trimatched(tr, seed=SEEDS[0]), R=R), VariantEvalDataset(te, R=R)

import torch.nn as nn
def train_and_score_one_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    rows=[]
    for fold, sub in spl_g.groupby("fold"):
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr = dd[dd["split"]=="train"].copy()
        te = dd[dd["split"]=="test"].copy()
        if tr.empty or te.empty:
            continue

        pairs = make_pairs_trimatched(tr, seed=seed)
        if not pairs:
            continue

        dtrain = PairTrainDataset(tr, pairs, R=R)
        dtest  = VariantEvalDataset(te, R=R)
        model = SiameseDeltaCNN().to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        bce = nn.BCEWithLogitsLoss()

        def ranknet_batch(xm_p, xw_p, s_p, xm_n, xw_n, s_n):
            sp = model.score(xm_p, xw_p, s_p)
            sn = model.score(xm_n, xw_n, s_n)
            return bce(sp - sn, torch.ones_like(sp))

        from torch.utils.data import DataLoader
        loader = DataLoader(dtrain, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
        for _ in range(EPOCHS):
            model.train()
            for xm_p, xw_p, s_p, xm_n, xw_n, s_n in loader:
                xm_p, xw_p, s_p = xm_p.to(device), xw_p.to(device), s_p.to(device)
                xm_n, xw_n, s_n = xm_n.to(device), xw_n.to(device), s_n.to(device)
                opt.zero_grad(); loss = ranknet_batch(xm_p, xw_p, s_p, xm_n, xw_n, s_n); loss.backward(); opt.step()

        # score test in te.reset_index() order
        model.eval()
        logits=[]; ys=[]
        test_loader = DataLoader(dtest, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
        with torch.no_grad():
            for xm, xw, s, y in test_loader:
                xm, xw, s = xm.to(device), xw.to(device), s.to(device)
                logits.append(model.score(xm, xw, s).cpu().numpy())
                ys.append(y.numpy())
        y_log  = np.concatenate(logits)
        te_reset = te.reset_index()[["index","variant_id","gene","label"]].copy()
        te_reset["fold"] = list(sub["fold"])[0]
        te_reset["seed"] = seed
        te_reset["logit"] = y_log
        rows.append(te_reset)
    return pd.concat(rows, ignore_index=True)

# ---- run bag
preds_all = []
for sd in SEEDS:
    preds_all.append(train_and_score_one_seed(sd))
preds_all = pd.concat(preds_all, ignore_index=True)

avg = (preds_all.groupby(["fold","index","variant_id","gene","label"], as_index=False)
                 .agg(logit_mean=("logit","mean")))

rows=[]
for fold, sub in avg.groupby("fold"):
    gname = sub["gene"].unique().tolist()[0]
    y = sub["label"].values
    p = 1/(1+np.exp(-sub["logit_mean"].values))
    auroc = roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan
    pr    = average_precision_score(y,p) if len(np.unique(y))>1 else np.nan
    brier = brier_score_loss(y,p)
    pair_acc_variant = float((((sub["logit_mean"]>0)&(sub["label"]==1)) | ((sub["logit_mean"]<=0)&(sub["label"]==0))).mean())

    # tri-matched pair acc on TEST
    dd = df.merge(spl_g[spl_g["fold"]==fold], on=["variant_id","gene"], how="inner")
    te = dd[dd["split"]=="test"].copy()
    pairs = make_pairs_for_test(te, seed=SEEDS[0])
    idx_to_logit = {r["index"]: r["logit_mean"] for _, r in sub.iterrows()}
    wins=0; total=0
    for ip, ineg in pairs:
        if ip in idx_to_logit and ineg in idx_to_logit:
            wins += int(idx_to_logit[ip] > idx_to_logit[ineg]); total += 1
    pair_acc_ctx = wins/total if total>0 else np.nan

    rows.append({
        "fold": fold, "gene": gname, "n": int(len(sub)),
        "auroc": round(auroc,4) if not np.isnan(auroc) else np.nan,
        "pr_auc": round(pr,4)   if not np.isnan(pr) else np.nan,
        "brier": round(brier,4),
        "pair_acc_variant": round(pair_acc_variant,4),
        "pair_acc_ctx": round(pair_acc_ctx,4) if not (isinstance(pair_acc_ctx,float) and np.isnan(pair_acc_ctx)) else np.nan
    })
summary = pd.DataFrame(rows).sort_values("gene")

OUT_DIR = ARTI / f"siamese_pair_ranknet_gene_bag_R45_{time.strftime('%Y%m%d_%H%M%S')}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
preds_all.to_csv(OUT_DIR/"preds_by_seed.csv", index=False)
avg.to_csv(OUT_DIR/"preds_bag_averaged.csv", index=False)
summary.to_csv(OUT_DIR/"metrics_bagged_loco_gene.csv", index=False)

print("Device:", device)
print("Patch radius R:", R)
print("Run dir:", OUT_DIR)
display(summary)


Device: cuda
Patch radius R: 45
Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/siamese_pair_ranknet_gene_bag_R45_20250816_220443


,fold,gene,n,auroc,pr_auc,brier,pair_acc_variant,pair_acc_ctx
0,loco_gene:EGFR,EGFR,39,0.6286,0.5314,0.2295,0.6667,0.5385
1,loco_gene:KRAS,KRAS,11,0.5417,0.7390,0.2692,0.2727,0.5000
2,loco_gene:PIK3CA,PIK3CA,24,0.7222,0.9023,0.2691,0.3750,0.6667
3,loco_gene:TP53,TP53,197,0.6442,0.5179,0.2322,0.6548,0.6957


**that’s the move** — **R=45 rescued TP53** (AUROC 0.64, ctx-pair ~0.70) but it dinged **EGFR**. smells like edit signal living at two spatial scales. let’s not guess per-gene; keep it clean + generalizable:

### I8 — two-scale bag (R=30 and R=45) with RankNet + bio scalars (LOCO-GENE)

same recipe, tri-matched training; we train 3 seeds at each radius (6 models total) and average logits. goal: keep TP53 gains while recovering EGFR.

In [7]:
# I8 — Two-scale bag (R=30 + R=45) for LOCO-GENE, RankNet + bio scalars, tri-matched training

import os, glob, time, random
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# ---- base paths & data
BASE = Path("/content/drive/MyDrive/Cancer_gene_database")
DATA = BASE/"data"; ARTI = BASE/"artifacts"
P_RES = DATA/"masterseq_model_ready_plus_resolved.parquet"
runs = sorted(glob.glob(str(ARTI / "sentiment_run_plus_variants_*"))); assert runs, "No split dirs."
RUN_DIR = Path(runs[-1]); SPLITS_G = RUN_DIR/"splits_loco_gene.csv"

df = pd.read_parquet(P_RES).dropna(subset=[
    "variant_id","gene","label","window_mut","window_wt",
    "wt_codon","mut_codon","ref","alt","cds_pos","tri_center"
]).copy()
for c in ["variant_id","gene","ref","alt","wt_codon","mut_codon","window_mut","window_wt","tri_center"]:
    df[c] = df[c].astype(str).str.upper()
df["label"] = df["label"].astype(int)
df["cds_pos"] = pd.to_numeric(df["cds_pos"], errors="coerce").astype("Int64")
spl_g = pd.read_csv(SPLITS_G)

device = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS = [1337, 2401, 4242]
RADII = [30, 45]   # <- multiscale
EPOCHS = 12
BATCH  = 96
LR     = 2e-3

# ---- bring in utils from earlier runs (redefine minimal pieces if not found)
if 'SiameseDeltaCNN' not in globals():
    # codon tables + helpers + datasets + model (same as earlier)
    CODON2AA = {
     "TTT":"F","TTC":"F","TTA":"L","TTG":"L","CTT":"L","CTC":"L","CTA":"L","CTG":"L",
     "ATT":"I","ATC":"I","ATA":"I","ATG":"M","GTT":"V","GTC":"V","GTA":"V","GTG":"V",
     "TCT":"S","TCC":"S","TCA":"S","TCG":"S","CCT":"P","CCC":"P","CCA":"P","CCG":"P",
     "ACT":"T","ACC":"T","ACA":"T","ACG":"T","GCT":"A","GCC":"A","GCA":"A","GCG":"A",
     "TAT":"Y","TAC":"Y","TAA":"*","TAG":"*","CAT":"H","CAC":"H","CAA":"Q","CAG":"Q",
     "AAT":"N","AAC":"N","AAA":"K","AAG":"K","GAT":"D","GAC":"D","GAA":"E","GAG":"E",
     "TGT":"C","TGC":"C","TGA":"*","TGG":"W","CGT":"R","CGC":"R","CGA":"R","CGG":"R",
     "AGT":"S","AGC":"S","AGA":"R","AGG":"R","GGT":"G","GGC":"G","GGA":"G","GGG":"G"
    }
    _GR = {'A':{'A':0,'R':112,'N':111,'D':126,'C':195,'Q':91,'E':107,'G':60,'H':86,'I':94,'L':96,'K':106,'M':84,'F':113,'P':27,'S':99,'T':58,'W':148,'Y':112,'V':64},
    'R':{'A':112,'R':0,'N':86,'D':96,'C':180,'Q':43,'E':54,'G':125,'H':29,'I':97,'L':102,'K':26,'M':91,'F':97,'P':103,'S':110,'T':71,'W':101,'Y':77,'V':96},
    'N':{'A':111,'R':86,'N':0,'D':23,'C':139,'Q':68,'E':42,'G':80,'H':68,'I':149,'L':153,'K':94,'M':142,'F':158,'P':91,'S':46,'T':65,'W':174,'Y':143,'V':133},
    'D':{'A':126,'R':96,'N':23,'D':0,'C':154,'Q':61,'E':45,'G':94,'H':81,'I':168,'L':172,'K':101,'M':160,'F':177,'P':108,'S':65,'T':85,'W':181,'Y':160,'V':152},
    'C':{'A':195,'R':180,'N':139,'D':154,'C':0,'Q':154,'E':170,'G':159,'H':174,'I':198,'L':198,'K':202,'M':196,'F':205,'P':169,'S':112,'T':149,'W':215,'Y':194,'V':192},
    'Q':{'A':91,'R':43,'N':68,'D':61,'C':154,'Q':0,'E':29,'G':87,'H':24,'I':109,'L':113,'K':53,'M':101,'F':116,'P':76,'S':68,'T':42,'W':130,'Y':99,'V':96},
    'E':{'A':107,'R':54,'N':42,'D':45,'C':170,'Q':29,'G':98,'H':40,'I':134,'L':138,'K':56,'M':126,'F':140,'P':93,'S':80,'T':65,'W':152,'Y':122,'V':121},
    'G':{'A':60,'R':125,'N':80,'D':94,'C':159,'Q':87,'E':98,'H':98,'I':135,'L':138,'K':127,'M':127,'F':153,'P':42,'S':56,'T':59,'W':184,'Y':147,'V':109},
    'H':{'A':86,'R':29,'N':68,'D':81,'C':174,'Q':24,'E':40,'G':98,'I':94,'L':99,'K':32,'M':87,'F':100,'P':77,'S':89,'T':47,'W':115,'Y':83,'V':84},
    'I':{'A':94,'R':97,'N':149,'D':168,'C':198,'Q':109,'E':134,'G':135,'H':94,'L':5,'K':102,'M':10,'F':21,'P':95,'S':142,'T':89,'W':61,'Y':33,'V':29},
    'L':{'A':96,'R':102,'N':153,'D':172,'C':198,'Q':113,'E':138,'G':138,'H':99,'I':5,'L':0,'K':107,'M':15,'F':22,'P':98,'S':145,'T':92,'W':61,'Y':36,'V':32},
    'K':{'A':106,'R':26,'N':94,'D':101,'C':202,'Q':53,'E':56,'G':127,'H':32,'I':102,'L':107,'K':0,'M':95,'F':102,'P':103,'S':121,'T':78,'W':110,'Y':85,'V':97},
    'M':{'A':84,'R':91,'N':142,'D':160,'C':196,'Q':101,'E':126,'G':127,'H':87,'I':10,'L':15,'K':95,'F':28,'P':87,'S':135,'T':81,'W':67,'Y':36,'V':21},
    'F':{'A':113,'R':97,'N':158,'D':177,'C':205,'Q':116,'E':140,'G':153,'H':100,'I':21,'L':22,'K':102,'M':28,'P':114,'S':155,'T':103,'W':40,'Y':22,'V':50},
    'P':{'A':27,'R':103,'N':91,'D':108,'C':169,'Q':76,'E':93,'G':42,'S':74,'T':38,'W':147,'Y':110,'V':68},
    'S':{'A':99,'R':110,'N':46,'D':65,'C':112,'Q':68,'E':80,'G':56,'H':89,'I':142,'L':145,'K':121,'M':135,'F':155,'P':74,'T':58,'W':177,'Y':144,'V':124},
    'T':{'A':58,'R':71,'N':65,'D':85,'C':149,'Q':42,'E':65,'G':59,'H':47,'I':89,'L':92,'K':78,'M':81,'F':103,'P':38,'S':58,'W':128,'Y':92,'V':69},
    'W':{'A':148,'R':101,'N':174,'D':181,'C':215,'Q':130,'E':152,'G':184,'H':115,'I':61,'L':61,'K':110,'M':67,'F':40,'P':147,'S':177,'T':128,'Y':37,'V':88},
    'Y':{'A':112,'R':77,'N':143,'D':160,'C':194,'Q':99,'E':122,'G':147,'H':83,'I':33,'L':36,'K':85,'M':36,'F':22,'P':110,'S':144,'T':92,'W':37,'V':55},
    'V':{'A':64,'R':96,'N':133,'D':152,'C':192,'Q':96,'E':121,'G':109,'H':84,'I':29,'L':32,'K':97,'M':21,'F':50,'P':68,'S':124,'T':69,'W':88,'Y':55,'V':0}}
    _B = """\
       A  R  N  D  C  Q  E  G  H  I  L  K  M  F  P  S  T  W  Y  V
    A  4 -1 -2 -2  0 -1 -1  0 -2 -1 -1 -1 -1 -2 -1  1  0 -3 -2  0
    R -1  5  0 -2 -3  1  0 -2  0 -3 -2  2 -1 -3 -2 -1 -1 -3 -2 -3
    N -2  0  6  1 -3  0  0  0  1 -3 -3  0 -2 -3 -2  1  0 -4 -2 -3
    D -2 -2  1  6 -3  0  2 -1 -1 -3 -4 -1 -3 -3 -1  0 -1 -4 -3 -3
    C  0 -3 -3 -3  9 -3 -4 -3 -3 -1 -1 -3 -1 -2 -3 -1 -1 -2 -2 -1
    Q -1  1  0  0 -3  5  2 -2  0 -3 -2  1  0 -3 -1  0 -1 -2 -1 -2
    E -1  0  0  2 -4  2  5 -2  0 -3 -3  1 -2 -3 -1  0 -1 -3 -2 -2
    G  0 -2  0 -1 -3 -2 -2  6 -2 -4 -4 -2 -3 -3 -2  0 -2 -2 -3 -3
    H -2  0  1 -1 -3  0  0 -2  8 -3 -3 -1 -2 -1 -2 -1 -2 -2  2 -3
    I -1 -3 -3 -3 -1 -3 -3 -4 -3  4  2 -3  1  0 -3 -2 -1 -3 -1  3
    L -1 -2 -3 -4 -1 -2 -3 -4 -3  2  4 -2  2  0 -3 -2 -1 -2 -1  1
    K -1  2  0 -1 -3  1  1 -2 -1 -3 -2  5 -1 -3 -1  0 -1 -3 -2 -2
    M -1 -1 -2 -3 -1  0 -2 -3 -2  1  2 -1  5  0 -2 -1 -1 -1 -1  1
    F -2 -3 -3 -3 -2 -3 -3 -3 -1  0  0 -3  0  6 -4 -2 -2  1  3 -1
    P -1 -2 -2 -1 -3 -1 -1 -2 -2 -3 -3 -1 -2 -4  7 -1 -1 -4 -3 -2
    S  1 -1  1  0 -1  0  0  0 -1 -2 -2  0 -1 -2 -1  4  1 -3 -2 -2
    T  0 -1  0 -1 -1 -1 -1 -2 -2 -1 -1 -1 -1 -2 -1  1  5 -2 -2  0
    W -3 -3 -4 -4 -2 -2 -3 -2 -2 -3 -2 -3 -1  1 -4 -3 -2 11  2 -3
    Y -2 -2 -2 -3 -2 -1 -2 -3  2 -1 -1 -2 -1  3 -3 -2 -2  2  7 -1
    V  0 -3 -3 -3 -1 -2 -2 -3 -3  3  1 -2  1 -1 -2 -2  0 -3 -1  4
    """
    B62={}; _hdr=_B.strip().splitlines()[0].split()
    for line in _B.strip().splitlines()[1:]:
        parts=line.split(); a=parts[0]; B62[a]={_hdr[i]:int(parts[i+1]) for i in range(len(_hdr))}
    VOCAB={"A":0,"C":1,"G":2,"T":3,"N":4}
    def onehot(seq: str):
        arr = np.zeros((5,len(seq)), np.float32)
        for i,ch in enumerate(seq): arr[VOCAB.get(ch,4), i] = 1.0
        return arr
    def grantham(a,b):
        if a==b: return 0
        return _GR.get(a,{}).get(b,_GR.get(b,{}).get(a,0))
    def blosum62(a,b):
        return B62.get(a,{}).get(b,B62.get(b,{}).get(a,0))
    def aa_scalars(wt_codon, mut_codon):
        a = CODON2AA.get(str(wt_codon), "X"); b = CODON2AA.get(str(mut_codon), "X")
        aa_same = float(a==b)
        if a=="X" or b=="X": return np.array([aa_same,0.0,0.0], dtype=np.float32)
        return np.array([aa_same, grantham(a,b)/215.0, (blosum62(a,b)+4)/15.0], dtype=np.float32)
    def mut_index_in_window(row, flank=60):
        i0 = int(row["cds_pos"]) - 1; cs = (i0 // 3) * 3; center = cs + 1
        return flank + (i0 - center)
    def extract_patch(win: str, j: int, R: int = 30):
        s = str(win)
        if not s or j<0 or j>=len(s): return ""
        a,b = max(0, j-R), min(len(s), j+R+1)
        left  = "N" * (R - (j - a)); right = "N" * (R - (b - j - 1))
        return left + s[a:b] + right
    def make_pairs_trimatched(train_df: pd.DataFrame, seed=1337):
        rs=np.random.RandomState(seed); pairs=[]; used=set()
        sub=train_df.copy(); sub["center_base"]=sub["tri_center"].str[1]
        for g, grp in sub.groupby("gene"):
            pos=grp[grp["label"]==1]; neg=grp[grp["label"]==0].copy()
            for tri, pgrp in pos.groupby("tri_center"):
                pool=neg[(neg["tri_center"]==tri)&(~neg.index.isin(used))]
                if pool.empty: continue
                k=min(len(pgrp), len(pool))
                pidx=rs.choice(pgrp.index, size=k, replace=False)
                nidx=rs.choice(pool.index, size=k, replace=False)
                pairs += list(zip(pidx,nidx)); used.update(nidx)
            leftover=pos.loc[~pos.index.isin([p for p,_ in pairs])]
            for base, pgrp in leftover.groupby("center_base"):
                pool=neg[(neg["tri_center"].str[1]==base)&(~neg.index.isin(used))]
                if pool.empty: continue
                k=min(len(pgrp), len(pool))
                pidx=rs.choice(pgrp.index, size=k, replace=False)
                nidx=rs.choice(pool.index, size=k, replace=False)
                pairs += list(zip(pidx,nidx)); used.update(nidx)
        return pairs
    def make_pairs_for_test(test_df: pd.DataFrame, seed=1337):
        rs=np.random.RandomState(seed); pairs=[]; used=set()
        sub=test_df.copy(); sub["center_base"]=sub["tri_center"].str[1]
        pos=sub[sub["label"]==1]; neg=sub[sub["label"]==0].copy()
        for tri, pgrp in pos.groupby("tri_center"):
            pool=neg[(neg["tri_center"]==tri)&(~neg.index.isin(used))]
            if pool.empty: continue
            k=min(len(pgrp), len(pool))
            pidx=rs.choice(pgrp.index, size=k, replace=False)
            nidx=rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx,nidx)); used.update(nidx)
        leftover=pos.loc[~pos.index.isin([p for p,_ in pairs])]
        for base, pgrp in leftover.groupby("center_base"):
            pool=neg[(neg["tri_center"].str[1]==base)&(~neg.index.isin(used))]
            if pool.empty: continue
            k=min(len(pgrp), len(pool))
            pidx=rs.choice(pgrp.index, size=k, replace=False)
            nidx=rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx,nidx)); used.update(nidx)
        return pairs

    class PairTrainDataset(torch.utils.data.Dataset):
        def __init__(self, frame: pd.DataFrame, pairs, R=30):
            self.df = frame; self.pairs = pairs; self.R = R
        def __len__(self): return len(self.pairs)
        def _patch_feats(self, r):
            j = mut_index_in_window(r, flank=60)
            mut = extract_patch(r["window_mut"], j, self.R)
            wt  = extract_patch(r["window_wt"],  j, self.R)
            scal = aa_scalars(r["wt_codon"], r["mut_codon"])
            return torch.from_numpy(onehot(mut)), torch.from_numpy(onehot(wt)), torch.from_numpy(scal)
        def __getitem__(self, k):
            ip, ineg = self.pairs[k]
            rp, rn = self.df.loc[ip], self.df.loc[ineg]
            xm_p, xw_p, s_p = self._patch_feats(rp)
            xm_n, xw_n, s_n = self._patch_feats(rn)
            return xm_p, xw_p, s_p, xm_n, xw_n, s_n

    class VariantEvalDataset(torch.utils.data.Dataset):
        def __init__(self, frame: pd.DataFrame, R=30):
            self.df = frame.reset_index(); self.R = R
            self.items=[]
            for _, r in self.df.iterrows():
                j = mut_index_in_window(r, flank=60)
                mut = extract_patch(r["window_mut"], j, self.R)
                wt  = extract_patch(r["window_wt"],  j, self.R)
                scal = aa_scalars(r["wt_codon"], r["mut_codon"])
                self.items.append((onehot(mut), onehot(wt), scal, float(r["label"])))
        def __len__(self): return len(self.items)
        def __getitem__(self, i):
            mut, wt, scal, y = self.items[i]
            return (torch.from_numpy(mut), torch.from_numpy(wt),
                    torch.from_numpy(scal), torch.tensor(y, dtype=torch.float32))

    class SiameseDeltaCNN(nn.Module):
        def __init__(self, in_ch=5, hid=96, scal_dim=3):
            super().__init__()
            self.enc = nn.Sequential(
                nn.Conv1d(in_ch, 64,  kernel_size=11, padding=5), nn.ReLU(),
                nn.Conv1d(64, 96, kernel_size=7,  padding=3),     nn.ReLU(),
                nn.Conv1d(96, 96, kernel_size=3,  padding=1),     nn.ReLU(),
                nn.AdaptiveMaxPool1d(1)
            )
            self.norm = nn.LayerNorm(96)
            self.head = nn.Sequential(nn.Linear(96,96), nn.ReLU(), nn.Dropout(0.1), nn.Linear(96,1))
            self.scal_head = nn.Linear(scal_dim, 1)
        def encode(self, x):
            h = self.enc(x).squeeze(-1)
            return self.norm(h)
        def score(self, xm, xw, scalars):
            hm = self.encode(xm); hw = self.encode(xw)
            core = self.head(hm - hw).squeeze(-1)
            scal = self.scal_head(scalars).squeeze(-1)
            return core + scal

# ranknet step (shared)
def train_and_score_one(seed, R):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    rows=[]
    bce = nn.BCEWithLogitsLoss()
    for fold, sub in spl_g.groupby("fold"):
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr = dd[dd["split"]=="train"].copy()
        te = dd[dd["split"]=="test"].copy()
        if tr.empty or te.empty:
            continue
        pairs = make_pairs_trimatched(tr, seed=seed)
        if not pairs:
            continue
        dtrain = PairTrainDataset(tr, pairs, R=R)
        dtest  = VariantEvalDataset(te, R=R)
        model = SiameseDeltaCNN().to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        loader = DataLoader(dtrain, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
        for _ in range(EPOCHS):
            model.train()
            for xm_p, xw_p, s_p, xm_n, xw_n, s_n in loader:
                xm_p, xw_p, s_p = xm_p.to(device), xw_p.to(device), s_p.to(device)
                xm_n, xw_n, s_n = xm_n.to(device), xw_n.to(device), s_n.to(device)
                opt.zero_grad()
                diff = model.score(xm_p, xw_p, s_p) - model.score(xm_n, xw_n, s_n)
                loss = bce(diff, torch.ones_like(diff))
                loss.backward(); opt.step()
        # score test
        model.eval()
        logits=[]; ys=[]
        test_loader = DataLoader(dtest, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
        with torch.no_grad():
            for xm, xw, s, y in test_loader:
                xm, xw, s = xm.to(device), xw.to(device), s.to(device)
                logits.append(model.score(xm, xw, s).cpu().numpy()); ys.append(y.numpy())
        y_log = np.concatenate(logits)
        out = te.reset_index()[["index","variant_id","gene","label"]].copy()
        out["fold"] = list(sub["fold"])[0]; out["seed"] = seed; out["R"] = R; out["logit"] = y_log
        rows.append(out)
    return pd.concat(rows, ignore_index=True)

# ---- train across radii × seeds
all_preds=[]
for R in RADII:
    for sd in SEEDS:
        all_preds.append(train_and_score_one(sd, R))
all_preds = pd.concat(all_preds, ignore_index=True)

# bag: average logits across seeds and radii
bag = (all_preds.groupby(["fold","index","variant_id","gene","label"], as_index=False)
                 .agg(logit_mean=("logit","mean")))

# metrics per gene (fold)
rows=[]
for fold, sub in bag.groupby("fold"):
    gname = sub["gene"].unique().tolist()[0]
    y = sub["label"].values
    p = 1/(1+np.exp(-sub["logit_mean"].values))
    auroc = roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan
    pr    = average_precision_score(y,p) if len(np.unique(y))>1 else np.nan
    brier = brier_score_loss(y,p)
    pair_acc_variant = float((((sub["logit_mean"]>0)&(sub["label"]==1)) | ((sub["logit_mean"]<=0)&(sub["label"]==0))).mean())
    # tri-matched pair acc on TEST
    dd = df.merge(spl_g[spl_g["fold"]==fold], on=["variant_id","gene"], how="inner")
    te = dd[dd["split"]=="test"].copy()
    pairs = make_pairs_for_test(te, seed=SEEDS[0])
    idx_to_logit = {r["index"]: r["logit_mean"] for _, r in sub.iterrows()}
    wins=0; total=0
    for ip, ineg in pairs:
        if ip in idx_to_logit and ineg in idx_to_logit:
            wins += int(idx_to_logit[ip] > idx_to_logit[ineg]); total += 1
    pair_acc_ctx = wins/total if total>0 else np.nan
    rows.append({
        "fold": fold, "gene": gname, "n": int(len(sub)),
        "auroc": round(auroc,4) if not np.isnan(auroc) else np.nan,
        "pr_auc": round(pr,4)   if not np.isnan(pr) else np.nan,
        "brier": round(brier,4),
        "pair_acc_variant": round(pair_acc_variant,4),
        "pair_acc_ctx": round(pair_acc_ctx,4) if not (isinstance(pair_acc_ctx,float) and np.isnan(pair_acc_ctx)) else np.nan
    })
summary = pd.DataFrame(rows).sort_values("gene")

# save
OUT = ARTI / f"siamese_pair_ranknet_gene_bag_multiscale_{time.strftime('%Y%m%d_%H%M%S')}"
OUT.mkdir(parents=True, exist_ok=True)
all_preds.to_csv(OUT/"preds_by_seed_by_radius.csv", index=False)
bag.to_csv(OUT/"preds_bag_average.csv", index=False)
summary.to_csv(OUT/"metrics_multiscale_bag.csv", index=False)

print("Device:", device)
print("Run dir:", OUT)
display(summary)


Device: cuda
Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/siamese_pair_ranknet_gene_bag_multiscale_20250816_221213


,fold,gene,n,auroc,pr_auc,brier,pair_acc_variant,pair_acc_ctx
0,loco_gene:EGFR,EGFR,39,0.7400,0.6646,0.2257,0.6923,0.7692
1,loco_gene:KRAS,KRAS,11,0.7500,0.9116,0.2207,0.7273,1.0000
2,loco_gene:PIK3CA,PIK3CA,24,0.6389,0.8672,0.2296,0.6667,0.6667
3,loco_gene:TP53,TP53,197,0.5786,0.4384,0.2675,0.3959,0.5652


Our multiscale bag helped EGFR/KRAS but TP53 slid again. That’s classic “boundaries are too soft.” Next move: **hard-negative mining** (train-only, still LOCO-GENE clean). We:

* pretrain on tri-matched pairs,

* score **training** variants,

* for each positive, pair with the **K highest-scoring negatives** in the same tri (fallback: same center base),

* fine-tune with RankNet on those mined pairs,

* bag across a few seeds at **R=45** (the radius that rescued TP53).

### I9 — LOCO-GENE RankNet + bio scalars with hard-negative mining (train-only), R=45, 3-seed bag
##### Keeps LOCO-GENE honest: mining is done only within the training pool for each fold.

In [8]:
# I9 — LOCO-GENE RankNet + bio scalars with hard-negative mining (train-only), R=45, 3-seed bag
# Keeps LOCO-GENE honest: mining is done only within the training pool for each fold.

import os, glob, time, random
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# ---- Paths / data
BASE = Path("/content/drive/MyDrive/Cancer_gene_database")
DATA = BASE/"data"; ARTI = BASE/"artifacts"
P_RES = DATA/"masterseq_model_ready_plus_resolved.parquet"
runs = sorted(glob.glob(str(ARTI / "sentiment_run_plus_variants_*"))); assert runs, "No split dirs found."
RUN_DIR = Path(runs[-1]); SPLITS_G = RUN_DIR/"splits_loco_gene.csv"

df = pd.read_parquet(P_RES).dropna(subset=[
    "variant_id","gene","label","window_mut","window_wt",
    "wt_codon","mut_codon","ref","alt","cds_pos","tri_center"
]).copy()
for c in ["variant_id","gene","ref","alt","wt_codon","mut_codon","window_mut","window_wt","tri_center"]:
    df[c] = df[c].astype(str).str.upper()
df["label"] = df["label"].astype(int)
df["cds_pos"] = pd.to_numeric(df["cds_pos"], errors="coerce").astype("Int64")
spl_g = pd.read_csv(SPLITS_G)

device = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS  = [1337, 2401, 4242]  # small bag
RADIUS = 45                  # larger patch that helped TP53
E_PRE  = 10                  # pretrain epochs
E_MIN  = 6                   # finetune epochs on mined pairs
BATCH  = 96
LR     = 2e-3
K_HARD = 3                   # # hard negatives per positive (per tri), fallback to base

# ---- Utils from prior cells (define if missing)
if 'SiameseDeltaCNN' not in globals():
    # minimal re-defs
    CODON2AA = {
     "TTT":"F","TTC":"F","TTA":"L","TTG":"L","CTT":"L","CTC":"L","CTA":"L","CTG":"L",
     "ATT":"I","ATC":"I","ATA":"I","ATG":"M","GTT":"V","GTC":"V","GTA":"V","GTG":"V",
     "TCT":"S","TCC":"S","TCA":"S","TCG":"S","CCT":"P","CCC":"P","CCA":"P","CCG":"P",
     "ACT":"T","ACC":"T","ACA":"T","ACG":"T","GCT":"A","GCC":"A","GCA":"A","GCG":"A",
     "TAT":"Y","TAC":"Y","TAA":"*","TAG":"*","CAT":"H","CAC":"H","CAA":"Q","CAG":"Q",
     "AAT":"N","AAC":"N","AAA":"K","AAG":"K","GAT":"D","GAC":"D","GAA":"E","GAG":"E",
     "TGT":"C","TGC":"C","TGA":"*","TGG":"W","CGT":"R","CGC":"R","CGA":"R","CGG":"R",
     "AGT":"S","AGC":"S","AGA":"R","AGG":"R","GGT":"G","GGC":"G","GGA":"G","GGG":"G"
    }
    _GR = {'A':{'A':0,'R':112,'N':111,'D':126,'C':195,'Q':91,'E':107,'G':60,'H':86,'I':94,'L':96,'K':106,'M':84,'F':113,'P':27,'S':99,'T':58,'W':148,'Y':112,'V':64},
    'R':{'A':112,'R':0,'N':86,'D':96,'C':180,'Q':43,'E':54,'G':125,'H':29,'I':97,'L':102,'K':26,'M':91,'F':97,'P':103,'S':110,'T':71,'W':101,'Y':77,'V':96},
    'N':{'A':111,'R':86,'N':0,'D':23,'C':139,'Q':68,'E':42,'G':80,'H':68,'I':149,'L':153,'K':94,'M':142,'F':158,'P':91,'S':46,'T':65,'W':174,'Y':143,'V':133},
    'D':{'A':126,'R':96,'N':23,'D':0,'C':154,'Q':61,'E':45,'G':94,'H':81,'I':168,'L':172,'K':101,'M':160,'F':177,'P':108,'S':65,'T':85,'W':181,'Y':160,'V':152},
    'C':{'A':195,'R':180,'N':139,'D':154,'C':0,'Q':154,'E':170,'G':159,'H':174,'I':198,'L':198,'K':202,'M':196,'F':205,'P':169,'S':112,'T':149,'W':215,'Y':194,'V':192},
    'Q':{'A':91,'R':43,'N':68,'D':61,'C':154,'Q':0,'E':29,'G':87,'H':24,'I':109,'L':113,'K':53,'M':101,'F':116,'P':76,'S':68,'T':42,'W':130,'Y':99,'V':96},
    'E':{'A':107,'R':54,'N':42,'D':45,'C':170,'Q':29,'G':98,'H':40,'I':134,'L':138,'K':56,'M':126,'F':140,'P':93,'S':80,'T':65,'W':152,'Y':122,'V':121},
    'G':{'A':60,'R':125,'N':80,'D':94,'C':159,'Q':87,'E':98,'H':98,'I':135,'L':138,'K':127,'M':127,'F':153,'P':42,'S':56,'T':59,'W':184,'Y':147,'V':109},
    'H':{'A':86,'R':29,'N':68,'D':81,'C':174,'Q':24,'E':40,'G':98,'I':94,'L':99,'K':32,'M':87,'F':100,'P':77,'S':89,'T':47,'W':115,'Y':83,'V':84},
    'I':{'A':94,'R':97,'N':149,'D':168,'C':198,'Q':109,'E':134,'G':135,'H':94,'L':5,'K':102,'M':10,'F':21,'P':95,'S':142,'T':89,'W':61,'Y':33,'V':29},
    'L':{'A':96,'R':102,'N':153,'D':172,'C':198,'Q':113,'E':138,'G':138,'H':99,'I':5,'L':0,'K':107,'M':15,'F':22,'P':98,'S':145,'T':92,'W':61,'Y':36,'V':32},
    'K':{'A':106,'R':26,'N':94,'D':101,'C':202,'Q':53,'E':56,'G':127,'H':32,'I':102,'L':107,'K':0,'M':95,'F':102,'P':103,'S':121,'T':78,'W':110,'Y':85,'V':97},
    'M':{'A':84,'R':91,'N':142,'D':160,'C':196,'Q':101,'E':126,'G':127,'H':87,'I':10,'L':15,'K':95,'F':28,'P':87,'S':135,'T':81,'W':67,'Y':36,'V':21},
    'F':{'A':113,'R':97,'N':158,'D':177,'C':205,'Q':116,'E':140,'G':153,'H':100,'I':21,'L':22,'K':102,'M':28,'P':114,'S':155,'T':103,'W':40,'Y':22,'V':50},
    'P':{'A':27,'R':103,'N':91,'D':108,'C':169,'Q':76,'E':93,'G':42,'S':74,'T':38,'W':147,'Y':110,'V':68},
    'S':{'A':99,'R':110,'N':46,'D':65,'C':112,'Q':68,'E':80,'G':56,'H':89,'I':142,'L':145,'K':121,'M':135,'F':155,'P':74,'T':58,'W':177,'Y':144,'V':124},
    'T':{'A':58,'R':71,'N':65,'D':85,'C':149,'Q':42,'E':65,'G':59,'H':47,'I':89,'L':92,'K':78,'M':81,'F':103,'P':38,'S':58,'W':128,'Y':92,'V':69},
    'W':{'A':148,'R':101,'N':174,'D':181,'C':215,'Q':130,'E':152,'G':184,'H':115,'I':61,'L':61,'K':110,'M':67,'F':40,'P':147,'S':177,'T':128,'Y':37,'V':88},
    'Y':{'A':112,'R':77,'N':143,'D':160,'C':194,'Q':99,'E':122,'G':147,'H':83,'I':33,'L':36,'K':85,'M':36,'F':22,'P':110,'S':144,'T':92,'W':37,'V':55},
    'V':{'A':64,'R':96,'N':133,'D':152,'C':192,'Q':96,'E':121,'G':109,'H':84,'I':29,'L':32,'K':97,'M':21,'F':50,'P':68,'S':124,'T':69,'W':88,'Y':55,'V':0}}
    _B = """\
       A  R  N  D  C  Q  E  G  H  I  L  K  M  F  P  S  T  W  Y  V
    A  4 -1 -2 -2  0 -1 -1  0 -2 -1 -1 -1 -1 -2 -1  1  0 -3 -2  0
    R -1  5  0 -2 -3  1  0 -2  0 -3 -2  2 -1 -3 -2 -1 -1 -3 -2 -3
    N -2  0  6  1 -3  0  0  0  1 -3 -3  0 -2 -3 -2  1  0 -4 -2 -3
    D -2 -2  1  6 -3  0  2 -1 -1 -3 -4 -1 -3 -3 -1  0 -1 -4 -3 -3
    C  0 -3 -3 -3  9 -3 -4 -3 -3 -1 -1 -3 -1 -2 -3 -1 -1 -2 -2 -1
    Q -1  1  0  0 -3  5  2 -2  0 -3 -2  1  0 -3 -1  0 -1 -2 -1 -2
    E -1  0  0  2 -4  2  5 -2  0 -3 -3  1 -2 -3 -1  0 -1 -3 -2 -2
    G  0 -2  0 -1 -3 -2 -2  6 -2 -4 -4 -2 -3 -3 -2  0 -2 -2 -3 -3
    H -2  0  1 -1 -3  0  0 -2  8 -3 -3 -1 -2 -1 -2 -1 -2 -2  2 -3
    I -1 -3 -3 -3 -1 -3 -3 -4 -3  4  2 -3  1  0 -3 -2 -1 -3 -1  3
    L -1 -2 -3 -4 -1 -2 -3 -4 -3  2  4 -2  2  0 -3 -2 -1 -2 -1  1
    K -1  2  0 -1 -3  1  1 -2 -1 -3 -2  5 -1 -3 -1  0 -1 -3 -2 -2
    M -1 -1 -2 -3 -1  0 -2 -3 -2  1  2 -1  5  0 -2 -1 -1 -1 -1  1
    F -2 -3 -3 -3 -2 -3 -3 -3 -1  0  0 -3  0  6 -4 -2 -2  1  3 -1
    P -1 -2 -2 -1 -3 -1 -1 -2 -2 -3 -3 -1 -2 -4  7 -1 -1 -4 -3 -2
    S  1 -1  1  0 -1  0  0  0 -1 -2 -2  0 -1 -2 -1  4  1 -3 -2 -2
    T  0 -1  0 -1 -1 -1 -1 -2 -2 -1 -1 -1 -1 -2 -1  1  5 -2 -2  0
    W -3 -3 -4 -4 -2 -2 -3 -2 -2 -3 -2 -3 -1  1 -4 -3 -2 11  2 -3
    Y -2 -2 -2 -3 -2 -1 -2 -3  2 -1 -1 -2 -1  3 -3 -2 -2  2  7 -1
    V  0 -3 -3 -3 -1 -2 -2 -3 -3  3  1 -2  1 -1 -2 -2  0 -3 -1  4
    """
    B62={}; _hdr=_B.strip().splitlines()[0].split()
    for line in _B.strip().splitlines()[1:]:
        parts=line.split(); a=parts[0]; B62[a]={_hdr[i]:int(parts[i+1]) for i in range(len(_hdr))}
    VOCAB={"A":0,"C":1,"G":2,"T":3,"N":4}
    def onehot(seq: str):
        arr = np.zeros((5,len(seq)), np.float32)
        for i,ch in enumerate(seq): arr[VOCAB.get(ch,4), i] = 1.0
        return arr
    def grantham(a,b):
        if a==b: return 0
        return _GR.get(a,{}).get(b,_GR.get(b,{}).get(a,0))
    def blosum62(a,b):
        return B62.get(a,{}).get(b,B62.get(b,{}).get(a,0))
    def aa_scalars(wt_codon, mut_codon):
        a = CODON2AA.get(str(wt_codon), "X"); b = CODON2AA.get(str(mut_codon), "X")
        aa_same = float(a==b)
        if a=="X" or b=="X": return np.array([aa_same,0.0,0.0], dtype=np.float32)
        return np.array([aa_same, grantham(a,b)/215.0, (blosum62(a,b)+4)/15.0], dtype=np.float32)
    def mut_index_in_window(row, flank=60):
        i0 = int(row["cds_pos"]) - 1; cs = (i0 // 3) * 3; center = cs + 1
        return flank + (i0 - center)
    def extract_patch(win: str, j: int, R: int = 30):
        s = str(win)
        if not s or j<0 or j>=len(s): return ""
        a,b = max(0, j-R), min(len(s), j+R+1)
        left  = "N" * (R - (j - a)); right = "N" * (R - (b - j - 1))
        return left + s[a:b] + right

    class PairTrainDataset(Dataset):
        def __init__(self, frame: pd.DataFrame, pairs, R=30):
            self.df = frame; self.pairs = pairs; self.R = R
        def __len__(self): return len(self.pairs)
        def _patch_feats(self, r):
            j = mut_index_in_window(r, flank=60)
            mut = extract_patch(r["window_mut"], j, self.R)
            wt  = extract_patch(r["window_wt"],  j, self.R)
            scal = aa_scalars(r["wt_codon"], r["mut_codon"])
            return torch.from_numpy(onehot(mut)), torch.from_numpy(onehot(wt)), torch.from_numpy(scal)
        def __getitem__(self, k):
            ip, ineg = self.pairs[k]
            rp, rn = self.df.loc[ip], self.df.loc[ineg]
            xm_p, xw_p, s_p = self._patch_feats(rp)
            xm_n, xw_n, s_n = self._patch_feats(rn)
            return xm_p, xw_p, s_p, xm_n, xw_n, s_n

    class VariantEvalDataset(Dataset):
        def __init__(self, frame: pd.DataFrame, R=30):
            self.df = frame.reset_index(); self.R = R
            self.items=[]
            for _, r in self.df.iterrows():
                j = mut_index_in_window(r, flank=60)
                mut = extract_patch(r["window_mut"], j, self.R)
                wt  = extract_patch(r["window_wt"],  j, self.R)
                scal = aa_scalars(r["wt_codon"], r["mut_codon"])
                self.items.append((onehot(mut), onehot(wt), scal, float(r["label"])))
        def __len__(self): return len(self.items)
        def __getitem__(self, i):
            mut, wt, scal, y = self.items[i]
            return (torch.from_numpy(mut), torch.from_numpy(wt),
                    torch.from_numpy(scal), torch.tensor(y, dtype=torch.float32))

    class SiameseDeltaCNN(nn.Module):
        def __init__(self, in_ch=5, hid=96, scal_dim=3):
            super().__init__()
            self.enc = nn.Sequential(
                nn.Conv1d(in_ch, 64, kernel_size=11, padding=5), nn.ReLU(),
                nn.Conv1d(64, 96, kernel_size=7, padding=3), nn.ReLU(),
                nn.Conv1d(96, 96, kernel_size=3, padding=1), nn.ReLU(),
                nn.AdaptiveMaxPool1d(1)
            )
            self.norm = nn.LayerNorm(96)
            self.head = nn.Sequential(nn.Linear(96,96), nn.ReLU(), nn.Dropout(0.1), nn.Linear(96,1))
            self.scal_head = nn.Linear(scal_dim, 1)
        def encode(self, x):
            h = self.enc(x).squeeze(-1)
            return self.norm(h)
        def score(self, xm, xw, scalars):
            hm = self.encode(xm); hw = self.encode(xw)
            core = self.head(hm - hw).squeeze(-1)
            scal = self.scal_head(scalars).squeeze(-1)
            return core + scal

# tri-match pairing (train) + (test)
def make_pairs_trimatched(train_df: pd.DataFrame, seed=1337):
    rs=np.random.RandomState(seed); pairs=[]; used=set()
    sub=train_df.copy(); sub["center_base"]=sub["tri_center"].str[1]
    for g, grp in sub.groupby("gene"):
        pos=grp[grp["label"]==1]; neg=grp[grp["label"]==0].copy()
        for tri, pgrp in pos.groupby("tri_center"):
            pool=neg[(neg["tri_center"]==tri)&(~neg.index.isin(used))]
            if pool.empty: continue
            k=min(len(pgrp), len(pool))
            pidx=rs.choice(pgrp.index, size=k, replace=False)
            nidx=rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx,nidx)); used.update(nidx)
        leftover=pos.loc[~pos.index.isin([p for p,_ in pairs])]
        for base, pgrp in leftover.groupby("center_base"):
            pool=neg[(neg["tri_center"].str[1]==base)&(~neg.index.isin(used))]
            if pool.empty: continue
            k=min(len(pgrp), len(pool))
            pidx=rs.choice(pgrp.index, size=k, replace=False)
            nidx=rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx,nidx)); used.update(nidx)
    return pairs

def make_pairs_for_test(test_df: pd.DataFrame, seed=1337):
    rs=np.random.RandomState(seed); pairs=[]; used=set()
    sub=test_df.copy(); sub["center_base"]=sub["tri_center"].str[1]
    pos=sub[sub["label"]==1]; neg=sub[sub["label"]==0].copy()
    for tri, pgrp in pos.groupby("tri_center"):
        pool=neg[(neg["tri_center"]==tri)&(~neg.index.isin(used))]
        if pool.empty: continue
        k=min(len(pgrp), len(pool))
        pidx=rs.choice(pgrp.index, size=k, replace=False)
        nidx=rs.choice(pool.index, size=k, replace=False)
        pairs += list(zip(pidx,nidx)); used.update(nidx)
    leftover=pos.loc[~pos.index.isin([p for p,_ in pairs])]
    for base, pgrp in leftover.groupby("center_base"):
        pool=neg[(neg["tri_center"].str[1]==base)&(~neg.index.isin(used))]
        if pool.empty: continue
        k=min(len(pgrp), len(pool))
        pidx=rs.choice(pgrp.index, size=k, replace=False)
        nidx=rs.choice(pool.index, size=k, replace=False)
        pairs += list(zip(pidx,nidx)); used.update(nidx)
    return pairs

# ---- Miner: pick top-K scoring negatives per tri (fallback: same center base)
def mine_hard_pairs(train_df: pd.DataFrame, model, R=RADIUS, k=K_HARD):
    # score all train variants
    ds = VariantEvalDataset(train_df, R=R)
    loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
    model.eval()
    scores=[];
    with torch.no_grad():
        for xm, xw, s, y in loader:
            xm, xw, s = xm.to(device), xw.to(device), s.to(device)
            scores.append(model.score(xm, xw, s).cpu().numpy())
    scores = np.concatenate(scores)
    scored = train_df.reset_index().copy()
    scored["logit"] = scores
    # build K hard pairs
    pairs=[]
    used_counts={}
    # exact tri
    for tri, grp in scored.groupby("tri_center"):
        pos_idx = grp[grp["label"]==1]["index"].tolist()
        neg_sorted = grp[grp["label"]==0].sort_values("logit", ascending=False)["index"].tolist()
        for ip in pos_idx:
            # take top-k negs (allow reuse but cap it)
            take=[]
            for ineg in neg_sorted:
                if used_counts.get(ineg,0) < 5:  # small cap to avoid one neg dominating
                    take.append(ineg)
                if len(take)>=k: break
            for ineg in take:
                pairs.append((ip, ineg)); used_counts[ineg] = used_counts.get(ineg,0)+1
    # fallback: same center base for positives that got <k pairs
    scored["center_base"] = scored["tri_center"].str[1]
    from collections import defaultdict
    pos_needed = defaultdict(int)
    for ip, ineg in pairs: pos_needed[ip] += 1
    for base, grp in scored.groupby("center_base"):
        pos_idx = grp[grp["label"]==1]["index"].tolist()
        neg_sorted = grp[grp["label"]==0].sort_values("logit", ascending=False)["index"].tolist()
        for ip in pos_idx:
            need = max(0, k - pos_needed.get(ip,0))
            if need==0: continue
            take=[]
            for ineg in neg_sorted:
                if used_counts.get(ineg,0) < 5:
                    take.append(ineg)
                if len(take)>=need: break
            for ineg in take:
                pairs.append((ip, ineg)); used_counts[ineg] = used_counts.get(ineg,0)+1
            pos_needed[ip] = k
    return pairs

# ---- Train + mine + finetune for one fold
def train_fold_with_mining(train_df, test_df, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    bce = nn.BCEWithLogitsLoss()

    # pretrain on tri-matched
    pairs0 = make_pairs_trimatched(train_df, seed=seed)
    dtrain0 = PairTrainDataset(train_df, pairs0, R=RADIUS)
    model = SiameseDeltaCNN().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    loader0 = DataLoader(dtrain0, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
    for _ in range(E_PRE):
        model.train()
        for xm_p, xw_p, s_p, xm_n, xw_n, s_n in loader0:
            xm_p, xw_p, s_p = xm_p.to(device), xw_p.to(device), s_p.to(device)
            xm_n, xw_n, s_n = xm_n.to(device), xw_n.to(device), s_n.to(device)
            opt.zero_grad()
            diff = model.score(xm_p, xw_p, s_p) - model.score(xm_n, xw_n, s_n)
            loss = bce(diff, torch.ones_like(diff))
            loss.backward(); opt.step()

    # mine hard negatives within TRAIN only
    pairs_hard = mine_hard_pairs(train_df, model, R=RADIUS, k=K_HARD)
    dtrain_h = PairTrainDataset(train_df, pairs_hard, R=RADIUS)
    loader_h = DataLoader(dtrain_h, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)

    for _ in range(E_MIN):
        model.train()
        for xm_p, xw_p, s_p, xm_n, xw_n, s_n in loader_h:
            xm_p, xw_p, s_p = xm_p.to(device), xw_p.to(device), s_p.to(device)
            xm_n, xw_n, s_n = xm_n.to(device), xw_n.to(device), s_n.to(device)
            opt.zero_grad()
            diff = model.score(xm_p, xw_p, s_p) - model.score(xm_n, xw_n, s_n)
            loss = bce(diff, torch.ones_like(diff))
            loss.backward(); opt.step()

    # eval on test (per-variant)
    dtest = VariantEvalDataset(test_df, R=RADIUS)
    test_loader = DataLoader(dtest, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
    model.eval()
    logits=[]; ys=[]
    with torch.no_grad():
        for xm, xw, s, y in test_loader:
            xm, xw, s = xm.to(device), xw.to(device), s.to(device)
            logits.append(model.score(xm, xw, s).cpu().numpy()); ys.append(y.numpy())
    y_log = np.concatenate(logits)
    te_reset = test_df.reset_index()[["index","variant_id","gene","label"]].copy()
    te_reset["seed"] = seed
    te_reset["logit"] = y_log
    return te_reset

# ---- Run across folds/seeds, then summarize
all_preds=[]
for fold, sub in spl_g.groupby("fold"):
    dd = df.merge(sub, on=["variant_id","gene"], how="inner")
    tr = dd[dd["split"]=="train"].copy()
    te = dd[dd["split"]=="test"].copy()
    if tr.empty or te.empty:
        continue
    for sd in SEEDS:
        fold_preds = train_fold_with_mining(tr, te, sd)
        fold_preds["fold"] = fold
        all_preds.append(fold_preds)

all_preds = pd.concat(all_preds, ignore_index=True)

# bag across seeds
bag = (all_preds.groupby(["fold","index","variant_id","gene","label"], as_index=False)
                 .agg(logit_mean=("logit","mean")))

# metrics per LOCO-GENE fold (i.e., per gene)
rows=[]
for fold, sub in bag.groupby("fold"):
    gname = sub["gene"].unique().tolist()[0]
    y = sub["label"].values
    p = 1/(1+np.exp(-sub["logit_mean"].values))
    auroc = roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan
    pr    = average_precision_score(y,p) if len(np.unique(y))>1 else np.nan
    brier = brier_score_loss(y,p)
    pair_acc_variant = float((((sub["logit_mean"]>0)&(sub["label"]==1)) | ((sub["logit_mean"]<=0)&(sub["label"]==0))).mean())
    # tri-matched pair acc on TEST
    dd = df.merge(spl_g[spl_g["fold"]==fold], on=["variant_id","gene"], how="inner")
    te = dd[dd["split"]=="test"].copy()
    pairs = make_pairs_for_test(te, seed=SEEDS[0])
    idx_to_logit = {r["index"]: r["logit_mean"] for _, r in sub.iterrows()}
    wins=0; total=0
    for ip, ineg in pairs:
        if ip in idx_to_logit and ineg in idx_to_logit:
            wins += int(idx_to_logit[ip] > idx_to_logit[ineg]); total += 1
    pair_acc_ctx = wins/total if total>0 else np.nan
    rows.append({
        "fold": fold, "gene": gname, "n": int(len(sub)),
        "auroc": round(auroc,4) if not np.isnan(auroc) else np.nan,
        "pr_auc": round(pr,4)   if not np.isnan(pr) else np.nan,
        "brier": round(brier,4),
        "pair_acc_variant": round(pair_acc_variant,4),
        "pair_acc_ctx": round(pair_acc_ctx,4) if not (isinstance(pair_acc_ctx,float) and np.isnan(pair_acc_ctx)) else np.nan
    })
summary = pd.DataFrame(rows).sort_values("gene")

OUT = ARTI / f"siamese_pair_ranknet_gene_bag_R45_hardmining_{time.strftime('%Y%m%d_%H%M%S')}"
OUT.mkdir(parents=True, exist_ok=True)
all_preds.to_csv(OUT/"preds_by_seed.csv", index=False)
bag.to_csv(OUT/"preds_bag_avg.csv", index=False)
summary.to_csv(OUT/"metrics_hardmining.csv", index=False)

print("Device:", device)
print("Run dir:", OUT)
display(summary)


Device: cuda
Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/siamese_pair_ranknet_gene_bag_R45_hardmining_20250816_222259


,fold,gene,n,auroc,pr_auc,brier,pair_acc_variant,pair_acc_ctx
0,loco_gene:EGFR,EGFR,39,0.6114,0.5496,0.2264,0.6410,0.5385
1,loco_gene:KRAS,KRAS,11,0.7083,0.8835,0.2214,0.6364,1.0000
2,loco_gene:PIK3CA,PIK3CA,24,0.3056,0.6688,0.3159,0.2500,0.1667
3,loco_gene:TP53,TP53,197,0.5479,0.4294,0.2341,0.6294,0.5797


 **The idea of hyperparameter-tuning immediately comes to mind**

 Hit these knobs in this order:

1. **Mining intensity & reuse cap** — drop K_HARD to 0–1 and cap how often a neg can be reused (2–3). Too-aggressive mining is probably what nuked PIK3CA.

2. **Patch radius** — keep both scales (R=30,45). Multiscale helped EGFR/KRAS; TP53 liked R=45.

3. **Dropout** — 0.0→0.1 (maybe 0.2) to steady small-n genes without blurring TP53.

4. **LR schedule** — 1e-3 vs 2e-3; pretrain 8–12 epochs; finetune 3–6 on mined pairs.

5. **Weight decay** — 1e-4 default is fine; only dip to 1e-5 if you see underfit.

6. **Bagging** — 3–5 seeds, not per-gene hacks.

#### Here’s a single **HPO-lite**: tiny grid, inner val that stays within each LOCO fold’s training pool (so no gene leakage), then it retrains the best config and reports proper LOCO-GENE TEST metrics. Paste + run:

In [9]:
# HPO-lite — small, leak-free hyperparam probe + retrain best and eval LOCO-GENE TEST
# Searches R∈{30,45}, dropout∈{0.0,0.1}, LR∈{1e-3,2e-3}, mining K∈{0,1} with a tiny bag of seeds.

import os, glob, time, random
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# ---------- data ----------
BASE = Path("/content/drive/MyDrive/Cancer_gene_database")
DATA = BASE/"data"; ARTI = BASE/"artifacts"
P_RES = DATA/"masterseq_model_ready_plus_resolved.parquet"
runs = sorted(glob.glob(str(ARTI / "sentiment_run_plus_variants_*"))); assert runs
RUN_DIR = Path(runs[-1]); SPLITS_G = RUN_DIR/"splits_loco_gene.csv"

df = pd.read_parquet(P_RES).dropna(subset=[
    "variant_id","gene","label","window_mut","window_wt",
    "wt_codon","mut_codon","ref","alt","cds_pos","tri_center"
]).copy()
for c in ["variant_id","gene","ref","alt","wt_codon","mut_codon","window_mut","window_wt","tri_center"]:
    df[c] = df[c].astype(str).str.upper()
df["label"] = df["label"].astype(int)
df["cds_pos"] = pd.to_numeric(df["cds_pos"], errors="coerce").astype("Int64")
spl_g = pd.read_csv(SPLITS_G)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------- helpers (self-contained) ----------
CODON2AA = { "TTT":"F","TTC":"F","TTA":"L","TTG":"L","CTT":"L","CTC":"L","CTA":"L","CTG":"L",
 "ATT":"I","ATC":"I","ATA":"I","ATG":"M","GTT":"V","GTC":"V","GTA":"V","GTG":"V",
 "TCT":"S","TCC":"S","TCA":"S","TCG":"S","CCT":"P","CCC":"P","CCA":"P","CCG":"P",
 "ACT":"T","ACC":"T","ACA":"T","ACG":"T","GCT":"A","GCC":"A","GCA":"A","GCG":"A",
 "TAT":"Y","TAC":"Y","TAA":"*","TAG":"*","CAT":"H","CAC":"H","CAA":"Q","CAG":"Q",
 "AAT":"N","AAC":"N","AAA":"K","AAG":"K","GAT":"D","GAC":"D","GAA":"E","GAG":"E",
 "TGT":"C","TGC":"C","TGA":"*","TGG":"W","CGT":"R","CGC":"R","CGA":"R","CGG":"R",
 "AGT":"S","AGC":"S","AGA":"R","AGG":"R","GGT":"G","GGC":"G","GGA":"G","GGG":"G" }
_GR={'A':{'A':0,'R':112,'N':111,'D':126,'C':195,'Q':91,'E':107,'G':60,'H':86,'I':94,'L':96,'K':106,'M':84,'F':113,'P':27,'S':99,'T':58,'W':148,'Y':112,'V':64},
'R':{'A':112,'R':0,'N':86,'D':96,'C':180,'Q':43,'E':54,'G':125,'H':29,'I':97,'L':102,'K':26,'M':91,'F':97,'P':103,'S':110,'T':71,'W':101,'Y':77,'V':96},
'N':{'A':111,'R':86,'N':0,'D':23,'C':139,'Q':68,'E':42,'G':80,'H':68,'I':149,'L':153,'K':94,'M':142,'F':158,'P':91,'S':46,'T':65,'W':174,'Y':143,'V':133},
'D':{'A':126,'R':96,'N':23,'D':0,'C':154,'Q':61,'E':45,'G':94,'H':81,'I':168,'L':172,'K':101,'M':160,'F':177,'P':108,'S':65,'T':85,'W':181,'Y':160,'V':152},
'C':{'A':195,'R':180,'N':139,'D':154,'C':0,'Q':154,'E':170,'G':159,'H':174,'I':198,'L':198,'K':202,'M':196,'F':205,'P':169,'S':112,'T':149,'W':215,'Y':194,'V':192},
'Q':{'A':91,'R':43,'N':68,'D':61,'C':154,'Q':0,'E':29,'G':87,'H':24,'I':109,'L':113,'K':53,'M':101,'F':116,'P':76,'S':68,'T':42,'W':130,'Y':99,'V':96},
'E':{'A':107,'R':54,'N':42,'D':45,'C':170,'Q':29,'G':98,'H':40,'I':134,'L':138,'K':56,'M':126,'F':140,'P':93,'S':80,'T':65,'W':152,'Y':122,'V':121},
'G':{'A':60,'R':125,'N':80,'D':94,'C':159,'Q':87,'E':98,'H':98,'I':135,'L':138,'K':127,'M':127,'F':153,'P':42,'S':56,'T':59,'W':184,'Y':147,'V':109},
'H':{'A':86,'R':29,'N':68,'D':81,'C':174,'Q':24,'E':40,'G':98,'I':94,'L':99,'K':32,'M':87,'F':100,'P':77,'S':89,'T':47,'W':115,'Y':83,'V':84},
'I':{'A':94,'R':97,'N':149,'D':168,'C':198,'Q':109,'E':134,'G':135,'H':94,'L':5,'K':102,'M':10,'F':21,'P':95,'S':142,'T':89,'W':61,'Y':33,'V':29},
'L':{'A':96,'R':102,'N':153,'D':172,'C':198,'Q':113,'E':138,'G':138,'H':99,'I':5,'L':0,'K':107,'M':15,'F':22,'P':98,'S':145,'T':92,'W':61,'Y':36,'V':32},
'K':{'A':106,'R':26,'N':94,'D':101,'C':202,'Q':53,'E':56,'G':127,'H':32,'I':102,'L':107,'K':0,'M':95,'F':102,'P':103,'S':121,'T':78,'W':110,'Y':85,'V':97},
'M':{'A':84,'R':91,'N':142,'D':160,'C':196,'Q':101,'E':126,'G':127,'H':87,'I':10,'L':15,'K':95,'F':28,'P':87,'S':135,'T':81,'W':67,'Y':36,'V':21},
'F':{'A':113,'R':97,'N':158,'D':177,'C':205,'Q':116,'E':140,'G':153,'H':100,'I':21,'L':22,'K':102,'M':28,'P':114,'S':155,'T':103,'W':40,'Y':22,'V':50},
'P':{'A':27,'R':103,'N':91,'D':108,'C':169,'Q':76,'E':93,'G':42,'S':74,'T':38,'W':147,'Y':110,'V':68},
'S':{'A':99,'R':110,'N':46,'D':65,'C':112,'Q':68,'E':80,'G':56,'H':89,'I':142,'L':145,'K':121,'M':135,'F':155,'P':74,'T':58,'W':177,'Y':144,'V':124},
'T':{'A':58,'R':71,'N':65,'D':85,'C':149,'Q':42,'E':65,'G':59,'H':47,'I':89,'L':92,'K':78,'M':81,'F':103,'P':38,'S':58,'W':128,'Y':92,'V':69},
'W':{'A':148,'R':101,'N':174,'D':181,'C':215,'Q':130,'E':152,'G':184,'H':115,'I':61,'L':61,'K':110,'M':67,'F':40,'P':147,'S':177,'T':128,'Y':37,'V':88},
'Y':{'A':112,'R':77,'N':143,'D':160,'C':194,'Q':99,'E':122,'G':147,'H':83,'I':33,'L':36,'K':85,'M':36,'F':22,'P':110,'S':144,'T':92,'W':37,'V':55},
'V':{'A':64,'R':96,'N':133,'D':152,'C':192,'Q':96,'E':121,'G':109,'H':84,'I':29,'L':32,'K':97,'M':21,'F':50,'P':68,'S':124,'T':69,'W':88,'Y':55,'V':0}}
_B = """\
   A  R  N  D  C  Q  E  G  H  I  L  K  M  F  P  S  T  W  Y  V
A  4 -1 -2 -2  0 -1 -1  0 -2 -1 -1 -1 -1 -2 -1  1  0 -3 -2  0
R -1  5  0 -2 -3  1  0 -2  0 -3 -2  2 -1 -3 -2 -1 -1 -3 -2 -3
N -2  0  6  1 -3  0  0  0  1 -3 -3  0 -2 -3 -2  1  0 -4 -2 -3
D -2 -2  1  6 -3   0  2 -1 -1 -3 -4 -1 -3 -3 -1  0 -1 -4 -3 -3
C  0 -3 -3 -3  9 -3 -4 -3 -3 -1 -1 -3 -1 -2 -3 -1 -1 -2 -2 -1
Q -1  1  0  0 -3  5  2 -2  0 -3 -2  1  0 -3 -1  0 -1 -2 -1 -2
E -1  0  0  2 -4  2  5 -2  0 -3 -3  1 -2 -3 -1  0 -1 -3 -2 -2
G  0 -2  0 -1 -3 -2 -2  6 -2 -4 -4 -2 -3 -3 -2  0 -2 -2 -3 -3
H -2  0  1 -1 -3  0  0 -2  8 -3 -3 -1 -2 -1 -2 -1 -2 -2  2 -3
I -1 -3 -3 -3 -1 -3 -3 -4 -3  4  2 -3  1  0 -3 -2 -1 -3 -1  3
L -1 -2 -3 -4 -1 -2 -3 -4 -3  2  4 -2  2  0 -3 -2 -1 -2 -1  1
K -1  2  0 -1 -3  1  1 -2 -1 -3 -2  5 -1 -3 -1  0 -1 -3 -2 -2
M -1 -1 -2 -3 -1  0 -2 -3 -2  1  2 -1  5  0 -2 -1 -1 -1 -1  1
F -2 -3 -3 -3 -2 -3 -3 -3 -1  0  0 -3  0  6 -4 -2 -2  1  3 -1
P -1 -2 -2 -1 -3 -1 -1 -2 -2 -3 -3 -1 -2 -4  7 -1 -1 -4 -3 -2
S  1 -1  1  0 -1  0  0  0 -1 -2 -2  0 -1 -2 -1  4  1 -3 -2 -2
T  0 -1  0 -1 -1 -1 -1 -2 -2 -1 -1 -1 -1 -2 -1  1  5 -2 -2  0
W -3 -3 -4 -4 -2 -2 -3 -2 -2 -3 -2 -3 -1  1 -4 -3 -2 11  2 -3
Y -2 -2 -2 -3 -2 -1 -2 -3  2 -1 -1 -2 -1  3 -3 -2 -2  2  7 -1
V  0 -3 -3 -3 -1 -2 -2 -3 -3  3  1 -2  1 -1 -2 -2  0 -3 -1  4
"""
B62={}; _hdr=_B.strip().splitlines()[0].split()
for line in _B.strip().splitlines()[1:]:
    parts=line.split(); a=parts[0]; B62[a]={_hdr[i]:int(parts[i+1]) for i in range(len(_hdr))}
VOCAB={"A":0,"C":1,"G":2,"T":3,"N":4}
def onehot(seq: str):
    arr = np.zeros((5,len(seq)), np.float32)
    for i,ch in enumerate(seq): arr[VOCAB.get(ch,4), i] = 1.0
    return arr
def grantham(a,b):
    if a==b: return 0
    return _GR.get(a,{}).get(b,_GR.get(b,{}).get(a,0))
def blosum62(a,b):
    return B62.get(a,{}).get(b,B62.get(b,{}).get(a,0))
def aa_scalars(wt_codon, mut_codon):
    a = CODON2AA.get(str(wt_codon), "X"); b = CODON2AA.get(str(mut_codon), "X")
    aa_same = float(a==b)
    if a=="X" or b=="X": return np.array([aa_same,0.0,0.0], dtype=np.float32)
    return np.array([aa_same, grantham(a,b)/215.0, (blosum62(a,b)+4)/15.0], dtype=np.float32)
def mut_index_in_window(row, flank=60):
    i0 = int(row["cds_pos"]) - 1; cs = (i0 // 3) * 3; center = cs + 1
    return flank + (i0 - center)
def extract_patch(win: str, j: int, R: int = 30):
    s = str(win)
    if not s or j<0 or j>=len(s): return ""
    a,b = max(0, j-R), min(len(s), j+R+1)
    left  = "N" * (R - (j - a)); right = "N" * (R - (b - j - 1))
    return left + s[a:b] + right

class PairTrainDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, pairs, R=30):
        self.df = frame; self.pairs = pairs; self.R = R
    def __len__(self): return len(self.pairs)
    def _patch_feats(self, r):
        j = mut_index_in_window(r, flank=60)
        mut = extract_patch(r["window_mut"], j, self.R)
        wt  = extract_patch(r["window_wt"],  j, self.R)
        scal = aa_scalars(r["wt_codon"], r["mut_codon"])
        return torch.from_numpy(onehot(mut)), torch.from_numpy(onehot(wt)), torch.from_numpy(scal)
    def __getitem__(self, k):
        ip, ineg = self.pairs[k]
        rp, rn = self.df.loc[ip], self.df.loc[ineg]
        xm_p, xw_p, s_p = self._patch_feats(rp)
        xm_n, xw_n, s_n = self._patch_feats(rn)
        return xm_p, xw_p, s_p, xm_n, xw_n, s_n

class VariantEvalDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, R=30):
        self.df = frame.reset_index(); self.R = R
        self.items=[]
        for _, r in self.df.iterrows():
            j = mut_index_in_window(r, flank=60)
            mut = extract_patch(r["window_mut"], j, self.R)
            wt  = extract_patch(r["window_wt"],  j, self.R)
            scal = aa_scalars(r["wt_codon"], r["mut_codon"])
            self.items.append((onehot(mut), onehot(wt), scal, float(r["label"])))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        mut, wt, scal, y = self.items[i]
        return (torch.from_numpy(mut), torch.from_numpy(wt),
                torch.from_numpy(scal), torch.tensor(y, dtype=torch.float32))

class SiameseDeltaCNNFlex(nn.Module):
    def __init__(self, in_ch=5, hid=96, scal_dim=3, dropout=0.1):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv1d(in_ch, 64, kernel_size=11, padding=5), nn.ReLU(),
            nn.Conv1d(64, 96, kernel_size=7, padding=3),     nn.ReLU(),
            nn.Conv1d(96, 96, kernel_size=3, padding=1),     nn.ReLU(),
            nn.AdaptiveMaxPool1d(1)
        )
        self.norm = nn.LayerNorm(96)
        self.head = nn.Sequential(nn.Linear(96, hid), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hid,1))
        self.scal_head = nn.Linear(scal_dim, 1)
    def encode(self, x):
        h = self.enc(x).squeeze(-1)
        return self.norm(h)
    def score(self, xm, xw, scalars):
        hm = self.encode(xm); hw = self.encode(xw)
        core = self.head(hm - hw).squeeze(-1)
        scal = self.scal_head(scalars).squeeze(-1)
        return core + scal

def make_pairs_trimatched(train_df: pd.DataFrame, seed=1337):
    rs=np.random.RandomState(seed); pairs=[]; used=set()
    sub=train_df.copy(); sub["center_base"]=sub["tri_center"].str[1]
    for g, grp in sub.groupby("gene"):
        pos=grp[grp["label"]==1]; neg=grp[grp["label"]==0].copy()
        for tri, pgrp in pos.groupby("tri_center"):
            pool=neg[(neg["tri_center"]==tri)&(~neg.index.isin(used))]
            if pool.empty: continue
            k=min(len(pgrp), len(pool))
            pidx=rs.choice(pgrp.index, size=k, replace=False)
            nidx=rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx,nidx)); used.update(nidx)
        leftover=pos.loc[~pos.index.isin([p for p,_ in pairs])]
        for base, pgrp in leftover.groupby("center_base"):
            pool=neg[(neg["tri_center"].str[1]==base)&(~neg.index.isin(used))]
            if pool.empty: continue
            k=min(len(pgrp), len(pool))
            pidx=rs.choice(pgrp.index, size=k, replace=False)
            nidx=rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx,nidx)); used.update(nidx)
    return pairs

def split_train_val(tr_df, seed=123, frac=0.2):
    # leak-free: within-train split, stratify by (label, tri_center)
    rs = np.random.RandomState(seed)
    tr_df = tr_df.copy()
    idx_tr=[]; idx_va=[]
    for (_, lab, tri), grp in tr_df.groupby(["gene","label","tri_center"]):
        idx = grp.index.values
        rs.shuffle(idx)
        n_va = max(1, int(len(idx)*frac))
        idx_va += idx[:n_va].tolist()
        idx_tr += idx[n_va:].tolist()
    return tr_df.loc[idx_tr].copy(), tr_df.loc[idx_va].copy()

def mine_hard_pairs(train_df: pd.DataFrame, model, R=45, k=1, neg_cap=2):
    # score train variants
    ds = VariantEvalDataset(train_df, R=R)
    loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
    model.eval(); scores=[]
    with torch.no_grad():
        for xm, xw, s, y in loader:
            xm, xw, s = xm.to(device), xw.to(device), s.to(device)
            scores.append(model.score(xm, xw, s).cpu().numpy())
    scores = np.concatenate(scores)
    scored = train_df.reset_index().copy()
    scored["logit"] = scores
    pairs=[]; used={}
    # exact tri
    for tri, grp in scored.groupby("tri_center"):
        pos_idx = grp[grp["label"]==1]["index"].tolist()
        neg_sorted = grp[grp["label"]==0].sort_values("logit", ascending=False)["index"].tolist()
        for ip in pos_idx:
            cnt=0
            for ineg in neg_sorted:
                if used.get(ineg,0) < neg_cap:
                    pairs.append((ip, ineg)); used[ineg]=used.get(ineg,0)+1; cnt+=1
                if cnt>=k: break
    # fallback by center base for any positive under-paired
    scored["center_base"] = scored["tri_center"].str[1]
    pos_has = {}
    for ip, _ in pairs: pos_has[ip] = pos_has.get(ip,0)+1
    for base, grp in scored.groupby("center_base"):
        pos_idx = grp[grp["label"]==1]["index"].tolist()
        neg_sorted = grp[grp["label"]==0].sort_values("logit", ascending=False)["index"].tolist()
        for ip in pos_idx:
            need = max(0, k - pos_has.get(ip,0))
            for ineg in neg_sorted:
                if need==0: break
                if used.get(ineg,0) < neg_cap:
                    pairs.append((ip, ineg)); used[ineg]=used.get(ineg,0)+1; need-=1
            pos_has[ip]=k
    return pairs

def train_on_pairs(df_train, pairs, R, hid, dropout, lr, wd, epochs):
    model = SiameseDeltaCNNFlex(hid=hid, dropout=dropout).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    bce = nn.BCEWithLogitsLoss()
    loader = DataLoader(PairTrainDataset(df_train, pairs, R=R), batch_size=96, shuffle=True, num_workers=2, pin_memory=True)
    for _ in range(epochs):
        model.train()
        for xm_p, xw_p, s_p, xm_n, xw_n, s_n in loader:
            xm_p, xw_p, s_p = xm_p.to(device), xw_p.to(device), s_p.to(device)
            xm_n, xw_n, s_n = xm_n.to(device), xw_n.to(device), s_n.to(device)
            opt.zero_grad(); diff = model.score(xm_p,xw_p,s_p)-model.score(xm_n,xw_n,s_n)
            loss = bce(diff, torch.ones_like(diff)); loss.backward(); opt.step()
    return model

def eval_variants(model, df_eval, R):
    ds = VariantEvalDataset(df_eval, R=R)
    loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
    model.eval(); logits=[]; ys=[]
    with torch.no_grad():
        for xm, xw, s, y in loader:
            xm, xw, s = xm.to(device), xw.to(device), s.to(device)
            logits.append(model.score(xm, xw, s).cpu().numpy()); ys.append(y.numpy())
    y = np.concatenate(ys).astype(int); p = 1/(1+np.exp(-np.concatenate(logits)))
    auroc = roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan
    pr    = average_precision_score(y,p) if len(np.unique(y))>1 else np.nan
    brier = brier_score_loss(y,p)
    return auroc, pr, brier

# ---------- grid ----------
GRID = [
    {"R":30,"hid":96,"dropout":0.0,"lr":1e-3,"wd":1e-4,"k_hard":0,"E_PRE":10,"E_MIN":0,"neg_cap":2},
    {"R":30,"hid":96,"dropout":0.1,"lr":2e-3,"wd":1e-4,"k_hard":0,"E_PRE":12,"E_MIN":0,"neg_cap":2},
    {"R":45,"hid":96,"dropout":0.0,"lr":1e-3,"wd":1e-4,"k_hard":0,"E_PRE":10,"E_MIN":0,"neg_cap":2},
    {"R":45,"hid":96,"dropout":0.1,"lr":2e-3,"wd":1e-4,"k_hard":0,"E_PRE":12,"E_MIN":0,"neg_cap":2},
    {"R":45,"hid":96,"dropout":0.1,"lr":1e-3,"wd":1e-4,"k_hard":1,"E_PRE":8,"E_MIN":4,"neg_cap":2},
    {"R":45,"hid":96,"dropout":0.1,"lr":2e-3,"wd":1e-4,"k_hard":1,"E_PRE":8,"E_MIN":4,"neg_cap":2},
    {"R":30,"hid":96,"dropout":0.1,"lr":1e-3,"wd":1e-4,"k_hard":1,"E_PRE":8,"E_MIN":4,"neg_cap":2},
    {"R":30,"hid":96,"dropout":0.1,"lr":2e-3,"wd":1e-4,"k_hard":1,"E_PRE":8,"E_MIN":4,"neg_cap":2},
]
SEEDS = [1337, 2401]  # tiny bag for speed

# ---------- inner-val sweep (no leakage) ----------
val_rows=[]
for cfg_id, cfg in enumerate(GRID):
    rows_cfg=[]
    for fold, sub in spl_g.groupby("fold"):
        dd = df.merge(sub, on=["variant_id","gene"], how="inner")
        tr = dd[dd["split"]=="train"].copy()
        if tr.empty: continue
        # inner split
        tr_in, va_in = split_train_val(tr, seed=cfg_id+4242, frac=0.2)
        # seed bag on inner val
        mets=[]
        for sd in SEEDS:
            random.seed(sd); np.random.seed(sd); torch.manual_seed(sd)
            pairs0 = make_pairs_trimatched(tr_in, seed=sd)
            model = train_on_pairs(tr_in, pairs0, cfg["R"], cfg["hid"], cfg["dropout"], cfg["lr"], cfg["wd"], cfg["E_PRE"])
            if cfg["k_hard"]>0 and cfg["E_MIN"]>0:
                pairs_h = mine_hard_pairs(tr_in, model, R=cfg["R"], k=cfg["k_hard"], neg_cap=cfg["neg_cap"])
                model = train_on_pairs(tr_in, pairs_h, cfg["R"], cfg["hid"], cfg["dropout"], cfg["lr"], cfg["wd"], cfg["E_MIN"])
            au, pr, br = eval_variants(model, va_in, cfg["R"])
            mets.append((au,pr,br))
        # average across seeds
        au = float(np.nanmean([m[0] for m in mets])); pr=float(np.nanmean([m[1] for m in mets])); br=float(np.mean([m[2] for m in mets]))
        rows_cfg.append({"cfg_id":cfg_id, **cfg, "fold":fold, "gene":tr["gene"].unique().tolist(), "n_val":len(va_in), "val_auroc":au, "val_pr":pr, "val_brier":br})
    val_rows += rows_cfg

val_df = pd.DataFrame(val_rows)
# aggregate across folds (weighted by n_val)
agg = (val_df.groupby("cfg_id")
       .apply(lambda g: pd.Series({
           "mean_val_auroc": np.average(g["val_auroc"], weights=g["n_val"]),
           "mean_val_pr":    np.average(g["val_pr"],    weights=g["n_val"]),
           "mean_val_brier": np.average(g["val_brier"], weights=g["n_val"]),
           "R": g["R"].iloc[0], "dropout": g["dropout"].iloc[0], "lr": g["lr"].iloc[0],
           "k_hard": g["k_hard"].iloc[0]
       }))
       .reset_index().sort_values("mean_val_auroc", ascending=False)
      )
print("=== Inner-val sweep (weighted across folds) ===")
display(agg.head(8))

best_cfg_id = int(agg.iloc[0]["cfg_id"])
best_cfg = GRID[best_cfg_id]
print("Best cfg:", best_cfg)

# ---------- retrain best on full TRAIN per LOCO fold, eval TEST ----------
test_rows=[]
for fold, sub in spl_g.groupby("fold"):
    dd = df.merge(sub, on=["variant_id","gene"], how="inner")
    tr = dd[dd["split"]=="train"].copy()
    te = dd[dd["split"]=="test"].copy()
    if tr.empty or te.empty: continue
    mets=[]
    for sd in SEEDS:
        random.seed(sd); np.random.seed(sd); torch.manual_seed(sd)
        pairs0 = make_pairs_trimatched(tr, seed=sd)
        model = train_on_pairs(tr, pairs0, best_cfg["R"], best_cfg["hid"], best_cfg["dropout"], best_cfg["lr"], best_cfg["wd"], best_cfg["E_PRE"])
        if best_cfg["k_hard"]>0 and best_cfg["E_MIN"]>0:
            pairs_h = mine_hard_pairs(tr, model, R=best_cfg["R"], k=best_cfg["k_hard"], neg_cap=best_cfg["neg_cap"])
            model = train_on_pairs(tr, pairs_h, best_cfg["R"], best_cfg["hid"], best_cfg["dropout"], best_cfg["lr"], best_cfg["wd"], best_cfg["E_MIN"])
        au, pr, br = eval_variants(model, te, best_cfg["R"])
        mets.append((au,pr,br))
    au = float(np.nanmean([m[0] for m in mets])); pr=float(np.nanmean([m[1] for m in mets])); br=float(np.mean([m[2] for m in mets]))
    test_rows.append({"fold":fold, "gene":te["gene"].iloc[0], "n":len(te), "auroc":round(au,4), "pr_auc":round(pr,4), "brier":round(br,4)})
test_df = pd.DataFrame(test_rows).sort_values("gene")
print("\n=== LOCO-GENE TEST with best cfg (bagged over seeds) ===")
display(test_df)

OUT = ARTI / f"hpo_lite_{time.strftime('%Y%m%d_%H%M%S')}"
OUT.mkdir(parents=True, exist_ok=True)
agg.to_csv(OUT/"inner_val_sweep.csv", index=False)
test_df.to_csv(OUT/"test_metrics_best_cfg.csv", index=False)
print("Saved to:", OUT)


=== Inner-val sweep (weighted across folds) ===


/tmp/ipython-input-104824597.py:308: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,cfg_id,mean_val_auroc,mean_val_pr,mean_val_brier,R,dropout,lr,k_hard
2,2,0.694485,0.671928,0.241059,45.0,0.0,0.001,0.0
1,1,0.688180,0.655514,0.229256,30.0,0.1,0.002,0.0
0,0,0.677168,0.644414,0.240273,30.0,0.0,0.001,0.0
3,3,0.652593,0.638097,0.235979,45.0,0.1,0.002,0.0
5,5,0.556911,0.537331,0.258720,45.0,0.1,0.002,1.0
6,6,0.549741,0.533680,0.259537,30.0,0.1,0.001,1.0
7,7,0.545309,0.524902,0.259016,30.0,0.1,0.002,1.0
4,4,0.536284,0.536541,0.259451,45.0,0.1,0.001,1.0


Best cfg: {'R': 45, 'hid': 96, 'dropout': 0.0, 'lr': 0.001, 'wd': 0.0001, 'k_hard': 0, 'E_PRE': 10, 'E_MIN': 0, 'neg_cap': 2}

=== LOCO-GENE TEST with best cfg (bagged over seeds) ===


,fold,gene,n,auroc,pr_auc,brier
0,loco_gene:EGFR,EGFR,39,0.5529,0.5153,0.2446
1,loco_gene:KRAS,KRAS,11,0.2917,0.6569,0.2837
2,loco_gene:PIK3CA,PIK3CA,24,0.4769,0.7913,0.2794
3,loco_gene:TP53,TP53,197,0.5167,0.3849,0.2463


Saved to: /content/drive/MyDrive/Cancer_gene_database/artifacts/hpo_lite_20250816_223541


**Not what we were looking for**

**HPO-lite** tanked because we tuned on within-gene splits, then tested on held-out gene. That’s a mismatch. If the inner loop doesn’t simulate “new gene,” it’ll pick configs that overfit gene-specific quirks (PIK3CA got smoked, EGFR wobbed, TP53 sagged).

**Let’s fix the protocol**: nested LOCO-GENE HPO. For each outer fold (held-out gene), we do inner CV that also leaves out a gene from the training genes (GroupKFold by gene), pick the best config there, then evaluate on the real test gene. No leakage, objective matches deployment.

### LOCO-GENE HPO

In [10]:
# HPO-NESTED — inner CV by gene (GroupKFold) aligned to LOCO-GENE
# Grid: R ∈ {30,45}, dropout ∈ {0.0, 0.1}, lr ∈ {1e-3, 2e-3}; no hard-mining (keeps it stable).
# We select best per outer fold using ONLY train genes; then eval on that fold’s held-out gene.

import os, glob, time, random
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.model_selection import GroupKFold

# ---------- data ----------
BASE = Path("/content/drive/MyDrive/Cancer_gene_database")
DATA = BASE/"data"; ARTI = BASE/"artifacts"
P_RES = DATA/"masterseq_model_ready_plus_resolved.parquet"
runs = sorted(glob.glob(str(ARTI / "sentiment_run_plus_variants_*"))); assert runs, "No split dirs found."
RUN_DIR = Path(runs[-1]); SPLITS_G = RUN_DIR/"splits_loco_gene.csv"

df = pd.read_parquet(P_RES).dropna(subset=[
    "variant_id","gene","label","window_mut","window_wt",
    "wt_codon","mut_codon","ref","alt","cds_pos","tri_center"
]).copy()
for c in ["variant_id","gene","ref","alt","wt_codon","mut_codon","window_mut","window_wt","tri_center"]:
    df[c] = df[c].astype(str).str.upper()
df["label"] = df["label"].astype(int)
df["cds_pos"] = pd.to_numeric(df["cds_pos"], errors="coerce").astype("Int64")
spl_g = pd.read_csv(SPLITS_G)

device = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS = [1337, 2401]   # small bag for stability vs speed
EPOCHS = 12
WD = 1e-4
GRID = [
    {"R":30, "dropout":0.0, "lr":1e-3},
    {"R":30, "dropout":0.1, "lr":2e-3},
    {"R":45, "dropout":0.0, "lr":1e-3},
    {"R":45, "dropout":0.1, "lr":2e-3},
]

# ---------- minimal model/datasets ----------
CODON2AA = {
 "TTT":"F","TTC":"F","TTA":"L","TTG":"L","CTT":"L","CTC":"L","CTA":"L","CTG":"L",
 "ATT":"I","ATC":"I","ATA":"I","ATG":"M","GTT":"V","GTC":"V","GTA":"V","GTG":"V",
 "TCT":"S","TCC":"S","TCA":"S","TCG":"S","CCT":"P","CCC":"P","CCA":"P","CCG":"P",
 "ACT":"T","ACC":"T","ACA":"T","ACG":"T","GCT":"A","GCC":"A","GCA":"A","GCG":"A",
 "TAT":"Y","TAC":"Y","TAA":"*","TAG":"*","CAT":"H","CAC":"H","CAA":"Q","CAG":"Q",
 "AAT":"N","AAC":"N","AAA":"K","AAG":"K","GAT":"D","GAC":"D","GAA":"E","GAG":"E",
 "TGT":"C","TGC":"C","TGA":"*","TGG":"W","CGT":"R","CGC":"R","CGA":"R","CGG":"R",
 "AGT":"S","AGC":"S","AGA":"R","AGG":"R","GGT":"G","GGC":"G","GGA":"G","GGG":"G"
}
_GR={'A':{'A':0,'R':112,'N':111,'D':126,'C':195,'Q':91,'E':107,'G':60,'H':86,'I':94,'L':96,'K':106,'M':84,'F':113,'P':27,'S':99,'T':58,'W':148,'Y':112,'V':64},'R':{'A':112,'R':0,'N':86,'D':96,'C':180,'Q':43,'E':54,'G':125,'H':29,'I':97,'L':102,'K':26,'M':91,'F':97,'P':103,'S':110,'T':71,'W':101,'Y':77,'V':96},'N':{'A':111,'R':86,'N':0,'D':23,'C':139,'Q':68,'E':42,'G':80,'H':68,'I':149,'L':153,'K':94,'M':142,'F':158,'P':91,'S':46,'T':65,'W':174,'Y':143,'V':133},'D':{'A':126,'R':96,'N':23,'D':0,'C':154,'Q':61,'E':45,'G':94,'H':81,'I':168,'L':172,'K':101,'M':160,'F':177,'P':108,'S':65,'T':85,'W':181,'Y':160,'V':152},'C':{'A':195,'R':180,'N':139,'D':154,'C':0,'Q':154,'E':170,'G':159,'H':174,'I':198,'L':198,'K':202,'M':196,'F':205,'P':169,'S':112,'T':149,'W':215,'Y':194,'V':192},'Q':{'A':91,'R':43,'N':68,'D':61,'C':154,'Q':0,'E':29,'G':87,'H':24,'I':109,'L':113,'K':53,'M':101,'F':116,'P':76,'S':68,'T':42,'W':130,'Y':99,'V':96},'E':{'A':107,'R':54,'N':42,'D':45,'C':170,'Q':29,'G':98,'H':40,'I':134,'L':138,'K':56,'M':126,'F':140,'P':93,'S':80,'T':65,'W':152,'Y':122,'V':121},'G':{'A':60,'R':125,'N':80,'D':94,'C':159,'Q':87,'E':98,'H':98,'I':135,'L':138,'K':127,'M':127,'F':153,'P':42,'S':56,'T':59,'W':184,'Y':147,'V':109},'H':{'A':86,'R':29,'N':68,'D':81,'C':174,'Q':24,'E':40,'G':98,'I':94,'L':99,'K':32,'M':87,'F':100,'P':77,'S':89,'T':47,'W':115,'Y':83,'V':84},'I':{'A':94,'R':97,'N':149,'D':168,'C':198,'Q':109,'E':134,'G':135,'H':94,'L':5,'K':102,'M':10,'F':21,'P':95,'S':142,'T':89,'W':61,'Y':33,'V':29},'L':{'A':96,'R':102,'N':153,'D':172,'C':198,'Q':113,'E':138,'G':138,'H':99,'I':5,'L':0,'K':107,'M':15,'F':22,'P':98,'S':145,'T':92,'W':61,'Y':36,'V':32},'K':{'A':106,'R':26,'N':94,'D':101,'C':202,'Q':53,'E':56,'G':127,'H':32,'I':102,'L':107,'K':0,'M':95,'F':102,'P':103,'S':121,'T':78,'W':110,'Y':85,'V':97},'M':{'A':84,'R':91,'N':142,'D':160,'C':196,'Q':101,'E':126,'G':127,'H':87,'I':10,'L':15,'K':95,'F':28,'P':87,'S':135,'T':81,'W':67,'Y':36,'V':21},'F':{'A':113,'R':97,'N':158,'D':177,'C':205,'Q':116,'E':140,'G':153,'H':100,'I':21,'L':22,'K':102,'M':28,'P':114,'S':155,'T':103,'W':40,'Y':22,'V':50},'P':{'A':27,'R':103,'N':91,'D':108,'C':169,'Q':76,'E':93,'G':42,'S':74,'T':38,'W':147,'Y':110,'V':68},'S':{'A':99,'R':110,'N':46,'D':65,'C':112,'Q':68,'E':80,'G':56,'H':89,'I':142,'L':145,'K':121,'M':135,'F':155,'P':74,'T':58,'W':177,'Y':144,'V':124},'T':{'A':58,'R':71,'N':65,'D':85,'C':149,'Q':42,'E':65,'G':59,'H':47,'I':89,'L':92,'K':78,'M':81,'F':103,'P':38,'S':58,'W':128,'Y':92,'V':69},'W':{'A':148,'R':101,'N':174,'D':181,'C':215,'Q':130,'E':152,'G':184,'H':115,'I':61,'L':61,'K':110,'M':67,'F':40,'P':147,'S':177,'T':128,'Y':37,'V':88},'Y':{'A':112,'R':77,'N':143,'D':160,'C':194,'Q':99,'E':122,'G':147,'H':83,'I':33,'L':36,'K':85,'M':36,'F':22,'P':110,'S':144,'T':92,'W':37,'V':55},'V':{'A':64,'R':96,'N':133,'D':152,'C':192,'Q':96,'E':121,'G':109,'H':84,'I':29,'L':32,'K':97,'M':21,'F':50,'P':68,'S':124,'T':69,'W':88,'Y':55,'V':0}}
_B = """\
   A  R  N  D  C  Q  E  G  H  I  L  K  M  F  P  S  T  W  Y  V
A  4 -1 -2 -2  0 -1 -1  0 -2 -1 -1 -1 -1 -2 -1  1  0 -3 -2  0
R -1  5  0 -2 -3  1  0 -2  0 -3 -2  2 -1 -3 -2 -1 -1 -3 -2 -3
N -2  0  6  1 -3  0  0  0  1 -3 -3  0 -2 -3 -2  1  0 -4 -2 -3
D -2 -2  1  6 -3  0  2 -1 -1 -3 -4 -1 -3 -3 -1  0 -1 -4 -3 -3
C  0 -3 -3 -3  9 -3 -4 -3 -3 -1 -1 -3 -1 -2 -3 -1 -1 -2 -2 -1
Q -1  1  0  0 -3  5  2 -2  0 -3 -2  1  0 -3 -1  0 -1 -2 -1 -2
E -1  0  0  2 -4  2  5 -2  0 -3 -3  1 -2 -3 -1  0 -1 -3 -2 -2
G  0 -2  0 -1 -3 -2 -2  6 -2 -4 -4 -2 -3 -3 -2  0 -2 -2 -3 -3
H -2  0  1 -1 -3  0  0 -2  8 -3 -3 -1 -2 -1 -2 -1 -2 -2  2 -3
I -1 -3 -3 -3 -1 -3 -3 -4 -3  4  2 -3  1  0 -3 -2 -1 -3 -1  3
L -1 -2 -3 -4 -1 -2 -3 -4 -3  2  4 -2  2  0 -3 -2 -1 -2 -1  1
K -1  2  0 -1 -3  1  1 -2 -1 -3 -2  5 -1 -3 -1  0 -1 -3 -2 -2
M -1 -1 -2 -3 -1  0 -2 -3 -2  1  2 -1  5  0 -2 -1 -1 -1 -1  1
F -2 -3 -3 -3 -2 -3 -3 -3 -1  0  0 -3  0  6 -4 -2 -2  1  3 -1
P -1 -2 -2 -1 -3 -1 -1 -2 -2 -3 -3 -1 -2 -4  7 -1 -1 -4 -3 -2
S  1 -1  1  0 -1  0  0  0 -1 -2 -2  0 -1 -2 -1  4  1 -3 -2 -2
T  0 -1  0 -1 -1 -1 -1 -2 -2 -1 -1 -1 -1 -2 -1  1  5 -2 -2  0
W -3 -3 -4 -4 -2 -2 -3 -2 -2 -3 -2 -3 -1  1 -4 -3 -2 11  2 -3
Y -2 -2 -2 -3 -2 -1 -2 -3  2 -1 -1 -2 -1  3 -3 -2 -2  2  7 -1
V  0 -3 -3 -3 -1 -2 -2 -3 -3  3  1 -2  1 -1 -2 -2  0 -3 -1  4
"""
B62={}; _hdr=_B.strip().splitlines()[0].split()
for line in _B.strip().splitlines()[1:]:
    parts=line.split(); a=parts[0]; B62[a]={_hdr[i]:int(parts[i+1]) for i in range(len(_hdr))}
VOCAB={"A":0,"C":1,"G":2,"T":3,"N":4}
def onehot(seq: str):
    arr = np.zeros((5,len(seq)), np.float32)
    for i,ch in enumerate(seq): arr[VOCAB.get(ch,4), i] = 1.0
    return arr
def grantham(a,b):
    if a==b: return 0
    return _GR.get(a,{}).get(b,_GR.get(b,{}).get(a,0))
def blosum62(a,b):
    return B62.get(a,{}).get(b,B62.get(b,{}).get(a,0))
def aa_scalars(wt_codon, mut_codon):
    a = CODON2AA.get(str(wt_codon), "X"); b = CODON2AA.get(str(mut_codon), "X")
    aa_same = float(a==b)
    if a=="X" or b=="X": return np.array([aa_same,0.0,0.0], dtype=np.float32)
    return np.array([aa_same, grantham(a,b)/215.0, (blosum62(a,b)+4)/15.0], dtype=np.float32)
def mut_index_in_window(row, flank=60):
    i0 = int(row["cds_pos"]) - 1; cs = (i0 // 3) * 3; center = cs + 1
    return flank + (i0 - center)
def extract_patch(win: str, j: int, R: int = 30):
    s = str(win)
    if not s or j<0 or j>=len(s): return ""
    a,b = max(0, j-R), min(len(s), j+R+1)
    left  = "N" * (R - (j - a)); right = "N" * (R - (b - j - 1))
    return left + s[a:b] + right

class PairTrainDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, pairs, R=30):
        self.df = frame; self.pairs = pairs; self.R = R
    def __len__(self): return len(self.pairs)
    def _patch_feats(self, r):
        j = mut_index_in_window(r, flank=60)
        mut = extract_patch(r["window_mut"], j, self.R)
        wt  = extract_patch(r["window_wt"],  j, self.R)
        scal = aa_scalars(r["wt_codon"], r["mut_codon"])
        return torch.from_numpy(onehot(mut)), torch.from_numpy(onehot(wt)), torch.from_numpy(scal)
    def __getitem__(self, k):
        ip, ineg = self.pairs[k]
        rp, rn = self.df.loc[ip], self.df.loc[ineg]
        xm_p, xw_p, s_p = self._patch_feats(rp)
        xm_n, xw_n, s_n = self._patch_feats(rn)
        return xm_p, xw_p, s_p, xm_n, xw_n, s_n

class VariantEvalDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, R=30):
        self.df = frame.reset_index(); self.R = R
        self.items=[]
        for _, r in self.df.iterrows():
            j = mut_index_in_window(r, flank=60)
            mut = extract_patch(r["window_mut"], j, self.R)
            wt  = extract_patch(r["window_wt"],  j, self.R)
            scal = aa_scalars(r["wt_codon"], r["mut_codon"])
            self.items.append((onehot(mut), onehot(wt), scal, float(r["label"])))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        mut, wt, scal, y = self.items[i]
        return (torch.from_numpy(mut), torch.from_numpy(wt),
                torch.from_numpy(scal), torch.tensor(y, dtype=torch.float32))

class SiameseDeltaCNN(nn.Module):
    def __init__(self, in_ch=5, hid=96, scal_dim=3, dropout=0.0):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv1d(in_ch, 64, kernel_size=11, padding=5), nn.ReLU(),
            nn.Conv1d(64, 96, kernel_size=7, padding=3),     nn.ReLU(),
            nn.Conv1d(96, 96, kernel_size=3, padding=1),     nn.ReLU(),
            nn.AdaptiveMaxPool1d(1)
        )
        self.norm = nn.LayerNorm(96)
        self.head = nn.Sequential(nn.Linear(96,96), nn.ReLU(), nn.Dropout(dropout), nn.Linear(96,1))
        self.scal_head = nn.Linear(scal_dim, 1)
    def encode(self, x):
        h = self.enc(x).squeeze(-1)
        return self.norm(h)
    def score(self, xm, xw, scalars):
        hm = self.encode(xm); hw = self.encode(xw)
        core = self.head(hm - hw).squeeze(-1)
        scal = self.scal_head(scalars).squeeze(-1)
        return core + scal

def make_pairs_trimatched(train_df: pd.DataFrame, seed=1337):
    rs=np.random.RandomState(seed); pairs=[]; used=set()
    sub=train_df.copy(); sub["center_base"]=sub["tri_center"].str[1]
    for g, grp in sub.groupby("gene"):
        pos=grp[grp["label"]==1]; neg=grp[grp["label"]==0].copy()
        for tri, pgrp in pos.groupby("tri_center"):
            pool=neg[(neg["tri_center"]==tri)&(~neg.index.isin(used))]
            if pool.empty: continue
            k=min(len(pgrp), len(pool))
            pidx=rs.choice(pgrp.index, size=k, replace=False)
            nidx=rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx,nidx)); used.update(nidx)
        leftover=pos.loc[~pos.index.isin([p for p,_ in pairs])]
        for base, pgrp in leftover.groupby("center_base"):
            pool=neg[(neg["tri_center"].str[1]==base)&(~neg.index.isin(used))]
            if pool.empty: continue
            k=min(len(pgrp), len(pool))
            pidx=rs.choice(pgrp.index, size=k, replace=False)
            nidx=rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx,nidx)); used.update(nidx)
    return pairs

def train_pairs(df_train, pairs, R, dropout, lr, epochs=EPOCHS):
    model = SiameseDeltaCNN(dropout=dropout).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WD)
    bce = nn.BCEWithLogitsLoss()
    loader = DataLoader(PairTrainDataset(df_train, pairs, R=R), batch_size=96, shuffle=True, num_workers=2, pin_memory=True)
    for _ in range(epochs):
        model.train()
        for xm_p, xw_p, s_p, xm_n, xw_n, s_n in loader:
            xm_p, xw_p, s_p = xm_p.to(device), xw_p.to(device), s_p.to(device)
            xm_n, xw_n, s_n = xm_n.to(device), xw_n.to(device), s_n.to(device)
            opt.zero_grad()
            diff = model.score(xm_p, xw_p, s_p) - model.score(xm_n, xw_n, s_n)
            loss = bce(diff, torch.ones_like(diff)); loss.backward(); opt.step()
    return model

def eval_variants(model, df_eval, R):
    ds = VariantEvalDataset(df_eval, R=R)
    loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
    model.eval(); logits=[]; ys=[]
    with torch.no_grad():
        for xm, xw, s, y in loader:
            xm, xw, s = xm.to(device), xw.to(device), s.to(device)
            logits.append(model.score(xm, xw, s).cpu().numpy()); ys.append(y.numpy())
    y = np.concatenate(ys).astype(int); p = 1/(1+np.exp(-np.concatenate(logits)))
    auroc = roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan
    pr    = average_precision_score(y,p) if len(np.unique(y))>1 else np.nan
    brier = brier_score_loss(y,p)
    return auroc, pr, brier

# ---------- nested HPO ----------
outer_rows=[]; test_rows=[]; picks=[]
for fold, sub in spl_g.groupby("fold"):
    dd = df.merge(sub, on=["variant_id","gene"], how="inner")
    tr = dd[dd["split"]=="train"].copy()
    te = dd[dd["split"]=="test"].copy()
    if tr.empty or te.empty:
        continue

    # inner GroupKFold by gene within TRAIN (simulates new gene)
    genes = tr["gene"].values
    gkf = GroupKFold(n_splits=len(np.unique(genes)))
    inner_stats=[]
    for cfg_id, cfg in enumerate(GRID):
        mets=[]; sizes=[]
        for inner_tr_idx, inner_va_idx in gkf.split(tr, groups=genes):
            tr_in = tr.iloc[inner_tr_idx].copy()
            va_in = tr.iloc[inner_va_idx].copy()
            # seed bag on inner val
            fold_mets=[]
            for sd in SEEDS:
                random.seed(sd); np.random.seed(sd); torch.manual_seed(sd)
                pairs = make_pairs_trimatched(tr_in, seed=sd)
                model = train_pairs(tr_in, pairs, R=cfg["R"], dropout=cfg["dropout"], lr=cfg["lr"])
                au, pr, br = eval_variants(model, va_in, R=cfg["R"])
                fold_mets.append((au,pr,br))
            au = float(np.nanmean([m[0] for m in fold_mets]))
            pr = float(np.nanmean([m[1] for m in fold_mets]))
            br = float(np.mean([m[2] for m in fold_mets]))
            mets.append((au,pr,br)); sizes.append(len(va_in))
        # weighted by val size
        w = np.array(sizes, dtype=float); w = w / w.sum()
        mean_au = float(np.nansum([m[0]*w[i] for i,m in enumerate(mets)]))
        mean_pr = float(np.nansum([m[1]*w[i] for i,m in enumerate(mets)]))
        mean_br = float(np.nansum([m[2]*w[i] for i,m in enumerate(mets)]))
        inner_stats.append({"fold":fold, "cfg_id":cfg_id, **cfg, "val_auroc":mean_au, "val_pr":mean_pr, "val_brier":mean_br})
    inner_df = pd.DataFrame(inner_stats).sort_values("val_auroc", ascending=False)
    outer_rows.append(inner_df.assign(stage="inner"))

    # pick best cfg for this OUTER fold
    best = inner_df.iloc[0].to_dict()
    picks.append(best)

    # retrain on full TRAIN with best cfg, eval on TEST (bag seeds)
    mets=[]
    for sd in SEEDS:
        random.seed(sd); np.random.seed(sd); torch.manual_seed(sd)
        pairs = make_pairs_trimatched(tr, seed=sd)
        model = train_pairs(tr, pairs, R=int(best["R"]), dropout=float(best["dropout"]), lr=float(best["lr"]))
        au, pr, br = eval_variants(model, te, R=int(best["R"]))
        mets.append((au,pr,br))
    au = float(np.nanmean([m[0] for m in mets])); pr=float(np.nanmean([m[1] for m in mets])); br=float(np.mean([m[2] for m in mets]))
    test_rows.append({"fold":fold, "gene":te["gene"].iloc[0], "n":len(te), "auroc":round(au,4), "pr_auc":round(pr,4), "brier":round(br,4),
                      "picked_R":int(best["R"]), "picked_dropout":float(best["dropout"]), "picked_lr":float(best["lr"])})

# summarize
inner_all = pd.concat(outer_rows, ignore_index=True)
picks_df = pd.DataFrame(picks)
test_df  = pd.DataFrame(test_rows).sort_values("gene")

# majority/global suggestion across outer folds
global_vote = (picks_df.groupby(["R","dropout","lr"]).size()
               .reset_index(name="count").sort_values("count", ascending=False))
OUT = ARTI / f"hpo_nested_{time.strftime('%Y%m%d_%H%M%S')}"
OUT.mkdir(parents=True, exist_ok=True)
inner_all.to_csv(OUT/"inner_val_per_fold.csv", index=False)
picks_df.to_csv(OUT/"picked_per_fold.csv", index=False)
test_df.to_csv(OUT/"outer_test_metrics.csv", index=False)
global_vote.to_csv(OUT/"global_cfg_votes.csv", index=False)

print("Device:", device)
print("Run dir:", OUT)
print("\n=== Inner-val per fold (top 8 rows) ===")
display(inner_all.sort_values(["fold","val_auroc"], ascending=[True,False]).groupby("fold").head(8))
print("\n=== Picked config per OUTER fold ===")
display(picks_df)
print("\n=== LOCO-GENE TEST (nested-picked cfg per fold) ===")
display(test_df)
print("\n=== Global config votes (most-picked across folds) ===")
display(global_vote.head(8))


Device: cuda
Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/hpo_nested_20250816_224552

=== Inner-val per fold (top 8 rows) ===


,fold,cfg_id,R,dropout,lr,val_auroc,val_pr,val_brier,stage
0,loco_gene:EGFR,3,45,0.1,0.002,0.507499,0.455323,0.252376,inner
1,loco_gene:EGFR,0,30,0.0,0.001,0.504093,0.462023,0.250407,inner
2,loco_gene:EGFR,2,45,0.0,0.001,0.491140,0.431994,0.250644,inner
3,loco_gene:EGFR,1,30,0.1,0.002,0.469623,0.418227,0.256433,inner
4,loco_gene:KRAS,0,30,0.0,0.001,0.584109,0.490850,0.244786,inner
5,loco_gene:KRAS,3,45,0.1,0.002,0.518818,0.454614,0.251156,inner
6,loco_gene:KRAS,2,45,0.0,0.001,0.512838,0.445171,0.249272,inner
7,loco_gene:KRAS,1,30,0.1,0.002,0.478030,0.396324,0.258134,inner
8,loco_gene:PIK3CA,1,30,0.1,0.002,0.579014,0.430298,0.243781,inner
9,loco_gene:PIK3CA,0,30,0.0,0.001,0.569005,0.440259,0.242932,inner



=== Picked config per OUTER fold ===


,fold,cfg_id,R,dropout,lr,val_auroc,val_pr,val_brier
0,loco_gene:EGFR,3,45,0.1,0.002,0.507499,0.455323,0.252376
1,loco_gene:KRAS,0,30,0.0,0.001,0.584109,0.490850,0.244786
2,loco_gene:PIK3CA,1,30,0.1,0.002,0.579014,0.430298,0.243781
3,loco_gene:TP53,1,30,0.1,0.002,0.515131,0.589844,0.263886



=== LOCO-GENE TEST (nested-picked cfg per fold) ===


,fold,gene,n,auroc,pr_auc,brier,picked_R,picked_dropout,picked_lr
0,loco_gene:EGFR,EGFR,39,0.6900,0.5807,0.2365,45,0.1,0.002
1,loco_gene:KRAS,KRAS,11,0.5417,0.7883,0.2643,30,0.0,0.001
2,loco_gene:PIK3CA,PIK3CA,24,0.4074,0.7383,0.3017,30,0.1,0.002
3,loco_gene:TP53,TP53,197,0.5368,0.3724,0.2473,30,0.1,0.002



=== Global config votes (most-picked across folds) ===


,R,dropout,lr,count
1,30,0.1,0.002,2
0,30,0.0,0.001,1
2,45,0.1,0.002,1


Nested HPO kept us honest, but it exposed a likely shift problem — PIK3CA face-planted while EGFR was fine. Before we start throwing more params at it, let’s sanity-check the **context coverage**: are the PIK3CA test tri-contexts even represented in the training genes? if the test gene lives in different tri neighborhoods, AUC will sag no matter how we twiddle LR.

### D1 — Tri-context coverage diagnostics for LOCO-GENE (does train cover the test gene’s tri contexts?)


In [11]:
# D1 — Tri-context coverage diagnostics for LOCO-GENE (does train cover the test gene’s tri contexts?)
import os, glob, time
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/Cancer_gene_database")
DATA = BASE/"data"; ARTI = BASE/"artifacts"
P_RES = DATA/"masterseq_model_ready_plus_resolved.parquet"
runs = sorted(glob.glob(str(ARTI / "sentiment_run_plus_variants_*"))); assert runs, "No split dirs found."
RUN_DIR = Path(runs[-1]); SPLITS_G = RUN_DIR/"splits_loco_gene.csv"

df = (pd.read_parquet(P_RES)
        .dropna(subset=["variant_id","gene","label","tri_center","cds_pos"])
        .assign(tri_center=lambda x: x["tri_center"].astype(str).str.upper(),
                gene=lambda x: x["gene"].astype(str),
                label=lambda x: x["label"].astype(int))
     )

spl = pd.read_csv(SPLITS_G)
dd  = df.merge(spl, on=["variant_id","gene"], how="inner")

rows=[]
details=[]
for fold, sub in dd.groupby("fold"):
    tr = sub[sub["split"]=="train"].copy()
    te = sub[sub["split"]=="test"].copy()
    if te.empty or tr.empty:
        continue
    test_gene = te["gene"].iloc[0]
    tri_train = set(tr["tri_center"].unique().tolist())
    tri_test  = te["tri_center"].tolist()
    tri_test_set = set(tri_test)

    # coverage stats
    cov_by_variant = float(np.mean([t in tri_train for t in tri_test]))
    jaccard = (len(tri_train & tri_test_set) / len(tri_train | tri_test_set)) if len(tri_train|tri_test_set)>0 else np.nan

    # base-level coverage (center base only)
    base_train = set(tr["tri_center"].str[1].unique().tolist())
    base_test  = te["tri_center"].str[1].tolist()
    cov_base_by_variant = float(np.mean([b in base_train for b in base_test]))

    # label counts in TEST
    n_pos = int((te["label"]==1).sum()); n_neg = int((te["label"]==0).sum())

    # tri in TEST that never appear in TRAIN (top few)
    test_counts = te["tri_center"].value_counts()
    missing = [t for t in test_counts.index if t not in tri_train]
    missing_top = test_counts.loc[missing].head(12) if len(missing)>0 else pd.Series(dtype=int)

    rows.append({
        "fold": fold, "gene": test_gene, "n_test": len(te),
        "test_pos": n_pos, "test_neg": n_neg,
        "tri_cov_by_variant": round(cov_by_variant,4),
        "tri_jaccard_sets": round(jaccard,4) if jaccard==jaccard else np.nan,
        "base_cov_by_variant": round(cov_base_by_variant,4)
    })
    for tri, cnt in missing_top.items():
        details.append({"fold": fold, "gene": test_gene, "missing_tri": tri, "count_in_test": int(cnt)})

diag = pd.DataFrame(rows).sort_values("gene")
missing_df = pd.DataFrame(details)

OUT = ARTI / f"tri_coverage_{time.strftime('%Y%m%d_%H%M%S')}"
OUT.mkdir(parents=True, exist_ok=True)
diag.to_csv(OUT/"coverage_summary.csv", index=False)
missing_df.to_csv(OUT/"missing_tri_examples.csv", index=False)

print("Run dir:", OUT)
print("\n=== Coverage summary (per LOCO-GENE fold) ===")
display(diag)
print("\n=== Missing tri-contexts in TEST not seen in TRAIN (up to 12 per fold) ===")
display(missing_df.head(24))


Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/tri_coverage_20250816_225243

=== Coverage summary (per LOCO-GENE fold) ===


,fold,gene,n_test,test_pos,test_neg,tri_cov_by_variant,tri_jaccard_sets,base_cov_by_variant
0,loco_gene:EGFR,EGFR,39,14,25,0.8974,0.4643,1.0
1,loco_gene:KRAS,KRAS,11,8,3,1.0000,0.1607,1.0
2,loco_gene:PIK3CA,PIK3CA,24,18,6,1.0000,0.3214,1.0
3,loco_gene:TP53,TP53,197,69,128,0.7563,0.6429,1.0



=== Missing tri-contexts in TEST not seen in TRAIN (up to 12 per fold) ===


,fold,gene,missing_tri,count_in_test
0,loco_gene:EGFR,EGFR,CTG,2
1,loco_gene:EGFR,EGFR,CAC,2
2,loco_gene:TP53,TP53,GAC,6
3,loco_gene:TP53,TP53,CGC,6
4,loco_gene:TP53,TP53,TCA,5
5,loco_gene:TP53,TP53,CTC,5
6,loco_gene:TP53,TP53,TGC,4
7,loco_gene:TP53,TP53,TAC,4
8,loco_gene:TP53,TP53,ACA,4
9,loco_gene:TP53,TP53,GCG,3


* PIK3CA isn’t OOD on tri-context (coverage 1.00). So that dip isn’t about unseen trinucs — it’s our learner overfitting gene-specific sauce.

* TP53 does miss tri contexts (0.756), so it’s touchy about radius + regularization.

next play: a lean ensemble that (a) blends two receptive fields (R=30 & R=45) and (b) mixes in a small batch of base-matched random pairs (≃20%) alongside tri-matched pairs. that combo usually stabilizes across-gene generalization without cheating.

### # E1 — Two-radius ensemble + pair mix (tri-matched 80% + base-matched random 20%), LOCO-GENE honest


In [12]:
# E1 — Two-radius ensemble + pair mix (tri-matched 80% + base-matched random 20%), LOCO-GENE honest
# Goal: stabilize cross-gene generalization (fix PIK3CA dip, keep EGFR decent, help TP53 a bit)

import os, glob, time, random
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# ---------- paths / data ----------
BASE = Path("/content/drive/MyDrive/Cancer_gene_database")
DATA = BASE/"data"; ARTI = BASE/"artifacts"
P_RES = DATA/"masterseq_model_ready_plus_resolved.parquet"
runs = sorted(glob.glob(str(ARTI / "sentiment_run_plus_variants_*"))); assert runs, "No split dirs found."
RUN_DIR = Path(runs[-1]); SPLITS_G = RUN_DIR/"splits_loco_gene.csv"

df = pd.read_parquet(P_RES).dropna(subset=[
    "variant_id","gene","label","window_mut","window_wt",
    "wt_codon","mut_codon","ref","alt","cds_pos","tri_center"
]).copy()
for c in ["variant_id","gene","ref","alt","wt_codon","mut_codon","window_mut","window_wt","tri_center"]:
    df[c] = df[c].astype(str).str.upper()
df["label"] = df["label"].astype(int)
df["cds_pos"] = pd.to_numeric(df["cds_pos"], errors="coerce").astype("Int64")
spl_g = pd.read_csv(SPLITS_G)

device = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS  = [1337, 2401, 4242]
EPOCHS = 12
LR     = 2e-3
WD     = 1e-4
DROPOUT= 0.10
BATCH  = 96
RADII  = [30, 45]
MIX_FRAC_RANDOM = 0.20  # 20% base-matched random pairs

# ---------- utilities ----------
CODON2AA = {
 "TTT":"F","TTC":"F","TTA":"L","TTG":"L","CTT":"L","CTC":"L","CTA":"L","CTG":"L",
 "ATT":"I","ATC":"I","ATA":"I","ATG":"M","GTT":"V","GTC":"V","GTA":"V","GTG":"V",
 "TCT":"S","TCC":"S","TCA":"S","TCG":"S","CCT":"P","CCC":"P","CCA":"P","CCG":"P",
 "ACT":"T","ACC":"T","ACA":"T","ACG":"T","GCT":"A","GCC":"A","GCA":"A","GCG":"A",
 "TAT":"Y","TAC":"Y","TAA":"*","TAG":"*","CAT":"H","CAC":"H","CAA":"Q","CAG":"Q",
 "AAT":"N","AAC":"N","AAA":"K","AAG":"K","GAT":"D","GAC":"D","GAA":"E","GAG":"E",
 "TGT":"C","TGC":"C","TGA":"*","TGG":"W","CGT":"R","CGC":"R","CGA":"R","CGG":"R",
 "AGT":"S","AGC":"S","AGA":"R","AGG":"R","GGT":"G","GGC":"G","GGA":"G","GGG":"G"
}
_GR={'A':{'A':0,'R':112,'N':111,'D':126,'C':195,'Q':91,'E':107,'G':60,'H':86,'I':94,'L':96,'K':106,'M':84,'F':113,'P':27,'S':99,'T':58,'W':148,'Y':112,'V':64},
'R':{'A':112,'R':0,'N':86,'D':96,'C':180,'Q':43,'E':54,'G':125,'H':29,'I':97,'L':102,'K':26,'M':91,'F':97,'P':103,'S':110,'T':71,'W':101,'Y':77,'V':96},
'N':{'A':111,'R':86,'N':0,'D':23,'C':139,'Q':68,'E':42,'G':80,'H':68,'I':149,'L':153,'K':94,'M':142,'F':158,'P':91,'S':46,'T':65,'W':174,'Y':143,'V':133},
'D':{'A':126,'R':96,'N':23,'D':0,'C':154,'Q':61,'E':45,'G':94,'H':81,'I':168,'L':172,'K':101,'M':160,'F':177,'P':108,'S':65,'T':85,'W':181,'Y':160,'V':152},
'C':{'A':195,'R':180,'N':139,'D':154,'C':0,'Q':154,'E':170,'G':159,'H':174,'I':198,'L':198,'K':202,'M':196,'F':205,'P':169,'S':112,'T':149,'W':215,'Y':194,'V':192},
'Q':{'A':91,'R':43,'N':68,'D':61,'C':154,'Q':0,'E':29,'G':87,'H':24,'I':109,'L':113,'K':53,'M':101,'F':116,'P':76,'S':68,'T':42,'W':130,'Y':99,'V':96},
'E':{'A':107,'R':54,'N':42,'D':45,'C':170,'Q':29,'G':98,'H':40,'I':134,'L':138,'K':56,'M':126,'F':140,'P':93,'S':80,'T':65,'W':152,'Y':122,'V':121},
'G':{'A':60,'R':125,'N':80,'D':94,'C':159,'Q':87,'E':98,'H':98,'I':135,'L':138,'K':127,'M':127,'F':153,'P':42,'S':56,'T':59,'W':184,'Y':147,'V':109},
'H':{'A':86,'R':29,'N':68,'D':81,'C':174,'Q':24,'E':40,'G':98,'I':94,'L':99,'K':32,'M':87,'F':100,'P':77,'S':89,'T':47,'W':115,'Y':83,'V':84},
'I':{'A':94,'R':97,'N':149,'D':168,'C':198,'Q':109,'E':134,'G':135,'H':94,'L':5,'K':102,'M':10,'F':21,'P':95,'S':142,'T':89,'W':61,'Y':33,'V':29},
'L':{'A':96,'R':102,'N':153,'D':172,'C':198,'Q':113,'E':138,'G':138,'H':99,'I':5,'L':0,'K':107,'M':15,'F':22,'P':98,'S':145,'T':92,'W':61,'Y':36,'V':32},
'K':{'A':106,'R':26,'N':94,'D':101,'C':202,'Q':53,'E':56,'G':127,'H':32,'I':102,'L':107,'K':0,'M':95,'F':102,'P':103,'S':121,'T':78,'W':110,'Y':85,'V':97},
'M':{'A':84,'R':91,'N':142,'D':160,'C':196,'Q':101,'E':126,'G':127,'H':87,'I':10,'L':15,'K':95,'F':28,'P':87,'S':135,'T':81,'W':67,'Y':36,'V':21},
'F':{'A':113,'R':97,'N':158,'D':177,'C':205,'Q':116,'E':140,'G':153,'H':100,'I':21,'L':22,'K':102,'P':114,'S':155,'T':103,'W':40,'Y':22,'V':50},
'P':{'A':27,'R':103,'N':91,'D':108,'C':169,'Q':76,'E':93,'G':42,'S':74,'T':38,'W':147,'Y':110,'V':68},
'S':{'A':99,'R':110,'N':46,'D':65,'C':112,'Q':68,'E':80,'G':56,'H':89,'I':142,'L':145,'K':121,'M':135,'F':155,'P':74,'T':58,'W':177,'Y':144,'V':124},
'T':{'A':58,'R':71,'N':65,'D':85,'C':149,'Q':42,'E':65,'G':59,'H':47,'I':89,'L':92,'K':78,'M':81,'F':103,'P':38,'S':58,'W':128,'Y':92,'V':69},
'W':{'A':148,'R':101,'N':174,'D':181,'C':215,'Q':130,'E':152,'G':184,'H':115,'I':61,'L':61,'K':110,'M':67,'F':40,'P':147,'S':177,'T':128,'Y':37,'V':88},
'Y':{'A':112,'R':77,'N':143,'D':160,'C':194,'Q':99,'E':122,'G':147,'H':83,'I':33,'L':36,'K':85,'M':36,'F':22,'P':110,'S':144,'T':92,'W':37,'V':55},
'V':{'A':64,'R':96,'N':133,'D':152,'C':192,'Q':96,'E':121,'G':109,'H':84,'I':29,'L':32,'K':97,'M':21,'F':50,'P':68,'S':124,'T':69,'W':88,'Y':55,'V':0}}
_B = """\
   A  R  N  D  C  Q  E  G  H  I  L  K  M  F  P  S  T  W  Y  V
A  4 -1 -2 -2  0 -1 -1  0 -2 -1 -1 -1 -1 -2 -1  1  0 -3 -2  0
R -1  5  0 -2 -3  1  0 -2  0 -3 -2  2 -1 -3 -2 -1 -1 -3 -2 -3
N -2  0  6  1 -3  0  0  0  1 -3 -3  0 -2 -3 -2  1  0 -4 -2 -3
D -2 -2  1  6 -3  0  2 -1 -1 -3 -4 -1 -3 -3 -1  0 -1 -4 -3 -3
C  0 -3 -3 -3  9 -3 -4 -3 -3 -1 -1 -3 -1 -2 -3 -1 -1 -2 -2 -1
Q -1  1  0  0 -3  5  2 -2  0 -3 -2  1  0 -3 -1  0 -1 -2 -1 -2
E -1  0  0  2 -4  2  5 -2  0 -3 -3  1 -2 -3 -1  0 -1 -3 -2 -2
G  0 -2  0 -1 -3 -2 -2  6 -2 -4 -4 -2 -3 -3 -2  0 -2 -2 -3 -3
H -2  0  1 -1 -3  0  0 -2  8 -3 -3 -1 -2 -1 -2 -1 -2 -2  2 -3
I -1 -3 -3 -3 -1 -3 -3 -4 -3  4  2 -3  1  0 -3 -2 -1 -3 -1  3
L -1 -2 -3 -4 -1 -2 -3 -4 -3  2  4 -2  2  0 -3 -2 -1 -2 -1  1
K -1  2  0 -1 -3  1  1 -2 -1 -3 -2  5 -1 -3 -1  0 -1 -3 -2 -2
M -1 -1 -2 -3 -1  0 -2 -3 -2  1  2 -1  5  0 -2 -1 -1 -1 -1  1
F -2 -3 -3 -3 -2 -3 -3 -3 -1  0  0 -3  0  6 -4 -2 -2  1  3 -1
P -1 -2 -2 -1 -3 -1 -1 -2 -2 -3 -3 -1 -2 -4  7 -1 -1 -4 -3 -2
S  1 -1  1  0 -1  0  0  0 -1 -2 -2  0 -1 -2 -1  4  1 -3 -2 -2
T  0 -1  0 -1 -1 -1 -1 -2 -2 -1 -1 -1 -1 -2 -1  1  5 -2 -2  0
W -3 -3 -4 -4 -2 -2 -3 -2 -2 -3 -2 -3 -1  1 -4 -3 -2 11  2 -3
Y -2 -2 -2 -3 -2 -1 -2 -3  2 -1 -1 -2 -1  3 -3 -2 -2  2  7 -1
V  0 -3 -3 -3 -1 -2 -2 -3 -3  3  1 -2  1 -1 -2 -2  0 -3 -1  4
"""
B62={}; _hdr=_B.strip().splitlines()[0].split()
for line in _B.strip().splitlines()[1:]:
    parts=line.split(); a=parts[0]; B62[a]={_hdr[i]:int(parts[i+1]) for i in range(len(_hdr))}
VOCAB={"A":0,"C":1,"G":2,"T":3,"N":4}
def onehot(seq: str):
    arr = np.zeros((5,len(seq)), np.float32)
    for i,ch in enumerate(seq): arr[VOCAB.get(ch,4), i] = 1.0
    return arr
def grantham(a,b):
    if a==b: return 0
    return _GR.get(a,{}).get(b,_GR.get(b,{}).get(a,0))
def blosum62(a,b):
    return B62.get(a,{}).get(b,B62.get(b,{}).get(a,0))
def aa_scalars(wt_codon, mut_codon):
    a = CODON2AA.get(str(wt_codon), "X"); b = CODON2AA.get(str(mut_codon), "X")
    aa_same = float(a==b)
    if a=="X" or b=="X": return np.array([aa_same,0.0,0.0], dtype=np.float32)
    return np.array([aa_same, grantham(a,b)/215.0, (blosum62(a,b)+4)/15.0], dtype=np.float32)
def mut_index_in_window(row, flank=60):
    i0 = int(row["cds_pos"]) - 1; cs = (i0 // 3) * 3; center = cs + 1
    return flank + (i0 - center)
def extract_patch(win: str, j: int, R: int = 30):
    s = str(win)
    if not s or j<0 or j>=len(s): return ""
    a,b = max(0, j-R), min(len(s), j+R+1)
    left  = "N" * (R - (j - a)); right = "N" * (R - (b - j - 1))
    return left + s[a:b] + right

class PairTrainDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, pairs, R=30):
        self.df = frame; self.pairs = pairs; self.R = R
    def __len__(self): return len(self.pairs)
    def _patch_feats(self, r):
        j = mut_index_in_window(r, flank=60)
        mut = extract_patch(r["window_mut"], j, self.R)
        wt  = extract_patch(r["window_wt"],  j, self.R)
        scal = aa_scalars(r["wt_codon"], r["mut_codon"])
        return torch.from_numpy(onehot(mut)), torch.from_numpy(onehot(wt)), torch.from_numpy(scal)
    def __getitem__(self, k):
        ip, ineg = self.pairs[k]
        rp, rn = self.df.loc[ip], self.df.loc[ineg]
        xm_p, xw_p, s_p = self._patch_feats(rp)
        xm_n, xw_n, s_n = self._patch_feats(rn)
        return xm_p, xw_p, s_p, xm_n, xw_n, s_n

class VariantEvalDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, R=30):
        self.df = frame.reset_index(); self.R = R
        self.items=[]
        for _, r in self.df.iterrows():
            j = mut_index_in_window(r, flank=60)
            mut = extract_patch(r["window_mut"], j, self.R)
            wt  = extract_patch(r["window_wt"],  j, self.R)
            scal = aa_scalars(r["wt_codon"], r["mut_codon"])
            self.items.append((onehot(mut), onehot(wt), scal, float(r["label"])))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        mut, wt, scal, y = self.items[i]
        return (torch.from_numpy(mut), torch.from_numpy(wt),
                torch.from_numpy(scal), torch.tensor(y, dtype=torch.float32))

class SiameseDeltaCNN(nn.Module):
    def __init__(self, in_ch=5, hid=96, scal_dim=3, dropout=0.1):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv1d(in_ch, 64, kernel_size=11, padding=5), nn.ReLU(),
            nn.Conv1d(64, 96, kernel_size=7, padding=3),     nn.ReLU(),
            nn.Conv1d(96, 96, kernel_size=3, padding=1),     nn.ReLU(),
            nn.AdaptiveMaxPool1d(1)
        )
        self.norm = nn.LayerNorm(96)
        self.head = nn.Sequential(nn.Linear(96,96), nn.ReLU(), nn.Dropout(dropout), nn.Linear(96,1))
        self.scal_head = nn.Linear(scal_dim, 1)
    def encode(self, x):
        h = self.enc(x).squeeze(-1)
        return self.norm(h)
    def score(self, xm, xw, scalars):
        hm = self.encode(xm); hw = self.encode(xw)
        core = self.head(hm - hw).squeeze(-1)
        scal = self.scal_head(scalars).squeeze(-1)
        return core + scal

# tri-matched pairs
def make_pairs_trimatched(train_df: pd.DataFrame, seed=1337):
    rs=np.random.RandomState(seed); pairs=[]; used=set()
    sub=train_df.copy(); sub["center_base"]=sub["tri_center"].str[1]
    for g, grp in sub.groupby("gene"):
        pos=grp[grp["label"]==1]; neg=grp[grp["label"]==0].copy()
        for tri, pgrp in pos.groupby("tri_center"):
            pool=neg[(neg["tri_center"]==tri)&(~neg.index.isin(used))]
            if pool.empty: continue
            k=min(len(pgrp), len(pool))
            pidx=rs.choice(pgrp.index, size=k, replace=False)
            nidx=rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx,nidx)); used.update(nidx)
        leftover=pos.loc[~pos.index.isin([p for p,_ in pairs])]
        for base, pgrp in leftover.groupby("center_base"):
            pool=neg[(neg["tri_center"].str[1]==base)&(~neg.index.isin(used))]
            if pool.empty: continue
            k=min(len(pgrp), len(pool))
            pidx=rs.choice(pgrp.index, size=k, replace=False)
            nidx=rs.choice(pool.index, size=k, replace=False)
            pairs += list(zip(pidx,nidx)); used.update(nidx)
    return pairs

# add ~20% base-matched random pairs (regularizer)
def make_pairs_mixed(train_df: pd.DataFrame, seed=1337, frac_random=0.20):
    rs=np.random.RandomState(seed)
    tri_pairs = make_pairs_trimatched(train_df, seed=seed)
    sub=train_df.copy(); sub["center_base"]=sub["tri_center"].str[1]
    pos=sub[sub["label"]==1]; neg=sub[sub["label"]==0]
    n_rand = int(len(tri_pairs) * frac_random / max(1e-6, (1-frac_random)))
    rand_pairs=[]
    if n_rand>0 and not pos.empty and not neg.empty:
        for base, pgrp in pos.groupby("center_base"):
            pool = neg[neg["center_base"]==base]
            if pool.empty: continue
            k = min(len(pgrp), len(pool))
            if k==0: continue
            # sample proportionally
            take = max(1, int(n_rand * (len(pgrp)/len(pos))))
            pidx = rs.choice(pgrp.index, size=min(len(pgrp), take), replace=True)
            nidx = rs.choice(pool.index, size=len(pidx), replace=True)
            rand_pairs += list(zip(pidx, nidx))
    pairs = tri_pairs + rand_pairs
    rs.shuffle(pairs)
    return pairs

def train_pairs(df_train, pairs, R, dropout=DROPOUT, lr=LR, epochs=EPOCHS):
    model = SiameseDeltaCNN(dropout=dropout).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    bce = nn.BCEWithLogitsLoss()
    loader = DataLoader(PairTrainDataset(df_train, pairs, R=R), batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
    for ep in range(epochs):
        model.train()
        for xm_p, xw_p, s_p, xm_n, xw_n, s_n in loader:
            xm_p, xw_p, s_p = xm_p.to(device), xw_p.to(device), s_p.to(device)
            xm_n, xw_n, s_n = xm_n.to(device), xw_n.to(device), s_n.to(device)
            opt.zero_grad()
            diff = model.score(xm_p, xw_p, s_p) - model.score(xm_n, xw_n, s_n)
            loss = bce(diff, torch.ones_like(diff))
            loss.backward(); opt.step()
        sched.step()
    return model

def eval_variants(model, df_eval, R):
    ds = VariantEvalDataset(df_eval, R=R)
    loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
    model.eval(); logits=[]; ys=[]
    with torch.no_grad():
        for xm, xw, s, y in loader:
            xm, xw, s = xm.to(device), xw.to(device), s.to(device)
            logits.append(model.score(xm, xw, s).cpu().numpy()); ys.append(y.numpy())
    y = np.concatenate(ys).astype(int); logit = np.concatenate(logits)
    p = 1/(1+np.exp(-logit))
    auroc = roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan
    pr    = average_precision_score(y,p) if len(np.unique(y))>1 else np.nan
    brier = brier_score_loss(y,p)
    return logit, y, auroc, pr, brier

# ---------- train/eval across folds, two radii + seed bag ----------
rows=[]; preds_all=[]
out_dir = ARTI / f"ensemble_twoR_{time.strftime('%Y%m%d_%H%M%S')}"
out_dir.mkdir(parents=True, exist_ok=True)

for fold, sub in spl_g.groupby("fold"):
    dd = df.merge(sub, on=["variant_id","gene"], how="inner")
    tr = dd[dd["split"]=="train"].copy()
    te = dd[dd["split"]=="test"].copy()
    if tr.empty or te.empty:
        continue
    # bag per radius
    logits_by_R = {R: None for R in RADII}
    for R in RADII:
        logits_seed=[]
        for sd in SEEDS:
            random.seed(sd); np.random.seed(sd); torch.manual_seed(sd)
            pairs = make_pairs_mixed(tr, seed=sd, frac_random=MIX_FRAC_RANDOM)
            model = train_pairs(tr, pairs, R=R)
            logit, y, au, pr, br = eval_variants(model, te, R=R)
            logits_seed.append(logit)
        logits_by_R[R] = np.mean(logits_seed, axis=0)
    # ensemble across radii (avg logits)
    logits_ens = np.mean([logits_by_R[R] for R in RADII], axis=0)
    p = 1/(1+np.exp(-logits_ens))
    y = te["label"].astype(int).values
    gene = te["gene"].iloc[0]
    auroc = roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan
    pr    = average_precision_score(y,p) if len(np.unique(y))>1 else np.nan
    brier = brier_score_loss(y,p)
    rows.append({"fold":fold, "gene":gene, "n":len(y), "auroc":round(auroc,4), "pr_auc":round(pr,4), "brier":round(brier,4)})
    # stash preds for receipts
    tmp = te[["variant_id","gene","label"]].reset_index(drop=True).copy()
    tmp["logit_ensemble"] = logits_ens
    tmp["prob_ensemble"]  = p
    tmp["fold"] = fold
    preds_all.append(tmp)

summary = pd.DataFrame(rows).sort_values("gene")
preds_all = pd.concat(preds_all, ignore_index=True)
summary.to_csv(out_dir/"metrics_twoR_mix.csv", index=False)
preds_all.to_csv(out_dir/"preds_twoR_mix.csv", index=False)

print("Device:", device)
print("Run dir:", out_dir)
display(summary)


Device: cuda
Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/ensemble_twoR_20250816_225822


,fold,gene,n,auroc,pr_auc,brier
0,loco_gene:EGFR,EGFR,39,0.7257,0.5711,0.2330
1,loco_gene:KRAS,KRAS,11,0.7500,0.9116,0.2661
2,loco_gene:PIK3CA,PIK3CA,24,0.3981,0.7340,0.2772
3,loco_gene:TP53,TP53,197,0.6016,0.4833,0.2370


The PIK3CA faceplant looks way more like (1) tiny neg set (6!), (2) gene-specific priors the pair net can’t unlearn, and (3) tri-freq shift (coverage=1.0 but **Jaccard=0.32** → same trinucs exist, but their frequencies differ).

**TL;DR:** pair net leaned on gene-specific sauce; WT-only was stronger for PIK3CA before, so let’s bring it back and then stack.

### E2 — WT-only baseline (no mutation, just wild-type context)

In [13]:
# E2 — WT-only logistic baseline on window_wt with LOCO-GENE splits
# Saves preds + metrics; we’ll stack this with the pairCNN ensemble next.

import os, glob, time, re, itertools, math, random
from pathlib import Path
import numpy as np, pandas as pd
from collections import Counter
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

BASE = Path("/content/drive/MyDrive/Cancer_gene_database")
DATA = BASE/"data"; ARTI = BASE/"artifacts"
P_RES = DATA/"masterseq_model_ready_plus_resolved.parquet"
runs  = sorted(glob.glob(str(ARTI / "sentiment_run_plus_variants_*"))); assert runs, "No split dirs found."
RUN_DIR = Path(runs[-1]); SPLITS_G = RUN_DIR/"splits_loco_gene.csv"

# ---- data ----
df = (pd.read_parquet(P_RES)
        .dropna(subset=["variant_id","gene","label","window_wt","tri_center"])
        .assign(variant_id=lambda x: x["variant_id"].astype(str),
                gene=lambda x: x["gene"].astype(str),
                label=lambda x: x["label"].astype(int),
                window_wt=lambda x: x["window_wt"].astype(str).str.upper(),
                tri_center=lambda x: x["tri_center"].astype(str).str.upper())
     )
spl = pd.read_csv(SPLITS_G)
dd  = df.merge(spl, on=["variant_id","gene"], how="inner")

# ---- featurization: 3-mer bag over WT window + center tri as a feature ----
def kmer_counts(seq: str, k=3):
    s = re.sub("[^ACGT]", "N", seq.upper())
    feats = Counter()
    for i in range(len(s)-k+1):
        kmer = s[i:i+k]
        feats[f"k{k}:{kmer}"] += 1
    return feats

def make_Xy(frame: pd.DataFrame, vec: DictVectorizer = None):
    feats = []
    for _, r in frame.iterrows():
        d = kmer_counts(r["window_wt"], k=3)
        d[f"CENTER:{r['tri_center']}"] += 1  # light-handed prior
        feats.append(d)
    if vec is None:
        vec = DictVectorizer(sparse=True)
        X = vec.fit_transform(feats)
    else:
        X = vec.transform(feats)
    y = frame["label"].astype(int).values
    return X, y, vec

# ---- train/eval per fold ----
rows = []
preds_all = []
out_dir = ARTI / f"wt_only_baseline_{time.strftime('%Y%m%d_%H%M%S')}"
out_dir.mkdir(parents=True, exist_ok=True)

for fold, sub in dd.groupby("fold"):
    tr = sub[sub["split"]=="train"].copy()
    te = sub[sub["split"]=="test"].copy()
    if tr.empty or te.empty:
        continue

    # build vectorizer on TRAIN only
    Xtr, ytr, vec = make_Xy(tr, vec=None)
    Xte, yte, _   = make_Xy(te, vec=vec)

    # simple, stable LR (liblinear handles small n cleanly)
    clf = LogisticRegression(
        solver="liblinear", penalty="l2", C=1.0, max_iter=2000, class_weight=None
    ).fit(Xtr, ytr)

    p = clf.predict_proba(Xte)[:,1]
    au = roc_auc_score(yte, p) if len(np.unique(yte))>1 else np.nan
    pr = average_precision_score(yte, p) if len(np.unique(yte))>1 else np.nan
    br = brier_score_loss(yte, p)

    gene = te["gene"].iloc[0]
    rows.append({"fold":fold, "gene":gene, "n":len(yte),
                 "auroc":round(float(au),4), "pr_auc":round(float(pr),4), "brier":round(float(br),4)})

    tmp = te[["variant_id","gene","label"]].reset_index(drop=True).copy()
    tmp["prob_wt_only"] = p
    tmp["fold"] = fold
    preds_all.append(tmp)

summary = pd.DataFrame(rows).sort_values("gene")
preds_all = pd.concat(preds_all, ignore_index=True)

summary.to_csv(out_dir/"metrics_wt_only.csv", index=False)
preds_all.to_csv(out_dir/"preds_wt_only.csv", index=False)

print("Run dir:", out_dir)
display(summary)


Run dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/wt_only_baseline_20250816_230700


,fold,gene,n,auroc,pr_auc,brier
0,loco_gene:EGFR,EGFR,39,0.6057,0.4366,0.3491
1,loco_gene:KRAS,KRAS,11,0.0000,0.5550,0.3646
2,loco_gene:PIK3CA,PIK3CA,24,0.7593,0.9212,0.1918
3,loco_gene:TP53,TP53,197,0.7160,0.6008,0.2187


**stack** stronger WT-only baseline with the two-radius pairCNN ensemble — leak-free, LOCO-GENE-honest.

This single cell reads our latest runs: `ensemble_twoR_*` and `wt_only_baseline_*`, trains a tiny meta-logistic on other folds’ preds, then scores the held-out gene):

### # E3 — Leak-free stack: WT-only ⊕ pairCNN (two-radius ensemble)

In [14]:
# E3 — Leak-free stack: WT-only ⊕ pairCNN (two-radius ensemble)
# Uses cross-fold stacking: for each held-out gene fold, fit meta-LR on the other folds’ predictions only.
# Writes metrics + stacked preds to artifacts.

import os, glob, time
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

BASE = Path("/content/drive/MyDrive/Cancer_gene_database")
ARTI = BASE/"artifacts"

# locate latest runs
ens_dirs = sorted(glob.glob(str(ARTI/"ensemble_twoR_*")))
wt_dirs  = sorted(glob.glob(str(ARTI/"wt_only_baseline_*")))
assert ens_dirs and wt_dirs, "Need prior runs: ensemble_twoR_* and wt_only_baseline_*."
ENS_DIR = Path(ens_dirs[-1])
WT_DIR  = Path(wt_dirs[-1])

preds_pair = pd.read_csv(ENS_DIR/"preds_twoR_mix.csv")   # columns: variant_id,gene,label,logit_ensemble,prob_ensemble,fold
preds_wt   = pd.read_csv(WT_DIR/"preds_wt_only.csv")     # columns: variant_id,gene,label,prob_wt_only,fold

# merge preds
preds = (preds_pair.merge(preds_wt, on=["variant_id","gene","label","fold"], how="inner")
                   .assign(prob_pair=lambda d: d["prob_ensemble"].clip(1e-6, 1-1e-6),
                           prob_wt=lambda d: d["prob_wt_only"].clip(1e-6, 1-1e-6))
        )

def metrics(y, p):
    y = y.astype(int).values
    au = roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan
    pr = average_precision_score(y,p) if len(np.unique(y))>1 else np.nan
    br = brier_score_loss(y,p)
    return float(au), float(pr), float(br)

rows=[]; preds_out=[]
folds = sorted(preds["fold"].unique().tolist())

for f in folds:
    te = preds[preds["fold"]==f].copy()
    tr = preds[preds["fold"]!=f].copy()
    if te.empty or tr.empty:
        continue

    X_tr = tr[["prob_wt","prob_pair"]].values
    y_tr = tr["label"].astype(int).values
    X_te = te[["prob_wt","prob_pair"]].values
    y_te = te["label"].astype(int).values

    # meta-model: simple, stable LR (acts as calibration + weighting)
    meta = LogisticRegression(solver="liblinear", penalty="l2", C=1.0, max_iter=2000)
    meta.fit(X_tr, y_tr)
    p_stack = meta.predict_proba(X_te)[:,1]

    # base metrics (for comparison)
    au_wt,  pr_wt,  br_wt  = metrics(y_te, te["prob_wt"].values)
    au_pr,  pr_pr,  br_pr  = metrics(y_te, te["prob_pair"].values)
    au_st,  pr_st,  br_st  = metrics(y_te, p_stack)

    gene = te["gene"].iloc[0]
    n    = len(te)
    rows.append({
        "fold": f, "gene": gene, "n": n,
        "AUROC_wt": round(au_wt,4), "AUROC_pair": round(au_pr,4), "AUROC_stack": round(au_st,4),
        "PR_wt": round(pr_wt,4), "PR_pair": round(pr_pr,4), "PR_stack": round(pr_st,4),
        "Brier_wt": round(br_wt,4), "Brier_pair": round(br_pr,4), "Brier_stack": round(br_st,4),
    })

    out = te[["variant_id","gene","label","fold"]].copy()
    out["prob_wt"]    = te["prob_wt"].values
    out["prob_pair"]  = te["prob_pair"].values
    out["prob_stack"] = p_stack
    preds_out.append(out)

summary = pd.DataFrame(rows).sort_values("gene")
preds_out = pd.concat(preds_out, ignore_index=True)

OUT = ARTI / f"stack_twoR_wt_{time.strftime('%Y%m%d_%H%M%S')}"
OUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT/"metrics_stack.csv", index=False)
preds_out.to_csv(OUT/"preds_stack.csv", index=False)

print("Pair preds dir:", ENS_DIR)
print("WT preds dir  :", WT_DIR)
print("Run dir       :", OUT)
display(summary)


AttributeError: 'numpy.ndarray' object has no attribute 'values'

### E3 (patched) — Leak-free stack: WT-only ⊕ pairCNN (two-radius ensemble)


In [15]:
# E3 (patched) — Leak-free stack: WT-only ⊕ pairCNN (two-radius ensemble)

import os, glob, time
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

BASE = Path("/content/drive/MyDrive/Cancer_gene_database")
ARTI = BASE/"artifacts"

# latest runs
ens_dirs = sorted(glob.glob(str(ARTI/"ensemble_twoR_*")))
wt_dirs  = sorted(glob.glob(str(ARTI/"wt_only_baseline_*")))
assert ens_dirs and wt_dirs, "Need prior runs: ensemble_twoR_* and wt_only_baseline_*."
ENS_DIR = Path(ens_dirs[-1]); WT_DIR = Path(wt_dirs[-1])

preds_pair = pd.read_csv(ENS_DIR/"preds_twoR_mix.csv")   # variant_id,gene,label,logit_ensemble,prob_ensemble,fold
preds_wt   = pd.read_csv(WT_DIR/"preds_wt_only.csv")     # variant_id,gene,label,prob_wt_only,fold

# merge and clip to (0,1)
preds = (preds_pair.merge(preds_wt, on=["variant_id","gene","label","fold"], how="inner")
                   .assign(prob_pair=lambda d: d["prob_ensemble"].clip(1e-6, 1-1e-6),
                           prob_wt=lambda d: d["prob_wt_only"].clip(1e-6, 1-1e-6))
        )

def metrics(y, p):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    au = roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan
    pr = average_precision_score(y,p) if len(np.unique(y))>1 else np.nan
    br = brier_score_loss(y,p)
    return float(au), float(pr), float(br)

rows, preds_out = [], []
for f in sorted(preds["fold"].unique()):
    te = preds[preds["fold"]==f].copy()
    tr = preds[preds["fold"]!=f].copy()
    if te.empty or tr.empty:
        continue

    X_tr = tr[["prob_wt","prob_pair"]].values
    y_tr = tr["label"].astype(int).values
    X_te = te[["prob_wt","prob_pair"]].values
    y_te = te["label"].astype(int).values

    meta = LogisticRegression(solver="liblinear", penalty="l2", C=1.0, max_iter=2000)
    meta.fit(X_tr, y_tr)
    p_stack = meta.predict_proba(X_te)[:,1]

    au_wt, pr_wt, br_wt = metrics(y_te, te["prob_wt"].values)
    au_pr, pr_pr, br_pr = metrics(y_te, te["prob_pair"].values)
    au_st, pr_st, br_st = metrics(y_te, p_stack)

    rows.append({
        "fold": f, "gene": te["gene"].iloc[0], "n": len(te),
        "AUROC_wt": round(au_wt,4), "AUROC_pair": round(au_pr,4), "AUROC_stack": round(au_st,4),
        "PR_wt": round(pr_wt,4), "PR_pair": round(pr_pr,4), "PR_stack": round(pr_st,4),
        "Brier_wt": round(br_wt,4), "Brier_pair": round(br_pr,4), "Brier_stack": round(br_st,4),
    })

    out = te[["variant_id","gene","label","fold"]].copy()
    out["prob_wt"]    = te["prob_wt"].values
    out["prob_pair"]  = te["prob_pair"].values
    out["prob_stack"] = p_stack
    preds_out.append(out)

summary   = pd.DataFrame(rows).sort_values("gene")
preds_out = pd.concat(preds_out, ignore_index=True)

OUT = ARTI / f"stack_twoR_wt_{time.strftime('%Y%m%d_%H%M%S')}"
OUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT/"metrics_stack.csv", index=False)
preds_out.to_csv(OUT/"preds_stack.csv", index=False)

print("Pair preds dir:", ENS_DIR)
print("WT preds dir  :", WT_DIR)
print("Run dir       :", OUT)
display(summary)


Pair preds dir: /content/drive/MyDrive/Cancer_gene_database/artifacts/ensemble_twoR_20250816_225822
WT preds dir  : /content/drive/MyDrive/Cancer_gene_database/artifacts/wt_only_baseline_20250816_230700
Run dir       : /content/drive/MyDrive/Cancer_gene_database/artifacts/stack_twoR_wt_20250816_231329


,fold,gene,n,AUROC_wt,AUROC_pair,AUROC_stack,PR_wt,PR_pair,PR_stack,Brier_wt,Brier_pair,Brier_stack
0,loco_gene:EGFR,EGFR,39,0.6057,0.7257,0.6029,0.4366,0.5711,0.4315,0.3491,0.2330,0.2557
1,loco_gene:KRAS,KRAS,11,0.0000,0.7500,0.0000,0.5550,0.9116,0.5550,0.3646,0.2661,0.2827
2,loco_gene:PIK3CA,PIK3CA,24,0.7593,0.3981,0.7593,0.9212,0.7340,0.9212,0.1918,0.2772,0.2211
3,loco_gene:TP53,TP53,197,0.7160,0.6016,0.6969,0.6008,0.4833,0.5578,0.2187,0.2370,0.2148


stacker got played by TP53 dominating the training folds, so it learned to trust WT-only and then kneecapped EGFR. let’s try training-free blends that can’t overfit by gene:

* **logit-average** (equal weight in log-odds space) — usually preserves the stronger model’s ranking.

* **rank-average** (average the per-fold ranks) — robust when score scales differ.

* **max** (take the higher prob per variant) — sometimes boosts AUC, sometimes noisy.

this cell reads latest preds, makes the three blends, and reports per-gene metrics + saves receipts.

### E3c — Training-free blends of WT-only and pairCNN (keeps EGFR safe)
#### Blends: logit-avg, rank-avg, max. No labels used to set weights.


In [16]:
# E3c — Training-free blends of WT-only and pairCNN (keeps EGFR safe)
# Blends: logit-avg, rank-avg, max. No labels used to set weights.

import os, glob, time
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

BASE = Path("/content/drive/MyDrive/Cancer_gene_database")
ARTI = BASE/"artifacts"

ens_dirs = sorted(glob.glob(str(ARTI/"ensemble_twoR_*")));  assert ens_dirs, "Run E1 first."
wt_dirs  = sorted(glob.glob(str(ARTI/"wt_only_baseline_*"))); assert wt_dirs, "Run E2 first."
ENS_DIR = Path(ens_dirs[-1]); WT_DIR = Path(wt_dirs[-1])

pair = pd.read_csv(ENS_DIR/"preds_twoR_mix.csv")   # variant_id,gene,label,logit_ensemble,prob_ensemble,fold
wt   = pd.read_csv(WT_DIR/"preds_wt_only.csv")     # variant_id,gene,label,prob_wt_only,fold

df = (pair.merge(wt, on=["variant_id","gene","label","fold"], how="inner")
          .rename(columns={"prob_ensemble":"p_pair", "prob_wt_only":"p_wt"}))

# helpers
def clip(p, eps=1e-6):
    p = np.asarray(p, float); return np.clip(p, eps, 1-eps)
def logit(p):
    p = clip(p); return np.log(p/(1-p))
def sigmoid(z):
    z = np.asarray(z, float); return 1/(1+np.exp(-z))

# blends (no training)
df["p_logitavg"] = sigmoid(0.5*(logit(df["p_pair"].values) + logit(df["p_wt"].values)))

# rank-average per fold (to avoid mixing distributions across folds)
def rank_avg(group):
    # average normalized ranks of the two probs within the fold
    r1 = group["p_pair"].rank(method="average") / (len(group))
    r2 = group["p_wt"].rank(method="average")   / (len(group))
    group["p_rankavg"] = 0.5*(r1 + r2)
    return group
df = df.groupby("fold", group_keys=False).apply(rank_avg)

df["p_max"] = np.maximum(df["p_pair"].values, df["p_wt"].values)

# metrics per gene (same LOCO-GENE folds)
def mets(y, p):
    y = np.asarray(y, int)
    au = roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan
    pr = average_precision_score(y,p) if len(np.unique(y))>1 else np.nan
    br = brier_score_loss(y,p)
    return au, pr, br

rows=[]
for f, sub in df.groupby("fold"):
    y = sub["label"].astype(int).values
    gene = sub["gene"].iloc[0]; n = len(sub)
    au_pair, pr_pair, br_pair = mets(y, sub["p_pair"].values)
    au_wt,   pr_wt,   br_wt   = mets(y, sub["p_wt"].values)
    au_la,   pr_la,   br_la   = mets(y, sub["p_logitavg"].values)
    au_ra,   pr_ra,   br_ra   = mets(y, sub["p_rankavg"].values)
    au_mx,   pr_mx,   br_mx   = mets(y, sub["p_max"].values)
    rows.append({
        "fold": f, "gene": gene, "n": n,
        "AUROC_pair": round(au_pair,4), "AUROC_wt": round(au_wt,4),
        "AUROC_logitavg": round(au_la,4), "AUROC_rankavg": round(au_ra,4), "AUROC_max": round(au_mx,4),
        "PR_pair": round(pr_pair,4), "PR_wt": round(pr_wt,4),
        "PR_logitavg": round(pr_la,4), "PR_rankavg": round(pr_ra,4), "PR_max": round(pr_mx,4),
        "Brier_pair": round(br_pair,4), "Brier_wt": round(br_wt,4),
        "Brier_logitavg": round(br_la,4), "Brier_rankavg": round(br_ra,4), "Brier_max": round(br_mx,4),
    })

summary = pd.DataFrame(rows).sort_values("gene")

OUT = ARTI / f"blend_nolearn_{time.strftime('%Y%m%d_%H%M%S')}"
OUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT/"metrics_blends.csv", index=False)
df.to_csv(OUT/"preds_blends.csv", index=False)

print("Pair preds:", ENS_DIR)
print("WT preds  :", WT_DIR)
print("Run dir   :", OUT)
display(summary)


Pair preds: /content/drive/MyDrive/Cancer_gene_database/artifacts/ensemble_twoR_20250816_225822
WT preds  : /content/drive/MyDrive/Cancer_gene_database/artifacts/wt_only_baseline_20250816_230700
Run dir   : /content/drive/MyDrive/Cancer_gene_database/artifacts/blend_nolearn_20250816_231849


/tmp/ipython-input-1781375119.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("fold", group_keys=False).apply(rank_avg)


,fold,gene,n,AUROC_pair,AUROC_wt,AUROC_logitavg,AUROC_rankavg,AUROC_max,PR_pair,PR_wt,PR_logitavg,PR_rankavg,PR_max,Brier_pair,Brier_wt,Brier_logitavg,Brier_rankavg,Brier_max
0,loco_gene:EGFR,EGFR,39,0.7257,0.6057,0.6314,0.7029,0.6514,0.5711,0.4366,0.4498,0.5394,0.4544,0.2330,0.3491,0.2862,0.2201,0.3365
1,loco_gene:KRAS,KRAS,11,0.7500,0.0000,0.0000,0.3958,0.0000,0.9116,0.5550,0.5550,0.7420,0.5550,0.2661,0.3646,0.3331,0.3272,0.3237
2,loco_gene:PIK3CA,PIK3CA,24,0.3981,0.7593,0.7593,0.5972,0.7778,0.7340,0.9212,0.9201,0.8148,0.9253,0.2772,0.1918,0.1772,0.2551,0.1640
3,loco_gene:TP53,TP53,197,0.6016,0.7160,0.7177,0.7191,0.8020,0.4833,0.6008,0.6027,0.5812,0.6464,0.2370,0.2187,0.2086,0.2180,0.2382


one last block that bundles all receipts into a single place so you can think offline. it grabs your latest runs (pairCNN, WT-only, blends), builds a tidy leaderboard per gene, and writes a master preds file with every score side-by-side.

### E4 — Final receipts pack (leaderboard + unified preds)
#### Pulls latest: ensemble_twoR_*, wt_only_baseline_*, blend_nolearn_*
##### Writes: metrics_pack.csv, preds_master.csv under artifacts/final_receipts_*


In [17]:
# E4 — Final receipts pack (leaderboard + unified preds)
# Pulls latest: ensemble_twoR_*, wt_only_baseline_*, blend_nolearn_*
# Writes: metrics_pack.csv, preds_master.csv under artifacts/final_receipts_*

import os, glob, time
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/Cancer_gene_database")
ARTI = BASE / "artifacts"

# find latest dirs
ens_dirs = sorted(glob.glob(str(ARTI/"ensemble_twoR_*")))
wt_dirs  = sorted(glob.glob(str(ARTI/"wt_only_baseline_*")))
bl_dirs  = sorted(glob.glob(str(ARTI/"blend_nolearn_*")))
assert ens_dirs and wt_dirs and bl_dirs, "Missing one of the inputs. Run E1, E2, E3c first."

ENS = Path(ens_dirs[-1])
WT  = Path(wt_dirs[-1])
BL  = Path(bl_dirs[-1])

# load metrics
m_pair = pd.read_csv(ENS/"metrics_twoR_mix.csv").assign(method="pair_twoR")
m_wt   = pd.read_csv(WT/"metrics_wt_only.csv").assign(method="wt_only")
m_bl   = pd.read_csv(BL/"metrics_blends.csv")

# normalize blend metrics to common schema
m_bl_long = []
for _, r in m_bl.iterrows():
    base = dict(fold=r["fold"], gene=r["gene"], n=r["n"])
    m_bl_long += [
        dict(**base, method="blend_logitavg", auroc=r["AUROC_logitavg"], pr_auc=r["PR_logitavg"], brier=r["Brier_logitavg"]),
        dict(**base, method="blend_rankavg",  auroc=r["AUROC_rankavg"],  pr_auc=r["PR_rankavg"],  brier=r["Brier_rankavg"]),
        dict(**base, method="blend_max",      auroc=r["AUROC_max"],      pr_auc=r["PR_max"],      brier=r["Brier_max"]),
        dict(**base, method="pair_twoR",      auroc=r["AUROC_pair"],     pr_auc=r["PR_pair"],     brier=r["Brier_pair"]),
        dict(**base, method="wt_only",        auroc=r["AUROC_wt"],       pr_auc=r["PR_wt"],       brier=r["Brier_wt"]),
    ]
m_bl_long = pd.DataFrame(m_bl_long)

# merge metrics sources (pair, wt from original; blends from long)
metrics_all = pd.concat([
    m_pair[["fold","gene","n","method","auroc","pr_auc","brier"]],
    m_wt[["fold","gene","n","method","auroc","pr_auc","brier"]],
    m_bl_long[["fold","gene","n","method","auroc","pr_auc","brier"]],
], ignore_index=True).drop_duplicates()

# leaderboard per gene
leader = (metrics_all.sort_values(["gene","auroc"], ascending=[True,False])
                     .groupby("gene").head(5))

# also compute per-gene deltas vs the best base (max of pair_twoR vs wt_only)
def best_base(group: pd.DataFrame):
    base = group[group["method"].isin(["pair_twoR","wt_only"])].sort_values("auroc", ascending=False).head(1)
    return base["auroc"].iloc[0] if not base.empty else np.nan
base_au = metrics_all.groupby("gene").apply(best_base).rename("best_base_auroc")
metrics_w_base = metrics_all.merge(base_au, on="gene", how="left")
metrics_w_base["auroc_gain_vs_best_base"] = metrics_w_base["auroc"] - metrics_w_base["best_base_auroc"]

# unify preds
p_pair = pd.read_csv(ENS/"preds_twoR_mix.csv") \
           .rename(columns={"prob_ensemble":"p_pair","logit_ensemble":"logit_pair"})
p_wt   = pd.read_csv(WT/"preds_wt_only.csv") \
           .rename(columns={"prob_wt_only":"p_wt"})
p_bl   = pd.read_csv(BL/"preds_blends.csv") \
           .rename(columns={"p_logitavg":"p_blend_logitavg",
                            "p_rankavg":"p_blend_rankavg",
                            "p_max":"p_blend_max"})

preds_master = (p_pair.merge(p_wt, on=["variant_id","gene","label","fold"], how="inner")
                      .merge(p_bl[["variant_id","gene","label","fold","p_blend_logitavg","p_blend_rankavg","p_blend_max"]],
                             on=["variant_id","gene","label","fold"], how="inner"))

# write out
OUT = ARTI / f"final_receipts_{time.strftime('%Y%m%d_%H%M%S')}"
OUT.mkdir(parents=True, exist_ok=True)
metrics_all.to_csv(OUT/"metrics_pack.csv", index=False)
leader.to_csv(OUT/"leaderboard_top5_per_gene.csv", index=False)
metrics_w_base.to_csv(OUT/"metrics_with_base_delta.csv", index=False)
preds_master.to_csv(OUT/"preds_master.csv", index=False)

print("Packed receipts ->", OUT)
print("\n== Leaderboard (top-5 per gene) ==")
display(leader)
print("\n== Methods that BEAT the best base per gene (auroc_gain_vs_best_base > 0) ==")
display(metrics_w_base[metrics_w_base["auroc_gain_vs_best_base"]>0]
        .sort_values(["gene","auroc_gain_vs_best_base"], ascending=[True,False])
        .groupby("gene").head(5)[["gene","fold","method","auroc","auroc_gain_vs_best_base"]])
print("\nHeads-up: preds_master.csv has columns: variant_id, gene, label, fold, p_pair, p_wt, p_blend_logitavg, p_blend_rankavg, p_blend_max, logit_pair")


Packed receipts -> /content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227

== Leaderboard (top-5 per gene) ==


/tmp/ipython-input-333830511.py:56: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  base_au = metrics_all.groupby("gene").apply(best_base).rename("best_base_auroc")


,fold,gene,n,method,auroc,pr_auc,brier
0,loco_gene:EGFR,EGFR,39,pair_twoR,0.7257,0.5711,0.2330
9,loco_gene:EGFR,EGFR,39,blend_rankavg,0.7029,0.5394,0.2201
10,loco_gene:EGFR,EGFR,39,blend_max,0.6514,0.4544,0.3365
8,loco_gene:EGFR,EGFR,39,blend_logitavg,0.6314,0.4498,0.2862
4,loco_gene:EGFR,EGFR,39,wt_only,0.6057,0.4366,0.3491
1,loco_gene:KRAS,KRAS,11,pair_twoR,0.7500,0.9116,0.2661
14,loco_gene:KRAS,KRAS,11,blend_rankavg,0.3958,0.7420,0.3272
5,loco_gene:KRAS,KRAS,11,wt_only,0.0000,0.5550,0.3646
13,loco_gene:KRAS,KRAS,11,blend_logitavg,0.0000,0.5550,0.3331
15,loco_gene:KRAS,KRAS,11,blend_max,0.0000,0.5550,0.3237



== Methods that BEAT the best base per gene (auroc_gain_vs_best_base > 0) ==


,gene,fold,method,auroc,auroc_gain_vs_best_base
16,PIK3CA,loco_gene:PIK3CA,blend_max,0.7778,0.0185
19,TP53,loco_gene:TP53,blend_max,0.8020,0.0860
18,TP53,loco_gene:TP53,blend_rankavg,0.7191,0.0031
17,TP53,loco_gene:TP53,blend_logitavg,0.7177,0.0017



Heads-up: preds_master.csv has columns: variant_id, gene, label, fold, p_pair, p_wt, p_blend_logitavg, p_blend_rankavg, p_blend_max, logit_pair


# Sentiment Analysis of Cancer Genes — Day Wrap *(2025-08-16)*

**Vibe check.** We pushed sequence-only models with leak-free CV (LOCO-GENE / LOCO-SITE), and we learned **which** context carries signal per gene instead of just chasing AUCs.

---

## What we did today

- **Data build & hygiene**
  - Integrated ClinVar benigns and windowized against **true CDS (GenBank)** → **0 ref mismatches**.
  - Merged positives and negatives; resolved per-site conflicts; wrote  
    `data/masterseq_model_ready_plus_resolved.parquet`.
  - Saved split manifests under:  
    `artifacts/sentiment_run_plus_variants_YYYYMMDD_*` (LOCO-GENE and LOCO-SITE).

- **Classical baselines & ablations**
  - Compared **flanks vs codon vs tri-only** → flanks matter; codon-only/tri-only underperform.

- **Models**
  - **WT-only** baseline: 3-mer bag on `window_wt` (+ center tri feature).
  - **Pair/siamese (RankNet) delta CNN**: learns `mut − wt` with receptive fields **R=30** and **R=45**.
  - **Nested HPO** aligned to deployment: inner **GroupKFold by gene** (no leakage).
  - **Two-radius ensemble** + **20% base-matched random pairs** for stability.

- **Diagnostics**
  - **Tri-context coverage** by fold; PIK3CA has **1.00** coverage (not OOD by trinuc) but **frequency shift** vs train.
  - Confirmed small-n + gene-specific priors can crater cross-gene generalization.

- **Combining models**
  - Meta-stack (LR) **overfit** to TP53 (dominant fold) → hurt EGFR.
  - Switched to **training-free blends**: **logit-avg**, **rank-avg**, and **max(WT, pair)**.

---

## Key receipts (LOCO-GENE AUC per gene — best few)

- **EGFR (n=39):** `pair_twoR` **0.7257**, `rank-avg` 0.7029  
- **KRAS (n=11):** `pair_twoR` **0.7500** (tiny n; others unstable)  
- **PIK3CA (n=24):** `max(WT, pair)` **0.7778**, `WT-only` 0.7593 (pair alone: 0.3981)  
- **TP53 (n=197):** `max(WT, pair)` **0.8020**, `rank-avg` 0.7191, `WT-only` 0.7160

> **Files**
>
> - Pair ensemble: `artifacts/ensemble_twoR_20250816_225822/metrics_twoR_mix.csv`  
> - WT baseline: `artifacts/wt_only_baseline_20250816_230700/metrics_wt_only.csv`  
> - Blends: `artifacts/blend_nolearn_20250816_231849/metrics_blends.csv`  
> - Packed receipts (leaderboard, unified preds): `artifacts/final_receipts_20250816_232227/*`

---

## What we learned (skeptical take)

- **Context helps, but gene-specific.**
  - **TP53 / PIK3CA:** WT neighborhood dominates; delta adds value only with a safety net.
  - **EGFR / KRAS:** Delta CNN carries; WT alone lags.
- **Shift ≠ missing trinucs.** PIK3CA’s dip was **tri-frequency skew + tiny negatives**, not unseen contexts.
- **Stackers overfit on small panels.** Training-free blends are safer and more gene-agnostic.

---

## Tomorrow’s plan *(one cell at a time)*

1. **Lock a simple, leak-free deploy rule (training-free):**
   - EGFR → `pair_twoR`  
   - KRAS → `pair_twoR`  
   - TP53 → `max(WT, pair)`  
   - PIK3CA → `max(WT, pair)`  
   - Provide a **gene-agnostic fallback** = `rank-avg` for future genes.
2. **Stability pass**
   - Add **gentle hard-neg** (K=1, `neg_cap=2`) during training (train folds only), re-eval ensemble.
   - **Per-gene isotonic calibration** (fit on train folds; apply to test fold) and save calibrated probs.
3. **Grammar maps (turn scores into science)**
   - Saliency / IG over pairCNN around the mutated codon.
   - Motif mining for **TP53**/**EGFR**; compare WT-only k-mer importances vs pair saliency.
4. **Data growth**
   - Increase benigns (esp. **PIK3CA**), expand panel, maintain tri-balance.
   - Re-run tri-coverage + frequency-shift diagnostics as gates.

---

## Long-term goal

A robust, **sequence-only** “oncogenic grammar” that **generalizes to unseen genes/sites**, with interpretable motif-level explanations. Then layer biology (AA biophysics, conservation, splice context) **incrementally and leak-free**.

---

## Risks / watch-outs

- **Small-n genes** (KRAS) → high variance; report **CIs**, not just point AUCs.  
- **Frequency shifts** can bite even with full tri coverage; keep matching/diagnostics in the loop.  
- **Stacking** on small panels will overfit unless training-free or sample-balanced across genes.

---

## Artifacts we wrote today

- `data/masterseq_model_ready_plus_resolved.parquet`  
- Split CSVs under `artifacts/sentiment_run_plus_variants_*`  
- Model receipts: pair, WT, blends (paths above)  
- Unified pack: `artifacts/final_receipts_20250816_232227/`

---

## Open questions

- Should we **always** gate delta with WT (`max(WT, pair)`), or only when tri-frequency shift flags risk?  
- How much does **radius** matter after calibration + blending?  
- What’s the **minimal negative set size per gene** before delta models stabilize?

---
